# Pipeline de Previsão de Demanda Intermitente (TCC)

Este notebook executa uma pipeline de previsão para produtos de cauda longa no e-commerce brasileiro, com foco em demanda intermitente.

São comparados modelos de Machine Learning (Random Forest e XGBoost) com baselines estatísticos (SBA e TSB), em avaliação walk-forward com janela expandida.

## Como usar

1. Edite apenas a célula de parâmetros (seção 2).
2. Execute o notebook na sequência.
3. Consulte os arquivos gerados na pasta de execução, nesta ordem:
   - `RELATORIO_DECISAO.md`
   - `selecao_variaveis.md`
   - `RESULTADOS_CONSOLIDADO.md`
   - `METADATA.md`

## Estrutura do notebook

- **Seções 1–3**: preparação do ambiente (Drive, parâmetros e dependências).
- **Seções 4–6**: geração dos módulos e execução da pipeline.
- **Seções 7–8**: conferência dos resultados e visualização das figuras.

## Observação

As decisões de modelagem e seleção de variáveis são registradas automaticamente nos relatórios de saída para garantir rastreabilidade metodológica.

## 1. Montar Google Drive (Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Parâmetros da execução

Defina caminhos e opções da rodada.

> Recomendação: altere somente esta seção no uso diário.

In [ ]:
"""Parametros da execucao. Esta e a unica celula que voce edita no dia a dia.

Como ler cada bloco abaixo: o que o parametro faz, os valores aceitos, um
exemplo e o que muda na saida.
"""
from pathlib import Path

# =============================================================================
# CAMINHOS
# =============================================================================

# CSV da base de vendas, no Drive montado. Colunas esperadas: Ped_Data,
# Prod_SKU, Qtde_VendaTotal, Qtde_PedidosUnicos, Prod_DataCadastro,
# Cat_CategoriaNome, Ped_Campanha.
DADOS = "/content/drive/MyDrive/data/base_venda.csv"

# Pasta raiz dos resultados. Cada execucao cria uma subpasta propria,
# <data>_<hora>_<ROTULO>, entao rodadas anteriores nunca sao sobrescritas.
DRIVE_OUT = "/content/drive/MyDrive/outputs"

# =============================================================================
# ESCOPO DA EXECUCAO
# =============================================================================

# Sufixo do nome da pasta desta execucao. Use para achar a rodada depois.
# Ex.: "parcimonia" -> outputs/20260804_101500_parcimonia/
ROTULO = "parcimonia"

# Quantos SKUs modelar. None = todos os elegiveis.
# Ex.: 50 -> amostra estratificada por quadrante ADI/CV2, para teste rapido.
AMOSTRA = None

# Quais experimentos rodar.
#   ("1",)      -> so o Experimento 1 (comparacao entre os 4 modelos)
#   ("1", "2")  -> acrescenta o Experimento 2 (efeito das variaveis exogenas)
# O Exp2 depende do Exp1: os cenarios do Exp2 reusam a mesma base de features.
EXPERIMENTOS = ("1", "2")

# Atalho para a cadencia do walk-forward.
#   True  -> STRIDE_ORIGEM_TREINO=1, JANELAS_POR_REFIT=1
#   False -> STRIDE_ORIGEM_TREINO=7, JANELAS_POR_REFIT=4
# Os dois parametros abaixo, quando diferentes de None, tem PRECEDENCIA sobre
# este atalho — e e assim que se roda a configuracao recomendada.
MODO_FIEL = False

# --- Cadencia do walk-forward ------------------------------------------------
# Duas grandezas independentes, que respondem a perguntas diferentes.
#
# JANELAS_POR_REFIT — de quantas em quantas semanas do teste o modelo e
# retreinado. E DESENHO EXPERIMENTAL:
#   1 -> reajuste semanal   (53 treinos no ano de teste)
#   2 -> quinzenal          (27 treinos)
#   4 -> mensal             (14 treinos)
#
# STRIDE_ORIGEM_TREINO — de quantos em quantos dias se toma uma data de origem
# para montar a tabela de treino. E DENSIDADE DA AMOSTRA, nao cobertura:
# nenhuma venda e descartada em passo nenhum, porque toda venda continua
# entrando nos lags e medias moveis de todas as origens posteriores. O que muda
# e quantas "fotos" quase identicas do historico viram linha de treino — a foto
# de 15/03 e a de 16/03 diferem em um dia de janela movel.
#
# NAO USE MULTIPLO DE 7. O passo e aplicado sobre a lista ordenada de datas,
# entao um multiplo de 7 poe todas as origens no mesmo dia da semana; como o
# alvo e origem + h (h = 1..7), o dia da semana do alvo vira funcao exata do
# horizonte. `dia_semana_alvo` deixa de acrescentar qualquer coisa ao
# `horizonte`, e a feature perde o efeito de dar ao ML o padrao semanal que
# SBA/TSB ja recebem via proporcao_dia_semana. O pipeline rejeita esse caso com
# erro.
#
# Custo medido na execucao 20260818_114211_exp1_selecao_wmape
# (3.684 SKUs, Exp1 + Exp2, stride=3, refit=1): 79.562,7 s (~22,1 h).
# Nesta rodada nao foram medidos os cenarios stride=3/refit=2-4 nem
# stride=1/refit=1.
#
# Ex.: (3, 1) e a configuracao recomendada — reajuste semanal, sem aliasing.
# Ex.: (None, None) devolve o controle ao MODO_FIEL.
STRIDE_ORIGEM_TREINO = 3
JANELAS_POR_REFIT = 1

# Libera passo multiplo de 7. Existe apenas para reproduzir execucoes
# anteriores de proposito; nao use em rodada nova.
PERMITIR_ALIAS_DIA_SEMANA = False

# Filtro de campanha sobre a coluna Ped_Campanha. Os dois sao tuplas vazias por
# padrao = usa a base inteira como veio.
# Ex.: CAMPANHAS_INCLUIDAS = ("TESTE",) -> mantem so as linhas dessa campanha.
# Ex.: CAMPANHAS_EXCLUIDAS = ("TESTE",) -> descarta essas linhas.
CAMPANHAS_EXCLUIDAS = ()
CAMPANHAS_INCLUIDAS = ()

# =============================================================================
# DESEMPENHO
# =============================================================================
import os

# Nucleos usados por RF, XGBoost e pelo laco por SKU do TSB. -1 = todos.
N_JOBS = -1

# Iteracoes do RandomizedSearchCV. Mais iteracoes = melhor cobertura da grade,
# custo linear no tempo de tuning. Com a grade "completa" do RF (108
# combinacoes), 50 iteracoes cobrem 46,3% do espaco.
# Custo medido na execucao 20260818_114211_exp1_selecao_wmape (janela 0):
# RF tuning = 6.634,97 s (~110,6 min) e XGBoost tuning = 136,52 s (~2,3 min).
# Esses valores estao em tempos_execucao.csv e variam com MAX_LINHAS_TUNING,
# grade ativa e tamanho da base.
N_ITER_BUSCA = 50

# Mesmo que o anterior, so para o Random Forest. None = usa N_ITER_BUSCA.
N_ITER_BUSCA_RF = None

# Grade de hiperparametros do Random Forest.
#   "completa" -> 108 combinacoes
#   "reduzida" ->  54 combinacoes (metade do espaco, mesmo tempo por iteracao)
GRADE_RF = "completa"

# Numero de dobras do TimeSeriesSplit dentro da busca de hiperparametros.
CV_SPLITS = 3

# Teto de linhas da tabela de treino usadas na BUSCA de hiperparametros. O
# ajuste final SEMPRE usa a tabela inteira -- este teto afeta apenas sobre
# quantas linhas os candidatos sao comparados entre si.
#   None    -> sem teto; a busca ve a tabela de treino completa
#   inteiro -> amostra aleatoria desse tamanho, preservando a ordem
#              cronologica exigida pelo TimeSeriesSplit
# Por que importa: a capacidade otima de um modelo cresce com o tamanho da
# amostra. Uma busca feita sobre poucas linhas tende a eleger modelos mais
# rasos que os adequados a tabela completa -- vies sistematico, e na direcao
# que enfraquece o ML na comparacao com os baselines.
# O teto so morde quando e MENOR que a tabela: qualquer valor acima do numero
# de linhas dela produz exatamente a mesma busca. Na execucao
# 20260818_114211_exp1_selecao_wmape, a tabela de tuning da janela 0 tinha
# 4.031.860 linhas e o teto usado foi 1.000.000 (MAX_LINHAS_TUNING).
# Com esse teto, os tempos observados de tuning foram:
#   RF ~110,6 min | XGBoost ~2,3 min.
# O valor usado em cada execucao e registrado em METADATA.md.
MAX_LINHAS_TUNING = 1_000_000

# Faz os modelos de um mesmo experimento/cenario (XGBoost e RF) dividirem UMA
# montagem da tabela de treino por janela, em vez de montarem a mesma tabela
# duas vezes. Sao 6 montagens por janela hoje (Exp1, Cenario 1 e Cenario 2 x 2
# modelos) contra 3 com o parametro ligado.
# Por que nao ha risco de vazamento: a tabela de uma janela e funcao de
# (features, corte, passo, horizonte, exogenas) e dos codificadores ajustados
# ate o corte DAQUELA janela -- nada disso depende do modelo, que entra so na
# selecao de colunas. O compartilhamento e entre MODELOS da mesma janela,
# nunca entre janelas: os codificadores continuam sendo reajustados a cada
# janela. Cada modelo ainda recebe uma COPIA das suas colunas.
# Na execucao 20260818_114211_exp1_selecao_wmape esse parametro ficou em
# False; portanto nao houve medicao direta do ganho com compartilhamento de
# tabela nessa rodada.
# ATENCAO: com True o tempo de montagem deixa de pertencer a um modelo (vai
# para a linha CACHE de tempos_execucao.csv) e o custo_total deixa de ser
# comparavel ao de uma execucao com False. Para medir custo computacional,
# mantenha False.
REAPROVEITAR_TABELA_TREINO = False

# True faz o XGBoost tentar device='cuda'. Cai para CPU sozinho se nao houver
# GPU no runtime. Atencao ao comparar tempos entre execucoes: uma rodada com
# GPU e outra sem tornam a tabela de custo computacional incomparavel.
GPU_XGB = False

# =============================================================================
# ANALISES DE APOIO E JANELAS
# =============================================================================

# Matriz e heatmaps de correlacao entre features e alvo, por quadrante ADI/CV2.
# Gera correlacao_features.md, que documenta a evidencia usada para remover
# features redundantes.
GERAR_CORRELACAO = True

# Figuras de resultado (boxplots, real x previsto, tempo, importancia).
GERAR_GRAFICOS = True

# Janelas de treino e teste. None em qualquer uma delas = usa o valor de
# config.py. O teste comeca no dia seguinte ao fim do treino e nada do periodo
# de teste entra em feature, hiperparametro ou criterio de selecao.
TREINO_INICIO = "2022-07-01"
TREINO_FIM = "2025-06-30"
TESTE_INICIO = "2025-07-01"
TESTE_FIM = "2026-06-30"

# =============================================================================
# VARIAVEIS EXOGENAS (Experimento 2)
# =============================================================================

# Calendario diario em formato wide: uma linha por data, uma coluna por evento.
# Serve ao Cenario 2. Ver a lista de colunas em config.COLUNAS_CALENDARIO_RICO.
CALENDARIO_EXOGENAS = "/content/drive/MyDrive/mba/tcc/calendario_diario_wide.csv"

# Cenarios do Experimento 2.
#   "1" -> calendario simples (dia da semana, mes, feriado nacional)
#   "2" -> calendario rico (eventos comerciais)
# Ex.: ("1", "2") isola o efeito das exogenas comparando as duas pontas.
CENARIOS_EXP2 = ("1", "2")

# =============================================================================
# METRICAS E DIAGNOSTICOS
# =============================================================================

# Conta e reporta, por modelo, quantos SKUs ficam com WMAPE indefinido
# (denominador zero = SKU sem venda no teste). Sem isso a mediana do WMAPE e
# calculada sobre um subconjunto diferente para cada modelo, sem avisar.
DIAGNOSTICO_WMAPE_ZERO = True

# Acrescenta MAE, RMSSE e MASE escalado pela media do treino. Necessario para
# comparar o estrato "Cessado" (sem venda no teste), que so pode ser avaliado
# por MAE.
# O RMSSE e a unica metrica de erro quadratico do conjunto, e por isso a unica
# em que o otimo e a media condicional e nao a mediana (= zero, em serie com
# ~95% de zeros). E onde os modelos separam do baseline Zero.
METRICAS_EXTRAS = True

# Avalia tambem o total semanal, alem do diario.
# E o nivel em que os 4 modelos operam nativamente (SBA e TSB preveem semanal e
# sao desagregados para o dia) e equivale ao lead time de 7 dias da reposicao.
AVALIAR_SEMANAL = True

# Acrescenta tamanho de efeito (r bisserial pareado) ao Wilcoxon e corrige o
# p-valor por Holm-Bonferroni.
# Convencao de sinal nos relatorios: r > 0 = o SEGUNDO modelo do par e melhor.
TAMANHO_EFEITO = True

# Inclui dia_semana_alvo nas features de ML. E informacao de calendario,
# conhecida com antecedencia, entao nao ha look-ahead. Sem ela os baselines
# recebem o perfil semanal (via proporcao_dia_semana) e o ML nao.
ADICIONAR_DIA_SEMANA = True

# Funcao objetivo do XGBoost. Muda para onde o modelo converge:
#   "reg:squarederror"   MSE            -> otimo = media condicional (padrao)
#   "reg:absoluteerror"  MAE            -> otimo = mediana condicional (= 0 aqui)
#   "count:poisson"      Poisson
#   "reg:tweedie"        Poisson-Gama composta, a mesma hipotese estrutural de
#                        SBA e TSB
# Ex.: "reg:absoluteerror" melhora WMAPE e piora RMSSE, por construcao.
XGB_OBJETIVO = "reg:squarederror"

# =============================================================================
# BLOCO A - CORRECOES DE COMPORTAMENTO.  DEFAULT: True
#
# Recomenda-se manter todas ligadas. Desligar serve so para medir o efeito de
# cada uma isoladamente.
# =============================================================================

# Ordena a tabela longa por (data_origem, horizonte) antes da busca de
# hiperparametros, e faz o scorer penalizar dobras sem demanda.
# Por que importa: o TimeSeriesSplit corta por POSICAO de linha. Com a tabela
# agrupada por horizonte, ele treina em h=1 e valida em h=2/h=3 -- a venda de um
# mesmo dia aparece dos dois lados, com features quase identicas, e o score de
# validacao fica otimista.
# Como conferir: coluna `detalhe` da etapa 'tuning' em tempos_execucao.csv.
ORDENAR_TABELA_CRONOLOGICA = True

# Corta de cada serie os dias anteriores a existencia do produto.
# A grade diaria e expandida de DATA_INICIO a DATA_FIM para todo SKU, entao sem
# esse corte um produto lancado em 2024 recebe ~600 zeros que nunca existiram --
# inflando ADI e CV2, contaminando o denominador do MASE e puxando o ML para
# prever zero.
# Inicio da serie = max(DATA_INICIO, min(data_cadastro, 1a venda observada)).
# O minimo garante que nenhuma venda real seja apagada quando o cadastro do ERP
# for posterior a ela.
# (ARQUIVO_FUNIL logo abaixo e apenas informativo, nao usado pelo pipeline.)
ARQUIVO_FUNIL = None
TRUNCAR_PRE_LANCAMENTO = True

# Usa media geometrica no score relativo, em vez de aritmetica.
# Razoes de erro sao multiplicativas: quando um modelo preve ~0 ele vira
# denominador e a media aritmetica explode. Ver Hyndman e Koehler (2006) e
# Fildes (1992).
SCORE_RELATIVO_GEOMETRICO = True

# Restringe ao treino o calculo dos atributos derivados de venda (primeira
# venda observada, moda da categoria). Sem isso eles enxergam o periodo de
# teste.
CORTE_ATRIBUTOS_TREINO = True

# Salva as previsoes ponto a ponto, alem das metricas.
#   metricas_exp*_<modelo>.csv        uma linha por SKU
#   previsoes_exp*_<modelo>.parquet   uma linha por SKU x data x horizonte
# Sem isso nao ha como refazer nenhuma analise sem reexecutar o pipeline.
SALVAR_PREVISOES_BRUTAS = True

# =============================================================================
# BLOCO B - ANALISES NOVAS.  DEFAULT: True
#
# Nao mudam modelo, feature nem metrica: so acrescentam recortes de relatorio.
# =============================================================================

# Separa os resultados entre SKUs Ativos e Cessados no teste.
#   Ativo   = vendeu >= 1 unidade no periodo de teste
#   Cessado = nenhuma venda no periodo de teste
# Por que importa: com quase metade dos SKUs sem venda no teste, a mediana entre
# SKUs cai bem na fronteira entre os dois grupos -- ela passa a medir "quao
# perto de zero o modelo preve num SKU morto", nao "quem preve melhor demanda".
# Nao e vazamento: todos os SKUs foram modelados igualmente e o recorte existe
# so na apresentacao. Nenhuma decisao de modelagem usa o periodo de teste.
ESTRATIFICAR_ATIVOS_CESSADOS = True

# Acrescenta o baseline "Zero", que preve 0 sempre.
# Nao e um modelo proposto: e o controle, analogo ao classificador de classe
# majoritaria. Se o Zero empata com o TSB em todas as metricas, entao "o TSB
# venceu" significa "prever zero venceu", e o achado passa a ser sobre a
# inadequacao das metricas pontuais em cauda longa.
BASELINE_ZERO = True

# Acrescenta metricas agregadas sobre todas as observacoes juntas:
#   WMAPE_pooled = soma|y-yhat| / soma(y)
#   MAE_pooled   = soma|y-yhat| / N
# Ficam definidas enquanto algum SKU vender, entao nao sofrem do problema de
# denominador zero que afeta a versao por SKU.
METRICAS_POOLED = True

# Roda a classificacao ADI/CV2 tambem em granularidade diaria, como tabela de
# robustez. Os cortes 1,32 e 0,49 de Syntetos, Boylan e Croston (2005)
# pressupoem que se classifique na mesma granularidade em que se preve;
# classificar em semanas enquanto se modela em dias comprime o ADI.
# Esperado: quase tudo cai em Lumpy no diario -- e um achado a reportar, e
# justifica manter o semanal como classificacao principal.
CLASSIFICACAO_ROBUSTEZ_DIARIA = True

# Gera RELATORIO_DECISAO.md: parametros ativos, ranking por metrica, resultado
# contra o baseline Zero, estratos e um bloco sobre o que os numeros sustentam
# e o que nao sustentam.
# E o primeiro arquivo a ler depois de cada execucao.
RELATORIO_DECISAO = True

# =============================================================================
# BLOCO B2 - RECORTE GERENCIAL.  DEFAULT: True
# =============================================================================

# Estratifica os resultados por volume de pedidos (curva ABC).
#   Faixa A = os 20% de SKUs com mais pedidos
#   Faixa B = os 30% seguintes
#   Faixa C = a cauda restante
# Responde a pergunta gerencial que a mediana entre SKUs nao responde: a mediana
# trata igualmente um produto com milhares de pedidos e outro com 3 no ano.
# Volume medido por Qtde_PedidosUnicos (pedidos distintos = clientes atendidos)
# e, na falta dela, por dias com venda -- sempre no periodo de TREINO, que e a
# informacao que o gestor teria em maos.
# Gera: estratos_volume_pedidos.csv, concentracao_volume_pedidos.csv,
#       resumo_experimento1_por_volume.csv, testes_wilcoxon_por_volume.md,
#       cruzamento_volume_x_obsolescencia.csv
ESTRATIFICAR_POR_VOLUME = True

# Grava em markdown tudo que sai no console.
#   LOG_EXECUCAO.md            captura literal da execucao
#   RESULTADOS_CONSOLIDADO.md  as tabelas formatadas, com notas de leitura
# A mesma chamada imprime e registra, entao nenhuma tabela aparece no console
# sem aparecer no markdown.
CONSOLIDAR_MARKDOWN = True

# Figuras dos recortes por estrato:
#   fig_pareto_pedidos.png                 concentracao dos pedidos
#   fig_*_por_estrato_teste.png            Ativo x Cessado
#   fig_*_por_estrato_volume.png           por faixa ABC
#   fig_boxplot_*_volume.png               distribuicao dentro de cada faixa
#   fig_realvsprevisto_TOP<i>_<sku>.png    os SKUs de maior numero de pedidos
# As figuras de real x previsto padrao mostram melhor e pior caso por quadrante
# ADI/CV2, que costumam cair em SKUs de giro baixissimo; estas mostram os que
# concentram volume.
GRAFICOS_POR_ESTRATO = True

# Quantos SKUs do topo da curva ABC recebem figura propria.
N_SKUS_TOP_FIGURAS = 6

# =============================================================================
# BLOCO C - MODELAGEM.  DEFAULT: False
#
# Mudam o que entra no modelo.
# =============================================================================

# Acrescenta as features especificas de demanda intermitente (Kourentzes,
# 2013; vencedores da M5):
#   dias_desde_ultima_venda, n_dias_venda_{28,91}, taxa_zeros_{28,91},
#   tam_medio_demanda, lag_28, roll_{mean,std}_{28,91}, mes_alvo, semana_ano_alvo
# Todas construidas sobre a serie deslocada em 1 dia, sem look-ahead.
# Argumento: sao a decomposicao de Croston (intervalo entre demandas + tamanho
# da demanda) oferecida ao ML -- deixam a comparacao com SBA/TSB mais justa.
FEATURES_INTERMITENCIA = False

# Como o SKU entra no modelo.
#   "ordinal" -> LabelEncoder alfabetico. A arvore so consegue perguntar
#                "sku_enc <= 137,5?", que agrupa SKUs por ordem alfabetica --
#                um corte sem relacao com demanda.
#   "perfil"  -> substitui o ID por sku_media_hist, sku_taxa_zeros e
#                sku_tam_medio, calculados so no treino de cada janela; um unico
#                split ja separa giro baixo de giro alto.
# Continua prevendo por SKU nos dois casos.
ENCODING_SKU = "ordinal"

# Grade de hiperparametros do XGBoost.
#   "completa"    -> 2187 combinacoes; com N_ITER_BUSCA=50 a cobertura e 2,3%
#   "equilibrada" ->  162 combinacoes; cobertura 30,9%, mesma ordem da do RF
#                    (108 combinacoes, 46,3% com N_ITER_BUSCA=50). Fixa
#                    subsample e colsample e acrescenta min_child_weight, o
#                    hiperparametro mais relevante quando ~95% dos alvos sao zero.
# O tempo de execucao nao muda: continua sendo N_ITER_BUSCA iteracoes.
# Default "equilibrada": equilibra a cobertura da busca entre RF e XGBoost.
GRADE_XGB = "equilibrada"

# Reaproveita no Experimento 2 os hiperparametros ja buscados no Experimento 1,
# em vez de tunar de novo em cada cenario.
# Por que importa: sem isso a diferenca entre os cenarios 1 e 2 mistura "efeito
# do calendario" com "efeito de ter sorteado outros hiperparametros".
TUNAR_UMA_VEZ_GLOBAL = True

# =============================================================================
# BLOCO D - PARCIMONIA DO CONJUNTO DE VARIAVEIS.  DEFAULT: True
#
# Criterio que as duas opcoes abaixo respeitam: a exclusao e ESTRUTURAL
# (identidade algebrica, janela equivalente) ou MEDIDA NO TREINO (dias com o
# evento ativo). Nenhum corte usa metrica, significancia ou importancia
# calculada sobre o periodo de teste -- selecionar variavel pelo desempenho no
# teste e depois reportar o desempenho nesse mesmo teste seria vazamento.
#
# Cada execucao grava selecao_variaveis.md com o que entrou, o que saiu, a
# justificativa de cada exclusao e o suporte amostral das exogenas recalculado
# sobre a janela de treino da propria execucao.
# =============================================================================

# Remove features que sao funcao (quase) deterministica de outra ja presente.
# Todas verificaveis em correlacao_features.md:
#   taxa_zeros_28  = 1 - n_dias_venda_28/28   r = -1,000  identidade exata
#   taxa_zeros_91  = 1 - n_dias_venda_91/91   r = -1,000  identidade exata
#   roll_mean_30   ~ roll_mean_28             r =  0,999  janela equivalente
#   roll_std_30    ~ roll_std_28              r =  0,997  janela equivalente
#   mes_alvo       ~ semana_ano_alvo          r =  0,975
# A remocao so ocorre se a substituta estiver ativa, entao com
# FEATURES_INTERMITENCIA=False nada e removido (as janelas de 28 e 91 dias nem
# chegam a existir).
# dia_semana_alvo NAO entra nesta lista: ele varia livremente na tabela de
# treino real. Ver a nota sobre o passo da analise de correlacao em config.py.
# Ex.: False -> mantem as 26 features do conjunto completo.
REMOVER_FEATURES_REDUNDANTES = True

# Exclusoes adicionais, alem das automaticas acima. Nao exigem substituta ativa.
# Ex.: ("lag_30",) para testar um conjunto ainda mais enxuto.
FEATURES_EXCLUIDAS_EXTRA = ()

# Colunas do calendario rico que ficam de FORA do Cenario 2.
# Escolher quais variaveis de calendario fazem sentido para o contexto e parte
# do desenho do cenario das exogenas: continua sendo o Cenario 2.
#
# Excluidas por padrao, criterio medido no treino:
#   ev_Cyber_Monday          3 dias ativos em 3 anos -> sem suporte amostral
#   ev_Dia_dos_Pais          importancia RF 7,1e-07 (5 ordens abaixo de lag_1)
#   ev_Dia_das_Maes          importancia RF 3,8e-05
#   ev_Dia_das_Criancas      importancia RF 4,4e-05
#   ev_Dia_dos_Namorados     importancia RF 6,0e-06
#   ev_Semana_do_Consumidor  importancia RF 1,5e-04
#   flag_campanha            redundante com intensidade_max: e praticamente
#                            1{intensidade_max >= 2} (117 excecoes em 2.191
#                            dias, r = 0,858) e fica ativa em 58,6% dos dias de
#                            treino. Nao marca eventos, marca TEMPORADAS (01/11
#                            a 24/12 e 02/01 a 15/02, todo ano); a versao
#                            graduada da mesma informacao ja esta em
#                            intensidade_max, com 6 niveis.
#
# Mantidas (7): intensidade_max, flag_feriado_nacional, ev_Black_Friday,
#               ev_Black_November_Esquenta, ev_Natal, ev_Pascoa, ev_Copa_do_Mundo
#
# Um nome que nao exista em config.COLUNAS_CALENDARIO_RICO gera erro na
# validacao, em vez de ser ignorado em silencio.
# Ex.: () -> Cenario 2 com as 14 exogenas do calendario completo.
EXOGENAS_EXCLUIDAS = (
    "ev_Cyber_Monday",
    "ev_Dia_dos_Pais",
    "ev_Dia_das_Maes",
    "ev_Dia_das_Criancas",
    "ev_Dia_dos_Namorados",
    "ev_Semana_do_Consumidor",
    "flag_campanha",
)

# --- Metrica de selecao dos hiperparametros -----------------------------------
# "WMAPE" seleciona os hiperparametros pela mesma metrica usada na avaliacao
# final. "RMSE" seleciona por uma metrica diferente da avaliacao final,
# evitando a circularidade de tunar e avaliar pela mesma metrica. Rode as duas
# configuracoes separadamente, trocando tambem o ROTULO para que os resultados
# caiam em pastas distintas e nada se misture.
METRICA_SELECAO_HIPER = "WMAPE"
DIAGNOSTICO_SELECAO_HIPER = True

# Ajusta o rotulo da execucao para carregar a metrica, garantindo pastas
# distintas entre as duas execucoes (sobrescreve o ROTULO definido acima).
ROTULO = f"exp1_selecao_{METRICA_SELECAO_HIPER.lower()}"


## 3. Instalar dependências

In [ ]:
%pip install -q xgboost statsforecast holidays pyarrow
print('dependencias ok')

## 4. Gerar módulos da pipeline

As células seguintes escrevem os módulos Python em `/content/src/`.

Esses módulos implementam as etapas de preparação dos dados, modelagem, avaliação e geração dos relatórios/finalizações usados no trabalho.

In [ ]:
import os
os.makedirs('/content/src', exist_ok=True)
open('/content/src/__init__.py', 'w').close()
print('pacote src criado')

In [ ]:
%%writefile /content/src/config.py
"""
Configuracao central do pipeline: janelas, limiares, grades de hiperparametros,
nomes de colunas e as chaves que ligam ou desligam cada analise.

COMO OS VALORES SAO RESOLVIDOS
  Este modulo guarda os DEFAULTS. A celula de parametros do notebook passa os
  valores da execucao para `pipeline_colab.rodar()`, que reatribui os atributos
  deste modulo antes de qualquer processamento e registra o resultado em
  METADATA.md. Por isso um default aqui pode divergir do que a execucao usou --
  METADATA.md e a fonte da verdade sobre o que rodou.

  `config` e um modulo, ou seja, estado global do processo. Toda chamada de
  `rodar()` reatribui as opcoes de forma incondicional, justamente para que duas
  execucoes na mesma sessao do Colab (o caso normal de um A/B) nao herdem
  configuracao uma da outra.

ORGANIZACAO DAS OPCOES
  Metricas e diagnosticos
  Bloco A   correcoes de comportamento; ligadas por padrao
  Bloco B   recortes de relatorio; nao mudam modelo nem metrica
  Bloco B2  recorte gerencial (curva ABC) e consolidacao
  Bloco C   mudam o que entra no modelo
  Bloco D   parcimonia do conjunto de variaveis

REGRA DO BLOCO D, valida tambem para qualquer selecao futura: o criterio de
exclusao e estrutural (identidade algebrica, janela equivalente) ou medido no
TREINO (dias com o evento ativo). Nenhum corte usa metrica, significancia ou
importancia calculada sobre o periodo de teste -- selecionar variavel pelo
desempenho no teste e depois reportar o desempenho nesse mesmo teste seria
vazamento.
"""
from __future__ import annotations

import sys
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path


# ---------------------------------------------------------------------------
# Encoding da saida de texto
# ---------------------------------------------------------------------------
def configurar_saida_utf8() -> None:
    """Poe stdout e stderr em UTF-8, ignorando ambientes que nao permitem."""
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8", errors="replace")
        except (AttributeError, ValueError, OSError):
            pass


# ---------------------------------------------------------------------------
# Fonte e codificacao das figuras
# ---------------------------------------------------------------------------
def configurar_matplotlib_pt_br() -> None:
    """Fixa fonte e flags do matplotlib para os titulos e rotulos acentuados
    (portugues) das figuras geradas pelo pipeline.

    DejaVu Sans (fonte padrao do matplotlib) cobre o alfabeto latino
    estendido usado no portugues; fixar isso explicitamente evita que um
    rcParams global de outro ambiente (tema do Colab, estilo customizado)
    troque a fonte por uma sem esses glifos."""
    import matplotlib
    matplotlib.rcParams["font.family"] = "DejaVu Sans"
    matplotlib.rcParams["axes.unicode_minus"] = False


# ---------------------------------------------------------------------------
# Caminhos do projeto
# ---------------------------------------------------------------------------
BASE_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = BASE_DIR / "data"
OUTPUTS_DIR = BASE_DIR / "outputs"
SQL_DIR = BASE_DIR / "sql"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

EXECUCOES_DIR = OUTPUTS_DIR / "execucoes"
EXECUCOES_DIR.mkdir(parents=True, exist_ok=True)

# Uma linha por execucao ja realizada, para comparar rodadas entre si.
HISTORICO_CSV = OUTPUTS_DIR / "historico_execucoes.csv"


def criar_pasta_execucao(rotulo: str | None = None) -> Path:
    """Cria a pasta desta execucao, nomeada <data>_<hora>_<rotulo>.

    O timestamp garante que rodadas anteriores nunca sejam sobrescritas."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    nome = timestamp if not rotulo else f"{timestamp}_{rotulo}"
    pasta = EXECUCOES_DIR / nome
    pasta.mkdir(parents=True, exist_ok=True)
    return pasta

# ---------------------------------------------------------------------------
# Periodo do estudo
# ---------------------------------------------------------------------------
# Janela total dos dados. A grade diaria de cada SKU e expandida dentro dela.
DATA_INICIO = "2022-07-01"
DATA_FIM = "2026-06-30"

# Treino: 3 anos. Tudo que e estimado (features, hiperparametros, criterios de
# selecao, classificacao ADI/CV2) enxerga somente este intervalo.
TREINO_INICIO = "2022-07-01"
TREINO_FIM = "2025-06-30"

# Teste: 1 ano, usado exclusivamente para medir. Comeca no dia seguinte ao fim
# do treino.
TESTE_INICIO = "2025-07-01"
TESTE_FIM = "2026-06-30"

# Horizonte de previsao em dias. Cada origem gera previsoes para h = 1..7, o
# que corresponde ao lead time de reposicao semanal.
HORIZONTE_PREVISAO_DIAS = 7

# ---------------------------------------------------------------------------
# Regras de extracao
# ---------------------------------------------------------------------------
PAIS_FILTRO = "BR"

# Filtros sobre a coluna de campanha. Vazios = usa a base inteira.
CAMPANHAS_EXCLUIDAS = ()
CAMPANHAS_INCLUIDAS = ()

# ---------------------------------------------------------------------------
# Criterio de elegibilidade
# ---------------------------------------------------------------------------
# Vendas acima desta quantidade de desvios-padrao da media historica do SKU sao
# apenas MARCADAS como candidatas a outlier. Nada e removido automaticamente.
OUTLIER_N_DESVIOS_PADRAO = 3

# ---------------------------------------------------------------------------
# Classificacao ADI / CV2
# ---------------------------------------------------------------------------
# Cortes de Syntetos, Boylan e Croston (2005). ADI = intervalo medio entre
# demandas; CV2 = variabilidade do tamanho da demanda quando ela ocorre.
ADI_LIMIAR = 1.32
CV2_LIMIAR = 0.49

# Granularidade da classificacao. "W-MON" e a mesma grade semanal que
# baselines.py usa para agregar as series -- os dois modulos precisam concordar,
# senao a classificacao descreve uma serie diferente da que e modelada.
GRANULARIDADE_CLASSIFICACAO = "W-MON"

# Apenas estes quadrantes seguem para a modelagem; e o recorte do objeto de
# estudo (demanda intermitente), nao um filtro de conveniencia.
QUADRANTES_FOCO = ("Intermitente", "Lumpy")

# ---------------------------------------------------------------------------
# Feature engineering
# ---------------------------------------------------------------------------
# Conjunto base, sempre ativo.
LAGS_DIAS = (1, 7, 14, 30)
JANELAS_ROLLING = (7, 30)

# Conjunto estendido, ativado por FEATURES_INTERMITENCIA. As janelas de 28 e
# 91 dias correspondem a 4 e 13 semanas exatas.
LAGS_DIAS_ESTENDIDO = (1, 7, 14, 28, 30)
JANELAS_ROLLING_ESTENDIDO = (7, 28, 30, 91)


def lags_dias() -> tuple:
    """Lags ativos nesta execucao: estendidos se FEATURES_INTERMITENCIA estiver ligado."""
    return LAGS_DIAS_ESTENDIDO if FEATURES_INTERMITENCIA else LAGS_DIAS


def janelas_rolling() -> tuple:
    """Janelas rolling ativas nesta execucao: estendidas se FEATURES_INTERMITENCIA estiver ligado."""
    return JANELAS_ROLLING_ESTENDIDO if FEATURES_INTERMITENCIA else JANELAS_ROLLING


# Datas comemorativas de dia fixo, marcadas como feature de calendario.
DATAS_COMEMORATIVAS_FIXAS = {
    (2, 14): "dia_dos_namorados_alternativo",
    (6, 12): "dia_dos_namorados",
    (10, 12): "dia_das_criancas",
}

# ---------------------------------------------------------------------------
# Analise estratificada por esparsidade
# ---------------------------------------------------------------------------
# Faixas de proporcao de dias sem venda, usadas so no relatorio.
FAIXAS_ESPARSIDADE = [
    (0.0, 0.40, "Baixa (0-40%)"),
    (0.40, 0.70, "Moderada (40-70%)"),
    (0.70, 1.01, "Alta (>70%)"),
]

# ---------------------------------------------------------------------------
# Busca de hiperparametros — RandomizedSearchCV + TimeSeriesSplit
# ---------------------------------------------------------------------------
N_SPLITS_TIME_SERIES_CV = 3
RANDOM_STATE = 42
N_JOBS = 1
USE_GPU_XGB = False

# Teto de linhas amostradas na busca de hiperparametros; o ajuste final sempre
# usa a tabela inteira. None = sem teto.
# Este e apenas o DEFAULT do modulo: a execucao pelo notebook sobrescreve com o
# parametro `max_linhas_tuning`, cujo valor efetivo fica gravado no METADATA.md.
MAX_LINHAS_TUNING = 40_000

N_ITER_RANDOM_SEARCH = 15
N_ITER_RANDOM_SEARCH_RF = None

# ---------------------------------------------------------------------------
# Metrica de selecao da busca de hiperparametros
# ---------------------------------------------------------------------------
# O WMAPE tem otimo na mediana condicional, que e zero numa serie ~95% esparsa:
# selecionar por ele orienta a busca a favorecer configuracoes que aproximam a
# previsao de zero -- o mesmo efeito que o baseline Zero evidencia. O RMSE tem
# otimo na media condicional e nao carrega esse vies.
#
# "WMAPE" seleciona os hiperparametros pela mesma metrica da avaliacao final;
# "RMSE" seleciona por uma metrica diferente, evitando essa circularidade. As
# duas execucoes devem ir para pastas distintas (ver ROTULO na celula 4).
METRICA_SELECAO_HIPER: str = "WMAPE"
# Nomes aceitos, para validacao na entrada do pipeline.
METRICAS_SELECAO_VALIDAS = ("WMAPE", "RMSE")
# Grava, a cada busca, qual candidato CADA criterio escolheria. Custa apenas a
# pontuacao extra dos candidatos ja ajustados.
DIAGNOSTICO_SELECAO_HIPER: bool = True

# ---------------------------------------------------------------------------
# Grades do Random Forest
# ---------------------------------------------------------------------------
# 3 x 4 x 3 x 3 = 108 combinacoes.
GRADE_HIPERPARAMETROS_RF_COMPLETA = {
    "n_estimators": [100, 200, 300],
    "max_depth": [6, 10, 15, None],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.5, 1.0],
}

# 3 x 3 x 3 x 2 = 54 combinacoes, para rodadas rapidas.
GRADE_HIPERPARAMETROS_RF_REDUZIDA = {
    "n_estimators": [100, 200, 300],
    "max_depth": [6, 10, 15],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.5],
}

GRADE_RF_MODO = "completa"

GRADE_HIPERPARAMETROS_RF = GRADE_HIPERPARAMETROS_RF_COMPLETA

_GRADES_RF = {
    "completa": GRADE_HIPERPARAMETROS_RF_COMPLETA,
    "reduzida": GRADE_HIPERPARAMETROS_RF_REDUZIDA,
}


def grade_rf(modo: str | None = None) -> dict:
    """Devolve a grade do RF pelo nome ('completa', 'reduzida' ou 'ativa')."""
    escolhido = (modo or GRADE_RF_MODO or "completa").lower()
    if escolhido == "ativa":
        return GRADE_HIPERPARAMETROS_RF
    if escolhido not in _GRADES_RF:
        raise ValueError(
            f"GRADE_RF_MODO invalido: {escolhido!r}. Use 'completa', 'reduzida' ou 'ativa'."
        )
    return _GRADES_RF[escolhido]


def definir_grade_rf(modo: str) -> dict:
    """Fixa a grade do RF para o resto da execucao e devolve a grade escolhida."""
    global GRADE_RF_MODO, GRADE_HIPERPARAMETROS_RF
    grade = grade_rf(modo)
    GRADE_RF_MODO = modo.lower()
    GRADE_HIPERPARAMETROS_RF = grade
    return grade


def n_iter_rf() -> int:
    """Iteracoes de busca do RF, caindo no valor do XGBoost quando nao definido."""
    return N_ITER_RANDOM_SEARCH_RF or N_ITER_RANDOM_SEARCH


# ---------------------------------------------------------------------------
# Grades do XGBoost
# ---------------------------------------------------------------------------
# A diferenca entre as duas nao e o tempo de execucao -- ambas fazem
# N_ITER_RANDOM_SEARCH sorteios -- e sim a COBERTURA do espaco de busca:
#
#   COMPLETA     3^7 = 2187 combinacoes. Com 15 iteracoes, cobre 0,7% da grade,
#                contra 13,9% da grade do RF (108 combinacoes). "Mesmo numero de
#                iteracoes" nao significa busca simetrica.
#   EQUILIBRADA  162 combinacoes -> cobertura 9,3%, mesma ordem do RF. Fixa
#                subsample e colsample no default de literatura e acrescenta
#                min_child_weight, que controla o minimo de observacoes por
#                folha e e o hiperparametro mais relevante quando ~95% dos
#                alvos sao zero.
GRADE_HIPERPARAMETROS_XGB_COMPLETA = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_alpha": [0, 0.1, 1.0],
    "reg_lambda": [1.0, 2.0, 5.0],
}

GRADE_HIPERPARAMETROS_XGB_EQUILIBRADA = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 5, 20],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_lambda": [1.0, 5.0],
}

GRADE_XGB_MODO = "completa"

# Alias para codigo que importa o nome sem sufixo.
GRADE_HIPERPARAMETROS_XGB = GRADE_HIPERPARAMETROS_XGB_COMPLETA

_GRADES_XGB = {
    "completa": GRADE_HIPERPARAMETROS_XGB_COMPLETA,
    "equilibrada": GRADE_HIPERPARAMETROS_XGB_EQUILIBRADA,
}


def grade_xgb(modo: str | None = None) -> dict:
    """Devolve a grade do XGBoost pelo nome ('completa' ou 'equilibrada')."""
    escolhido = (modo or GRADE_XGB_MODO or "completa").lower()
    if escolhido not in _GRADES_XGB:
        raise ValueError(
            f"GRADE_XGB_MODO invalido: {escolhido!r}. Use 'completa' ou 'equilibrada'."
        )
    return _GRADES_XGB[escolhido]


def definir_grade_xgb(modo: str) -> dict:
    """Fixa a grade do XGBoost para o resto da execucao e devolve a escolhida."""
    global GRADE_XGB_MODO, GRADE_HIPERPARAMETROS_XGB
    grade = grade_xgb(modo)
    GRADE_XGB_MODO = modo.lower()
    GRADE_HIPERPARAMETROS_XGB = grade
    return grade


def _tamanho_grade(grade: dict) -> int:
    """Numero de combinacoes de uma grade (produto do tamanho de cada lista)."""
    total = 1
    for valores in grade.values():
        total *= max(len(valores), 1)
    return total


def descrever_busca_rf() -> str:
    """Resume a busca dos dois modelos com a COBERTURA de cada grade.

    Cobertura = iteracoes / combinacoes. Com grades de tamanhos diferentes, o
    mesmo numero de iteracoes nao caracteriza busca simetrica. A string
    devolvida avisa quando as coberturas diferem por mais de 3x."""
    g_rf, g_xgb = grade_rf(), grade_xgb()
    n_rf, n_xgb = _tamanho_grade(g_rf), _tamanho_grade(g_xgb)
    cob_rf = n_iter_rf() / n_rf * 100
    cob_xgb = N_ITER_RANDOM_SEARCH / n_xgb * 100
    razao = max(cob_rf, cob_xgb) / max(min(cob_rf, cob_xgb), 1e-9)
    simetria = "cobertura equiparavel" if razao <= 3 else "COBERTURA ASSIMETRICA (declarar na Secao 4.3.1)"
    return (
        f"RF: grade '{GRADE_RF_MODO}' ({n_rf} comb., {n_iter_rf()} iter., "
        f"cobertura {cob_rf:.1f}%) | "
        f"XGBoost: grade '{GRADE_XGB_MODO}' ({n_xgb} comb., {N_ITER_RANDOM_SEARCH} iter., "
        f"cobertura {cob_xgb:.1f}%) | {simetria}"
    )


# ---------------------------------------------------------------------------
# Analise de correlacao features x target (src/correlacao.py)
# ---------------------------------------------------------------------------
# Passo entre as datas de origem amostradas para o relatorio de correlacao.
# NAO pode ser multiplo de 7: isso poria todas as origens no mesmo dia da
# semana, tornando o dia da semana do alvo uma funcao exata do horizonte
# (alvo = origem + h, h = 1..7) e fabricando um "r = 1,000" entre horizonte e
# dia_semana_alvo que e artefato da amostragem do relatorio, nao propriedade da
# tabela de treino. 13 e primo com 7 e cobre os sete dias de forma equilibrada.
CORRELACAO_STRIDE_ORIGENS_DIAS = 13

# Teto de linhas da amostra usada no relatorio de correlacao.
CORRELACAO_MAX_LINHAS = 500_000

# Pares acima deste |r| sao listados como multicolinearidade no relatorio. E a
# evidencia que sustenta as exclusoes de features redundantes.
LIMIAR_MULTICOLINEARIDADE = 0.85

# ---------------------------------------------------------------------------
# Colunas esperadas na entrada (schema minimo)
# ---------------------------------------------------------------------------
COLUNAS_ENTRADA_MINIMAS = (
    "sku",
    "Data",
    "Quantidade_Vendida",
    "categoria_produto",
)

TARGET = "Quantidade_Vendida"


@dataclass
class Colunas:
    """Nomes de colunas usados internamente pelo pipeline (evita strings soltas)."""

    sku: str = "sku"
    data: str = "Data"
    qtd: str = "Quantidade_Vendida"
    categoria: str = "categoria_produto"
    idade: str = "idade_produto_dias"
    campanha: str = "Campanha"
    promo: str = "promo_campanha"
    especial_br: str = "data_especial_br"
    black_friday: str = "semana_black_friday"
    data_cadastro: str = "data_cadastro_produto"
    tipo_produto: str = "tipo_produto"
    categoria_id: str = "categoria_id"


COLS = Colunas()

# ---------------------------------------------------------------------------
# Calendario externo de feriados e campanhas (Experimento 2 — Cenario 2)
# ---------------------------------------------------------------------------
# Preenchido pelo pipeline quando um calendario e informado nos parametros.
CALENDARIO_EXOGENAS_PATH: Path | None = None

# Todas as colunas que o CSV do calendario deve conter. A leitura e a validacao
# usam esta lista inteira; a exclusao de exogenas (EXOGENAS_EXCLUIDAS) age
# depois, sobre o que vira variavel.
COLUNAS_CALENDARIO_RICO = [
    "flag_feriado_nacional", "flag_campanha", "intensidade_max",
    "ev_Black_Friday", "ev_Black_November_Esquenta", "ev_Natal",
    "ev_Dia_das_Maes", "ev_Dia_dos_Pais", "ev_Pascoa",
    "ev_Dia_das_Criancas", "ev_Dia_dos_Namorados",
    "ev_Semana_do_Consumidor", "ev_Cyber_Monday", "ev_Copa_do_Mundo",
]

# Sufixo `_alvo`: valor da exogena na DATA-ALVO da previsao (origem + h), nao na
# data de origem. Calendario e conhecido com antecedencia, entao usar o valor
# futuro nao e look-ahead.
COLUNAS_CALENDARIO_RICO_ALVO = [c + "_alvo" for c in COLUNAS_CALENDARIO_RICO]

# ---------------------------------------------------------------------------
# Quais exogenas entram de fato no Cenario 2
# ---------------------------------------------------------------------------
# O Cenario 2 usa as variaveis de calendario que fazem sentido para o
# contexto; a escolha e explicita e justificada abaixo.
#
# CRITERIO, medido apenas no treino:
#   (a) suporte amostral — quantos dias o evento fica ativo no periodo de
#       treino. Com 3 dias em 3 anos nao existe estimador honesto de efeito;
#   (b) redundancia — a variavel e quase uma transformacao de outra ja presente.
#
# Justificativa de cada exclusao (base teste, treino 2022-07 a 2025-06). Os
# numeros sao recalculados em selecao_variaveis.md a cada execucao, entao nao
# dependem deste comentario estar atualizado:
#   ev_Cyber_Monday          3 dias ativos no treino  -> sem suporte amostral
#   ev_Dia_dos_Pais         54 dias, importancia RF 7,1e-07
#   ev_Dia_das_Maes         54 dias, importancia RF 3,8e-05
#   ev_Dia_das_Criancas     36 dias, importancia RF 4,4e-05
#   ev_Dia_dos_Namorados    36 dias, importancia RF 6,0e-06
#   ev_Semana_do_Consumidor 42 dias, importancia RF 1,5e-04
#   flag_campanha           redundante com intensidade_max: e praticamente
#                           1{intensidade_max >= 2} (117 excecoes em 2.191 dias,
#                           r = 0,858) e fica ativa em 58,6% dos dias de treino.
#                           Nao marca eventos, marca TEMPORADAS (01/11 a 24/12 e
#                           02/01 a 15/02, todo ano). Uma binaria ligada mais da
#                           metade do tempo discrimina pouco por construcao, e a
#                           versao graduada da mesma informacao ja esta em
#                           intensidade_max, com 6 niveis.
#
# As 7 mantidas: intensidade_max, flag_feriado_nacional, ev_Black_Friday,
# ev_Black_November_Esquenta, ev_Natal, ev_Pascoa e ev_Copa_do_Mundo.
#
# LIMITACAO: ev_Copa_do_Mundo e a unica exogena cujo suporte no teste (51 dias,
# Copa de 2026) e proximo do suporte no treino (60 dias, Catar 2022). O efeito
# esta sendo extrapolado de um unico evento anterior.
EXOGENAS_EXCLUIDAS_PADRAO: tuple = (
    "ev_Cyber_Monday",
    "ev_Dia_dos_Pais",
    "ev_Dia_das_Maes",
    "ev_Dia_das_Criancas",
    "ev_Dia_dos_Namorados",
    "ev_Semana_do_Consumidor",
    "flag_campanha",
)

# Valor efetivo da execucao, sempre reatribuido pelo pipeline (caindo no PADRAO
# quando o parametro vem como None).
#
# Por que existe o par PADRAO/efetivo: `config` e estado global do processo. Sem
# a reatribuicao incondicional, uma execucao que passasse `exogenas_excluidas=()`
# deixaria a lista vazia para todas as chamadas seguintes de `rodar()` na mesma
# sessao -- a execucao seguinte herdaria 14 exogenas em silencio e registraria
# isso no METADATA como se tivesse sido pedido. Em Colab, rodar duas
# configuracoes na mesma sessao para compara-las e o caso normal.
EXOGENAS_EXCLUIDAS: tuple = EXOGENAS_EXCLUIDAS_PADRAO


def colunas_calendario_rico() -> list:
    """Exogenas que o Cenario 2 usa de fato nesta execucao.

    O CSV do calendario continua sendo lido e validado com as 14 colunas, e
    calendario_exogenas_usado.csv registra todas elas. A exclusao age apenas
    sobre o que entra como variavel do modelo, para que o relatorio de selecao
    consiga comparar o que ficou com o que saiu."""
    excluidas = set(EXOGENAS_EXCLUIDAS or ())
    return [c for c in COLUNAS_CALENDARIO_RICO if c not in excluidas]


def colunas_calendario_rico_alvo() -> list:
    """Mesma lista de `colunas_calendario_rico()`, com o sufixo `_alvo`."""
    return [c + "_alvo" for c in colunas_calendario_rico()]

# =============================================================================
# OPCOES DE ANALISE — METRICAS E DIAGNOSTICOS
# =============================================================================
# Reporta, por modelo, quantos SKUs ficam com WMAPE indefinido (denominador
# zero = SKU sem venda no teste). Sem isso, a mediana do WMAPE e calculada
# sobre um subconjunto diferente para cada modelo, sem aviso.
DIAGNOSTICO_WMAPE_ZERO: bool = True

# MAE, RMSSE e MASE escalado pela media do treino (Kolassa, 2016). Necessario
# para comparar o estrato Cessado (ESTRATIFICAR_ATIVOS_CESSADOS): sem venda no
# teste, so pode ser comparado por MAE. O RMSSE e a unica metrica de erro
# quadratico do conjunto, logo a unica cujo otimo e a media condicional e nao
# a mediana (= zero, numa serie com ~95% de zeros).
METRICAS_EXTRAS: bool = False

# Avalia tambem o total semanal, agregando as previsoes diarias. E o nivel em
# que os 4 modelos operam nativamente e equivale ao lead time de 7 dias.
AVALIAR_SEMANAL: bool = False

# Tamanho de efeito (r bisserial pareado) e correcao de Holm-Bonferroni nos
# testes de Wilcoxon.
TAMANHO_EFEITO: bool = False

# dia_semana_alvo como feature dos modelos de ML. E calendario, entao nao ha
# look-ahead; sem ela, so os baselines recebem o perfil semanal.
ADICIONAR_DIA_SEMANA: bool = False

# Funcao objetivo do XGBoost. Determina para qual estatistica o modelo
# converge, e portanto qual metrica ele tende a favorecer:
#   "reg:squarederror"   MSE      -> otimo = media condicional
#   "reg:absoluteerror"  MAE      -> otimo = mediana condicional (= 0 aqui)
#   "count:poisson"      Poisson
#   "reg:tweedie"        Poisson-Gama composta
#
# Sobre a Tweedie: com 1 < p < 2 ela e uma Poisson-Gama composta -- massa de
# probabilidade em zero mais densidade continua nos positivos. E exatamente a
# hipotese estrutural que SBA e TSB assumem por construcao (eventos esporadicos
# de demanda, cada um com um tamanho). Usa-la alinha a verossimilhanca do ML a
# dos baselines, e a comparacao passa a ser de capacidade de aprendizado em vez
# de premissa distribucional. Quando XGB_OBJETIVO comeca com "reg:tweedie",
# ml_models acrescenta tweedie_variance_power a grade de busca.
XGB_OBJETIVO: str = "reg:squarederror"
XGB_TWEEDIE_POWER_GRADE = [1.1, 1.3, 1.5]

# =============================================================================
# BLOCO A — CORRECOES DE COMPORTAMENTO. Default True.
# =============================================================================
# Corrige a validacao cruzada da busca de hiperparametros, em dois pontos:
#   (a) ORDENACAO: montar_tabela_horizontes concatena blocos por horizonte
#       (h=1 inteiro, depois h=2, ...) e o TimeSeriesSplit corta por POSICAO de
#       linha. Sem ordenar por data, a "validacao temporal" treina em h=1 e
#       valida em h=2/h=3 com o mesmo valor de y dos dois lados: a venda de um
#       dia entra como (origem=d-1, h=1) no treino e como (origem=d-3, h=3) na
#       validacao, com features quase identicas. O score sai otimista e a busca
#       escolhe hiperparametros que decoram.
#   (b) SCORER: em dobras sem nenhuma demanda o WMAPE e indefinido. Devolver
#       0.0 daria nota maxima a qualquer candidato; o scorer cai para MAE puro,
#       que preserva a ordenacao entre candidatos.
# Como conferir: coluna `detalhe` da etapa 'tuning' em tempos_execucao.csv.
ORDENAR_TABELA_CRONOLOGICA: bool = True

# Corta de cada serie os dias anteriores a existencia do produto.
# A grade diaria e expandida de DATA_INICIO a DATA_FIM para todo SKU, entao sem
# o corte um produto lancado em 2024 recebe ~600 dias de demanda zero que nunca
# existiram: infla ADI e CV2, enviesa o ML para zero e contamina o denominador
# do MASE.
TRUNCAR_PRE_LANCAMENTO: bool = True

# Media geometrica no score relativo, com um piso no denominador.
# Razoes de erro sao multiplicativas e o TSB preve valores proximos de zero, que
# viram denominador; com media aritmetica o resultado explode. Media geometrica
# e o padrao para razoes de erro (Fildes, 1992; Davydenko e Fildes, 2013).
SCORE_RELATIVO_GEOMETRICO: bool = True
PISO_RELATIVO_SCORE: float = 1e-3

# Restringe ao treino os atributos derivados de venda (primeira venda
# observada, moda da categoria), que de outro modo enxergam o periodo de teste.
CORTE_ATRIBUTOS_TREINO: bool = True

# Grava as previsoes ponto a ponto, alem das metricas por SKU:
#   metricas_exp*_<modelo>.csv       uma linha por SKU
#   previsoes_exp*_<modelo>.parquet  uma linha por SKU x data x horizonte
# Sem o parquet, qualquer reanalise exige re-executar o pipeline inteiro.
SALVAR_PREVISOES_BRUTAS: bool = True

# =============================================================================
# BLOCO B — ANALISES NOVAS. Default True.
# Nao alteram modelo, feature nem metrica: acrescentam recortes de relatorio.
# =============================================================================
# Estratificacao ativos x cessados.
#   Ativo   = vendeu >= 1 unidade no teste -> WMAPE, MASE, RMSSE e MAE validos
#   Cessado = zero vendas no teste         -> MAE e RMSSE validos, WMAPE nao
# Com quase metade dos SKUs sem venda no teste, a mediana entre SKUs cai bem na
# fronteira entre os dois grupos: MAE_mediana e RMSSE_mediana passam a medir
# "quao perto de zero o modelo preve num SKU morto", nao "quem preve melhor
# demanda".
# E estratificacao de RELATORIO, nao criterio de selecao: todos os SKUs foram
# modelados e o recorte existe so na apresentacao. Nenhuma decisao de modelagem
# usa o periodo de teste.
ESTRATIFICAR_ATIVOS_CESSADOS: bool = True

# Baseline "Zero", que preve 0 sempre. Nao e um modelo proposto: e o controle
# que permite interpretar os demais, analogo ao classificador de classe
# majoritaria. Se Zero empata com TSB em todas as metricas, entao "o TSB
# venceu" significa "prever zero venceu", e o achado passa a ser sobre a
# inadequacao das metricas pontuais em cauda longa -- com evidencia propria, em
# vez de citacao.
BASELINE_ZERO: bool = True

# Metricas agregadas sobre todos os SKUs e dias de uma vez:
#   WMAPE_pooled = soma|y - yhat| / soma(y)      MAE_pooled = soma|y - yhat| / N
# Ficam definidas enquanto algum SKU vender, entao nao sofrem do denominador
# zero que afeta a versao por SKU. E tambem a definicao original do WMAPE -- a
# versao por SKU e a adaptacao -- e a que interessa ao negocio.
METRICAS_POOLED: bool = True

# Classificacao ADI/CV2 tambem em granularidade diaria, como tabela de
# robustez. Os cortes 1,32 e 0,49 pressupoem classificar na mesma granularidade
# em que se preve; classificar em semanas enquanto se modela em dias comprime o
# ADI. Esperado: quase tudo cai em Lumpy no diario, o que e um achado a reportar
# e justifica manter o semanal como classificacao principal.
CLASSIFICACAO_ROBUSTEZ_DIARIA: bool = True

# RELATORIO_DECISAO.md: parametros ativos, ranking por metrica, Wilcoxon,
# comparacao contra o baseline Zero e estratificacao, num arquivo so.
# E o primeiro arquivo a abrir depois de cada execucao, sem precisar abrir os
# ~20 CSVs da pasta.
RELATORIO_DECISAO: bool = True

# =============================================================================
# BLOCO C — MODELAGEM. Default False.
# Mudam o que entra no modelo.
# =============================================================================
# Features de intermitencia (Kourentzes, 2013; vencedores da M5):
#   dias_desde_ultima_venda, n_dias_venda_{28,91}, taxa_zeros_{28,91},
#   tam_medio_demanda, lag_28, roll_{mean,std}_{28,91},
#   mes_alvo e semana_ano_alvo (calendario puro, mesmo status de dia_semana_alvo)
# `dias_desde_ultima_venda` costuma dominar a importancia em serie intermitente.
# Sao a decomposicao de Croston (intervalo entre demandas + tamanho da
# demanda) oferecida ao ML, o que torna a comparacao com SBA/TSB mais justa,
# nao menos.
FEATURES_INTERMITENCIA: bool = False

# Como a identidade do SKU entra como variavel.
#   "ordinal": LabelEncoder alfabetico. A arvore so consegue perguntar
#       "sku_enc <= 137,5?", agrupando os SKUs alfabeticamente primeiros contra
#       o resto -- um corte sem relacao com demanda.
#   "perfil": substitui o ID por sku_media_hist, sku_taxa_zeros e sku_tam_medio,
#       calculados somente sobre o treino de cada janela. Um unico split
#       ("sku_media_hist <= 0,05") ja separa giro baixo de giro alto.
# Continua prevendo por SKU nos dois casos.
ENCODING_SKU: str = "ordinal"   # "ordinal" | "perfil"

# Grade do XGBoost, definida acima ("completa" | "equilibrada"). A grade
# equilibrada iguala a cobertura da busca entre RF e XGBoost (ver
# descrever_busca_rf).
GRADE_XGB_MODO_PADRAO: str = "completa"

# Reaproveita no Experimento 2 os hiperparametros ja buscados no Experimento 1,
# em vez de 3 buscas independentes.
# Sem isso, a diferenca entre Exp1 e Cenario 1 mistura "efeito do calendario"
# com "efeito de ter sorteado outros hiperparametros" -- o bastante para o
# Cenario 1 sair PIOR que o Exp1 (WMAPE 191,6 contra 185,8) mesmo recebendo
# informacao adicional legitima, e para o RF do Cenario 1 levar 545s contra 233s
# com os mesmos dados.
TUNAR_UMA_VEZ_GLOBAL: bool = False

# =============================================================================
# BLOCO D — PARCIMONIA DO CONJUNTO DE VARIAVEIS. Default True.
# =============================================================================
# Remove features que sao funcao (quase) deterministica de outra ja presente.
# O criterio e estrutural (identidade algebrica ou janela equivalente) e
# verificavel na matriz de correlacao que o pipeline gera sobre o periodo de
# treino, em correlacao_features.md.
#
# Arvores toleram preditores colineares -- nao ha erro de estimacao aqui, como
# haveria numa regressao. O ganho e outro:
#   1. a importancia deixa de se diluir entre variaveis equivalentes
#      (roll_mean_7/28/30 repartem entre si a mesma informacao, o que torna a
#      figura de importancia dificil de interpretar);
#   2. o custo cai -- o RF respondeu por ~85% do tempo da ultima execucao;
#   3. o RF degradou com o conjunto ampliado do Cenario 2 (WMAPE r = -0,688 a
#      favor do Cenario 1), entao parcimonia tem evidencia previa nesta base.
REMOVER_FEATURES_REDUNDANTES: bool = True

# {feature_removida: (feature_que_a_substitui, justificativa)}.
# A remocao so acontece se a substituta estiver ativa nesta execucao -- com
# FEATURES_INTERMITENCIA=False as janelas de 28 e 91 dias nem existem, e nada e
# removido por engano.
FEATURES_REDUNDANTES: dict = {
    # Identidades algebricas exatas: taxa_zeros_j = 1 - n_dias_venda_j / j.
    # O r = -1,000 nao e "correlacao alta": e a mesma variavel reescalada e
    # invertida.
    "taxa_zeros_28": ("n_dias_venda_28",
                      "identidade: taxa_zeros_28 = 1 - n_dias_venda_28/28 (r = -1,000)"),
    "taxa_zeros_91": ("n_dias_venda_91",
                      "identidade: taxa_zeros_91 = 1 - n_dias_venda_91/91 (r = -1,000)"),
    # Janelas equivalentes. Mantem-se a de 28 dias (4 semanas exatas, alinhada
    # ao padrao semanal) e descarta-se a de 30.
    "roll_mean_30": ("roll_mean_28",
                     "janela equivalente (r = 0,999 intermitente / 0,997 lumpy); "
                     "28 dias = 4 semanas exatas"),
    "roll_std_30": ("roll_std_28",
                    "janela equivalente (r = 0,997 intermitente / 0,993 lumpy)"),
    # Quase determinada pela semana do ano, que e mais fina.
    "mes_alvo": ("semana_ano_alvo",
                 "quase determinada pela semana do ano (r = 0,975)"),
}

# Exclusoes ad-hoc, aplicadas alem do dicionario acima e sem exigir substituta.
FEATURES_EXCLUIDAS_EXTRA: tuple = ()

# Por que `dia_semana_alvo` NAO esta na lista de redundantes.
# Um relatorio de correlacao com passo multiplo de 7 entre as origens reporta
# `horizonte x dia_semana_alvo: r = 1,000`, o que sugere redundancia exata. Isso
# e artefato da amostragem do relatorio, nao propriedade dos dados: com todas as
# origens no mesmo dia da semana, o dia da semana do alvo fica determinado por h.
# Na tabela de treino do modo fiel as origens sao diarias e a feature varia
# livremente. Ver CORRELACAO_STRIDE_ORIGENS_DIAS.
#
# O aliasing e real, porem, na tabela de PREVISAO: as origens de previsao avancam
# de 7 em 7 dias (ml_models.executar_walk_forward), entao no teste cada dia da
# semana aparece em um unico horizonte. Nao invalida as previsoes, mas impede
# separar "efeito do dia da semana" de "efeito do horizonte" nos resultados --
# e uma limitacao do desenho de avaliacao.


def features_a_excluir(colunas_ativas) -> dict:
    """Quais features remover de `colunas_ativas`, e por que.

    Devolve {feature_removida: justificativa}. As de FEATURES_REDUNDANTES so
    saem quando a substituta esta presente; as de FEATURES_EXCLUIDAS_EXTRA saem
    incondicionalmente, se existirem."""
    ativas = set(colunas_ativas)
    remover: dict = {}
    if REMOVER_FEATURES_REDUNDANTES:
        for feature, (substituta, motivo) in FEATURES_REDUNDANTES.items():
            if feature in ativas and substituta in ativas:
                remover[feature] = f"{motivo}; mantida `{substituta}`"
    for feature in FEATURES_EXCLUIDAS_EXTRA or ():
        if feature in ativas:
            remover.setdefault(feature, "exclusao manual (FEATURES_EXCLUIDAS_EXTRA)")
    return remover


# =============================================================================
# BLOCO B2 — RECORTE GERENCIAL E CONSOLIDACAO. Default True.
# Recortes de relatorio e figuras; nao alteram modelo nem metrica.
# =============================================================================
# Estratificacao por volume de pedidos (curva ABC / Pareto).
#
# O valor economico da previsao nao esta distribuido uniformemente entre os
# SKUs: um erro num SKU que concentra milhares de pedidos afeta muito mais
# clientes finais do que o mesmo erro percentual num SKU de cauda que vendeu 3
# vezes no ano. A mediana entre SKUs trata os dois igualmente, o que responde a
# pergunta de desempenho geral mas nao a gerencial ("onde a previsao vale a
# pena?").
#
# O recorte responde: nos 20% de SKUs com mais pedidos, qual modelo preve melhor,
# e quanto da demanda total esses 20% concentram.
#
# O volume e medido no periodo de TREINO -- e a informacao que um gestor teria em
# maos ao decidir onde investir em previsao, e nao ha vazamento.
#
# Fonte do volume, em ordem de preferencia:
#   1. Qtde_PedidosUnicos — pedidos DISTINTOS, ou seja clientes atendidos, e nao
#      unidades vendidas;
#   2. numero de dias com venda no treino, quando a coluna acima nao existe.
# A fonte efetivamente usada aparece no log e no relatorio.
ESTRATIFICAR_POR_VOLUME: bool = True

FAIXAS_VOLUME_PEDIDOS = [
    (0.00, 0.20, "A - Top 20% (maior nº de pedidos)"),
    (0.20, 0.50, "B - Intermediários (20-50%)"),
    (0.50, 1.01, "C - Cauda (50-100%)"),
]

# Consolidacao das saidas em markdown, alem dos CSVs:
#   LOG_EXECUCAO.md            captura literal de tudo que foi impresso
#   RESULTADOS_CONSOLIDADO.md  as tabelas formatadas, com notas de leitura
# Consolidado.mostrar imprime E registra na mesma chamada, entao nenhuma tabela
# aparece no console sem aparecer no markdown.
CONSOLIDAR_MARKDOWN: bool = True

# Figuras dos recortes por estrato (src/graficos_estratos.py): curva de
# Pareto dos pedidos, barras e boxplots por estrato, e real x previsto dos N SKUs
# com maior numero de pedidos. As figuras padrao mostram melhor e pior caso por
# quadrante ADI/CV2, que costumam cair em SKUs de giro baixissimo.
GRAFICOS_POR_ESTRATO: bool = True
N_SKUS_TOP_FIGURAS: int = 6


In [ ]:
%%writefile /content/src/exogenas.py
"""
src/exogenas.py — Calendario externo de feriados e campanhas promocionais.

Carrega `calendario_diario_wide.csv` e expoe a funcao `carregar_calendario()`
que retorna um DataFrame com index = data (datetime), pronto para ser
mesclado ao df_modelagem via adicionar_calendario_exogeno() em features.py.
"""
from __future__ import annotations

import warnings
from pathlib import Path

import pandas as pd

from . import config


def carregar_calendario(caminho) -> pd.DataFrame:
    """Carrega o CSV de calendario externo, valida cobertura e retorna
    DataFrame com index = data (datetime), contendo as colunas de
    config.COLUNAS_CALENDARIO_RICO.

    Validacoes obrigatorias:
      1. Arquivo existe (FileNotFoundError informativo).
      2. Coluna 'data' presente e convertivel para datetime.
      3. Todas as colunas de COLUNAS_CALENDARIO_RICO presentes (KeyError).
      4. Cobertura: warning se DATA_INICIO < min(data) ou DATA_FIM > max(data).
      5. Sem datas duplicadas (o CSV wide tem 1 linha/dia).
    """
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(
            f"Calendario externo nao encontrado: {caminho}\n"
            "Suba o arquivo 'calendario_diario_wide.csv' para o Drive e ajuste "
            "CALENDARIO_EXOGENAS na celula de parametros do notebook."
        )

    cal = pd.read_csv(caminho)

    if "data" not in cal.columns:
        raise KeyError(
            f"Coluna 'data' nao encontrada em {caminho.name}. "
            f"Colunas presentes: {list(cal.columns)[:10]}"
        )

    cal["data"] = pd.to_datetime(cal["data"])

    colunas_rico = config.COLUNAS_CALENDARIO_RICO
    faltando = [c for c in colunas_rico if c not in cal.columns]
    if faltando:
        raise KeyError(
            f"Colunas ausentes em {caminho.name}: {faltando}\n"
            "Verifique se o arquivo e o calendario_diario_wide.csv correto."
        )

    # Verificar cobertura temporal do pipeline
    data_inicio_ts = pd.Timestamp(config.DATA_INICIO)
    data_fim_ts = pd.Timestamp(config.DATA_FIM)
    cal_min = cal["data"].min()
    cal_max = cal["data"].max()
    if data_inicio_ts < cal_min:
        warnings.warn(
            f"[exogenas] DATA_INICIO ({config.DATA_INICIO}) anterior ao "
            f"inicio do calendario ({cal_min.date()}). Dias sem cobertura "
            "receberao 0 no merge (sem evento)."
        )
    if data_fim_ts > cal_max:
        warnings.warn(
            f"[exogenas] DATA_FIM ({config.DATA_FIM}) posterior ao "
            f"fim do calendario ({cal_max.date()}). Dias sem cobertura "
            "receberao 0 no merge (sem evento)."
        )

    # Tratar duplicatas
    n_dup = cal["data"].duplicated().sum()
    if n_dup > 0:
        warnings.warn(
            f"[exogenas] {n_dup} data(s) duplicada(s) no calendario — "
            "mantendo apenas o primeiro registro por data."
        )
        cal = cal.drop_duplicates(subset="data", keep="first")

    # Selecionar apenas as colunas necessarias
    cal = cal[["data"] + colunas_rico].copy()
    cal = cal.set_index("data").sort_index()

    return cal


def merge_calendario_em_df(df: pd.DataFrame, cal: pd.DataFrame, col_data: str = "Data") -> pd.DataFrame:
    """Faz left join de `cal` em `df` pela coluna de data.

    Preenche NaN com 0 (data fora do range do calendario = sem evento).
    Nao sobrescreve colunas ja existentes no df.
    """
    colunas_novas = [c for c in config.COLUNAS_CALENDARIO_RICO if c not in df.columns]
    if not colunas_novas:
        return df

    cal_sub = cal[colunas_novas].reset_index()
    cal_sub = cal_sub.rename(columns={"data": col_data})

    df = df.merge(cal_sub, on=col_data, how="left")
    for c in colunas_novas:
        if c in df.columns:
            df[c] = df[c].fillna(0)
    return df

In [ ]:
%%writefile /content/src/gerar_dados_sinteticos.py
"""
Gerador de dataset SINTETICO — apoio para desenvolver e testar o pipeline
antes de ter acesso aos dados reais extraidos via sql/extracao_dados.sql.

NAO FAZ PARTE DA MODELAGEM: serve apenas para validar que todo o pipeline
(Fases 1-9) roda corretamente de ponta a ponta. Deve ser descartado (ou
mantido só como fixture de teste) quando os dados reais estiverem
disponiveis.

Schema gerado (igual ao que a extracao SQL produz, no grao SKU x Dia,
ja com D2 aplicado — agregacao por campanha feita aqui):
    sku, Data, Quantidade_Vendida, categoria_produto, idade_produto_dias,
    promo_campanha (flag simulada)
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config


def gerar_dataset_sintetico(
    n_skus: int = 600,
    seed: int = config.RANDOM_STATE,
) -> pd.DataFrame:
    """Gera vendas diarias sinteticas para n_skus, no periodo total do estudo.

    Simula:
      - distribuicao de popularidade tipo Pareto/cauda longa;
      - esparsidade heterogenea entre SKUs;
      - categorias de produto e idade (dias desde a 1a venda);
      - SKUs com "nascimento" escalonado (catalogo dinamico);
      - flag de promo_campanha simulada (blocos de ~5 dias, ~6% dos dias).
    """
    rng = np.random.default_rng(seed)

    datas = pd.date_range(config.DATA_INICIO, config.DATA_FIM, freq="D")
    n_dias = len(datas)

    categorias = [
        "casa_e_decoracao", "eletronicos", "moda", "beleza",
        "esporte", "brinquedos", "livros", "pet_shop",
    ]

    linhas = []
    for i in range(n_skus):
        sku = f"SKU_{i:05d}"
        categoria = categorias[rng.integers(0, len(categorias))]

        # Popularidade tipo Pareto: poucos SKUs de "cabeca", muitos de nicho
        popularidade = rng.pareto(a=1.5) + 0.05
        popularidade = min(popularidade, 15.0)

        # Probabilidade diaria de venda (define esparsidade do SKU)
        prob_venda = np.clip(0.02 + popularidade * 0.02, 0.01, 0.85)

        # Nascimento escalonado: nem todo SKU existe desde o inicio
        dia_nascimento = rng.integers(0, int(n_dias * 0.7))
        idx_ativos = np.arange(n_dias) >= dia_nascimento

        vendas = np.zeros(n_dias)
        ocorre_venda = rng.random(n_dias) < prob_venda
        magnitudes = rng.poisson(lam=max(popularidade, 0.5), size=n_dias) + 1
        vendas[ocorre_venda & idx_ativos] = magnitudes[ocorre_venda & idx_ativos]

        primeira_venda_idx = np.argmax(vendas > 0) if vendas.sum() > 0 else dia_nascimento
        idade_dias = np.maximum(np.arange(n_dias) - primeira_venda_idx, 0)

        df_sku = pd.DataFrame({
            "sku": sku,
            "Data": datas,
            "Quantidade_Vendida": vendas,
            "categoria_produto": categoria,
            "idade_produto_dias": idade_dias,
        })
        linhas.append(df_sku)

    df = pd.concat(linhas, ignore_index=True)

    # Somente linhas com venda > 0 permanecem "observadas" — o restante
    # sera reconstituido como zero na etapa de expansao (Fase 2), tal como
    # ocorreria com a extracao SQL real (que so retorna dias com venda).
    df_observado = df[df["Quantidade_Vendida"] > 0].copy()

    # Flag sintetica de promo_campanha: blocos de ~5 dias em ~6% dos dias,
    # por SKU, apenas para testar a logica que consome essa coluna.
    df_observado["promo_campanha"] = 0
    for sku in df_observado["sku"].unique():
        n_blocos = rng.integers(3, 10)
        for _ in range(n_blocos):
            inicio = rng.integers(0, n_dias - 5)
            bloco_datas = datas[inicio: inicio + 5]
            mask = (df_observado["sku"] == sku) & (df_observado["Data"].isin(bloco_datas))
            df_observado.loc[mask, "promo_campanha"] = 1

    return df_observado.reset_index(drop=True)


def salvar_dataset_sintetico(caminho=None, **kwargs) -> pd.DataFrame:
    """Gera e grava o dataset sintetico em parquet. Repassa **kwargs para
    gerar_dataset_sintetico."""
    caminho = caminho or (config.DATA_DIR / "vendas_sinteticas.parquet")
    df = gerar_dataset_sintetico(**kwargs)
    df.to_parquet(caminho, index=False)
    print(f"Dataset sintetico salvo em {caminho} ({len(df):,} linhas, "
          f"{df['sku'].nunique()} SKUs)")
    return df


if __name__ == "__main__":
    salvar_dataset_sintetico()


In [ ]:
%%writefile /content/src/preprocess.py
"""
Fase 2/3 — Pre-processamento.

Ordem das etapas, e o que cada uma resolve:
  1. carregar_extracao            le a base e normaliza o schema
  2. agregar_por_sku              consolida em uma linha por SKU x dia (D2)
  3. expandir_series_diarias      cria a grade diaria continua, com zeros
  4. truncar_antes_do_lancamento  tira os zeros de antes do produto existir
  5. detectar_outliers            marca candidatos, sem remover nada
  6. aplicar_criterios_elegibilidade  seleciona os SKUs que entram no estudo

As etapas 3 e 4 andam juntas: a expansao fabrica zeros para toda a janela e a
truncagem devolve so os que correspondem a dias em que o produto ja existia.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config
from .config import COLS


# Colunas da extracao real (tbItemOrder/tbOrder/tbCampaign/Company/tbProduct/
# tbCatalogue) para os nomes internos do pipeline. Ver sql/extracao_dados.sql.
RENOMEIA_COLUNAS_EXTRACAO_REAL = {
    "Ped_Campanha": "Campanha",
    "Ped_Data": COLS.data,
    "Qtde_PedidosUnicos": "PedidosUnicos",
    "Qtde_VendaTotal": COLS.qtd,
    "Prod_SKU": COLS.sku,                    # proProviderCode, nao itoCustomProdCode
    "Prod_Id": "idProduct_apoio",
    "Prod_Tipo": COLS.tipo_produto,
    "Prod_DataCadastro": COLS.data_cadastro,
    "Cat_CategoriaId": COLS.categoria_id,
    "Cat_CategoriaNome": COLS.categoria,
    "Cat_DataCadastro": "categoria_data_cadastro",
}

# Colunas de data que podem vir como texto e precisam de conversao.
_COLUNAS_DE_DATA_CANDIDATAS = (COLS.data, "Ped_Data", COLS.data_cadastro, "Prod_DataCadastro", "categoria_data_cadastro", "Cat_DataCadastro")


def carregar_extracao(caminho) -> pd.DataFrame:
    """Le a extracao (parquet ou csv) e devolve no schema interno.

    Detecta sozinho se a entrada esta no schema da extracao real (colunas
    Ped_*/Prod_*/Cat_*) ou no schema interno simplificado (sku/Data/
    Quantidade_Vendida, que o gerador sintetico produz), e renomeia quando
    preciso. Ver RENOMEIA_COLUNAS_EXTRACAO_REAL."""
    caminho = str(caminho)
    if caminho.endswith(".parquet"):
        df = pd.read_parquet(caminho)
    else:
        df = pd.read_csv(caminho)

    if "Ped_Data" in df.columns or "Prod_SKU" in df.columns:
        colunas_presentes = {k: v for k, v in RENOMEIA_COLUNAS_EXTRACAO_REAL.items() if k in df.columns}
        df = df.rename(columns=colunas_presentes)

    for col in _COLUNAS_DE_DATA_CANDIDATAS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    return df


# Alias para deixar explicito, no codigo de chamada, que a entrada e a extracao
# real. Aponta para o mesmo auto-detector.
carregar_extracao_real = carregar_extracao


def _moda(serie: pd.Series):
    """Valor mais frequente da serie. `mode()` devolve ordenado, entao pegar o
    primeiro torna o desempate deterministico."""
    m = serie.dropna().mode()
    return m.iloc[0] if not m.empty else np.nan


def resolver_atributos_produto(df: pd.DataFrame, corte=None) -> pd.DataFrame:
    """Reduz os atributos de produto a um valor por SKU.

    Necessario porque a categoria vem de tbItemOrder.CatalogueId e pode variar
    entre pedidos do mesmo produto (campanhas ou momentos diferentes) -- usar a
    categoria do primeiro registro do dia daria resultados instaveis.

    Regra: categoria e tipo pela moda das vendas; data de cadastro pelo minimo
    (deveria ser constante, ja que vem de tbProduct, mas o min protege contra
    divergencia).

    `corte`: calcula a moda so com vendas ate essa data. SKUs que so
    aparecem no teste caem no fallback sobre a extracao inteira, senao ficariam
    sem categoria. E inofensivo: sao SKUs sem historico de treino, e categoria e
    atributo cadastral, nao sinal de demanda."""
    agregadores = {}
    if COLS.categoria in df.columns:
        agregadores[COLS.categoria] = _moda
    if COLS.tipo_produto in df.columns:
        agregadores[COLS.tipo_produto] = _moda
    if COLS.data_cadastro in df.columns:
        agregadores[COLS.data_cadastro] = "min"

    if not agregadores:
        return pd.DataFrame({COLS.sku: df[COLS.sku].unique()})

    atributos_full = df.groupby(COLS.sku).agg(agregadores).reset_index()
    if corte is None:
        return atributos_full

    df_corte = df[df[COLS.data] <= pd.Timestamp(corte)]
    if df_corte.empty:
        return atributos_full

    atributos_treino = df_corte.groupby(COLS.sku).agg(agregadores).reset_index()
    # SKUs ausentes do treino herdam o valor da extracao completa.
    faltantes = atributos_full[~atributos_full[COLS.sku].isin(atributos_treino[COLS.sku])]
    return pd.concat([atributos_treino, faltantes], ignore_index=True)


def agregar_por_sku(df: pd.DataFrame, corte_atributos=None) -> pd.DataFrame:
    """Consolida a base em uma linha por SKU x dia, somando as campanhas.

    A modelagem e por SKU, nao por SKU x campanha.

    Duas etapas:
      1. atributos de produto resolvidos por SKU (resolver_atributos_produto);
      2. Quantidade_Vendida somada por (SKU, Data); promo_campanha, se existir,
         vira 1 quando QUALQUER campanha do SKU tinha promocao ativa no dia."""
    atributos = resolver_atributos_produto(df, corte=corte_atributos)

    agregadores_dia = {COLS.qtd: "sum"}
    if COLS.promo in df.columns:
        agregadores_dia[COLS.promo] = "max"
    if "PedidosUnicos" in df.columns:
        agregadores_dia["PedidosUnicos"] = "sum"

    vendas_dia = df.groupby([COLS.sku, COLS.data], as_index=False).agg(agregadores_dia)

    df_agg = vendas_dia.merge(atributos, on=COLS.sku, how="left")
    return df_agg


def expandir_series_diarias(
    df: pd.DataFrame,
    data_inicio: str = config.DATA_INICIO,
    data_fim: str = config.DATA_FIM,
) -> pd.DataFrame:
    """Expande cada SKU para uma serie diaria continua, com zero nos dias sem
    venda (demanda intermitente).

    Gera a grade cheia para todos os SKUs; e truncar_antes_do_lancamento que
    depois remove os dias em que o produto ainda nao existia."""
    datas_completas = pd.date_range(start=data_inicio, end=data_fim, freq="D")
    skus_unicos = df[COLS.sku].unique()

    idx = pd.MultiIndex.from_product([skus_unicos, datas_completas], names=[COLS.sku, COLS.data])
    df_grid = pd.DataFrame(index=idx).reset_index()

    df_full = pd.merge(df_grid, df, on=[COLS.sku, COLS.data], how="left")
    df_full[COLS.qtd] = df_full[COLS.qtd].fillna(0)

    # Atributos estaticos precisam ser propagados por SKU: os dias sem venda nao
    # tem linha correspondente em `df` e sairiam do merge como NaN.
    if COLS.categoria in df_full.columns:
        df_full[COLS.categoria] = (
            df_full.groupby(COLS.sku)[COLS.categoria].transform(lambda s: s.ffill().bfill())
        )
        df_full[COLS.categoria] = df_full[COLS.categoria].fillna("SEM_CATEGORIA")

    if COLS.tipo_produto in df_full.columns:
        df_full[COLS.tipo_produto] = (
            df_full.groupby(COLS.sku)[COLS.tipo_produto].transform(lambda s: s.ffill().bfill())
        )

    if COLS.data_cadastro in df_full.columns:
        df_full[COLS.data_cadastro] = (
            df_full.groupby(COLS.sku)[COLS.data_cadastro].transform(lambda s: s.ffill().bfill())
        )

    if COLS.promo in df_full.columns:
        df_full[COLS.promo] = df_full[COLS.promo].fillna(0).astype(int)

    # A idade e recalculada em features.adicionar_idade_produto, sobre a serie
    # ja expandida. Descartar aqui evita que um valor da extracao sobreviva.
    if COLS.idade in df_full.columns:
        df_full = df_full.drop(columns=[COLS.idade])

    return df_full.sort_values([COLS.sku, COLS.data]).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Truncagem pre-lancamento
# ---------------------------------------------------------------------------
def truncar_antes_do_lancamento(df_full: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Remove de cada serie os dias anteriores a existencia do produto.

    Sem isso, a grade expandida da a todo SKU a janela inteira
    DATA_INICIO..DATA_FIM, e um produto lancado em 2024 recebe ~600 dias de
    demanda zero que nunca existiram. As consequencias:
      - ADI e CV2 inflados: o SKU parece mais intermitente do que e;
      - ML enviesado para prever zero;
      - denominador do MASE contaminado por zeros fabricados;
      - esparsidade superestimada.

    REGRA: cada serie comeca em
        max( DATA_INICIO , min(data_cadastro, 1a venda observada) )

    Por que o `min` interno, e nao o cadastro sozinho: `Prod_DataCadastro` nem
    sempre e a data de nascimento do produto. Em produto migrado, recadastrado
    ou com registro atualizado, o cadastro aparece DEPOIS de vendas que de fato
    ocorreram -- e quando cai depois de DATA_FIM, cortar por ele apaga a serie
    inteira. A venda e evidencia direta de que o produto existia naquele dia; o
    cadastro e evidencia indireta.

    Por que NAO e o `max`: o intervalo entre o cadastro e a primeira venda e
    zero legitimo (o produto existia e nao vendeu) -- exatamente o tipo de
    observacao que o trabalho quer modelar. So se remove o periodo em que o
    produto nao existia.

    Garantias asseguradas por assert ao final:
      1. nenhuma linha com venda > 0 e removida;
      2. nenhum SKU perde todas as linhas.

    Devolve (df_truncado, resumo); o resumo alimenta o funil de filtragem."""
    n_linhas_antes = len(df_full)
    n_skus_antes = df_full[COLS.sku].nunique()

    primeira_venda = (
        df_full[df_full[COLS.qtd] > 0].groupby(COLS.sku)[COLS.data].min().rename("_primeira_venda")
    )
    df = df_full.merge(primeira_venda, on=COLS.sku, how="left")

    n_skus_cadastro_apos_venda = 0
    if COLS.data_cadastro in df.columns:
        cadastro = df[COLS.data_cadastro]
        venda = df["_primeira_venda"]
        # Contagem de SKUs com cadastro posterior a venda: e diagnostico de
        # qualidade do cadastro no ERP, e vai para o relatorio.
        suspeitos = df.loc[cadastro.notna() & venda.notna() & (cadastro > venda), COLS.sku]
        n_skus_cadastro_apos_venda = int(suspeitos.nunique())
        # min(cadastro, 1a venda), tolerando ausencia de qualquer um dos dois.
        # A mesma expressao esta em features._origem_da_serie.
        inicio = cadastro.fillna(venda).where(
            venda.isna() | (cadastro.fillna(venda) <= venda), venda
        )
    else:
        inicio = df["_primeira_venda"]

    # SKUs sem nenhuma referencia (sem cadastro e sem venda) ficam intactos.
    inicio = inicio.fillna(pd.Timestamp(config.DATA_INICIO))
    inicio = inicio.clip(lower=pd.Timestamp(config.DATA_INICIO))

    manter = df[COLS.data] >= inicio
    n_vendas_removidas = int(((~manter) & (df[COLS.qtd] > 0)).sum())
    df = df[manter].drop(columns=["_primeira_venda"]).reset_index(drop=True)

    n_skus_depois = df[COLS.sku].nunique()
    resumo = {
        "linhas_antes": n_linhas_antes,
        "linhas_depois": len(df),
        "linhas_removidas": n_linhas_antes - len(df),
        "pct_removido": (n_linhas_antes - len(df)) / n_linhas_antes * 100 if n_linhas_antes else 0.0,
        "skus_antes": int(n_skus_antes),
        "skus_depois": int(n_skus_depois),
        "skus_esvaziados": int(n_skus_antes - n_skus_depois),
        "linhas_com_venda_removidas": n_vendas_removidas,
        "skus_cadastro_apos_1a_venda": n_skus_cadastro_apos_venda,
    }

    # Se qualquer invariante falhar, a truncagem esta descartando demanda real e
    # todo o resto fica invalido. Parar aqui e melhor do que produzir metricas
    # silenciosamente erradas.
    assert n_vendas_removidas == 0, (
        f"Item 12: a truncagem removeu {n_vendas_removidas} linhas com venda > 0. "
        "Isso nao deveria ser possivel — verifique COLS.data_cadastro na extracao."
    )
    assert resumo["skus_esvaziados"] == 0, (
        f"Item 12: a truncagem esvaziou {resumo['skus_esvaziados']} SKUs."
    )
    return df, resumo


def detectar_outliers(df: pd.DataFrame, treino_fim: str = config.TREINO_FIM) -> pd.DataFrame:
    """Marca candidatos a outlier na coluna `outlier_candidato`, sem remover.

    Criterio: venda acima de N desvios-padrao da media historica do SKU,
    calculada so sobre o treino. Esses pontos devem ser INSPECIONADOS
    (cruzando com o calendario de datas especiais) antes de qualquer exclusao
    -- num varejo com Black Friday, o pico geralmente e o fenomeno de
    interesse, nao ruido."""
    df = df.copy()
    treino = df[df[COLS.data] <= treino_fim]
    stats = treino.groupby(COLS.sku)[COLS.qtd].agg(["mean", "std"]).rename(
        columns={"mean": "_media_treino", "std": "_dp_treino"}
    )
    df = df.merge(stats, on=COLS.sku, how="left")
    limite = df["_media_treino"] + config.OUTLIER_N_DESVIOS_PADRAO * df["_dp_treino"].fillna(0)
    df["outlier_candidato"] = (df[COLS.qtd] > limite) & (df["_dp_treino"].fillna(0) > 0)
    return df.drop(columns=["_media_treino", "_dp_treino"])


def aplicar_criterios_elegibilidade(
    df_full: pd.DataFrame,
    treino_inicio: str = config.TREINO_INICIO,
    treino_fim: str = config.TREINO_FIM,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Seleciona os SKUs do estudo: ao menos uma venda no ultimo ano de treino.

    O "ultimo ano" e [treino_fim - 1 ano, treino_fim], calculado sem tocar no
    periodo de teste. Descarta produtos ja descontinuados antes do inicio da
    avaliacao.

    O criterio e deliberadamente frouxo. Endurece-lo reduziria a proporcao de
    SKUs sem demanda no teste, mas o cenario realista e justamente nao saber de
    antemao quais produtos vao morrer -- prever isso e parte do problema. O
    tratamento desses SKUs acontece na APRESENTACAO, via estratificacao ativos x
    cessados, que nao altera nenhuma decisao de modelagem e portanto nao
    introduz vazamento.

    Devolve (df_filtrado, resumo_por_sku)."""
    treino_fim_ts = pd.Timestamp(treino_fim)
    ultimo_ano_inicio = treino_fim_ts - pd.DateOffset(years=1)

    # Apenas dias com venda dentro do ultimo ano de treino.
    mask_ativo = (
        (df_full[COLS.data] >= ultimo_ano_inicio)
        & (df_full[COLS.data] <= treino_fim_ts)
        & (df_full[COLS.qtd] > 0)
    )
    df_ativo = df_full[mask_ativo]

    resumo = (
        df_ativo.groupby(COLS.sku)[COLS.data]
        .agg(
            primeira_venda_ultimo_ano="min",
            ultima_venda_ultimo_ano="max",
            dias_com_venda_ultimo_ano="nunique",
        )
        .reset_index()
    )
    resumo["elegivel"] = True  # estar na lista ja garante pelo menos 1 venda

    skus_elegiveis = resumo[COLS.sku]
    df_filtrado = df_full[df_full[COLS.sku].isin(skus_elegiveis)].copy()

    return df_filtrado, resumo


def funil_filtragem_texto(df_full: pd.DataFrame, resumo: pd.DataFrame,
                          resumo_truncagem: dict | None = None) -> str:
    """Monta o funil de filtragem em markdown: de quantos SKUs se partiu, o que
    a truncagem removeu e quantos sobraram elegiveis.

    E a tabela de rastreabilidade da amostra -- o caminho da base bruta ate os
    SKUs modelados, com o numero de cada etapa."""
    n_bruto = df_full[COLS.sku].nunique()
    n_elegiveis = len(resumo)
    treino_fim_ts = pd.Timestamp(config.TREINO_FIM)
    ultimo_ano_inicio = (treino_fim_ts - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
    linhas = [
        "## Funil de filtragem da amostra (Secao 4.1.2)",
        "",
        f"- SKUs na base expandida (bruta): **{n_bruto}**",
    ]
    if resumo_truncagem:
        linhas.append(
            f"- Item 12 — truncagem pre-lancamento: **{resumo_truncagem['linhas_removidas']:,}** "
            f"linhas removidas de {resumo_truncagem['linhas_antes']:,} "
            f"({resumo_truncagem['pct_removido']:.1f}%) — dias anteriores ao cadastro/1a venda "
            f"do produto, que a expansao da grade diaria havia preenchido com zero."
        )
        linhas.append(
            f"  - Nenhuma linha com venda > 0 removida "
            f"({resumo_truncagem.get('linhas_com_venda_removidas', 0)}); "
            f"SKUs preservados: {resumo_truncagem.get('skus_depois', '?')} de "
            f"{resumo_truncagem.get('skus_antes', '?')}."
        )
        n_susp = resumo_truncagem.get("skus_cadastro_apos_1a_venda", 0)
        if n_susp:
            linhas.append(
                f"  - **Qualidade do cadastro:** {n_susp} SKUs tem "
                f"`Prod_DataCadastro` POSTERIOR a uma venda ja registrada "
                f"(produto migrado/recadastrado). Para esses, a 1a venda prevalece "
                f"sobre o cadastro — ver `preprocess.truncar_antes_do_lancamento`."
            )
    linhas += [
        f"- Criterio: pelo menos uma venda no ultimo ano de treino "
        f"({ultimo_ano_inicio} a {config.TREINO_FIM})",
        f"- **SKUs elegiveis: {n_elegiveis}**",
        "",
    ]
    return "\n".join(linhas)


In [ ]:
%%writefile /content/src/eda.py
"""
Fase — Analise Exploratoria de Dados: esparsidade,
distribuicao de vendas, padroes temporais e qualidade dos dados.
Gera outputs/eda_relatorio.md e figuras de apoio.
"""
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config
from .config import COLS

config.configurar_matplotlib_pt_br()


def calcular_esparsidade_por_sku(df: pd.DataFrame, treino_fim: str = config.TREINO_FIM) -> pd.DataFrame:
    """Proporcao de dias sem venda de cada SKU no treino, com o total de dias.

    E o numero que caracteriza a amostra como demanda intermitente, e depende
    da truncagem pre-lancamento: sem ela, a esparsidade sai superestimada
    pelos zeros de antes do produto existir."""
    treino = df[df[COLS.data] <= treino_fim]
    esparsidade = (
        treino.groupby(COLS.sku)[COLS.qtd]
        .agg(pct_zeros=lambda s: float((s == 0).mean()), total_dias="size")
        .reset_index()
    )
    return esparsidade


def gerar_figura_esparsidade(esparsidade: pd.DataFrame, caminho_fig):
    """Histograma da esparsidade por SKU, com a marca de 95% destacada."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(esparsidade["pct_zeros"] * 100, bins=30, color="#4C72B0", edgecolor="white")
    ax.set_xlabel("% de dias sem venda (esparsidade) no treino")
    ax.set_ylabel("Número de SKUs")
    ax.set_title("Distribuição da esparsidade diária por SKU")
    ax.axvline(95, color="crimson", linestyle="--", label="Limiar de atenção (95%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)


def gerar_figura_padrao_semanal(df: pd.DataFrame, caminho_fig):
    """Volume total por dia da semana.

    E a evidencia de que existe padrao semanal a ser capturado -- o que
    justifica `dia_semana_alvo` como feature e o perfil de desagregacao dos
    baselines."""
    d = df.copy()
    d["dia_semana"] = d[COLS.data].dt.day_name()
    ordem = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    resumo = d.groupby("dia_semana")[COLS.qtd].sum().reindex(ordem)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(resumo.index, resumo.values, color="#55A868")
    ax.set_ylabel("Quantidade total vendida")
    ax.set_title("Volume de vendas por dia da semana (base filtrada)")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)


def gerar_figura_cauda_longa(df: pd.DataFrame, treino_fim: str, caminho_fig) -> dict:
    """Gera a figura que evidencia o TEMA CENTRAL do estudo: a cauda longa de
    vendas no e-commerce (Anderson, 2004) — poucos SKUs concentram a maior
    parte das vendas, e uma cauda extensa de SKUs de baixo volume responde
    por uma fracao pequena (mas coletivamente relevante) da demanda total.
    Usa a base BRUTA (antes dos criterios de elegibilidade), restrita ao
    periodo de treino, para retratar o catalogo como um todo.

    Painel esquerdo: ranking de SKUs (do mais vendido ao menos vendido) x
    quantidade total vendida, em escala log — o formato classico da cauda
    longa. Painel direito: curva de concentracao (Pareto) — % acumulado de
    SKUs x % acumulado de vendas, com a referencia 20%/80%.

    Retorna um dict com as estatisticas-chave (para incluir no relatorio)."""
    treino = df[df[COLS.data] <= treino_fim]
    total_por_sku = treino.groupby(COLS.sku)[COLS.qtd].sum().sort_values(ascending=False)
    total_por_sku = total_por_sku[total_por_sku > 0]
    n = len(total_por_sku)
    if n == 0:
        return {"n_skus_com_venda": 0, "pct_vendas_top20pct_skus": 0.0}

    vendas = total_por_sku.to_numpy(dtype=float)
    rank = np.arange(1, n + 1)
    cum_pct_vendas = np.cumsum(vendas) / vendas.sum() * 100
    cum_pct_skus = rank / n * 100

    idx20 = max(int(np.searchsorted(cum_pct_skus, 20, side="right")), 1)
    pct_vendas_top20 = float(cum_pct_vendas[idx20 - 1])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    cores = np.where(cum_pct_skus <= 20, "#C44E52", "#4C72B0")
    ax1.bar(rank, vendas, width=1.0, color=cores, edgecolor="none")
    ax1.set_yscale("log")
    ax1.set_xlabel("Ranking do SKU (do mais vendido ao menos vendido)")
    ax1.set_ylabel("Quantidade total vendida no treino (escala log)")
    ax1.set_title("Cauda longa: vendas por SKU")
    ax1.legend(handles=[
        plt.Rectangle((0, 0), 1, 1, color="#C44E52", label="Top 20% SKUs (\"cabeça\")"),
        plt.Rectangle((0, 0), 1, 1, color="#4C72B0", label="80% SKUs restantes (\"cauda longa\")"),
    ])

    ax2.plot(cum_pct_skus, cum_pct_vendas, color="#55A868", linewidth=2)
    ax2.plot([0, 100], [0, 100], color="gray", linestyle=":", linewidth=1, label="Distribuição uniforme (referência)")
    ax2.axvline(20, color="black", linestyle="--", linewidth=1)
    ax2.axhline(pct_vendas_top20, color="black", linestyle="--", linewidth=1)
    ax2.set_xlabel("% acumulado de SKUs (ordenados do mais vendido)")
    ax2.set_ylabel("% acumulado de vendas")
    ax2.set_title("Curva de concentração (Pareto)")
    ax2.text(
        0.97, 0.05, f"{pct_vendas_top20:.0f}% das vendas\nnos 20% SKUs top",
        transform=ax2.transAxes, ha="right", va="bottom", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="gray"),
    )
    ax2.legend(loc="upper left")

    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)

    return {"n_skus_com_venda": n, "pct_vendas_top20pct_skus": pct_vendas_top20}


def relatorio_qualidade_dados(df: pd.DataFrame) -> str:
    """Bloco de qualidade da base: linhas, SKUs, duplicatas SKU x Data,
    ausentes e percentual global de zeros.

    Duplicatas e ausentes devem sair zerados apos a agregacao; se nao sairem, o
    problema esta na extracao e nao no pipeline."""
    n_linhas = len(df)
    n_skus = df[COLS.sku].nunique()
    duplicatas = df.duplicated(subset=[COLS.sku, COLS.data]).sum()
    ausentes = df[COLS.qtd].isna().sum()
    pct_zeros_global = float((df[COLS.qtd] == 0).mean())

    linhas = [
        "## Qualidade dos dados",
        "",
        f"- Linhas na base expandida: **{n_linhas:,}**",
        f"- SKUs distintos: **{n_skus:,}**",
        f"- Duplicatas SKU x Data: **{duplicatas}**",
        f"- Valores ausentes em Quantidade_Vendida: **{ausentes}**",
        f"- % geral de dias com zero (toda a base expandida): **{pct_zeros_global:.1%}**",
        "",
    ]
    return "\n".join(linhas)


def gerar_relatorio_eda(
    df_full: pd.DataFrame,
    funil_texto: str,
    outputs_dir=config.OUTPUTS_DIR,
) -> str:
    """Monta o eda_relatorio.md e as figuras: funil da amostra,
    qualidade dos dados, cauda longa, esparsidade e padrao semanal.

    Devolve o texto gerado."""
    esparsidade = calcular_esparsidade_por_sku(df_full)
    fig_esparsidade = outputs_dir / "fig_esparsidade.png"
    fig_semanal = outputs_dir / "fig_padrao_semanal.png"
    fig_cauda_longa = outputs_dir / "fig_cauda_longa.png"
    gerar_figura_esparsidade(esparsidade, fig_esparsidade)
    gerar_figura_padrao_semanal(df_full, fig_semanal)
    stats_cauda_longa = gerar_figura_cauda_longa(df_full, config.TREINO_FIM, fig_cauda_longa)

    n_alta_esparsidade = int((esparsidade["pct_zeros"] > 0.95).sum())
    n_skus_pos_truncagem = df_full[COLS.sku].nunique()
    n_skus_sem_historico_treino = n_skus_pos_truncagem - len(esparsidade)

    partes = [
        "# Relatorio de EDA (Secao 4.1.3)\n",
        funil_texto,
        relatorio_qualidade_dados(df_full),
        "## Cauda longa (tema central do TCC)\n",
        f"- SKUs com pelo menos 1 venda no treino: **{stats_cauda_longa['n_skus_com_venda']:,}**",
        f"- Os 20% SKUs mais vendidos concentram **{stats_cauda_longa['pct_vendas_top20pct_skus']:.1f}%** "
        "das vendas totais no treino — o restante (a cauda longa propriamente dita) responde pelos "
        f"{100 - stats_cauda_longa['pct_vendas_top20pct_skus']:.1f}% remanescentes.",
        f"- Figura: `{fig_cauda_longa.name}`\n",
        "## Esparsidade\n",
        f"- SKUs pos-truncagem com ao menos 1 dia de historico no treino: **{len(esparsidade):,}** de {n_skus_pos_truncagem:,} "
        f"({n_skus_sem_historico_treino} SKU(s) cadastrado(s)/lancado(s) somente no periodo de teste, sem linhas no treino, "
        "e por isso fora desta tabela de esparsidade).",
        f"- SKUs com esparsidade > 95% no treino: **{n_alta_esparsidade}** de {len(esparsidade):,}",
        f"- Figura: `{fig_esparsidade.name}`\n",
        "## Padroes temporais\n",
        f"- Figura: `{fig_semanal.name}`\n",
        "## Demanda censurada\n",
        "- Dados de posicao de estoque nao disponiveis nesta extracao; ",
        "os periodos com zero sao tratados como ausencia de demanda observada ",
        "(limitacao documentada na Secao 4.3.2 do TCC).\n",
    ]
    texto = "\n".join(partes)
    (outputs_dir / "eda_relatorio.md").write_text(texto, encoding="utf-8")
    return texto


In [ ]:
%%writefile /content/src/sazonalidade.py
"""
Fase EDA — Sazonalidade e identificacao de datas de pico de vendas, geral e
por campanha.

Objetivo pratico: ajudar a olho a identificar dias/periodos do ano com
volume anormalmente alto (candidatos a datas de campanhas promocionais —
Black Friday, Dia das Maes, Natal, campanhas de parceiros especificas
etc.), tanto no agregado geral quanto quebrado por campanha. Isso apoia a
inspecao de outliers antes de qualquer exclusao e a definicao das variaveis
de calendario/promocao.

IMPORTANTE: estas funcoes operam sobre os dados BRUTOS (df_raw, grao SKU x
Campanha x Dia, ANTES de `preprocess.agregar_por_sku`), pois a agregacao
por SKU descarta a coluna de campanha. No dataset sintetico (sem colunas
de campanha) os graficos por campanha sao pulados automaticamente.
"""
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config
from .config import COLS

config.configurar_matplotlib_pt_br()


def calcular_volume_diario(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Volume total vendido por dia (soma de todos os SKUs e campanhas)."""
    volume = df_raw.groupby(COLS.data)[COLS.qtd].sum().rename("volume").reset_index()
    return volume.sort_values(COLS.data).reset_index(drop=True)


def identificar_top_dias(volume_diario: pd.DataFrame, n: int = 15) -> pd.DataFrame:
    """Retorna os N dias de maior volume, com data, dia da semana e quanto
    o volume do dia excede a media do periodo (em desvios-padrao e em %)."""
    media = volume_diario["volume"].mean()
    dp = volume_diario["volume"].std()

    top = volume_diario.sort_values("volume", ascending=False).head(n).copy()
    top["dia_semana"] = top[COLS.data].dt.day_name()
    top["pct_acima_media"] = (top["volume"] / media - 1) * 100 if media else np.nan
    top["desvios_padrao"] = (top["volume"] - media) / dp if dp else np.nan
    return top[[COLS.data, "dia_semana", "volume", "pct_acima_media", "desvios_padrao"]]


def gerar_figura_volume_diario(volume_diario: pd.DataFrame, caminho_fig, n_destaque: int = 10):
    """Serie temporal do volume diario total (todo o periodo), com media
    movel de 7 dias e os N dias de maior pico anotados com a data."""
    v = volume_diario.copy()
    v["media_movel_7d"] = v["volume"].rolling(7, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(v[COLS.data], v["volume"], color="#4C72B0", linewidth=0.6, alpha=0.5, label="Volume diário")
    ax.plot(v[COLS.data], v["media_movel_7d"], color="#C44E52", linewidth=1.6, label="Média móvel (7 dias)")

    picos = v.sort_values("volume", ascending=False).head(n_destaque)
    ax.scatter(picos[COLS.data], picos["volume"], color="black", zorder=5, s=18)
    for _, row in picos.iterrows():
        ax.annotate(
            row[COLS.data].strftime("%d/%m/%y"),
            (row[COLS.data], row["volume"]),
            textcoords="offset points", xytext=(0, 8), ha="center", fontsize=7, rotation=45,
        )

    ax.set_xlabel("Data")
    ax.set_ylabel("Quantidade total vendida no dia")
    ax.set_title("Volume de vendas diário — picos anotados (candidatos a datas de campanha)")
    ax.legend(loc="upper left")
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)


def gerar_figura_sazonalidade_dia_ano(volume_diario: pd.DataFrame, caminho_fig):
    """Sobrepoe o volume diario de cada ano por dia-do-ano (1-366), para
    revelar picos que se REPETEM no calendario entre anos (fortes
    candidatos a datas fixas de campanha/sazonalidade, em vez de eventos
    pontuais de um unico ano)."""
    v = volume_diario.copy()
    v["ano"] = v[COLS.data].dt.year
    v["dia_do_ano"] = v[COLS.data].dt.dayofyear
    v["media_movel_3d"] = v.groupby("ano")["volume"].transform(lambda s: s.rolling(3, min_periods=1).mean())

    fig, ax = plt.subplots(figsize=(13, 5))
    cores = plt.cm.viridis(np.linspace(0, 0.85, v["ano"].nunique()))
    for cor, (ano, grupo) in zip(cores, v.groupby("ano")):
        ax.plot(grupo["dia_do_ano"], grupo["media_movel_3d"], label=str(ano), color=cor, linewidth=1.3)

    meses_inicio = pd.date_range("2024-01-01", "2024-12-31", freq="MS").dayofyear
    meses_nomes = pd.date_range("2024-01-01", "2024-12-31", freq="MS").strftime("%b")
    ax.set_xticks(meses_inicio)
    ax.set_xticklabels(meses_nomes)
    ax.set_xlabel("Dia do ano (meses como referência)")
    ax.set_ylabel("Quantidade vendida (média móvel 3 dias)")
    ax.set_title("Sazonalidade: volume por dia-do-ano, sobrepondo os anos")
    ax.legend(title="Ano")
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)


def calcular_ranking_campanhas(df_raw: pd.DataFrame) -> pd.DataFrame | None:
    """Volume total e participacao de cada campanha. None quando a base nao tem
    coluna de campanha (o caso do dataset sintetico)."""
    if COLS.campanha not in df_raw.columns:
        return None
    ranking = (
        df_raw.groupby(COLS.campanha)[COLS.qtd].sum()
        .sort_values(ascending=False)
        .rename("volume_total")
        .reset_index()
    )
    ranking["pct_do_total"] = ranking["volume_total"] / ranking["volume_total"].sum() * 100
    return ranking


def gerar_figura_volume_por_campanha(df_raw: pd.DataFrame, caminho_fig, top_n: int = 8):
    """Volume MENSAL das top-N campanhas (por volume total), uma linha por
    campanha — evidencia em que epoca do ano cada campanha concentra
    vendas. Retorna None (sem gerar figura) se a coluna de campanha nao
    existir na entrada (ex.: dataset sintetico)."""
    if COLS.campanha not in df_raw.columns:
        return None

    ranking = calcular_ranking_campanhas(df_raw)
    top_campanhas = ranking.head(top_n)[COLS.campanha].tolist()

    df_top = df_raw[df_raw[COLS.campanha].isin(top_campanhas)].copy()
    df_top["mes"] = df_top[COLS.data].dt.to_period("M").dt.to_timestamp()
    mensal = df_top.groupby(["mes", COLS.campanha])[COLS.qtd].sum().reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))
    for campanha in top_campanhas:
        serie = mensal[mensal[COLS.campanha] == campanha]
        ax.plot(serie["mes"], serie[COLS.qtd], marker="o", markersize=3, linewidth=1.2, label=campanha)

    ax.set_xlabel("Mês")
    ax.set_ylabel("Quantidade vendida no mês")
    ax.set_title(f"Volume mensal das {top_n} campanhas de maior volume")
    ax.legend(title="Campanha", fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)
    return True


def gerar_relatorio_sazonalidade(df_raw: pd.DataFrame, outputs_dir=config.OUTPUTS_DIR) -> str:
    """Orquestra os graficos/analises de sazonalidade e escreve
    `sazonalidade_vendas.md` em outputs_dir. Pensado para rodar logo apos
    carregar os dados brutos (Fase 0), ANTES de `preprocess.agregar_por_sku`
    (que descarta a granularidade de campanha)."""
    volume_diario = calcular_volume_diario(df_raw)

    fig_diario = outputs_dir / "fig_volume_diario.png"
    fig_sazonal = outputs_dir / "fig_sazonalidade_dia_ano.png"
    gerar_figura_volume_diario(volume_diario, fig_diario)
    gerar_figura_sazonalidade_dia_ano(volume_diario, fig_sazonal)

    top_dias = identificar_top_dias(volume_diario, n=15)

    partes = [
        "# Sazonalidade e datas de pico de vendas\n",
        "Analise de apoio (Secao 4.1.3) para identificar visualmente dias/periodos "
        "do ano com volume anormalmente alto — candidatos a datas de campanhas "
        "promocionais, uteis para cruzar com o calendario de campanhas de "
        "parceiros (pendencia registrada no README/plano).\n",
        "## Volume diario geral\n",
        f"- Figura: `{fig_diario.name}` — serie completa com media movel de 7 dias "
        "e os 10 maiores picos anotados.\n",
        "## Top 15 dias de maior volume (geral, todas as campanhas)\n",
        top_dias.to_string(index=False, formatters={
            COLS.data: lambda d: d.strftime("%Y-%m-%d"),
            "volume": lambda v: f"{v:,.0f}",
            "pct_acima_media": lambda v: f"{v:+.0f}%",
            "desvios_padrao": lambda v: f"{v:.1f}",
        }),
        "\n",
        "## Sazonalidade por dia-do-ano (sobrepondo os anos)\n",
        f"- Figura: `{fig_sazonal.name}` — picos que se repetem no MESMO dia-do-ano "
        "em anos diferentes sao fortes candidatos a datas fixas de campanha "
        "(ex.: Black Friday, Dia das Maes, Natal), em vez de eventos pontuais.\n",
    ]

    ranking_campanhas = calcular_ranking_campanhas(df_raw)
    if ranking_campanhas is not None:
        fig_campanha = outputs_dir / "fig_volume_por_campanha.png"
        gerar_figura_volume_por_campanha(df_raw, fig_campanha, top_n=8)
        top_campanhas_texto = ranking_campanhas.head(15).to_string(
            index=False, formatters={
                "volume_total": lambda v: f"{v:,.0f}",
                "pct_do_total": lambda v: f"{v:.1f}%",
            }
        )
        partes += [
            "## Volume por campanha\n",
            f"- Figura: `{fig_campanha.name}` — volume mensal das 8 campanhas de maior volume.\n",
            "### Top 15 campanhas por volume total\n",
            top_campanhas_texto,
            "\n",
        ]
    else:
        partes += [
            "## Volume por campanha\n",
            "- Coluna de campanha nao disponivel nesta entrada (dataset sintetico) "
            "— grafico por campanha pulado. Com a extracao real (coluna `Campanha`), "
            "esta secao traz o volume mensal das principais campanhas.\n",
        ]

    texto = "\n".join(partes)
    (outputs_dir / "sazonalidade_vendas.md").write_text(texto, encoding="utf-8")
    return texto


In [ ]:
%%writefile /content/src/adi_cv2.py
"""
Fase 4 — Classificacao ADI / CV2.

Implementa a tipologia de Syntetos, Boylan e Croston (2005):
  ADI = periodos totais / periodos com venda    (intervalo medio entre demandas)
  CV2 = (desvio / media das vendas positivas)^2 (variabilidade do tamanho)

Cruzando os dois com os cortes ADI = 1,32 e CV2 = 0,49 saem quatro quadrantes:
Suave, Errática, Intermitente e Lumpy. O estudo retem Intermitente e Lumpy --
e o recorte do objeto de pesquisa, definido em config.QUADRANTES_FOCO.

Tudo e calculado SOMENTE sobre o periodo de treino: a classificacao seleciona
quais SKUs entram nos experimentos, entao usar o teste aqui seria vazamento.

A granularidade vem de config.GRANULARIDADE_CLASSIFICACAO e precisa coincidir
com a grade semanal de baselines.py -- se os dois modulos usarem grades
diferentes, a classificacao descreve uma serie que ninguem modelou.
"""
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config
from .config import COLS

config.configurar_matplotlib_pt_br()


def agregar_periodo(df: pd.DataFrame, freq: str | None = None,
                    treino_fim: str = config.TREINO_FIM) -> pd.DataFrame:
    """Agrega a serie diaria na granularidade `freq`, restrita ao treino.

    freq="D" devolve a propria serie diaria, sem reagregacao."""
    freq = freq or config.GRANULARIDADE_CLASSIFICACAO
    df_treino = df[df[COLS.data] <= treino_fim].copy()
    if str(freq).upper() == "D":
        return df_treino.groupby([COLS.sku, COLS.data], as_index=False)[COLS.qtd].sum()
    return (
        df_treino.groupby([COLS.sku, pd.Grouper(key=COLS.data, freq=freq)])[COLS.qtd]
        .sum()
        .reset_index()
    )


# Alias mantido para codigo que chama pelo nome antigo.
def agregar_semanal(df: pd.DataFrame, treino_fim: str = config.TREINO_FIM) -> pd.DataFrame:
    """Atalho para agregar_periodo na granularidade configurada."""
    return agregar_periodo(df, freq=config.GRANULARIDADE_CLASSIFICACAO, treino_fim=treino_fim)


def calcular_adi_cv2(df_agregado: pd.DataFrame) -> pd.DataFrame:
    """Calcula ADI, CV2 e o quadrante de cada SKU.

    Notas de leitura que importam para interpretar os resultados:
      - o ADI conta TODOS os periodos da serie no denominador, entao zeros no
        inicio da serie elevam o ADI. E por isso que a truncagem pre-lancamento
        afeta diretamente a classificacao;
      - SKUs sem nenhuma venda no treino sao descartados: ADI seria divisao por
        zero e nao ha o que classificar;
      - com uma unica venda o CV2 e definido como 0, por falta de dispersao
        estimavel. O SKU cai em Intermitente por construcao."""
    registros = []
    for sku, grupo in df_agregado.groupby(COLS.sku):
        total_periodos = len(grupo)
        positivas = grupo.loc[grupo[COLS.qtd] > 0, COLS.qtd]
        periodos_com_venda = len(positivas)

        if periodos_com_venda == 0:
            continue

        adi = total_periodos / periodos_com_venda

        if periodos_com_venda > 1:
            media = positivas.mean()
            std = positivas.std(ddof=1)
            cv2 = (std / media) ** 2 if media > 0 else 0.0
        else:
            cv2 = 0.0

        if adi > config.ADI_LIMIAR and cv2 <= config.CV2_LIMIAR:
            quadrante = "Intermitente"
        elif adi > config.ADI_LIMIAR and cv2 > config.CV2_LIMIAR:
            quadrante = "Lumpy"
        elif adi <= config.ADI_LIMIAR and cv2 <= config.CV2_LIMIAR:
            quadrante = "Suave"
        else:
            quadrante = "Errática"

        registros.append({
            COLS.sku: sku,
            "ADI": adi,
            "CV2": cv2,
            "total_semanas": total_periodos,       # nome mantido p/ compatibilidade
            "semanas_com_venda": periodos_com_venda,
            "quadrante": quadrante,
        })

    return pd.DataFrame(registros)


def filtrar_quadrantes_foco(df_classes: pd.DataFrame) -> pd.DataFrame:
    """Mantem so os quadrantes do objeto de estudo (Intermitente e Lumpy)."""
    return df_classes[df_classes["quadrante"].isin(config.QUADRANTES_FOCO)].copy()


def gerar_figura_dispersao(df_classes: pd.DataFrame, caminho_fig, titulo_extra: str = ""):
    """Dispersao ADI x CV2 com os cortes marcados, colorida por quadrante.

    Mostra onde a amostra cai no plano da tipologia."""
    cores = {"Suave": "#4C72B0", "Errática": "#DD8452", "Intermitente": "#55A868", "Lumpy": "#C44E52"}
    fig, ax = plt.subplots(figsize=(7, 6))
    for quad, cor in cores.items():
        sub = df_classes[df_classes["quadrante"] == quad]
        ax.scatter(sub["ADI"], sub["CV2"], s=12, alpha=0.6, label=quad, color=cor)

    ax.axvline(config.ADI_LIMIAR, color="black", linestyle="--", linewidth=1)
    ax.axhline(config.CV2_LIMIAR, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("ADI (Average Demand Interval)")
    ax.set_ylabel("CV² (coeficiente de variação ao quadrado)")
    ax.set_title(f"Classificação ADI × CV²{titulo_extra}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)


def resumo_quadrantes_texto(df_classes: pd.DataFrame, granularidade: str = "") -> str:
    """Tabela em markdown com a contagem e o percentual de cada quadrante, e
    quantos SKUs seguem para os experimentos."""
    contagem = df_classes["quadrante"].value_counts()
    total = len(df_classes)
    sufixo = f" — granularidade {granularidade}" if granularidade else ""
    linhas = [f"## Classificacao ADI/CV2 (Secao 4.1.4){sufixo}\n"]
    for quad in ["Suave", "Errática", "Intermitente", "Lumpy"]:
        n = int(contagem.get(quad, 0))
        pct = n / total * 100 if total else 0
        linhas.append(f"- {quad}: **{n}** SKUs ({pct:.1f}%)")
    n_foco = int(df_classes["quadrante"].isin(config.QUADRANTES_FOCO).sum())
    linhas.append(f"\n**SKUs retidos para os experimentos (Intermitente + Lumpy): {n_foco} de {total}**\n")
    return "\n".join(linhas)


def executar_classificacao(df_filtrado: pd.DataFrame, outputs_dir=config.OUTPUTS_DIR):
    """Classificacao PRINCIPAL: a que define quais SKUs entram nos experimentos.

    Gera a figura, o CSV e o markdown, e devolve
    (df_modelagem, df_classes, texto) -- onde df_modelagem ja vem restrito aos
    quadrantes de foco."""
    freq = config.GRANULARIDADE_CLASSIFICACAO
    df_agg = agregar_periodo(df_filtrado, freq=freq)
    df_classes = calcular_adi_cv2(df_agg)

    fig_path = outputs_dir / "fig_dispersao_adi_cv2.png"
    gerar_figura_dispersao(df_classes, fig_path, titulo_extra=f" (granularidade {freq}, treino)")

    df_classes.to_csv(outputs_dir / "classificacao_adi_cv2.csv", index=False)

    texto = resumo_quadrantes_texto(df_classes, granularidade=freq) + f"\n- Figura: `{fig_path.name}`\n"
    (outputs_dir / "classificacao_adi_cv2.md").write_text(texto, encoding="utf-8")

    df_foco = filtrar_quadrantes_foco(df_classes)
    skus_foco = df_foco[COLS.sku]
    df_modelagem = df_filtrado[df_filtrado[COLS.sku].isin(skus_foco)].copy()

    return df_modelagem, df_classes, texto


def executar_classificacao_robustez(df_filtrado: pd.DataFrame, outputs_dir,
                                    freq: str = "D") -> tuple[pd.DataFrame, str]:
    """Classificacao em outra granularidade, so para reportar.

    NAO altera a selecao de SKUs -- essa continua vindo de
    executar_classificacao.

    O ponto: os cortes 1,32 e 0,49 pressupoem classificar na mesma granularidade
    em que se preve. Classificar em semanas enquanto se modela em dias comprime
    o ADI. Rodar no diario tende a jogar quase tudo em Lumpy, o que mostra que a
    classificacao perde poder discriminante nessa granularidade -- e um achado a
    reportar, e o argumento para manter o semanal como principal.

    Devolve (df_classes, texto)."""
    df_agg = agregar_periodo(df_filtrado, freq=freq)
    df_classes = calcular_adi_cv2(df_agg)
    if df_classes.empty:
        return df_classes, ""

    sufixo = "diaria" if str(freq).upper() == "D" else str(freq).lower().replace("-", "")
    df_classes.to_csv(outputs_dir / f"classificacao_adi_cv2_{sufixo}.csv", index=False)
    fig_path = outputs_dir / f"fig_dispersao_adi_cv2_{sufixo}.png"
    gerar_figura_dispersao(df_classes, fig_path, titulo_extra=f" (granularidade {freq}, treino)")

    texto = (
        f"## Robustez da classificacao ADI/CV2 — granularidade {freq} (Item 19)\n\n"
        "Os cortes ADI = 1,32 e CV² = 0,49 (Syntetos, Boylan e Croston, 2005) foram derivados\n"
        "para o periodo de previsao. A classificacao principal deste trabalho usa granularidade\n"
        f"{config.GRANULARIDADE_CLASSIFICACAO}, enquanto a modelagem e diaria — esta tabela\n"
        "verifica o efeito dessa escolha.\n\n"
        + resumo_quadrantes_texto(df_classes, granularidade=str(freq))
        + f"\n- Figura: `{fig_path.name}`\n"
    )
    (outputs_dir / f"classificacao_adi_cv2_{sufixo}.md").write_text(texto, encoding="utf-8")
    return df_classes, texto


In [ ]:
%%writefile /content/src/features.py
"""
Fase 5 — Engenharia de Variaveis.

ESTRATEGIA DE PREVISAO: direta multi-horizonte, com o horizonte como feature.

Para cada DATA DE ORIGEM `o` (a virada de uma janela walk-forward, isto e, o
ultimo dia de treino disponivel), as features de historico (lags, rolling,
idade, categoria, SKU) sao calculadas UMA vez, usando apenas dados ate `o`.
Em seguida, para cada horizonte h = 1..7, gera-se uma linha com:
  - as mesmas features de historico, fixas para todos os h daquela origem;
  - o horizonte h como feature;
  - as exogenas de calendario na DATA-ALVO (o + h) -- legitimo, porque datas de
    calendario sao conhecidas com antecedencia e nao derivam do historico;
  - o alvo y = Quantidade_Vendida em (o + h).

O ganho sobre a previsao recursiva e nao precisar realimentar valores previstos
(nem, por descuido, valores reais futuros) como lag, mantendo o horizonte de 7
dias e o walk-forward expandido.

FEATURES DE CALENDARIO DA DATA-ALVO: `dia_semana_alvo`, `mes_alvo` e
`semana_ano_alvo` sao SEMPRE calculadas por montar_tabela_horizontes, mas so
entram no conjunto do modelo quando as flags correspondentes estao ligadas.
Sem `dia_semana_alvo` ha assimetria informacional: os baselines
recebem o perfil semanal via proporcao_dia_semana e o ML nao.

ORIGEM DA SERIE: `idade_produto_dias` e o inicio da serie usam a MESMA regra --
min(data de cadastro, 1a venda observada). Ver `_origem_da_serie` aqui e
`preprocess.truncar_antes_do_lancamento`.
"""
from __future__ import annotations

from datetime import timedelta

import numpy as np
import pandas as pd
import holidays

from . import config
from .config import COLS


# ---------------------------------------------------------------------------
# Features temporais baseadas no historico (lags e rolling)
# ---------------------------------------------------------------------------
def adicionar_features_temporais(df: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta lags e estatisticas moveis da propria serie do SKU.

    As rolling sao calculadas sobre a serie deslocada em 1 dia, entao a janela
    termina em d-1 e nunca inclui o dia que esta sendo previsto."""
    df = df.sort_values([COLS.sku, COLS.data]).reset_index(drop=True)
    grupo = df.groupby(COLS.sku)[COLS.qtd]

    for lag in config.lags_dias():
        df[f"lag_{lag}"] = grupo.shift(lag)

    base_deslocada = df.groupby(COLS.sku)[COLS.qtd].shift(1)
    for janela in config.janelas_rolling():
        df[f"roll_mean_{janela}"] = (
            base_deslocada.groupby(df[COLS.sku]).rolling(window=janela, min_periods=1)
            .mean().reset_index(level=0, drop=True)
        )
        df[f"roll_std_{janela}"] = (
            base_deslocada.groupby(df[COLS.sku]).rolling(window=janela, min_periods=1)
            .std().reset_index(level=0, drop=True).fillna(0)
        )
    return df


# ---------------------------------------------------------------------------
# Features especificas de demanda intermitente
# ---------------------------------------------------------------------------
COLUNAS_INTERMITENCIA = [
    "dias_desde_ultima_venda",
    "n_dias_venda_28", "n_dias_venda_91",
    "taxa_zeros_28", "taxa_zeros_91",
    "tam_medio_demanda",
]

# Calendario da data-alvo que entra junto com FEATURES_INTERMITENCIA. Mesmo
# status de dia_semana_alvo: conhecido com antecedencia, sem look-ahead.
COLUNAS_CALENDARIO_ALVO_EXTRA = ["mes_alvo", "semana_ano_alvo"]


def adicionar_features_intermitencia(df: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta as features de demanda intermitente.

      dias_desde_ultima_venda : 1 = vendeu ontem, 2 = anteontem, ...
                                E o intervalo entre demandas de Croston, e
                                costuma ser o preditor mais forte aqui.
      n_dias_venda_{28,91}    : quantos dos ultimos j dias tiveram venda
      taxa_zeros_{28,91}      : 1 - n_dias_venda_j / j
      tam_medio_demanda       : media das demandas NAO-NULAS dos ultimos 91
                                dias. E o tamanho da demanda de Croston.

    Juntas, sao a decomposicao de Croston oferecida ao ML -- o que torna a
    comparacao com SBA/TSB mais justa, nao menos.
    Todas partem da serie deslocada em 1 dia, portanto sem look-ahead."""
    df = df.sort_values([COLS.sku, COLS.data]).reset_index(drop=True)
    base = df.groupby(COLS.sku)[COLS.qtd].shift(1)   # somente o passado

    # --- dias desde a ultima venda -----------------------------------------
    # Um "bloco" muda toda vez que houve venda; dentro do bloco, a posicao
    # (cumcount) e o numero de dias decorridos desde aquela venda.
    teve_venda = (base > 0).astype(int)
    bloco = teve_venda.groupby(df[COLS.sku]).cumsum()
    df["dias_desde_ultima_venda"] = df.groupby([df[COLS.sku], bloco]).cumcount() + 1

    # --- frequencia de demanda e taxa de zeros -----------------------------
    # taxa_zeros_j e funcao exata de n_dias_venda_j; as duas so coexistem aqui
    # para que a matriz de correlacao exiba a identidade. O pipeline remove a
    # redundante do conjunto do modelo (config.REMOVER_FEATURES_REDUNDANTES).
    tv = teve_venda.astype(float)
    for j in (28, 91):
        df[f"n_dias_venda_{j}"] = (
            tv.groupby(df[COLS.sku]).rolling(window=j, min_periods=1)
            .sum().reset_index(level=0, drop=True)
        )
        df[f"taxa_zeros_{j}"] = 1.0 - df[f"n_dias_venda_{j}"] / float(j)

    # --- tamanho medio da demanda quando ela ocorre ------------------------
    positivos = base.where(base > 0)
    df["tam_medio_demanda"] = (
        positivos.groupby(df[COLS.sku]).rolling(window=91, min_periods=1)
        .mean().reset_index(level=0, drop=True).fillna(0.0)
    )
    return df


# ---------------------------------------------------------------------------
# Idade do produto
# ---------------------------------------------------------------------------
def _origem_da_serie(cadastro: pd.Series | None, primeira_venda: pd.Series) -> pd.Series:
    """Data de nascimento adotada para o produto: min(cadastro, 1a venda).

    Tolera a ausencia de qualquer um dos dois. E a mesma expressao de
    preprocess.truncar_antes_do_lancamento, de proposito: "quando a serie
    comeca" e "que idade o produto tem naquele dia" sao a mesma decisao e
    precisam da mesma resposta.

    O minimo existe porque a data de cadastro do ERP nem sempre e a data de
    nascimento do produto -- em produto migrado ou recadastrado ela aparece
    depois de vendas que de fato ocorreram, as vezes depois de toda a janela do
    estudo. Nesses casos a 1a venda observada e a melhor evidencia disponivel."""
    if cadastro is None:
        return primeira_venda
    return cadastro.fillna(primeira_venda).where(
        primeira_venda.isna() | (cadastro.fillna(primeira_venda) <= primeira_venda),
        primeira_venda,
    )


def adicionar_idade_produto(df: pd.DataFrame, corte=None) -> pd.DataFrame:
    """Calcula idade_produto_dias = data - min(cadastro, 1a venda observada).

    `corte`: data-limite para a 1a venda, de modo que ela nao enxergue
    o periodo de teste. O pipeline passa config.TREINO_FIM. A data de cadastro
    dispensa corte por ser atributo de produto, nao derivado de venda.

    O resultado e grampeado em 0, entao uma origem mal escolhida nao produz
    idade negativa -- produz uma constante zero, que passa despercebida. Por
    isso a origem e o minimo e nao o cadastro: ver `_origem_da_serie` e o
    diagnostico abaixo, que conta quantos SKUs caem em cada caso."""
    vendas_positivas = df[df[COLS.qtd] > 0]
    if corte is not None:
        vendas_positivas = vendas_positivas[vendas_positivas[COLS.data] <= pd.Timestamp(corte)]

    primeira_venda = vendas_positivas.groupby(COLS.sku)[COLS.data].min()
    df = df.merge(primeira_venda.rename("_primeira_venda"), on=COLS.sku, how="left")

    cadastro = df[COLS.data_cadastro] if COLS.data_cadastro in df.columns else None
    origem = _origem_da_serie(cadastro, df["_primeira_venda"])

    df[COLS.idade] = (df[COLS.data] - origem).dt.days.clip(lower=0)
    df[COLS.idade] = df[COLS.idade].fillna(0).astype(int)
    return df.drop(columns=["_primeira_venda"])


def diagnosticar_origem_idade(df: pd.DataFrame, corte=None) -> str:
    """Conta de onde veio a origem da idade de cada SKU, em tres grupos:
    cadastro usado, cadastro descartado por ser posterior a 1a venda, e cadastro
    ausente. Os dois ultimos caem no proxy da 1a venda.

    Serve para dimensionar a qualidade do campo de cadastro na base -- uma
    proporcao alta no segundo grupo e um achado sobre o ERP, e vale citar na
    seccao de tratamento dos dados."""
    if COLS.data_cadastro not in df.columns:
        return "[Item 14] data_cadastro_produto ausente — 100% dos SKUs usam o proxy (1a venda)."

    vendas_positivas = df[df[COLS.qtd] > 0]
    if corte is not None:
        vendas_positivas = vendas_positivas[vendas_positivas[COLS.data] <= pd.Timestamp(corte)]
    primeira_venda = vendas_positivas.groupby(COLS.sku)[COLS.data].min()

    cadastro = df.groupby(COLS.sku)[COLS.data_cadastro].first()
    pv = primeira_venda.reindex(cadastro.index)
    n_total = len(cadastro)
    n_ausente = int(cadastro.isna().sum())
    posterior = cadastro.notna() & pv.notna() & (cadastro > pv)
    n_posterior = int(posterior.sum())
    n_usado = n_total - n_ausente - n_posterior
    pct = lambda n: (n / n_total * 100 if n_total else 0.0)

    linhas = [
        f"[Items 14/30] Origem da idade do produto = min(cadastro, 1a venda no treino):",
        f"  - {n_usado}/{n_total} SKUs ({pct(n_usado):.1f}%) usam a data de cadastro;",
        f"  - {n_posterior}/{n_total} ({pct(n_posterior):.1f}%) tem cadastro POSTERIOR a 1a "
        f"venda e caem no proxy da 1a venda;",
        f"  - {n_ausente}/{n_total} ({pct(n_ausente):.1f}%) sem cadastro, tambem no proxy.",
    ]
    if n_posterior:
        linhas.append(
            f"  Nota: ate a v10 esses {n_posterior} SKUs ficavam com idade_produto_dias "
            f"constante em zero (cadastro no futuro + clip). Ver Item 30.")
    return "\n".join(linhas)


# ---------------------------------------------------------------------------
# Calendario brasileiro e Black Friday
# ---------------------------------------------------------------------------
def _segundo_domingo(ano: int, mes: int) -> pd.Timestamp:
    """Segundo domingo do mes (Dia das Maes em maio, Dia dos Pais em agosto)."""
    primeiro_dia = pd.Timestamp(ano, mes, 1)
    primeiro_domingo = primeiro_dia + pd.Timedelta(days=(6 - primeiro_dia.weekday()) % 7)
    return primeiro_domingo + pd.Timedelta(weeks=1)


def _ultima_sexta(ano: int, mes: int) -> pd.Timestamp:
    """Ultima sexta-feira do mes (em novembro, a Black Friday)."""
    ultimo_dia = pd.Timestamp(ano, mes, 1) + pd.offsets.MonthEnd(1)
    offset = (ultimo_dia.weekday() - 4) % 7
    return ultimo_dia - pd.Timedelta(days=offset)


def construir_datas_especiais_br(anos: list[int]) -> set[pd.Timestamp]:
    """Feriados nacionais (pacote `holidays`) mais as datas comerciais de varejo:
    Dia das Maes, Dia dos Pais, Dia dos Namorados, Dia das Criancas, vespera de
    Natal e vespera de Ano Novo."""
    datas = set()
    br_holidays = holidays.Brazil(years=anos)
    for d in br_holidays.keys():
        datas.add(pd.Timestamp(d))

    for ano in anos:
        datas.add(_segundo_domingo(ano, 5))
        datas.add(_segundo_domingo(ano, 8))
        datas.add(pd.Timestamp(ano, 6, 12))
        datas.add(pd.Timestamp(ano, 10, 12))
        datas.add(pd.Timestamp(ano, 12, 24))
        datas.add(pd.Timestamp(ano, 12, 31))

    return datas


def construir_semana_black_friday(anos: list[int]) -> set[pd.Timestamp]:
    """Janela de 10 dias da Black Friday: da segunda-feira da semana da ultima
    sexta de novembro em diante, cobrindo tambem a Cyber Monday."""
    dias = set()
    for ano in anos:
        sexta = _ultima_sexta(ano, 11)
        inicio_semana = sexta - pd.Timedelta(days=sexta.weekday())
        for i in range(10):
            dias.add(inicio_semana + pd.Timedelta(days=i))
    return dias


def adicionar_calendario(df: pd.DataFrame) -> pd.DataFrame:
    """Marca as duas flags de calendario simples usadas pelo Cenario 1:
    data especial brasileira e semana da Black Friday."""
    anos = sorted(df[COLS.data].dt.year.unique().tolist())
    datas_especiais = construir_datas_especiais_br(anos)
    dias_black_friday = construir_semana_black_friday(anos)

    df[COLS.especial_br] = df[COLS.data].isin(datas_especiais).astype(int)
    df[COLS.black_friday] = df[COLS.data].isin(dias_black_friday).astype(int)
    return df


# ---------------------------------------------------------------------------
# Calendario externo (Cenario 2 do Experimento 2)
# ---------------------------------------------------------------------------
def adicionar_calendario_exogeno(
    df: pd.DataFrame,
    cal: pd.DataFrame,
) -> pd.DataFrame:
    """Junta o calendario rico (CSV externo) a serie diaria, por data."""
    from .exogenas import merge_calendario_em_df
    return merge_calendario_em_df(df, cal, col_data=COLS.data)


# ---------------------------------------------------------------------------
# Encoding categorico sem vazamento — ajustado por janela de treino
# ---------------------------------------------------------------------------
class LabelEncoderJanela:
    """Label encoding ajustado somente sobre o treino disponivel ate a data de
    corte de cada janela walk-forward.

    Categorias nao vistas no treino recebem -1 ("desconhecida"), em vez de
    quebrar ou de ganhar um codigo novo que o modelo nunca viu."""

    def __init__(self):
        self.mapa: dict = {}

    def fit(self, valores: pd.Series) -> "LabelEncoderJanela":
        categorias = sorted(valores.dropna().unique().tolist())
        self.mapa = {cat: i for i, cat in enumerate(categorias)}
        return self

    def transform(self, valores: pd.Series) -> pd.Series:
        return valores.map(self.mapa).fillna(-1).astype(int)

    def fit_transform(self, valores: pd.Series) -> pd.Series:
        return self.fit(valores).transform(valores)


COLUNAS_PERFIL_SKU = ["sku_media_hist", "sku_taxa_zeros", "sku_tam_medio"]


class PerfilSkuJanela:
    """Codifica o SKU pelo perfil historico, em vez do codigo ordinal.

    Com o LabelEncoder alfabetico a arvore so consegue perguntar
    "sku_enc <= 137,5?", o que agrupa os SKUs alfabeticamente primeiros contra o
    resto -- corte sem relacao com o padrao de demanda, que obriga a arvore a
    gastar ~log2(n_skus) splits para isolar um unico SKU. Com o perfil, um unico
    split ("sku_media_hist <= 0,05") ja separa giro baixo de giro alto.

    Como o LabelEncoderJanela, e ajustado somente sobre o treino ate o corte da
    janela. SKUs nao vistos recebem media 0, taxa de zeros 1 e tamanho medio 0,
    que e o perfil de um produto sem historico."""

    def __init__(self):
        self.perfil: pd.DataFrame | None = None

    def fit(self, df_treino_ate_corte: pd.DataFrame) -> "PerfilSkuJanela":
        g = df_treino_ate_corte.groupby(COLS.sku)[COLS.qtd]
        self.perfil = pd.DataFrame({
            "sku_media_hist": g.mean(),
            "sku_taxa_zeros": g.apply(lambda s: float((s == 0).mean())),
            "sku_tam_medio": g.apply(lambda s: float(s[s > 0].mean()) if (s > 0).any() else 0.0),
        })
        return self

    def transform(self, tabela: pd.DataFrame) -> pd.DataFrame:
        if self.perfil is None:
            raise RuntimeError("PerfilSkuJanela.transform chamado antes de fit.")
        saida = tabela.merge(self.perfil, left_on=COLS.sku, right_index=True, how="left")
        saida["sku_media_hist"] = saida["sku_media_hist"].fillna(0.0)
        saida["sku_taxa_zeros"] = saida["sku_taxa_zeros"].fillna(1.0)
        saida["sku_tam_medio"] = saida["sku_tam_medio"].fillna(0.0)
        return saida


# ---------------------------------------------------------------------------
# Pipeline completo de features "de origem" (historico)
# ---------------------------------------------------------------------------
def colunas_features_historico() -> list:
    """Features de historico ativas nesta execucao.

    E funcao, e nao constante, porque o conjunto depende de
    config.FEATURES_INTERMITENCIA, definido em tempo de execucao pela celula de
    parametros."""
    cols = (
        [f"lag_{l}" for l in config.lags_dias()]
        + [f"roll_mean_{j}" for j in config.janelas_rolling()]
        + [f"roll_std_{j}" for j in config.janelas_rolling()]
        + [COLS.idade]
    )
    if config.FEATURES_INTERMITENCIA:
        cols = cols + list(COLUNAS_INTERMITENCIA)
    return cols


def colunas_dropna_historico() -> list:
    """Colunas consideradas no dropna que remove o aquecimento de cada serie.

    So os lags produzem NaN -- as rolling usam min_periods=1 e as features de
    intermitencia sao preenchidas. Restringir o dropna aos lags evita descartar
    linhas validas por engano."""
    return [f"lag_{l}" for l in config.lags_dias()]


# Alias para codigo que importa a lista base (sem as features de intermitencia).
COLUNAS_FEATURES_HISTORICO = (
    [f"lag_{l}" for l in config.LAGS_DIAS]
    + [f"roll_mean_{j}" for j in config.JANELAS_ROLLING]
    + [f"roll_std_{j}" for j in config.JANELAS_ROLLING]
    + [COLS.idade]
)


def construir_features_base(df: pd.DataFrame, corte_atributos=None) -> pd.DataFrame:
    """Monta todas as features "de origem": historico, idade e calendario simples.

    `corte_atributos` e a data-limite da 1a venda usada na idade; o
    pipeline passa config.TREINO_FIM quando CORTE_ATRIBUTOS_TREINO=True."""
    df = adicionar_features_temporais(df)
    if config.FEATURES_INTERMITENCIA:
        df = adicionar_features_intermitencia(df)
    df = adicionar_idade_produto(df, corte=corte_atributos)
    df = adicionar_calendario(df)
    return df


def montar_tabela_horizontes(
    df_features: pd.DataFrame,
    datas_origem: list[pd.Timestamp],
    horizonte: int = config.HORIZONTE_PREVISAO_DIAS,
    cols_exogenas_alvo: list[str] | None = None,
) -> pd.DataFrame:
    """Monta a tabela longa: uma linha por (SKU, data de origem, horizonte h).

    As features de historico ficam fixas na data de origem; as exogenas sao
    capturadas na data-alvo (origem + h) e recebem o sufixo `_alvo`.

    Calcula sempre dia_semana_alvo, mes_alvo e semana_ano_alvo. Sao calendario
    puro, sem look-ahead; entram no conjunto do modelo conforme as flags de
    config (ADICIONAR_DIA_SEMANA, FEATURES_INTERMITENCIA).

    ORDENACAO CRONOLOGICA -- os blocos sao construidos por horizonte
    (h=1 inteiro, depois h=2, ...). Devolver a tabela nessa ordem quebra a busca
    de hiperparametros: o TimeSeriesSplit corta por POSICAO de linha, sem olhar
    data, e acaba treinando em h=1 e validando em h=2/h=3. A venda de um mesmo
    dia cai dos dois lados do corte -- como (origem=d-1, h=1) no treino e
    (origem=d-3, h=3) na validacao, com features quase identicas -- o score fica
    otimista e a busca escolhe hiperparametros que decoram. Ordenar por
    data_origem faz a ordem posicional coincidir com a cronologica."""
    #df_idx não utilizado
    #df_idx = df_features.set_index([COLS.sku, COLS.data]).sort_index()
    datas_disponiveis = set(df_features[COLS.data].unique())

    blocos = []
    for h in range(1, horizonte + 1):
        origem_validas = [o for o in datas_origem if (o + timedelta(days=h)) in datas_disponiveis]
        if not origem_validas:
            continue

        base = df_features[df_features[COLS.data].isin(origem_validas)].copy()
        base["_data_alvo"] = base[COLS.data] + pd.Timedelta(days=h)
        base["horizonte"] = h

        _cols_base = [COLS.sku, COLS.data, COLS.qtd, COLS.especial_br, COLS.black_friday]
        _cols_extra = [c for c in (cols_exogenas_alvo or []) if c in df_features.columns]
        _alvo_cols = list(dict.fromkeys(_cols_base + _cols_extra))

        _rename_map = {
            COLS.data: "_data_alvo",
            COLS.qtd: "y",
            COLS.especial_br: f"{COLS.especial_br}_alvo",
            COLS.black_friday: f"{COLS.black_friday}_alvo",
            **{c: f"{c}_alvo" for c in _cols_extra},
        }
        alvo = df_features[_alvo_cols].rename(columns=_rename_map)

        base = base.merge(alvo, on=[COLS.sku, "_data_alvo"], how="inner")
        blocos.append(base)

    if not blocos:
        return pd.DataFrame()

    tabela = pd.concat(blocos, ignore_index=True)
    tabela = tabela.rename(columns={COLS.data: "data_origem"})

    # Calendario da data-alvo: sempre calculado, ativado nas features do modelo
    # via config (ADICIONAR_DIA_SEMANA, FEATURES_INTERMITENCIA).
    _alvo = tabela["data_origem"] + pd.to_timedelta(tabela["horizonte"], unit="D")
    tabela["dia_semana_alvo"] = _alvo.dt.weekday
    tabela["mes_alvo"] = _alvo.dt.month
    tabela["semana_ano_alvo"] = _alvo.dt.isocalendar().week.astype(int)

    # Ver a nota na docstring sobre ordenacao cronologica. Uma linha, custo desprezivel, e sem ela
    # a busca de hiperparametros valida contra si mesma.
    if config.ORDENAR_TABELA_CRONOLOGICA:
        tabela = tabela.sort_values(["data_origem", "horizonte"]).reset_index(drop=True)

    return tabela


# ---------------------------------------------------------------------------
# Conjuntos de features dos experimentos
# ---------------------------------------------------------------------------
def colunas_encoding_sku() -> list:
    """Colunas que representam a identidade do SKU, conforme config.ENCODING_SKU."""
    if str(config.ENCODING_SKU).lower() == "perfil":
        return list(COLUNAS_PERFIL_SKU)
    return ["sku_enc"]


def _features_calendario_extra() -> list:
    """Calendario da data-alvo que esta ativado por flag nesta execucao."""
    extras = []
    if config.ADICIONAR_DIA_SEMANA:
        extras.append("dia_semana_alvo")
    if config.FEATURES_INTERMITENCIA:
        extras.extend(COLUNAS_CALENDARIO_ALVO_EXTRA)
    return extras


def _aplicar_exclusoes(colunas: list) -> list:
    """Tira do conjunto as features redundantes.

    O filtro age sobre a LISTA DE ENTRADA DO MODELO, nao sobre a construcao: as
    colunas continuam sendo calculadas e aparecendo na matriz de correlacao, que
    e justamente o artefato que justifica a exclusao. Calcular duas rolling a
    mais custa pouco; o que pesa e o numero de variaveis que a arvore avalia em
    cada split, e esse cai."""
    remover = config.features_a_excluir(colunas)
    if not remover:
        return colunas
    return [c for c in colunas if c not in remover]


def features_experimento1(aplicar_exclusoes: bool = True) -> list:
    """Conjunto do Experimento 1: historico, horizonte, identidade do SKU
    e categoria. Sem calendario da data-alvo -- e isso que o distingue do
    Cenario 1.

    `aplicar_exclusoes=False` devolve o conjunto antes da remocao de features
    redundantes; e o que o relatorio selecao_variaveis.md usa para mostrar o
    que entrou e o que saiu."""
    colunas = (
        colunas_features_historico()
        + ["horizonte"]
        + colunas_encoding_sku()
        + ["categoria_enc"]
        + _features_calendario_extra()
    )
    return _aplicar_exclusoes(colunas) if aplicar_exclusoes else colunas


def features_cenario_1() -> list:
    """Cenario 1: Experimento 1 + calendario simples (feriados BR e BF)."""
    return features_experimento1() + [f"{COLS.especial_br}_alvo", f"{COLS.black_friday}_alvo"]


def features_cenario_2() -> list:
    """Cenario 2: Experimento 1 + calendario rico do CSV externo.

    As exogenas sao as filtradas por config.EXOGENAS_EXCLUIDAS: escolher quais
    variaveis de calendario fazem sentido para o contexto faz parte do
    desenho do cenario."""
    return features_experimento1() + list(config.colunas_calendario_rico_alvo())


# Aliases para codigo que importa os conjuntos base, sem features de
# intermitencia, remocao de redundantes ou exclusao de exogenas.
FEATURES_EXPERIMENTO1 = COLUNAS_FEATURES_HISTORICO + ["horizonte", "sku_enc", "categoria_enc"]
FEATURES_CENARIO_1 = FEATURES_EXPERIMENTO1 + [f"{COLS.especial_br}_alvo", f"{COLS.black_friday}_alvo"]
FEATURES_CENARIO_2 = (
    COLUNAS_FEATURES_HISTORICO
    + ["horizonte", "sku_enc", "categoria_enc"]
    + config.COLUNAS_CALENDARIO_RICO_ALVO
)


In [ ]:
%%writefile /content/src/baselines.py
"""
Fase 6 — Baselines estatisticos SBA e TSB.

Ambos sao metodos de Croston: separam a serie em "com que frequencia ha
demanda" e "de que tamanho ela e quando ocorre", e preveem o produto das duas
estimativas. A diferenca entre eles esta em como tratam a ausencia de demanda:
  SBA  atualiza o intervalo somente quando ha venda, e corrige o vies de
       Croston pelo fator (1 - alpha/2). Nao decai quando o produto para de
       vender -- a previsao congela no ultimo nivel estimado.
  TSB  atualiza a PROBABILIDADE de demanda a cada periodo, inclusive nos
       periodos zerados. Por isso decai gradualmente quando a demanda cessa, o
       que o torna o unico dos dois capaz de sinalizar obsolescencia.

GRANULARIDADE: os dois operam em series SEMANAIS (a modelagem de ML segue
diaria) e a previsao semanal e desagregada para dias pela proporcao media de
cada dia da semana, estimada no treino. Os parametros de suavizacao sao
otimizados por MSE sobre o treino.

ALINHAMENTO DA GRADE SEMANAL: a agregacao usa freq="W-MON" -- semanas de terca
a segunda -- para coincidir com a cadencia das janelas de teste, que comecam na
terca 01/07/2025 e avancam de 7 em 7 dias. Com o padrao do pandas ("W" = W-SUN)
o ultimo bin do treino fecharia com um unico dia e a primeira previsao cairia
fora do janelamento, cobrindo 1 dos 7 dias de cada janela -- o que tornaria as
metricas dos baselines incomparaveis com as do ML. Toda ocorrencia de W-MON
neste modulo existe por esse motivo.
"""
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from . import config
from .config import COLS

try:
    from statsforecast import StatsForecast
    from statsforecast.models import CrostonSBA as _CrostonSBA
    _TEM_STATSFORECAST = True
except ImportError:  # pragma: no cover - fallback documentado
    _TEM_STATSFORECAST = False
    warnings.warn(
        "statsforecast nao encontrado — SBA usara fallback manual em Python "
        "puro. Instale `statsforecast==2.0.3` (Secao 4.3.1 do TCC) para a "
        "implementacao de referencia."
    )


# ---------------------------------------------------------------------------
# Agregacao semanal e proporcao de desagregacao por dia da semana
# ---------------------------------------------------------------------------
def agregar_semanal_sku(df: pd.DataFrame) -> pd.DataFrame:
    """Soma a demanda por SKU e semana (terca a segunda). Ver a nota sobre
    W-MON no cabecalho do modulo."""
    semanal = (
        df.groupby([COLS.sku, pd.Grouper(key=COLS.data, freq="W-MON")])[COLS.qtd]
        .sum()
        .reset_index()
    )
    return semanal


def calcular_proporcao_dia_semana(df_treino: pd.DataFrame) -> pd.DataFrame:
    """Proporcao media de venda de cada dia da semana, por SKU.

    E o perfil que desagrega a previsao semanal em diaria. Semanas sem venda
    nao entram na media, e SKUs sem nenhuma semana util caem em 1/7 por dia
    (distribuicao igualitaria). O resultado e normalizado para somar 1 por SKU,
    de modo que a desagregacao preserve o total semanal previsto.

    E tambem a via pela qual os baselines recebem sazonalidade semanal -- o
    equivalente ao `dia_semana_alvo` dos modelos de ML."""
    d = df_treino.copy()
    d["dia_semana"] = d[COLS.data].dt.weekday
    d["semana"] = d[COLS.data].dt.to_period("W-MON")

    total_semana = d.groupby([COLS.sku, "semana"])[COLS.qtd].transform("sum")
    d["_prop"] = np.where(total_semana > 0, d[COLS.qtd] / total_semana, np.nan)

    prop_media = d.groupby([COLS.sku, "dia_semana"])["_prop"].mean().reset_index()
    prop_media["_prop"] = prop_media["_prop"].fillna(1 / 7)

    # normaliza para somar 1 por SKU
    soma_por_sku = prop_media.groupby(COLS.sku)["_prop"].transform("sum")
    prop_media["_prop"] = np.where(soma_por_sku > 0, prop_media["_prop"] / soma_por_sku, 1 / 7)
    return prop_media.rename(columns={"_prop": "proporcao"})


def desagregar_semana_para_dias(
    previsao_semanal: pd.DataFrame,  # colunas: sku, semana_fim (Data), yhat
    proporcoes: pd.DataFrame,        # colunas: sku, dia_semana, proporcao
) -> pd.DataFrame:
    """Espalha cada previsao semanal pelos 7 dias, conforme o perfil do SKU.

    Devolve uma linha por (SKU, dia), somando ao total semanal previsto."""
    linhas = []
    prop_idx = proporcoes.set_index([COLS.sku, "dia_semana"])["proporcao"]

    for _, row in previsao_semanal.iterrows():
        sku = row[COLS.sku]
        fim_semana = row[COLS.data]  # segunda-feira, fim da semana W-MON
        inicio_semana = fim_semana - pd.Timedelta(days=6)
        yhat_semana = row["yhat"]

        for i in range(7):
            dia = inicio_semana + pd.Timedelta(days=i)
            dia_semana = dia.weekday()
            prop = prop_idx.get((sku, dia_semana), 1 / 7)
            linhas.append({COLS.sku: sku, COLS.data: dia, "yhat": yhat_semana * prop})

    return pd.DataFrame(linhas)


# ---------------------------------------------------------------------------
# TSB — implementacao manual (Teunter, Syntetos e Babai, 2011)
# ---------------------------------------------------------------------------
def _tsb_prever_serie(y: np.ndarray, alpha: float, beta: float, h: int) -> np.ndarray:
    """Roda o TSB sobre uma serie ja agregada e devolve h previsoes.

    Mantem dois estados suavizados: `z` (tamanho da demanda, atualizado so
    quando ha venda, com alpha) e `p` (probabilidade de haver demanda,
    atualizada em TODO periodo, com beta). A previsao e p * z.

    Atualizar `p` tambem nos periodos zerados e o que diferencia o TSB do SBA:
    a probabilidade decai enquanto o produto nao vende, e a previsao converge
    para zero. E o mecanismo pelo qual o TSB detecta obsolescencia.

    A previsao pontual e repetida por h periodos -- pratica padrao para
    suavizacao exponencial em horizonte curto, ja que o metodo nao modela
    tendencia."""
    z = 0.0
    p = 0.0
    inicializado = False

    for v in y:
        if not inicializado:
            z = v if v > 0 else np.mean(y[y > 0]) if (y > 0).any() else 0.0
            p = 1.0 if v > 0 else 0.0
            inicializado = True
            continue
        if v > 0:
            z = alpha * v + (1 - alpha) * z
            p = beta * 1.0 + (1 - beta) * p
        else:
            p = beta * 0.0 + (1 - beta) * p

    previsao_pontual = p * z
    return np.full(h, previsao_pontual)


def _otimizar_alpha_beta_tsb(y: np.ndarray) -> tuple[float, float]:
    """Escolhe alpha e beta por menor MSE um passo a frente no treino.

    Busca exaustiva numa grade 7x7 (49 avaliacoes por SKU). E deliberadamente
    simples, para nao depender de otimizador externo -- e o custo disso que
    torna a paralelizacao por SKU necessaria."""
    melhor = (0.2, 0.2, np.inf)
    grade = np.linspace(0.05, 0.95, 7)
    for a in grade:
        for b in grade:
            z, p = 0.0, 0.0
            erros = []
            inicializado = False
            for i, v in enumerate(y):
                if not inicializado:
                    z = v if v > 0 else 0.0
                    p = 1.0 if v > 0 else 0.0
                    inicializado = True
                    continue
                pred = p * z
                erros.append((v - pred) ** 2)
                if v > 0:
                    z = a * v + (1 - a) * z
                    p = b * 1.0 + (1 - b) * p
                else:
                    p = b * 0.0 + (1 - b) * p
            mse = np.mean(erros) if erros else np.inf
            if mse < melhor[2]:
                melhor = (a, b, mse)
    return melhor[0], melhor[1]


def _tsb_um_sku(sku, y: np.ndarray, ultima_data, h_semanas: int) -> list[dict]:
    """Otimiza os parametros e preve um unico SKU.

    Isolado em funcao propria para poder rodar em paralelo: cada SKU e
    independente e a busca em grade e cara em Python puro."""
    if len(y) == 0:
        return []
    alpha, beta = _otimizar_alpha_beta_tsb(y)
    yhat = _tsb_prever_serie(y, alpha, beta, h_semanas)
    datas_futuras = pd.date_range(ultima_data + pd.Timedelta(weeks=1), periods=h_semanas, freq="W-MON")
    return [{COLS.sku: sku, COLS.data: d, "yhat": val} for d, val in zip(datas_futuras, yhat)]


def prever_tsb(df_semanal_treino: pd.DataFrame, skus: list, h_semanas: int) -> pd.DataFrame:
    """Previsao TSB semanal para uma lista de SKUs.

    Paraleliza por SKU com joblib (config.N_JOBS). Abaixo de 200 SKUs roda
    serial: o custo de criar processos superaria o ganho."""
    # Agrupa uma unica vez. Filtrar o DataFrame inteiro dentro do laco seria
    # O(n_skus x n_linhas).
    grupos = {
        sku: (g[COLS.qtd].to_numpy(), g[COLS.data].max())
        for sku, g in df_semanal_treino.groupby(COLS.sku)
    }
    tarefas = [(sku, grupos[sku][0], grupos[sku][1]) for sku in skus if sku in grupos]

    if config.N_JOBS == 1 or len(tarefas) < 200:
        blocos = [_tsb_um_sku(sku, y, ud, h_semanas) for sku, y, ud in tarefas]
    else:
        blocos = Parallel(n_jobs=config.N_JOBS, prefer="processes")(
            delayed(_tsb_um_sku)(sku, y, ud, h_semanas) for sku, y, ud in tarefas
        )

    resultados = [linha for bloco in blocos for linha in bloco]
    return pd.DataFrame(resultados)


def prever_sba(df_semanal_treino: pd.DataFrame, skus: list, h_semanas: int) -> pd.DataFrame:
    """Previsao SBA semanal, pelo `statsforecast` (implementacao de referencia).

    Cai para `_prever_sba_fallback` quando o pacote nao esta instalado."""
    if not _TEM_STATSFORECAST:
        return _prever_sba_fallback(df_semanal_treino, skus, h_semanas)

    sf_df = df_semanal_treino.rename(
        columns={COLS.sku: "unique_id", COLS.data: "ds", COLS.qtd: "y"}
    )[["unique_id", "ds", "y"]]
    sf_df = sf_df[sf_df["unique_id"].isin(skus)]

    if sf_df.empty:
        return pd.DataFrame(columns=[COLS.sku, COLS.data, "yhat"])

    sf = StatsForecast(models=[_CrostonSBA()], freq="W-MON", n_jobs=config.N_JOBS)
    sf.fit(sf_df)
    previsao = sf.predict(h=h_semanas).reset_index()
    previsao = previsao.rename(columns={"unique_id": COLS.sku, "ds": COLS.data, "CrostonSBA": "yhat"})
    return previsao[[COLS.sku, COLS.data, "yhat"]]


def _sba_fallback_um_sku(sku, y: np.ndarray, ultima_data, h_semanas: int) -> list[dict]:
    """Croston com correcao de Syntetos-Boylan para um unico SKU.

    Mantem `z` (tamanho da demanda) e `p` (intervalo medio entre demandas),
    ambos atualizados APENAS quando ha venda -- e por isso que o SBA nao decai
    em produto que parou de vender. A previsao e (1 - alpha/2) * z/p, em que o
    fator corrige o vies de alta do Croston original.

    `q` conta os periodos desde a ultima venda e alimenta a atualizacao de `p`."""
    if len(y) == 0:
        return []
    melhor_alpha, melhor_mse = 0.2, np.inf
    z_final, p_final = 0.0, 0.0
    for a in np.linspace(0.05, 0.95, 7):
        z, p, erros, inicializado = 0.0, 0.0, [], False
        q = 0
        for v in y:
            if not inicializado:
                z = v if v > 0 else 0.0
                p = 1.0
                inicializado = True
                q = 0
                continue
            q += 1
            pred = (1 - a / 2) * (z / p) if p > 0 else 0.0
            erros.append((v - pred) ** 2)
            if v > 0:
                z = a * v + (1 - a) * z
                p = a * q + (1 - a) * p
                q = 0
        mse = np.mean(erros) if erros else np.inf
        if mse < melhor_mse:
            melhor_alpha, melhor_mse = a, mse
            z_final, p_final = z, p

    yhat = (1 - melhor_alpha / 2) * (z_final / p_final) if p_final > 0 else 0.0
    datas_futuras = pd.date_range(ultima_data + pd.Timedelta(weeks=1), periods=h_semanas, freq="W-MON")
    return [{COLS.sku: sku, COLS.data: d, "yhat": yhat} for d in datas_futuras]


def _prever_sba_fallback(df_semanal_treino: pd.DataFrame, skus: list, h_semanas: int) -> pd.DataFrame:
    """SBA em Python puro, usado quando `statsforecast` nao esta disponivel.

    Paraleliza por SKU como o TSB. Os resultados sao proximos, mas nao
    identicos, aos do statsforecast -- se esta funcao for usada na rodada final,
    registre isso junto com as versoes das bibliotecas no METADATA."""
    grupos = {
        sku: (g[COLS.qtd].to_numpy(), g[COLS.data].max())
        for sku, g in df_semanal_treino.groupby(COLS.sku)
    }
    tarefas = [(sku, grupos[sku][0], grupos[sku][1]) for sku in skus if sku in grupos]

    if config.N_JOBS == 1 or len(tarefas) < 200:
        blocos = [_sba_fallback_um_sku(sku, y, ud, h_semanas) for sku, y, ud in tarefas]
    else:
        blocos = Parallel(n_jobs=config.N_JOBS, prefer="processes")(
            delayed(_sba_fallback_um_sku)(sku, y, ud, h_semanas) for sku, y, ud in tarefas
        )

    resultados = [linha for bloco in blocos for linha in bloco]
    return pd.DataFrame(resultados)


# ---------------------------------------------------------------------------
# Orquestracao walk-forward (Experimento 1)
# ---------------------------------------------------------------------------
def executar_walk_forward_baseline(
    df_modelagem: pd.DataFrame,
    metodo: str,  # "sba" ou "tsb"
    janelas_por_refit: int = 1,
    teste_inicio: str = config.TESTE_INICIO,
    teste_fim: str = config.TESTE_FIM,
) -> pd.DataFrame:
    """Walk-forward dos baselines, na mesma cadencia de 7 dias dos modelos de ML.

    Usar as mesmas janelas e o que permite a comparacao par a par por SKU dos
    testes de Wilcoxon: os quatro modelos preveem exatamente as mesmas datas.

    Em cada refit: reagrega o treino ate o corte, reestima o perfil semanal,
    preve as proximas `janelas_por_refit` semanas, desagrega para dias e casa
    com o valor real observado.

    Saida no mesmo formato do ML: uma linha por (SKU, data, y_real, y_pred)."""
    teste_inicio_ts = pd.Timestamp(teste_inicio)
    teste_fim_ts = pd.Timestamp(teste_fim)

    janelas = []
    cursor = teste_inicio_ts
    while cursor <= teste_fim_ts:
        janelas.append(cursor)
        cursor += pd.Timedelta(days=7)

    df_real = df_modelagem.set_index([COLS.sku, COLS.data])[COLS.qtd]
    resultados = []

    for i, inicio_janela in enumerate(janelas):
        if i % janelas_por_refit != 0:
            continue
        corte = inicio_janela - pd.Timedelta(days=1)

        df_treino_ate_corte = df_modelagem[df_modelagem[COLS.data] <= corte]
        semanal_treino = agregar_semanal_sku(df_treino_ate_corte)
        proporcoes = calcular_proporcao_dia_semana(df_treino_ate_corte)
        skus = sorted(df_treino_ate_corte[COLS.sku].unique())

        n_janelas_previsao = min(janelas_por_refit, len(janelas) - i)

        if metodo == "sba":
            previsao_semanal = prever_sba(semanal_treino, skus, h_semanas=n_janelas_previsao)
        else:
            previsao_semanal = prever_tsb(semanal_treino, skus, h_semanas=n_janelas_previsao)

        if previsao_semanal.empty:
            continue

        previsao_diaria = desagregar_semana_para_dias(previsao_semanal, proporcoes)

        # Recorta a previsao diaria por janela e junta o real observado.
        for k in range(n_janelas_previsao):
            data_ini_k = inicio_janela + pd.Timedelta(days=7 * k)
            data_fim_k = data_ini_k + pd.Timedelta(days=6)
            bloco = previsao_diaria[
                (previsao_diaria[COLS.data] >= data_ini_k) & (previsao_diaria[COLS.data] <= data_fim_k)
            ].copy()
            if bloco.empty:
                continue
            bloco["y_real"] = bloco.apply(
                lambda r: df_real.get((r[COLS.sku], r[COLS.data]), np.nan), axis=1
            )
            bloco = bloco.dropna(subset=["y_real"]).rename(columns={"yhat": "y_pred"})
            resultados.append(bloco[[COLS.sku, COLS.data, "y_real", "y_pred"]])

    if not resultados:
        return pd.DataFrame(columns=[COLS.sku, COLS.data, "y_real", "y_pred"])
    return pd.concat(resultados, ignore_index=True)


In [ ]:
%%writefile /content/src/ml_models.py
"""
Fases 7-8 — Modelos de ML globais (Random Forest e XGBoost): busca de
hiperparametros e walk-forward com janela expandida.

"Global" quer dizer um unico modelo treinado sobre todos os SKUs de uma vez,
com a identidade do SKU como variavel de entrada -- e nao um modelo por SKU.
E o que permite a um produto de giro baixissimo aproveitar o padrao aprendido
nos demais.

FLUXO DE UMA JANELA WALK-FORWARD
  corte = vespera do inicio da janela
  1. ajusta os codificadores so com dados ate `corte`  (_fit_encoders)
  2. monta a tabela longa de treino ate `corte`        (_preparar_tabela)
  3. busca hiperparametros, se ainda nao houver        (ajustar_hiperparametros)
  4. treina e preve as janelas ate o proximo refit
As janelas avancam de 7 em 7 dias; `janelas_por_refit` controla de quantas em
quantas o modelo e retreinado, e e o principal regulador do custo total.

REGISTROS: o modulo acumula tempos por etapa (_REGISTRO_TEMPOS) e importancia
de features por refit (_REGISTRO_IMPORTANCIAS). Alimentam o relatorio de
tempos e a figura de importancia. `resetar_registros()` deve ser chamado no
inicio de cada execucao, senao os numeros somam entre rodadas da mesma sessao.
"""
from __future__ import annotations

import time
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

try:
    from xgboost import XGBRegressor
    _TEM_XGBOOST = True
except ImportError:  # pragma: no cover - fallback documentado
    _TEM_XGBOOST = False
    from sklearn.ensemble import GradientBoostingRegressor
    warnings.warn(
        "xgboost nao encontrado — usando GradientBoostingRegressor como "
        "substituto aproximado. Instale `xgboost==3.0.5` (Secao 4.3.1) "
        "para a implementacao de referencia."
    )

from . import config
from .config import COLS
from .features import (
    construir_features_base,
    montar_tabela_horizontes,
    LabelEncoderJanela,
    PerfilSkuJanela,
    colunas_dropna_historico,
)


# ---------------------------------------------------------------------------
# Registro de tempos de execucao e de importancia de features
# ---------------------------------------------------------------------------
_REGISTRO_TEMPOS: list[dict] = []
_REGISTRO_IMPORTANCIAS: list[dict] = []
# Hiperparametros ja buscados nesta execucao, por tipo de modelo ("rf"/"xgb").
# So e consultado com TUNAR_UMA_VEZ_GLOBAL ligado.
_HIPER_CACHE: dict = {}

ETAPAS_TEMPO = ("features_base", "tuning", "fit_final", "predict", "walk_forward",
                "tabelas_compartilhadas")

# Uma linha por busca realizada, comparando o candidato que o WMAPE
# escolheria com o que o RMSE escolheria. Consumido por
# `relatorio_diagnostico_selecao()` ao final da execucao.
_DIAGNOSTICO_SELECAO: list = []


def resetar_registros() -> None:
    """Zera tempos, importancias e cache de hiperparametros.

    Obrigatorio no inicio de cada execucao: sao listas de modulo, e sem o reset
    duas rodadas na mesma sessao somam os numeros uma da outra."""
    _REGISTRO_TEMPOS.clear()
    _REGISTRO_IMPORTANCIAS.clear()
    _HIPER_CACHE.clear()
    _DIAGNOSTICO_SELECAO.clear()


def registrar_tempo(
    modelo: str,
    etapa: str,
    segundos: float,
    janela: int | None = None,
    data_corte=None,
    n_linhas: int | None = None,
    n_skus: int | None = None,
    detalhe: str | None = None,
) -> None:
    """Anota o tempo de uma etapa. `detalhe` guarda os hiperparametros
    escolhidos, o que torna tempos_execucao.csv auditavel."""
    _REGISTRO_TEMPOS.append({
        "modelo": modelo,
        "etapa": etapa,
        "janela": janela,
        "data_corte": None if data_corte is None else pd.Timestamp(data_corte).date().isoformat(),
        "n_linhas": n_linhas,
        "n_skus": n_skus,
        "segundos": round(float(segundos), 4),
        "detalhe": detalhe,
    })


def obter_tempos_df() -> pd.DataFrame:
    """Tempos registrados como DataFrame (vazio com as colunas certas, se nao
    houver registro)."""
    if not _REGISTRO_TEMPOS:
        return pd.DataFrame(columns=["modelo", "etapa", "janela", "data_corte",
                                     "n_linhas", "n_skus", "segundos", "detalhe"])
    return pd.DataFrame(_REGISTRO_TEMPOS)


def salvar_tempos_csv(caminho) -> pd.DataFrame:
    """Grava tempos_execucao.csv e devolve a tabela."""
    df = obter_tempos_df()
    df.to_csv(caminho, index=False)
    return df


def registrar_importancias(modelo: str, janela: int, features: list, importancias) -> None:
    """Guarda a importancia de cada feature em um refit. Uma linha por
    (modelo, janela, feature), para depois agregar com media e desvio."""
    for feature, valor in zip(features, np.asarray(importancias, dtype=float)):
        _REGISTRO_IMPORTANCIAS.append({
            "modelo": modelo, "janela": janela,
            "feature": feature, "importancia": float(valor),
        })


def obter_importancias_df(agregado: bool = True) -> pd.DataFrame:
    """Importancias por feature. Agregado devolve media, desvio e numero de
    refits -- o desvio mostra o quanto a importancia oscila entre janelas, que e
    o que diz se a leitura da figura e estavel."""
    if not _REGISTRO_IMPORTANCIAS:
        return pd.DataFrame(columns=["modelo", "feature", "importancia_media",
                                     "importancia_std", "n_refits"])
    df = pd.DataFrame(_REGISTRO_IMPORTANCIAS)
    if not agregado:
        return df
    return (
        df.groupby(["modelo", "feature"], as_index=False)["importancia"]
        .agg(importancia_media="mean", importancia_std="std", n_refits="count")
    )


def salvar_importancias_csv(caminho) -> pd.DataFrame:
    """Grava importancia_features.csv e devolve a tabela agregada."""
    df = obter_importancias_df(agregado=True)
    df.to_csv(caminho, index=False)
    return df


_gpu_disponivel_cache: bool | None = None


def _xgb_gpu_disponivel() -> bool:
    """Testa uma vez se ha GPU CUDA utilizavel, com um fit minimo.

    A deteccao precisa inspecionar os warnings: o XGBoost cai para CPU
    silenciosamente quando nao acha GPU, em vez de levantar excecao. Sem isso o
    pipeline reportaria uso de GPU numa execucao que rodou inteira em CPU, e a
    tabela de custo computacional ficaria incomparavel com outras rodadas."""
    global _gpu_disponivel_cache
    if _gpu_disponivel_cache is not None:
        return _gpu_disponivel_cache
    try:
        with warnings.catch_warnings(record=True) as capturados:
            warnings.simplefilter("always")
            teste = XGBRegressor(tree_method="hist", device="cuda", n_estimators=2, max_depth=2)
            teste.fit(np.zeros((4, 2)), np.zeros(4))
        sem_gpu = any(
            "gpu" in str(w.message).lower() and (
                "no visible" in str(w.message).lower() or "couldn't find" in str(w.message).lower()
            )
            for w in capturados
        )
        if sem_gpu:
            raise RuntimeError("xgboost reportou ausencia de GPU via warning no fit de teste")
        _gpu_disponivel_cache = True
        print("[GPU] XGBoost detectou GPU CUDA disponivel — usando device='cuda'.")
    except Exception as exc:
        _gpu_disponivel_cache = False
        print(
            f"[GPU] config.USE_GPU_XGB=True mas GPU CUDA indisponivel neste ambiente "
            f"({exc.__class__.__name__}: {exc}) — caindo para device='cpu'."
        )
    return _gpu_disponivel_cache


def _device_xgb() -> str:
    """'cuda' se pedido e disponivel, senao 'cpu'."""
    if _TEM_XGBOOST and config.USE_GPU_XGB and _xgb_gpu_disponivel():
        return "cuda"
    return "cpu"


def _wmape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """WMAPE usado como criterio INTERNO da busca de hiperparametros.

    Nao e a metrica de avaliacao final -- essa esta em evaluate.wmape.

    Dobras inteiras sem demanda sao frequentes numa serie com ~95% de zeros. Com
    denominador zero, devolver 0.0 daria nota perfeita a qualquer candidato e
    dissolveria a ordenacao da busca. O ramo ativo cai para MAE puro, que
    preserva a ordenacao (com y_true fixo dentro da dobra, WMAPE e transformacao
    monotona do MAE)."""
    soma_erro = np.sum(np.abs(y_true - y_pred))
    soma_real = np.sum(np.abs(y_true))
    if soma_real > 0:
        return soma_erro / soma_real
    if config.ORDENAR_TABELA_CRONOLOGICA:
        return float(np.mean(np.abs(y_true - y_pred)))
    return 0.0  # ramo sem a ordenacao cronologica, mantido para o A/B


def _amostrar_para_tuning(X: pd.DataFrame, y: pd.Series, max_linhas):
    """Amostra a tabela para caber em MAX_LINHAS_TUNING.

    O `np.sort` do indice e essencial: preserva a ordem cronologica da tabela,
    sem a qual os cortes posicionais do TimeSeriesSplit deixam de ser cortes
    temporais."""
    if max_linhas is None or len(X) <= max_linhas:
        return X, y
    idx = np.random.RandomState(config.RANDOM_STATE).choice(len(X), size=max_linhas, replace=False)
    idx = np.sort(idx)
    return X.iloc[idx], y.iloc[idx]


def _grade_xgb_efetiva() -> dict:
    """Grade do XGBoost desta execucao.

    Com funcao objetivo Tweedie, acrescenta tweedie_variance_power: o expoente
    entre 1 e 2 define a mistura Poisson-Gama e nao tem valor padrao razoavel
    para dados de demanda -- precisa ser buscado."""
    grade = dict(config.grade_xgb())
    if str(config.XGB_OBJETIVO).startswith("reg:tweedie"):
        grade["tweedie_variance_power"] = list(config.XGB_TWEEDIE_POWER_GRADE)
    return grade


def _scorers_selecao() -> dict:
    """Os dois criterios de selecao, avaliados na MESMA busca.

    Com scoring de multiplas metricas, o RandomizedSearchCV ajusta cada
    candidato UMA vez por dobra e o pontua com as duas metricas. O custo
    adicional e o da pontuacao, nao o do ajuste -- por isso o diagnostico sai
    praticamente de graca.

    Ambos os scorers sao "maior = melhor", como o sklearn exige: o WMAPE entra
    negado e o RMSE usa a metrica negativa ja embutida na biblioteca."""
    return {
        "wmape": make_scorer(lambda yt, yp: -_wmape(np.asarray(yt), np.asarray(yp))),
        "rmse": "neg_root_mean_squared_error",
    }


def ajustar_hiperparametros(X: pd.DataFrame, y: pd.Series, tipo_modelo: str):
    """Busca aleatoria com TimeSeriesSplit.

    A metrica de selecao e config.METRICA_SELECAO_HIPER. As duas
    metricas candidatas sao sempre avaliadas; apenas o `refit` muda entre as
    execucoes.

    X precisa chegar ordenado por (data_origem, horizonte) para que os cortes
    posicionais do TimeSeriesSplit sejam cortes temporais -- ver a nota sobre
    ordenacao cronologica em features.montar_tabela_horizontes.

    Devolve o dicionario de melhores hiperparametros."""
    X_amostra, y_amostra = _amostrar_para_tuning(X, y, config.MAX_LINHAS_TUNING)

    tscv = TimeSeriesSplit(n_splits=config.N_SPLITS_TIME_SERIES_CV)

    device_xgb = None
    if tipo_modelo == "rf":
        base = RandomForestRegressor(random_state=config.RANDOM_STATE, n_jobs=1)
        grade = config.grade_rf()
    else:
        if _TEM_XGBOOST:
            device_xgb = _device_xgb()
            base = XGBRegressor(
                random_state=config.RANDOM_STATE, tree_method="hist", device=device_xgb, n_jobs=1,
                objective=config.XGB_OBJETIVO,
            )
        else:
            base = GradientBoostingRegressor(random_state=config.RANDOM_STATE)
        grade = _grade_xgb_efetiva()  # inclui Tweedie quando aplicavel

    # Com GPU, paralelizar a busca disputaria o mesmo dispositivo entre
    # processos; o paralelismo fica dentro do proprio XGBoost.
    n_jobs_busca = 1 if device_xgb == "cuda" else config.N_JOBS

    n_iter = config.n_iter_rf() if tipo_modelo == "rf" else config.N_ITER_RANDOM_SEARCH
    n_iter = min(n_iter, _tamanho_grade(grade))

    # As duas metricas sao avaliadas sempre; `refit` define qual
    # delas decide o modelo desta execucao.
    criterio = str(config.METRICA_SELECAO_HIPER).upper()
    if criterio not in config.METRICAS_SELECAO_VALIDAS:
        raise ValueError(
            f"METRICA_SELECAO_HIPER='{criterio}' invalida; "
            f"use uma de {config.METRICAS_SELECAO_VALIDAS}")
    busca = RandomizedSearchCV(
        base, grade, n_iter=n_iter, cv=tscv,
        scoring=_scorers_selecao(),
        refit=criterio.lower(),
        random_state=config.RANDOM_STATE, n_jobs=n_jobs_busca,
    )
    busca.fit(X_amostra, y_amostra)
    if config.DIAGNOSTICO_SELECAO_HIPER:
        res = pd.DataFrame(busca.cv_results_)
        escolha_wmape = res.loc[res["rank_test_wmape"] == 1, "params"].iloc[0]
        escolha_rmse = res.loc[res["rank_test_rmse"] == 1, "params"].iloc[0]
        _DIAGNOSTICO_SELECAO.append({
            "modelo": tipo_modelo,
            "criterio_da_execucao": criterio,
            "n_candidatos": len(res),
            "escolha_wmape": escolha_wmape,
            "escolha_rmse": escolha_rmse,
            "criterios_concordam": escolha_wmape == escolha_rmse,
        })
    return busca.best_params_


def _tamanho_grade(grade: dict) -> int:
    """Numero de combinacoes da grade; limita n_iter para nao sortear repetido."""
    total = 1
    for valores in grade.values():
        total *= max(len(valores), 1)
    return total


@dataclass
class ConfigWalkForward:
    """Parametros do walk-forward.

    colunas_features            entrada do modelo
    stride_origem_treino_dias   passo entre as origens de TREINO. 1 = todas as
                                datas (modo fiel); 7 = uma por semana (rapido)
    janelas_por_refit           de quantas em quantas janelas o modelo e
                                retreinado. E o maior regulador do custo
    tunar_uma_vez               busca hiperparametros so no primeiro refit
    """
    colunas_features: list
    stride_origem_treino_dias: int = 1
    janelas_por_refit: int = 1
    tunar_uma_vez: bool = True


def _preparar_origens_treino(df_features: pd.DataFrame, corte: pd.Timestamp, stride: int) -> list:
    """Datas de origem de treino ate `corte`, com o passo pedido.

    A margem de 7 dias garante que exista alvo observado para todo h = 1..7:
    uma origem mais proxima do corte geraria linhas sem y."""
    todas = sorted(d for d in df_features[COLS.data].unique() if d <= corte - pd.Timedelta(days=7))
    return todas[::stride]


def _fit_encoders(df_features: pd.DataFrame, corte: pd.Timestamp):
    """Ajusta os codificadores somente sobre o treino disponivel ate `corte`.

    Refeito a cada refit -- e o que impede que a codificacao carregue
    informacao de datas que o modelo daquela janela ainda nao deveria ver.
    Devolve (encoder do SKU, encoder da categoria, perfil do SKU ou None)."""
    treino_ate_corte = df_features[df_features[COLS.data] <= corte]
    enc_sku = LabelEncoderJanela().fit(treino_ate_corte[COLS.sku])
    enc_cat = LabelEncoderJanela().fit(treino_ate_corte[COLS.categoria])
    perfil = None
    if str(config.ENCODING_SKU).lower() == "perfil":
        perfil = PerfilSkuJanela().fit(treino_ate_corte)
    return enc_sku, enc_cat, perfil


def _preparar_tabela(df_features, datas_origem, enc_sku, enc_cat, horizonte,
                     cols_exogenas_alvo=None, perfil_sku=None):
    """Monta a tabela longa e aplica os codificadores da janela.

    Serve tanto para treino quanto para previsao -- a diferenca esta apenas em
    quais datas de origem sao passadas."""
    tabela = montar_tabela_horizontes(df_features, datas_origem, horizonte=horizonte,
                                      cols_exogenas_alvo=cols_exogenas_alvo)
    if tabela.empty:
        return tabela
    tabela["sku_enc"] = enc_sku.transform(tabela[COLS.sku])
    tabela["categoria_enc"] = enc_cat.transform(tabela[COLS.categoria])
    if perfil_sku is not None:
        tabela = perfil_sku.transform(tabela)
    return tabela


@dataclass
class EspecificacaoModelo:
    """Um modelo do walk-forward, na forma em que ele chega ao executor.

    nome_modelo           "rf" ou "xgb" — decide o estimador e a grade de busca
    colunas_features      entrada do modelo; e sempre um SUBCONJUNTO das colunas
                          da tabela de treino, nunca algo que a modifique
    rotulo_tempos         nome usado nos registros de tempo/importancia e como
                          chave da saida
    max_estimators_final  teto de n_estimators no fit final; None = sem teto
    """
    nome_modelo: str
    colunas_features: list
    rotulo_tempos: str
    max_estimators_final: int | None = None


def _construir_estimador(nome_modelo: str, hiperparametros: dict):
    """Instancia o estimador da janela com os hiperparametros ja escolhidos."""
    if nome_modelo == "rf":
        return RandomForestRegressor(
            random_state=config.RANDOM_STATE, n_jobs=config.N_JOBS, **hiperparametros)
    if _TEM_XGBOOST:
        device_xgb = _device_xgb()
        n_jobs_fit = 1 if device_xgb == "cuda" else config.N_JOBS
        return XGBRegressor(
            random_state=config.RANDOM_STATE, tree_method="hist", device=device_xgb,
            n_jobs=n_jobs_fit,
            objective=config.XGB_OBJETIVO,
            **hiperparametros,
        )
    return GradientBoostingRegressor(random_state=config.RANDOM_STATE)


def executar_walk_forward_multi(
    df_modelagem: pd.DataFrame,
    especificacoes: list,
    cfg: ConfigWalkForward,
    teste_inicio: str = config.TESTE_INICIO,
    teste_fim: str = config.TESTE_FIM,
    horizonte: int = config.HORIZONTE_PREVISAO_DIAS,
    cols_exogenas_alvo: list | None = None,
    df_features_pronto: pd.DataFrame | None = None,
) -> dict:
    """Walk-forward de um ou mais modelos, devolvendo {rotulo_tempos: previsoes}.

    E a implementacao unica do laco: `executar_walk_forward` e um wrapper desta
    funcao com uma unica especificacao. Nao ha duas versoes do algoritmo para
    divergirem entre si.

    POR QUE COMPARTILHAR A TABELA E SEGURO
    A tabela de treino de uma janela e funcao de (df_features, corte, stride,
    horizonte, cols_exogenas_alvo) e dos codificadores ajustados ate `corte` —
    NENHUM desses termos depende do modelo. O modelo entra apenas na SELECAO de
    colunas (`tabela[colunas_features]`) e no estimador. Dois modelos na mesma
    chamada recebem, portanto, exatamente a tabela que receberiam se cada um a
    tivesse montado sozinho.
    Os codificadores continuam sendo reajustados a CADA janela, dentro do laco,
    sobre os dados ate o corte daquela janela — o compartilhamento e entre
    MODELOS da mesma janela, nunca entre janelas. E por isso que ele nao pode
    criar vazamento temporal: nao existe nenhum caminho pelo qual a tabela de
    uma janela alcance outra.

    A COPIA
    Com 2+ modelos, cada um recebe uma copia das suas colunas
    (`tabela[colunas].copy()`), em vez de uma fatia que compartilha memoria com
    a tabela comum. Hoje nenhum consumidor escreve na matriz de entrada, entao a
    copia nao muda resultado nenhum; ela existe para que isso continue verdade
    se algum consumidor futuro passar a escrever. O custo e uma fracao do que
    seria remontar a tabela.

    TEMPOS
    Com um unico modelo, `walk_forward` continua sendo o wall-clock inteiro da
    chamada — identico as versoes anteriores. Com 2+ modelos, o tempo de montar
    as tabelas nao pertence a nenhum modelo: vai para uma linha `CACHE` /
    `tabelas_compartilhadas` (que `graficos_resultados` ja ignora) e o
    `walk_forward` de cada modelo passa a somar apenas o que foi gasto NELE.
    Consequencia: o `custo_total` medido com tabela compartilhada NAO e
    comparavel ao de uma execucao sem compartilhamento.

    Demais parametros: ver `executar_walk_forward`."""
    especificacoes = list(especificacoes)
    if not especificacoes:
        raise ValueError("executar_walk_forward_multi exige ao menos uma especificacao.")
    rotulos = [e.rotulo_tempos for e in especificacoes]
    if len(set(rotulos)) != len(rotulos):
        raise ValueError(
            f"rotulo_tempos repetido em especificacoes: {rotulos}. Cada modelo "
            f"precisa de um rotulo unico — e a chave da saida e dos registros de tempo.")
    # So ha o que compartilhar quando ha mais de um modelo no grupo.
    compartilhada = len(especificacoes) > 1

    t_inicio = time.time()

    t = time.time()
    if df_features_pronto is not None:
        df_features = df_features_pronto
        for e in especificacoes:
            registrar_tempo(e.rotulo_tempos, "features_base", 0.0, n_linhas=len(df_features),
                            n_skus=int(df_features[COLS.sku].nunique()),
                            detalhe="reaproveitadas (cache)")
    else:
        corte_atributos = config.TREINO_FIM if config.CORTE_ATRIBUTOS_TREINO else None
        df_features = construir_features_base(df_modelagem, corte_atributos=corte_atributos)
        df_features = df_features.dropna(subset=colunas_dropna_historico()).reset_index(drop=True)
        _dt_features = time.time() - t
        for e in especificacoes:
            registrar_tempo(
                e.rotulo_tempos, "features_base", _dt_features,
                n_linhas=len(df_features), n_skus=int(df_features[COLS.sku].nunique()),
                detalhe=None if not compartilhada else "construidas 1x para o grupo",
            )

    teste_inicio_ts = pd.Timestamp(teste_inicio)
    teste_fim_ts = pd.Timestamp(teste_fim)

    # Janelas de 7 dias cobrindo o periodo de teste.
    janelas = []
    cursor = teste_inicio_ts
    while cursor <= teste_fim_ts:
        janelas.append(cursor)
        cursor = cursor + pd.Timedelta(days=7)

    # Reaproveita os hiperparametros ja buscados para este modelo em outro
    # experimento da mesma execucao, para que a comparacao entre cenarios do
    # Exp2 isole o efeito das exogenas em vez do sorteio da busca.
    hiperparametros = {}
    for e in especificacoes:
        hp = None
        if config.TUNAR_UMA_VEZ_GLOBAL and e.nome_modelo in _HIPER_CACHE:
            hp = dict(_HIPER_CACHE[e.nome_modelo])
            registrar_tempo(e.rotulo_tempos, "tuning", 0.0,
                            detalhe=f"reaproveitado do cache global (Item 24): {hp}")
        hiperparametros[e.rotulo_tempos] = hp

    resultados = {r: [] for r in rotulos}
    tempo_proprio = {r: 0.0 for r in rotulos}
    tempo_tabelas = 0.0

    for i, inicio_janela in enumerate(janelas):
        corte = inicio_janela - pd.Timedelta(days=1)

        precisa_refit = (i % cfg.janelas_por_refit == 0) or any(
            hp is None for hp in hiperparametros.values())
        if not precisa_refit:
            continue

        # ---- Parte comum da janela: codificadores + tabela de treino ----
        # Reajustados a cada janela, sobre os dados ate `corte`.
        t_tabela = time.time()
        enc_sku, enc_cat, perfil_sku = _fit_encoders(df_features, corte)

        datas_treino = _preparar_origens_treino(df_features, corte, cfg.stride_origem_treino_dias)
        tabela_treino = _preparar_tabela(df_features, datas_treino, enc_sku, enc_cat, horizonte,
                                         cols_exogenas_alvo=cols_exogenas_alvo, perfil_sku=perfil_sku)
        tempo_tabelas += time.time() - t_tabela
        if tabela_treino.empty:
            continue

        n_skus_janela = int(tabela_treino[COLS.sku].nunique())
        modelos_da_janela = {}

        # ---- Parte por modelo: selecao de colunas, busca, ajuste ----
        for e in especificacoes:
            t_modelo = time.time()
            X_treino = tabela_treino[e.colunas_features]
            y_treino = tabela_treino["y"]
            if compartilhada:
                # Cada modelo leva a SUA matriz. Ver "A COPIA" na docstring.
                X_treino = X_treino.copy()
                y_treino = y_treino.copy()

            hp = hiperparametros[e.rotulo_tempos]
            if hp is None or not cfg.tunar_uma_vez:
                t = time.time()
                hp = ajustar_hiperparametros(X_treino, y_treino, e.nome_modelo)
                registrar_tempo(
                    e.rotulo_tempos, "tuning", time.time() - t, janela=i, data_corte=corte,
                    n_linhas=len(X_treino), n_skus=n_skus_janela, detalhe=str(hp),
                )
                if e.max_estimators_final is not None and "n_estimators" in hp:
                    if hp["n_estimators"] > e.max_estimators_final:
                        hp = {**hp, "n_estimators": e.max_estimators_final}
                if config.TUNAR_UMA_VEZ_GLOBAL:
                    _HIPER_CACHE.setdefault(e.nome_modelo, dict(hp))
                hiperparametros[e.rotulo_tempos] = hp

            modelo = _construir_estimador(e.nome_modelo, hp)
            t = time.time()
            modelo.fit(X_treino, y_treino)
            registrar_tempo(
                e.rotulo_tempos, "fit_final", time.time() - t, janela=i, data_corte=corte,
                n_linhas=len(X_treino), n_skus=n_skus_janela,
            )

            if hasattr(modelo, "feature_importances_"):
                registrar_importancias(e.rotulo_tempos, i, list(e.colunas_features),
                                       modelo.feature_importances_)

            # A predicao e sequencial; liberar os nucleos aqui evita contencao.
            if hasattr(modelo, "n_jobs"):
                modelo.n_jobs = 1

            modelos_da_janela[e.rotulo_tempos] = modelo
            tempo_proprio[e.rotulo_tempos] += time.time() - t_modelo

        # ---- Parte comum: tabela de previsao ----
        # Estes modelos cobrem as janelas ate o proximo refit. As origens de
        # previsao avancam de 7 em 7 dias -- por isso, no teste, cada dia da
        # semana aparece em um unico horizonte (limitacao do desenho de
        # avaliacao).
        n_janelas_a_prever = min(cfg.janelas_por_refit, len(janelas) - i)
        origens_previsao = [corte + pd.Timedelta(days=7 * k) for k in range(n_janelas_a_prever)]
        t_tabela = time.time()
        tabela_previsao = _preparar_tabela(df_features, origens_previsao, enc_sku, enc_cat, horizonte,
                                           cols_exogenas_alvo=cols_exogenas_alvo, perfil_sku=perfil_sku)
        dt_tabela_previsao = time.time() - t_tabela
        if compartilhada:
            tempo_tabelas += dt_tabela_previsao

        for e in especificacoes:
            t = time.time()
            if not tabela_previsao.empty:
                X_previsao = tabela_previsao[e.colunas_features]
                if compartilhada:
                    X_previsao = X_previsao.copy()
                # Demanda nao e negativa; o clip evita previsoes sem sentido fisico.
                y_pred = np.clip(modelos_da_janela[e.rotulo_tempos].predict(X_previsao),
                                 a_min=0, a_max=None)

                saida = tabela_previsao[[COLS.sku]].copy()
                saida[COLS.data] = tabela_previsao["data_origem"] + pd.to_timedelta(
                    tabela_previsao["horizonte"], unit="D")
                saida["y_real"] = tabela_previsao["y"]
                saida["y_pred"] = y_pred
                resultados[e.rotulo_tempos].append(saida)
            # Com um modelo so, o tempo de montar a tabela de previsao continua
            # dentro da etapa 'predict', como nas versoes anteriores.
            dt_predict = time.time() - t + (0.0 if compartilhada else dt_tabela_previsao)
            registrar_tempo(
                e.rotulo_tempos, "predict", dt_predict, janela=i, data_corte=corte,
                n_linhas=len(tabela_previsao), detalhe=f"{n_janelas_a_prever} janela(s)",
            )
            tempo_proprio[e.rotulo_tempos] += dt_predict

    if compartilhada:
        registrar_tempo(
            "CACHE", "tabelas_compartilhadas", tempo_tabelas,
            detalhe=("tabelas de treino/previsao montadas 1x por janela para "
                     + ", ".join(rotulos)),
        )
        for e in especificacoes:
            registrar_tempo(
                e.rotulo_tempos, "walk_forward", tempo_proprio[e.rotulo_tempos],
                detalhe=(f"{len(janelas)} janelas, refit a cada {cfg.janelas_por_refit}; "
                         f"tabela compartilhada — NAO inclui a montagem (linha CACHE)"),
            )
    else:
        registrar_tempo(
            especificacoes[0].rotulo_tempos, "walk_forward", time.time() - t_inicio,
            detalhe=f"{len(janelas)} janelas, refit a cada {cfg.janelas_por_refit}",
        )

    saidas = {}
    for e in especificacoes:
        blocos = resultados[e.rotulo_tempos]
        saidas[e.rotulo_tempos] = (
            pd.concat(blocos, ignore_index=True) if blocos
            else pd.DataFrame(columns=[COLS.sku, COLS.data, "y_real", "y_pred"]))
    return saidas


def executar_walk_forward(
    df_modelagem: pd.DataFrame,
    colunas_features: list,
    nome_modelo: str,
    cfg: ConfigWalkForward,
    teste_inicio: str = config.TESTE_INICIO,
    teste_fim: str = config.TESTE_FIM,
    horizonte: int = config.HORIZONTE_PREVISAO_DIAS,
    max_estimators_final: int | None = None,
    rotulo_tempos: str | None = None,
    cols_exogenas_alvo: list | None = None,
    df_features_pronto: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Roda o walk-forward de UM modelo e devolve as previsoes.

    Saida: uma linha por (SKU, data, y_real, y_pred), onde data = origem + h.

    "Janela expandida" = o treino sempre comeca no inicio da serie e so cresce;
    nao ha janela deslizante que descarte historico antigo.

    `df_features_pronto` reaproveita as features de origem ja construidas por
    construir_cache_features. Sem isso elas seriam recalculadas a cada chamada,
    6 vezes sobre os mesmos dados (2 modelos no Exp1 + 4 no Exp2).

    `max_estimators_final` limita n_estimators no fit final, sem mexer no valor
    escolhido pela busca -- valvula de custo para o RF.

    E um caso particular de `executar_walk_forward_multi`: com um unico modelo
    nao ha tabela compartilhada nem copia, e o comportamento (previsoes, ordem
    e registros de tempo) e o mesmo das versoes anteriores."""
    rotulo = rotulo_tempos or nome_modelo.upper()
    saidas = executar_walk_forward_multi(
        df_modelagem,
        [EspecificacaoModelo(nome_modelo=nome_modelo, colunas_features=colunas_features,
                             rotulo_tempos=rotulo, max_estimators_final=max_estimators_final)],
        cfg,
        teste_inicio=teste_inicio, teste_fim=teste_fim, horizonte=horizonte,
        cols_exogenas_alvo=cols_exogenas_alvo, df_features_pronto=df_features_pronto,
    )
    return saidas[rotulo]


def construir_cache_features(df_modelagem: pd.DataFrame) -> pd.DataFrame:
    """Constroi as features de origem uma unica vez, para todos os modelos e
    experimentos que compartilham o mesmo df_modelagem.

    O dropna remove o aquecimento inicial de cada serie (linhas sem lag
    completo). Passe o resultado como `df_features_pronto`."""
    corte_atributos = config.TREINO_FIM if config.CORTE_ATRIBUTOS_TREINO else None
    t = time.time()
    df_features = construir_features_base(df_modelagem, corte_atributos=corte_atributos)
    df_features = df_features.dropna(subset=colunas_dropna_historico()).reset_index(drop=True)
    registrar_tempo("CACHE", "features_base", time.time() - t,
                    n_linhas=len(df_features), n_skus=int(df_features[COLS.sku].nunique()),
                    detalhe="construidas uma unica vez e compartilhadas")
    return df_features


def relatorio_diagnostico_selecao() -> str:
    """Relatorio de concordancia entre os dois criterios de selecao de
    hiperparametros.

    Se os dois criterios escolhem o mesmo candidato em todas as buscas, a
    circularidade de tunar e avaliar pela mesma metrica e potencial, mas nao
    efetiva nesta execucao."""
    if not _DIAGNOSTICO_SELECAO:
        return "# Item E4 — diagnostico da metrica de selecao\n\n(sem buscas registradas)\n"
    total = len(_DIAGNOSTICO_SELECAO)
    concordam = sum(1 for d in _DIAGNOSTICO_SELECAO if d["criterios_concordam"])
    linhas = [
        "# Item E4 — metrica de selecao dos hiperparametros\n",
        f"Criterio efetivo desta execucao: **{_DIAGNOSTICO_SELECAO[0]['criterio_da_execucao']}**\n",
        f"Buscas realizadas: **{total}** | "
        f"Buscas em que WMAPE e RMSE escolhem o MESMO candidato: "
        f"**{concordam} de {total}** ({concordam / total:.0%})\n",
        "\nAs duas metricas foram calculadas sobre os mesmos candidatos e as "
        "mesmas dobras, dentro de uma unica busca. A tabela abaixo mostra, por "
        "busca, o candidato que cada criterio elegeria.\n",
    ]
    for i, d in enumerate(_DIAGNOSTICO_SELECAO, start=1):
        marca = "IGUAIS" if d["criterios_concordam"] else "DIFERENTES"
        linhas.append(f"\n## Busca {i} — modelo `{d['modelo']}` ({d['n_candidatos']} candidatos) — {marca}\n")
        linhas.append(f"- Escolha por WMAPE: `{d['escolha_wmape']}`")
        linhas.append(f"- Escolha por RMSE:  `{d['escolha_rmse']}`")
    return "\n".join(linhas) + "\n"


def diagnostico_selecao_df() -> "pd.DataFrame":
    """Mesmo conteudo em formato tabular, para consolidar entre execucoes."""
    return pd.DataFrame(_DIAGNOSTICO_SELECAO)

In [ ]:
%%writefile /content/src/evaluate.py
"""
Fase 9 — Metricas, estratificacao e testes de significancia.

O QUE E PRECISO SABER PARA LER OS RESULTADOS

1. Metrica indefinida nao e erro zero. wmape() e smape() devolvem NaN quando a
   demanda total do conjunto avaliado e zero. Um SKU que nao vendeu no teste
   nao "acertou": nao ha o que medir. Como quase metade dos SKUs esta nessa
   situacao, cada modelo pode acabar tendo sua mediana calculada sobre um
   subconjunto diferente -- por isso o pipeline reporta a taxa de indefinicao.

2. Metricas de erro ABSOLUTO (MAE, WMAPE, MASE, sMAPE) tem otimo na mediana
   condicional. Numa serie com ~95% de zeros essa mediana e zero, entao o
   baseline que preve zero sempre vence todas elas. Nao e falha do baseline: e
   propriedade da metrica (Kolassa, 2016).

3. Metricas QUADRATICAS (RMSSE, RMSE_pooled) tem otimo na media condicional,
   que e positiva. Sao as unicas em que prever zero e penalizado, e portanto as
   unicas que discriminam capacidade preditiva nesta base.

4. WMAPE e MASE sao ambos MAE dividido por uma constante por SKU. No nivel de
   SKU eles produzem a MESMA ordenacao relativa entre modelos -- concordancia
   entre os dois nao e evidencia independente.

OPCOES (via config, ligadas na celula de parametros):
  METRICAS_EXTRAS               MAE, RMSSE, MASE pela media
  AVALIAR_SEMANAL                avaliacao no nivel semanal
  TAMANHO_EFEITO                 r bisserial + Holm-Bonferroni
  ESTRATIFICAR_ATIVOS_CESSADOS   recorte ativos x cessados
  BASELINE_ZERO                  controle "prever zero"
  METRICAS_POOLED                WMAPE/MAE agregados globais
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, rankdata

from . import config
from .config import COLS


# ---------------------------------------------------------------------------
# Metricas primarias
# ---------------------------------------------------------------------------
def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Erro percentual absoluto simetrico, em %.

    Descarta os pontos em que real e previsto sao ambos zero (denominador nulo);
    devolve NaN se nao sobrar nenhum ponto."""
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    validos = denom > 0
    if not validos.any():
        return np.nan
    return float(np.mean(np.abs(y_true[validos] - y_pred[validos]) / denom[validos]) * 100)


def wmape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Erro absoluto total dividido pela demanda total, em %.

    NaN quando a demanda total e zero -- indefinido, nao acerto perfeito."""
    soma_real = np.sum(np.abs(y_true))
    if soma_real == 0:
        return np.nan
    return float(np.sum(np.abs(y_true - y_pred)) / soma_real * 100)


def mase(y_true: np.ndarray, y_pred: np.ndarray, y_treino: np.ndarray) -> float | None:
    """MAE do modelo dividido pelo MAE do naive no TREINO.

    MASE < 1 significa "melhor que repetir o valor do dia anterior". O escalador
    vem do treino, entao e imune a demanda do teste -- mas fica contaminado se a
    serie de treino contiver zeros fabricados, e por isso a truncagem
    pre-lancamento importa aqui.

    None quando o naive nao tem variacao no treino (serie constante)."""
    if len(y_treino) < 2:
        return None
    mae_naive_treino = np.mean(np.abs(np.diff(y_treino)))
    if mae_naive_treino == 0:
        return None
    mae_modelo = np.mean(np.abs(y_true - y_pred))
    return float(mae_modelo / mae_naive_treino)


# ---------------------------------------------------------------------------
# Metricas extras (config.METRICAS_EXTRAS)
# ---------------------------------------------------------------------------
def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Erro absoluto medio. Sempre definido -- e a unica metrica utilizavel no
    estrato Cessado."""
    return float(np.mean(np.abs(y_true - y_pred)))


def rmsse(y_true: np.ndarray, y_pred: np.ndarray, y_treino: np.ndarray) -> float | None:
    """Raiz do erro quadratico escalado pela variacao naive no treino.

    Metrica oficial da M5 (Makridakis, Spiliotis e Assimakopoulos, 2022).

    E a metrica mais importante do conjunto para este trabalho: por ser
    quadratica, seu otimo e a MEDIA condicional, nao a mediana. E a unica que
    penaliza prever zero, e portanto a unica em que os modelos podem superar o
    baseline Zero.

    Ressalva ao interpretar: a vantagem da media sobre o zero em MSE e
    exatamente mean(y)^2. Com demanda media diaria na casa de 0,05 unidade essa
    margem e estreita, e o ruido das previsoes pode encobri-la. Leia o RMSSE
    junto com a estratificacao ativos x cessados, nao isolado."""
    if len(y_treino) < 2:
        return None
    mse_naive = np.mean(np.diff(y_treino) ** 2)
    if mse_naive == 0:
        return None
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2) / mse_naive))


def mase_intermitente(y_true: np.ndarray, y_pred: np.ndarray, y_treino: np.ndarray) -> float | None:
    """MASE escalado pela demanda MEDIA do treino, e nao pela variacao naive.

    Kolassa (2016) propoe essa variante porque, em serie intermitente, o naive
    e um comparador fraco: repetir o dia anterior acerta os zeros quase sempre."""
    escala = np.mean(np.abs(y_treino))
    if escala == 0:
        return None
    return float(np.mean(np.abs(y_true - y_pred)) / escala)


# ---------------------------------------------------------------------------
# Calculo de metricas por SKU
# ---------------------------------------------------------------------------
def calcular_metricas_por_sku(
    previsoes: pd.DataFrame,
    df_treino_historico: pd.DataFrame,
    metricas_extras: bool = False,
) -> pd.DataFrame:
    """Uma linha de metricas por SKU.

    `df_treino_historico` fornece o escalador do MASE e do RMSSE. Devolve
    tambem `n_observacoes`, util para conferir se todos os modelos foram
    avaliados exatamente sobre os mesmos pares SKU x data."""
    registros = []
    treino_idx = df_treino_historico.groupby(COLS.sku)[COLS.qtd].apply(lambda s: s.to_numpy())

    for sku, grupo in previsoes.groupby(COLS.sku):
        y_true = grupo["y_real"].to_numpy(dtype=float)
        y_pred = grupo["y_pred"].to_numpy(dtype=float)
        y_treino = treino_idx.get(sku, np.array([]))

        linha = {
            COLS.sku: sku,
            "sMAPE": smape(y_true, y_pred),
            "WMAPE": wmape(y_true, y_pred),
            "MASE": mase(y_true, y_pred, y_treino),
            "n_observacoes": len(y_true),
        }
        if metricas_extras:
            linha["MAE"] = mae(y_true, y_pred)
            linha["RMSSE"] = rmsse(y_true, y_pred, y_treino)
            linha["MASE_media"] = mase_intermitente(y_true, y_pred, y_treino)

        registros.append(linha)

    return pd.DataFrame(registros)


def resumir_medianas(dict_metricas_por_modelo: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Mediana de cada metrica por modelo, mais a taxa de indefinicao.

    Use a mediana e nao a media: as distribuicoes por SKU tem cauda longa e
    algumas metricas divergem em SKU de giro baixissimo.

    As colunas pct_*_indefinido sao parte do resultado, nao um detalhe tecnico:
    se dois modelos tem taxas diferentes, suas medianas nao foram calculadas
    sobre o mesmo conjunto de SKUs e nao sao diretamente comparaveis."""
    linhas = []
    for nome_modelo, df in dict_metricas_por_modelo.items():
        linha = {
            "modelo": nome_modelo,
            "sMAPE_mediana": df["sMAPE"].median(skipna=True),
            "WMAPE_mediana": df["WMAPE"].median(skipna=True),
            "MASE_mediana": df["MASE"].median(skipna=True),
            "pct_MASE_indefinido": df["MASE"].isna().mean() * 100,
            "pct_WMAPE_indefinido": df["WMAPE"].isna().mean() * 100,
            "n_skus": len(df),
        }
        if "MAE" in df.columns:
            linha["MAE_mediana"] = df["MAE"].median(skipna=True)
        if "RMSSE" in df.columns:
            linha["RMSSE_mediana"] = df["RMSSE"].median(skipna=True)
        if "MASE_media" in df.columns:
            linha["MASE_media_mediana"] = df["MASE_media"].median(skipna=True)
        linhas.append(linha)
    return pd.DataFrame(linhas)


# ===========================================================================
# Baseline Zero
# ===========================================================================
def construir_baseline_zero(previsoes_referencia: pd.DataFrame) -> pd.DataFrame:
    """Cria o baseline "prever zero sempre", sobre exatamente os mesmos pares
    SKU x data avaliados nos demais modelos.

    Nao e um modelo proposto: e o CONTROLE, analogo ao classificador de classe
    majoritaria. Se ele empata com o melhor modelo em WMAPE, MASE ou MAE, entao
    "o modelo X venceu" quer dizer "prever zero venceu", e a leitura correta
    passa a ser sobre a inadequacao das metricas pontuais em demanda
    intermitente -- com evidencia gerada nestes dados, em vez de citacao."""
    zero = previsoes_referencia[[COLS.sku, COLS.data, "y_real"]].copy()
    zero["y_pred"] = 0.0
    return zero


# ===========================================================================
# Metricas agregadas globais (pooled)
# ===========================================================================
def metricas_pooled(previsoes: pd.DataFrame) -> dict:
    """Metricas sobre todos os SKUs e dias de uma vez.

        WMAPE_pooled = soma|y - yhat| / soma(y)
        MAE_pooled   = soma|y - yhat| / N
        RMSE_pooled  = sqrt( media (y - yhat)^2 )

    Duas vantagens sobre a mediana por SKU: ficam definidas enquanto algum SKU
    vender, e respondem a pergunta de negocio ("qual o erro total sobre o volume
    total"). O WMAPE_pooled e, alem disso, a definicao original do WMAPE -- a
    versao por SKU e que e a adaptacao.

    RMSE_pooled e quadratico: e onde prever zero deixa de ser vantajoso."""
    y = previsoes["y_real"].to_numpy(dtype=float)
    p = previsoes["y_pred"].to_numpy(dtype=float)
    soma_real = float(np.sum(np.abs(y)))
    erro_abs = float(np.sum(np.abs(y - p)))
    return {
        "WMAPE_pooled": (erro_abs / soma_real * 100) if soma_real > 0 else np.nan,
        "MAE_pooled": float(np.mean(np.abs(y - p))) if len(y) else np.nan,
        "RMSE_pooled": float(np.sqrt(np.mean((y - p) ** 2))) if len(y) else np.nan,
        "n_obs": int(len(y)),
        "demanda_total": soma_real,
    }


def resumir_pooled(previsoes_por_modelo: dict) -> pd.DataFrame:
    """Uma linha de metricas pooled por modelo."""
    linhas = []
    for nome, prev in previsoes_por_modelo.items():
        if prev is None or prev.empty:
            continue
        linha = {"modelo": nome}
        linha.update(metricas_pooled(prev))
        linhas.append(linha)
    return pd.DataFrame(linhas)


# ===========================================================================
# Estratificacao ativos x cessados
# ===========================================================================
ESTRATO_ATIVO = "Ativo"
ESTRATO_CESSADO = "Cessado"


def classificar_estratos_teste(previsoes_referencia: pd.DataFrame) -> pd.DataFrame:
    """Rotula cada SKU como Ativo (vendeu no teste) ou Cessado (nao vendeu).

    E estratificacao de RELATORIO, nao criterio de selecao. Todos os SKUs foram
    modelados; o recorte existe apenas na apresentacao. Nenhuma decisao de
    modelagem -- elegibilidade, features, hiperparametros, janelas -- usou o
    periodo de teste, entao nao ha vazamento.

    Por que separar: com quase metade dos SKUs sem demanda no teste, a mediana
    entre SKUs cai bem na fronteira entre os dois grupos. MAE_mediana e
    RMSSE_mediana passam a medir "quao perto de zero o modelo preve num SKU
    morto", nao "quem preve melhor demanda".

    O que cada estrato responde:
      Ativo   -> qual modelo preve melhor a demanda que de fato ocorreu?
      Cessado -> qual modelo reconheceu a obsolescencia mais rapido? O WMAPE e
                 indefinido, mas MAE e RMSE nao: quanto menor a previsao, melhor.
    """
    total = previsoes_referencia.groupby(COLS.sku)["y_real"].sum()
    return pd.DataFrame({
        COLS.sku: total.index,
        "estrato_teste": np.where(total.to_numpy() > 0, ESTRATO_ATIVO, ESTRATO_CESSADO),
        "demanda_total_teste": total.to_numpy(),
    })


def resumir_por_estrato(
    dict_metricas_por_modelo: dict,
    estratos: pd.DataFrame,
    previsoes_por_modelo: dict | None = None,
    coluna_estrato: str = "estrato_teste",
    ordem: tuple | None = None,
) -> pd.DataFrame:
    """Metricas por (modelo x estrato), com as pooled dentro de cada estrato
    quando `previsoes_por_modelo` e informado.

    Generica: serve ao recorte ativo/cessado e as faixas de volume, bastando
    trocar `coluna_estrato`."""
    if estratos is None or estratos.empty or coluna_estrato not in estratos.columns:
        return pd.DataFrame()
    niveis = list(ordem) if ordem else list(pd.unique(estratos[coluna_estrato].dropna()))

    linhas = []
    for nome, df in dict_metricas_por_modelo.items():
        if df is None or df.empty:
            continue
        m = df.merge(estratos, on=COLS.sku, how="left")
        for estrato in niveis:
            sub = m[m[coluna_estrato] == estrato]
            if sub.empty:
                continue
            linha = {
                "modelo": nome,
                "estrato": estrato,
                "n_skus": len(sub),
                "sMAPE_mediana": sub["sMAPE"].median(skipna=True),
                "WMAPE_mediana": sub["WMAPE"].median(skipna=True),
                "MASE_mediana": sub["MASE"].median(skipna=True),
            }
            for extra in ("MAE", "RMSSE", "MASE_media"):
                if extra in sub.columns:
                    linha[f"{extra}_mediana"] = sub[extra].median(skipna=True)
            if previsoes_por_modelo and nome in previsoes_por_modelo:
                prev = previsoes_por_modelo[nome]
                skus_estrato = set(sub[COLS.sku])
                prev_sub = prev[prev[COLS.sku].isin(skus_estrato)]
                if not prev_sub.empty:
                    pooled = metricas_pooled(prev_sub)
                    linha["WMAPE_pooled"] = pooled["WMAPE_pooled"]
                    linha["MAE_pooled"] = pooled["MAE_pooled"]
                    linha["RMSE_pooled"] = pooled["RMSE_pooled"]
                    linha["demanda_total_teste"] = pooled["demanda_total"]
            linhas.append(linha)
    if not linhas:
        return pd.DataFrame()
    saida = pd.DataFrame(linhas)
    saida["_ordem"] = saida["estrato"].map({n: i for i, n in enumerate(niveis)})
    return saida.sort_values(["_ordem", "modelo"]).drop(columns="_ordem").reset_index(drop=True)


def filtrar_metricas_por_estrato(dict_metricas_por_modelo: dict, estratos: pd.DataFrame,
                                 estrato: str, coluna_estrato: str = "estrato_teste") -> dict:
    """Recorta o dicionario de metricas para um unico estrato, preservando o
    pareamento por SKU exigido pelos testes de Wilcoxon."""
    skus = set(estratos.loc[estratos[coluna_estrato] == estrato, COLS.sku])
    saida = {}
    for nome, df in dict_metricas_por_modelo.items():
        if df is None or df.empty:
            continue
        sub = df[df[COLS.sku].isin(skus)]
        if not sub.empty:
            saida[nome] = sub
    return saida


# ===========================================================================
# Estratificacao por VOLUME DE PEDIDOS (curva ABC / Pareto)
# ===========================================================================
# O valor economico da previsao nao esta distribuido uniformemente entre os
# SKUs: um erro num SKU que concentra milhares de pedidos afeta muito mais
# clientes finais do que o mesmo erro percentual num SKU de cauda que vendeu 3
# vezes no ano. A mediana entre SKUs trata os dois igualmente, o que responde
# a pergunta de desempenho geral mas nao a gerencial.
#
# O recorte responde: nos 20% de SKUs com mais pedidos, qual modelo preve
# melhor, e quanto da demanda total esses 20% concentram.
#
# O volume e medido no periodo de TREINO -- e a informacao que um gestor teria
# em maos ao decidir onde investir, e nao ha vazamento.

# Fontes de volume, em ordem de preferencia. PedidosUnicos conta pedidos
# DISTINTOS (clientes atendidos), e nao unidades vendidas.
_FONTES_VOLUME = [
    ("PedidosUnicos", "soma de pedidos distintos no treino (Qtde_PedidosUnicos)"),
    ("__dias_com_venda__", "numero de dias com venda no treino (proxy — coluna PedidosUnicos ausente)"),
]


def calcular_volume_por_sku(df_treino_historico: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    """Volume de cada SKU no treino, com fallback quando falta PedidosUnicos.

    Devolve (df[sku, volume_pedidos, demanda_treino], descricao_da_fonte). A
    descricao vai para o log e para o relatorio: qual fonte foi usada muda a
    interpretacao da curva ABC e precisa ser declarada."""
    fonte = None
    if "PedidosUnicos" in df_treino_historico.columns:
        vol = (df_treino_historico.groupby(COLS.sku)["PedidosUnicos"]
               .sum(min_count=1).fillna(0.0).rename("volume_pedidos"))
        if float(vol.sum()) > 0:
            fonte = _FONTES_VOLUME[0][1]
    if fonte is None:
        vol = (df_treino_historico.assign(_tem=(df_treino_historico[COLS.qtd] > 0).astype(int))
               .groupby(COLS.sku)["_tem"].sum().rename("volume_pedidos"))
        fonte = _FONTES_VOLUME[1][1]

    dem = df_treino_historico.groupby(COLS.sku)[COLS.qtd].sum().rename("demanda_treino")
    saida = pd.concat([vol, dem], axis=1).reset_index()
    return saida, fonte


def classificar_estratos_volume(
    df_treino_historico: pd.DataFrame,
    faixas: list | None = None,
) -> tuple[pd.DataFrame, str]:
    """Classifica os SKUs em faixas ABC pelo volume de pedidos no treino.

    A = os 20% com mais pedidos, B = os 30% seguintes, C = a cauda.

    Acrescenta `pct_volume_acumulado`, que permite a leitura de Pareto: "os 20%
    do topo concentram X% dos pedidos"."""
    faixas = faixas or config.FAIXAS_VOLUME_PEDIDOS
    base, fonte = calcular_volume_por_sku(df_treino_historico)
    if base.empty:
        return pd.DataFrame(), fonte

    base = base.sort_values("volume_pedidos", ascending=False).reset_index(drop=True)
    n = len(base)
    total_vol = float(base["volume_pedidos"].sum())

    base["posicao"] = np.arange(1, n + 1)
    base["rank_pct"] = (base["posicao"] - 1) / n          # 0 = maior volume
    base["pct_volume_acumulado"] = (
        base["volume_pedidos"].cumsum() / total_vol * 100 if total_vol > 0 else np.nan
    )

    rotulos = []
    for r in base["rank_pct"]:
        rotulo = faixas[-1][2]
        for lim_inf, lim_sup, nome in faixas:
            if lim_inf <= r < lim_sup:
                rotulo = nome
                break
        rotulos.append(rotulo)
    base["estrato_volume"] = rotulos
    return base, fonte


def resumir_concentracao_volume(estratos_volume: pd.DataFrame) -> pd.DataFrame:
    """Tabela de Pareto: quantos SKUs, quanto do volume de pedidos e quanto da
    demanda cada faixa concentra.

    E o argumento de negocio da secao de resultados -- mostra se o esforco de
    previsao pode ser concentrado em poucos SKUs."""
    if estratos_volume is None or estratos_volume.empty:
        return pd.DataFrame()
    total_vol = float(estratos_volume["volume_pedidos"].sum())
    total_dem = float(estratos_volume["demanda_treino"].sum())
    linhas = []
    for nome, g in estratos_volume.groupby("estrato_volume", sort=False):
        linhas.append({
            "estrato": nome,
            "n_skus": len(g),
            "pct_skus": len(g) / len(estratos_volume) * 100,
            "volume_pedidos": float(g["volume_pedidos"].sum()),
            "pct_volume_pedidos": float(g["volume_pedidos"].sum()) / total_vol * 100 if total_vol else np.nan,
            "demanda_treino": float(g["demanda_treino"].sum()),
            "pct_demanda_treino": float(g["demanda_treino"].sum()) / total_dem * 100 if total_dem else np.nan,
        })
    ordem = [f[2] for f in config.FAIXAS_VOLUME_PEDIDOS]
    saida = pd.DataFrame(linhas)
    saida["_o"] = saida["estrato"].map({n: i for i, n in enumerate(ordem)})
    return saida.sort_values("_o").drop(columns="_o").reset_index(drop=True)


def cruzar_estratos(estratos_teste: pd.DataFrame, estratos_volume: pd.DataFrame) -> pd.DataFrame:
    """Tabela cruzada Ativo/Cessado x faixa de volume.

    Mostra se a obsolescencia se concentra na cauda, como se espera, ou tambem
    atinge SKUs de alto giro -- o segundo caso seria um achado relevante."""
    if (estratos_teste is None or estratos_teste.empty
            or estratos_volume is None or estratos_volume.empty):
        return pd.DataFrame()
    j = estratos_teste.merge(estratos_volume[[COLS.sku, "estrato_volume"]], on=COLS.sku, how="inner")
    tabela = (j.groupby(["estrato_volume", "estrato_teste"], sort=False)[COLS.sku]
              .count().rename("n_skus").reset_index())
    total = tabela.groupby("estrato_volume")["n_skus"].transform("sum")
    tabela["pct_no_estrato_volume"] = tabela["n_skus"] / total * 100
    ordem = [f[2] for f in config.FAIXAS_VOLUME_PEDIDOS]
    tabela["_o"] = tabela["estrato_volume"].map({n: i for i, n in enumerate(ordem)})
    return tabela.sort_values(["_o", "estrato_teste"]).drop(columns="_o").reset_index(drop=True)


def skus_top_volume(estratos_volume: pd.DataFrame, n: int = 6) -> list:
    """Os n SKUs de maior volume de pedidos, para as figuras real x previsto."""
    if estratos_volume is None or estratos_volume.empty:
        return []
    return estratos_volume.nlargest(n, "volume_pedidos")[COLS.sku].tolist()


def calcular_esparsidade_treino(df_treino: pd.DataFrame) -> pd.DataFrame:
    """Proporcao de dias sem venda por SKU no treino."""
    return (
        df_treino.groupby(COLS.sku)[COLS.qtd]
        .apply(lambda s: (s == 0).mean())
        .rename("pct_zeros")
        .reset_index()
    )


def estratificar_por_esparsidade(
    dict_metricas_por_modelo: dict[str, pd.DataFrame],
    esparsidade: pd.DataFrame,
) -> pd.DataFrame:
    """Metricas por faixa de esparsidade.

    Responde se a vantagem de um modelo depende do grau de intermitencia da
    serie. As faixas vem de config.FAIXAS_ESPARSIDADE."""
    linhas = []
    for nome_modelo, df in dict_metricas_por_modelo.items():
        df_merge = df.merge(esparsidade, on=COLS.sku, how="left")
        for lim_inf, lim_sup, rotulo in config.FAIXAS_ESPARSIDADE:
            faixa = df_merge[(df_merge["pct_zeros"] >= lim_inf) & (df_merge["pct_zeros"] < lim_sup)]
            if faixa.empty:
                continue
            linhas.append({
                "modelo": nome_modelo,
                "faixa_esparsidade": rotulo,
                "sMAPE_mediana": faixa["sMAPE"].median(skipna=True),
                "WMAPE_mediana": faixa["WMAPE"].median(skipna=True),
                "MASE_mediana": faixa["MASE"].median(skipna=True),
                "n_skus": len(faixa),
            })
    return pd.DataFrame(linhas)


# ---------------------------------------------------------------------------
# Avaliacao no nivel semanal (config.AVALIAR_SEMANAL)
# ---------------------------------------------------------------------------
def avaliar_nivel_semanal(
    previsoes_por_modelo: dict,
    df_treino_historico: pd.DataFrame,
    outputs_dir=None,
) -> pd.DataFrame:
    """Reavalia os modelos no total semanal, agregando as previsoes ja feitas.

    Nao reexecuta modelo nenhum: soma as previsoes diarias por semana W-MON.

    E a comparacao mais justa entre os quatro modelos, e vale considerar
    reporta-la como principal. SBA e TSB preveem semanalmente e sao
    desagregados para o dia por um perfil fixo de dia da semana; a avaliacao
    diaria lhes cobra o erro dessa conversao, que nao e do metodo. O nivel
    semanal tambem corresponde ao agregado do lead time de 7 dias, que e o que a
    reposicao de estoque de fato precisa."""
    metricas_semanal: dict = {}
    treino_sem = df_treino_historico.copy()
    treino_sem["_semana"] = treino_sem[COLS.data].dt.to_period("W-MON")
    treino_semanal = treino_sem.groupby([COLS.sku, "_semana"])[COLS.qtd].sum().reset_index()
    treino_idx = treino_semanal.groupby(COLS.sku)[COLS.qtd].apply(lambda s: s.to_numpy())

    for nome, prev in previsoes_por_modelo.items():
        if prev is None or prev.empty:
            continue
        agg = prev.copy()
        agg["_semana"] = agg[COLS.data].dt.to_period("W-MON")
        agg_sem = agg.groupby([COLS.sku, "_semana"]).agg(
            y_real=("y_real", "sum"), y_pred=("y_pred", "sum")
        ).reset_index()

        registros = []
        for sku, grupo in agg_sem.groupby(COLS.sku):
            y_t = grupo["y_real"].to_numpy(dtype=float)
            y_p = grupo["y_pred"].to_numpy(dtype=float)
            y_tr = treino_idx.get(sku, np.array([]))
            registros.append({
                COLS.sku: sku,
                "sMAPE_sem": smape(y_t, y_p),
                "WMAPE_sem": wmape(y_t, y_p),
                "MASE_sem": mase(y_t, y_p, y_tr),
                "MAE_sem": mae(y_t, y_p),
                "n_semanas": len(y_t),
            })
        metricas_semanal[nome] = pd.DataFrame(registros)

    linhas = []
    for nome, df in metricas_semanal.items():
        linhas.append({
            "modelo": nome,
            "sMAPE_sem_mediana": df["sMAPE_sem"].median(skipna=True),
            "WMAPE_sem_mediana": df["WMAPE_sem"].median(skipna=True),
            "MASE_sem_mediana": df["MASE_sem"].median(skipna=True),
            "MAE_sem_mediana": df["MAE_sem"].median(skipna=True),
            "pct_WMAPE_sem_indefinido": df["WMAPE_sem"].isna().mean() * 100,
            "n_skus": len(df),
        })
    resumo = pd.DataFrame(linhas)
    if outputs_dir is not None:
        resumo.to_csv(outputs_dir / "resumo_semanal_agregado.csv", index=False)
    return resumo


# ---------------------------------------------------------------------------
# Tamanho de efeito e correcao de multiplas comparacoes
# ---------------------------------------------------------------------------
def r_bisserial_pareado(a: np.ndarray, b: np.ndarray) -> float | None:
    """Tamanho de efeito do Wilcoxon pareado, entre -1 e +1.

    O p-valor diz se ha diferenca; o r diz o quanto ela e sistematica.

    CONVENCAO DE SINAL -- vale para todos os relatorios deste pipeline. O sinal
    segue d = a - b sobre uma metrica de ERRO:
      r > 0  ->  `a` erra mais  ->  o SEGUNDO modelo do par e o melhor
      r < 0  ->  `a` erra menos ->  o PRIMEIRO modelo do par e o melhor
    +1,0 = `b` melhor em todos os pares; -1,0 = `a` melhor em todos.

    Pares empatados sao descartados, como no proprio teste."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    d = a - b
    d = d[~np.isnan(d) & (d != 0)]
    if len(d) == 0:
        return None
    postos = rankdata(np.abs(d))
    soma_pos = float(postos[d > 0].sum())
    soma_neg = float(postos[d < 0].sum())
    total = float(postos.sum())
    return (soma_pos - soma_neg) / total if total > 0 else None


def aplicar_holm_bonferroni(p_valores: list) -> list:
    """Ajusta os p-valores para multiplas comparacoes (metodo de Holm).

    Necessario porque cada relatorio faz varias comparacoes sobre os mesmos
    dados: sem correcao, algum par apareceria como significativo so por acaso.
    Holm e menos conservador que Bonferroni e nao exige independencia.

    Preserva a ordem da lista de entrada; posicoes None continuam None."""
    nulos = [i for i, p in enumerate(p_valores) if p is None]
    validos = [(i, p) for i, p in enumerate(p_valores) if p is not None]
    m = len(validos)
    if m == 0:
        return p_valores

    ordenados = sorted(validos, key=lambda x: x[1])
    p_adj = [None] * len(p_valores)
    max_adj = 0.0
    for rank, (orig_idx, p) in enumerate(ordenados):
        adj = p * (m - rank)
        # Monotonicidade: um p ajustado nunca pode ficar abaixo do anterior.
        adj = min(max(float(adj), max_adj), 1.0)
        max_adj = adj
        p_adj[orig_idx] = adj
    for i in nulos:
        p_adj[i] = None
    return p_adj


# ---------------------------------------------------------------------------
# Testes de Wilcoxon
# ---------------------------------------------------------------------------
def teste_wilcoxon_pareado(
    metricas_a: pd.DataFrame, metricas_b: pd.DataFrame, coluna: str = "WMAPE"
) -> dict:
    """Wilcoxon signed-rank sobre a diferenca de erro POR SKU.

    Pareado por SKU: cada SKU contribui com um par (erro do modelo A, erro do
    modelo B), o que controla a heterogeneidade enorme entre produtos. Nao
    parametrico, porque as distribuicoes de erro sao assimetricas.

    SKUs com metrica indefinida em qualquer dos dois modelos saem do teste. Com
    menos de 10 pares o resultado nao e reportado."""
    if coluna not in metricas_a.columns or coluna not in metricas_b.columns:
        return {"n_pares": 0, "estatistica": None, "p_valor": None,
                "significativo": None, "r_bisserial": None}

    merged = metricas_a[[COLS.sku, coluna]].merge(
        metricas_b[[COLS.sku, coluna]], on=COLS.sku, suffixes=("_a", "_b")
    ).dropna()

    if len(merged) < 10:
        return {"n_pares": len(merged), "estatistica": None, "p_valor": None,
                "significativo": None, "r_bisserial": None}

    diffs = merged[f"{coluna}_a"] - merged[f"{coluna}_b"]
    if (diffs == 0).all():
        return {"n_pares": len(merged), "estatistica": None, "p_valor": 1.0,
                "significativo": False, "r_bisserial": 0.0}

    estatistica, p_valor = wilcoxon(merged[f"{coluna}_a"], merged[f"{coluna}_b"])
    r_bs = r_bisserial_pareado(
        merged[f"{coluna}_a"].to_numpy(), merged[f"{coluna}_b"].to_numpy()
    )
    return {
        "n_pares": len(merged),
        "estatistica": float(estatistica),
        "p_valor": float(p_valor),
        "significativo": bool(p_valor < 0.05),
        "r_bisserial": r_bs,
    }


# Comparacoes entre os modelos concorrentes.
# As dez combinacoes par a par entre os cinco competidores.
# SBA vs TSB fecha a ordenacao: sao os dois metodos estatisticos especializados,
# e sem esse par os dois extremos do ranking reportado ficam sem teste.
PARES_WILCOXON = [
    ("RF", "XGBoost"), ("RF", "SBA"), ("RF", "TSB"),
    ("XGBoost", "SBA"), ("XGBoost", "TSB"),
    ("SBA", "TSB"),
]

# Comparacoes contra o controle. So entram quando o baseline Zero
# esta ativo -- sao as que dizem se a metrica discrimina alguma coisa.
PARES_WILCOXON_ZERO = [
    ("TSB", "Zero"), ("SBA", "Zero"), ("XGBoost", "Zero"), ("RF", "Zero"),
]


def _pares_disponiveis(dict_metricas_por_modelo: dict) -> list:
    """Pares testaveis com os modelos presentes neste dicionario."""
    pares = list(PARES_WILCOXON)
    if "Zero" in dict_metricas_por_modelo:
        pares = pares + list(PARES_WILCOXON_ZERO)
    return [(a, b) for a, b in pares
            if a in dict_metricas_por_modelo and b in dict_metricas_por_modelo]


def gerar_relatorio_wilcoxon(
    dict_metricas_por_modelo: dict[str, pd.DataFrame],
    coluna: str = "WMAPE",
    aplicar_correcao_multipla: bool = False,
    calcular_efeito: bool = False,
) -> str:
    """Bloco em markdown com todos os pares testados para UMA metrica."""
    pares_presentes = _pares_disponiveis(dict_metricas_por_modelo)
    resultados = {
        (a, b): teste_wilcoxon_pareado(dict_metricas_por_modelo[a], dict_metricas_por_modelo[b], coluna)
        for a, b in pares_presentes
    }

    if aplicar_correcao_multipla and resultados:
        p_vals = [resultados[(a, b)]["p_valor"] for a, b in pares_presentes]
        p_adj = aplicar_holm_bonferroni(p_vals)
        for (a, b), p in zip(pares_presentes, p_adj):
            resultados[(a, b)]["p_valor_adj"] = p
            if p is not None:
                resultados[(a, b)]["significativo_adj"] = p < 0.05

    linhas = [f"### Metrica: {coluna}"]
    if aplicar_correcao_multipla:
        linhas.append(f"_(correcao de Holm-Bonferroni aplicada — {len(pares_presentes)} comparacoes)_")
    linhas.append("")
    linhas.append("_Leitura do r bisserial: r > 0 = o SEGUNDO modelo do par e o melhor; "
                  "r < 0 = o PRIMEIRO e o melhor (a metrica e de erro)._")
    linhas.append("")
    for a, b in pares_presentes:
        r = resultados[(a, b)]
        linhas.append(_formatar_linha_wilcoxon(a, b, r, coluna, calcular_efeito, aplicar_correcao_multipla))
    return "\n".join(linhas)


# Metricas testadas por padrao. O pipeline acrescenta RMSSE e MAE quando
# METRICAS_EXTRAS esta ligado -- e importante que acrescente: WMAPE e MASE sao
# ambos MAE escalado, entao sozinhas elas nao testam nada que o Zero nao vence.
METRICAS_WILCOXON = ("WMAPE", "MASE")


def gerar_relatorio_wilcoxon_metricas(
    dict_metricas_por_modelo: dict[str, pd.DataFrame],
    colunas: tuple = METRICAS_WILCOXON,
    aplicar_correcao_multipla: bool = False,
    calcular_efeito: bool = False,
    titulo: str | None = None,
) -> str:
    """Relatorio de Wilcoxon para varias metricas, uma secao por metrica."""
    cabecalho = titulo or "## Testes de significancia — Wilcoxon signed-rank (alfa = 0,05), Secao 4.2.7"
    partes = [cabecalho + "\n"]
    if any("Zero" == m for m in dict_metricas_por_modelo):
        partes.append(
            "_O baseline **Zero** (Item 17) preve 0 sempre. Nao e um modelo proposto, e o "
            "controle: se um modelo nao vence o Zero, a metrica em questao nao discrimina "
            "capacidade preditiva nesta base._\n"
        )
    for col in colunas:
        partes.append(gerar_relatorio_wilcoxon(
            dict_metricas_por_modelo, coluna=col,
            aplicar_correcao_multipla=aplicar_correcao_multipla,
            calcular_efeito=calcular_efeito,
        ))
    return "\n".join(partes)


def gerar_relatorio_wilcoxon_por_estrato(
    dict_metricas_por_modelo: dict,
    estratos: pd.DataFrame,
    colunas: tuple = ("WMAPE", "MASE", "MAE"),
    aplicar_correcao_multipla: bool = False,
    calcular_efeito: bool = False,
    coluna_estrato: str = "estrato_teste",
    ordem: tuple | None = None,
    titulo: str | None = None,
    nota: str | None = None,
) -> str:
    """Wilcoxon dentro de cada estrato.

    No estrato Cessado o WMAPE e indefinido para todos os SKUs, entao a
    comparacao util ali e por MAE: menor MAE = reconheceu a obsolescencia mais
    rapido."""
    partes = [titulo or "# Testes de Wilcoxon por estrato (Item 16)\n"]
    if nota:
        partes.append(nota + "\n")
    else:
        partes.append("**Ativo** = SKU com pelo menos uma venda no teste; "
                      "**Cessado** = SKU sem nenhuma venda no teste.\n")
    partes.append("Estratificacao de RELATORIO — todos os SKUs foram modelados; o recorte e "
                  "feito apenas na apresentacao, sem qualquer decisao de modelagem baseada "
                  "no periodo de teste.\n")

    if ordem is None:
        ordem = (ESTRATO_ATIVO, ESTRATO_CESSADO)
    for estrato in ordem:
        sub = filtrar_metricas_por_estrato(dict_metricas_por_modelo, estratos, estrato,
                                           coluna_estrato=coluna_estrato)
        if not sub:
            continue
        n = len(next(iter(sub.values())))
        partes.append(f"\n## Estrato: {estrato} (n = {n} SKUs)\n")
        if estrato == ESTRATO_CESSADO:
            partes.append("_WMAPE e indefinido neste estrato (demanda total = 0). "
                          "A comparacao relevante e por **MAE**: menor = previu mais proximo "
                          "de zero, ou seja, reconheceu a obsolescencia mais rapido._\n")
        partes.append(gerar_relatorio_wilcoxon_metricas(
            sub, colunas=colunas,
            aplicar_correcao_multipla=aplicar_correcao_multipla,
            calcular_efeito=calcular_efeito,
            titulo=f"### Comparacoes — estrato {estrato}",
        ))
    return "\n".join(partes)


def _formatar_linha_wilcoxon(
    a: str, b: str, r: dict, coluna: str,
    calcular_efeito: bool = False,
    com_correcao: bool = False,
) -> str:
    """Formata o resultado de um par como uma linha de markdown.

    O rotulo [melhor: X] so aparece quando o resultado e significativo -- sem
    significancia, o sinal do r nao sustenta a afirmacao.

    Quando a correcao de Holm esta ligada, reporta SEMPRE o p-valor bruto ao
    lado do ajustado: o procedimento de Holm e sequencial e impoe
    monotonicidade sobre os p-valores ajustados -- um p ajustado nunca pode
    ficar abaixo do da
    comparacao anterior na mesma familia (ver aplicar_holm_bonferroni). Isso
    pode fazer comparacoes distintas dentro da mesma familia saturarem no
    mesmo teto, produzindo p-valores ajustados identicos -- coincidencia
    esperada do metodo, nao erro de calculo. Reportar so o ajustado escondia
    essa mecanica."""
    tem_p_adj = com_correcao and r.get("p_valor_adj") is not None
    sufixo_bruto = f" | p bruto={r['p_valor']:.4f}" if tem_p_adj else ""
    if r["p_valor"] is None:
        return f"- **{a} vs {b}** ({coluna}): pares insuficientes (n={r['n_pares']}) para o teste.\n"
    if r["estatistica"] is None:
        p_mostrado = r["p_valor_adj"] if tem_p_adj else r["p_valor"]
        sufixo_adj = " (p ajustado Holm)" if tem_p_adj else ""
        return (
            f"- **{a} vs {b}** ({coluna}, n={r['n_pares']}): diferencas identicas em "
            f"todos os pares (estatistica indefinida), p-valor={p_mostrado:.4f}{sufixo_adj}{sufixo_bruto} "
            f"→ nao significativo a 5%.\n"
        )
    p_reportado = r["p_valor_adj"] if tem_p_adj else r["p_valor"]
    sig_reportado = r.get("significativo_adj", r["significativo"]) if com_correcao else r["significativo"]
    significancia = "SIGNIFICATIVO" if sig_reportado else "nao significativo"
    sufixo_p = " (p ajustado Holm)" if tem_p_adj else ""
    efeito = ""
    vencedor = ""
    if r.get("r_bisserial") is not None:
        if calcular_efeito:
            efeito = f", r bisserial={r['r_bisserial']:.3f}"
        if sig_reportado:
            vencedor = f" [melhor: {b if r['r_bisserial'] > 0 else a}]"
    return (
        f"- **{a} vs {b}** ({coluna}, n={r['n_pares']}): "
        f"estatistica={r['estatistica']:.2f}, p-valor={p_reportado:.4f}{sufixo_p}{sufixo_bruto}{efeito} "
        f"→ {significancia} a 5%.{vencedor}\n"
    )


In [ ]:
%%writefile /content/src/historico.py
"""
Historico de execucoes — mantem um CSV/MD "mestre" (outputs/historico_execucoes.*)
com uma linha-resumo por (execucao x modelo), para permitir comparar a
evolucao dos resultados entre rodadas sucessivas do pipeline sem precisar
abrir cada pasta em outputs/execucoes/ individualmente.
"""
from __future__ import annotations

import pandas as pd

from . import config


def registrar_execucao(
    run_id: str,
    metadata: dict,
    resumo_exp1: pd.DataFrame | None,
    resumo_exp2: pd.DataFrame | None,
) -> None:
    """Acrescenta ao historico global (outputs/historico_execucoes.csv) uma
    linha por modelo/cenario desta execucao, com as medianas de sMAPE/WMAPE/
    MASE e os principais parametros de escopo usados (fonte dos dados, modo,
    n_iter, cv_splits, quantidade de SKUs). Nao apaga nem sobrescreve
    execucoes anteriores — apenas concatena."""
    linhas = []

    def _linhas_de(resumo: pd.DataFrame | None, experimento: str):
        if resumo is None or resumo.empty:
            return
        for _, row in resumo.iterrows():
            linhas.append({
                "run_id": run_id,
                "timestamp": metadata.get("timestamp"),
                "experimento": experimento,
                "modelo": row["modelo"],
                "sMAPE_mediana": row["sMAPE_mediana"],
                "WMAPE_mediana": row["WMAPE_mediana"],
                "MASE_mediana": row["MASE_mediana"],
                "pct_MASE_indefinido": row.get("pct_MASE_indefinido"),
                "n_skus": row["n_skus"],
                "fonte_dados": metadata.get("fonte_dados"),
                "modo": metadata.get("modo"),
                "n_iter_busca": metadata.get("n_iter_busca"),
                # Sem estes dois, duas execucoes do historico que buscaram
                # hiperparametros sobre amostras diferentes, ou que mediram o
                # custo de formas diferentes, ficam lado a lado sem aviso.
                "max_linhas_tuning": metadata.get("max_linhas_tuning"),
                "reaproveitar_tabela_treino": metadata.get("reaproveitar_tabela_treino"),
                "cv_splits": metadata.get("cv_splits"),
                "amostragem_skus": metadata.get("amostragem_skus"),
                "duracao_s": metadata.get("duracao_s"),
            })

    _linhas_de(resumo_exp1, "Experimento1")
    _linhas_de(resumo_exp2, "Experimento2")

    if not linhas:
        return

    df_novo = pd.DataFrame(linhas)
    if config.HISTORICO_CSV.exists():
        df_existente = pd.read_csv(config.HISTORICO_CSV)
        df_final = pd.concat([df_existente, df_novo], ignore_index=True)
    else:
        df_final = df_novo

    df_final.to_csv(config.HISTORICO_CSV, index=False)
    _atualizar_historico_md(df_final)


def _atualizar_historico_md(df_historico: pd.DataFrame) -> None:
    """Regera um resumo em markdown (mais facil de ler que o CSV bruto),
    com a evolucao do WMAPE/MASE mediano por modelo ao longo das execucoes."""
    linhas = ["# Historico de execucoes do pipeline\n",
              "Uma linha por (execucao x modelo x experimento). Ordenado da "
              "execucao mais recente para a mais antiga.\n"]
    cols = ["run_id", "experimento", "modelo", "sMAPE_mediana", "WMAPE_mediana",
            "MASE_mediana", "n_skus", "fonte_dados", "modo", "amostragem_skus"]
    tabela = df_historico[cols].sort_values("run_id", ascending=False)
    linhas.append(_tabela_markdown(tabela))
    config.OUTPUTS_DIR.joinpath("historico_execucoes.md").write_text(
        "\n".join(linhas) + "\n", encoding="utf-8"
    )


def _tabela_markdown(df: pd.DataFrame) -> str:
    """Formata um DataFrame como tabela markdown sem depender do pacote
    opcional `tabulate` (usado por DataFrame.to_markdown)."""
    def _fmt(v):
        if isinstance(v, float):
            return f"{v:.2f}"
        return "" if pd.isna(v) else str(v)

    cabecalho = "| " + " | ".join(df.columns) + " |"
    separador = "| " + " | ".join(["---"] * len(df.columns)) + " |"
    linhas_dados = [
        "| " + " | ".join(_fmt(v) for v in row) + " |"
        for row in df.itertuples(index=False)
    ]
    return "\n".join([cabecalho, separador] + linhas_dados)


In [ ]:
%%writefile /content/src/correlacao.py
"""
Fase 5b — Correlacao entre as features e o alvo.

Serve a dois propositos: sustentar empiricamente a escolha de variaveis, e
detectar redundancia entre lags e estatisticas moveis. O
`correlacao_features.md` que este modulo gera e a evidencia por tras da
remocao de features redundantes -- por isso as features removidas continuam
sendo calculadas e aparecendo aqui.

TRES DECISOES QUE AFETAM A LEITURA DOS COEFICIENTES

1. A correlacao usa a MESMA tabela longa que alimenta os modelos -- uma linha
   por (SKU, data de origem, horizonte h), features fixadas na origem, alvo na
   data (origem + h) -- e nao a serie diaria bruta. Assim os coeficientes
   descrevem a relacao que RF e XGBoost de fato enxergam no treino.

2. Tudo e restrito ao periodo de TREINO, com o corte aplicado ANTES de
   construir as features, de modo que nenhuma linha venha do teste, nem como
   feature nem como alvo. A analise e descritiva e nao alimenta modelo algum,
   mas manter o corte evita qualquer leitura de vazamento.

3. As origens sao amostradas de N em N dias (CORRELACAO_STRIDE_ORIGENS_DIAS),
   porque reconstruir a tabela para todas as origens diarias custaria como um
   refit. O passo NAO pode ser multiplo de 7: isso poria todas as origens no
   mesmo dia da semana e fabricaria correlacao perfeita entre horizonte e
   dia_semana_alvo. O relatorio avisa quando isso acontece.

Os coeficientes saem separados por quadrante ADI/CV2 (Intermitente e Lumpy),
porque a estrutura de autocorrelacao dos dois grupos e diferente -- o que ajuda
a explicar por que RF e XGBoost se comportam de forma distinta em cada um.
"""
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config
from .config import COLS

config.configurar_matplotlib_pt_br()
from .features import (
    colunas_dropna_historico,
    features_experimento1,
    LabelEncoderJanela,
    construir_features_base,
    montar_tabela_horizontes,
)

TARGET_CORRELACAO = "y"


def calcular_matriz_correlacao(
    df_features: pd.DataFrame,
    colunas: list,
    target: str = TARGET_CORRELACAO,
) -> pd.DataFrame:
    """Calcula a correlacao de Pearson entre as features numericas e o
    target, usando apenas as colunas presentes no DataFrame recebido.

    O recorte temporal (restricao ao treino) e responsabilidade de quem
    monta `df_features` — ver `montar_tabela_correlacao`."""
    colunas_presentes = [c for c in list(colunas) + [target] if c in df_features.columns]
    numericas = [
        c for c in colunas_presentes
        if pd.api.types.is_numeric_dtype(df_features[c])
    ]
    return df_features[numericas].corr(method="pearson")


def montar_tabela_correlacao(
    df_modelagem: pd.DataFrame,
    treino_fim: str | None = None,
    stride_origens: int | None = None,
    horizonte: int | None = None,
) -> pd.DataFrame:
    """Reconstroi a tabela longa (SKU x origem x horizonte) usada pelos
    modelos, restrita ao periodo de treino, para servir de base a analise de
    correlacao.

    `stride_origens` controla o passo entre as datas de origem consideradas
    (default config.CORRELACAO_STRIDE_ORIGENS_DIAS): reconstruir a tabela
    para todas as origens diarias do treino seria tao caro quanto um refit do
    walk-forward, sem alterar de forma relevante os coeficientes (origens
    consecutivas compartilham quase todo o historico)."""
    treino_fim = treino_fim or config.TREINO_FIM
    stride_origens = stride_origens or config.CORRELACAO_STRIDE_ORIGENS_DIAS
    horizonte = horizonte or config.HORIZONTE_PREVISAO_DIAS

    # Corte no treino ANTES de construir as features: garante que nem as
    # features nem os alvos y usem qualquer linha do periodo de teste.
    df_treino = df_modelagem[df_modelagem[COLS.data] <= pd.Timestamp(treino_fim)].copy()
    if df_treino.empty:
        return pd.DataFrame()

    df_features = construir_features_base(df_treino)
    df_features = df_features.dropna(subset=colunas_dropna_historico()).reset_index(drop=True)
    if df_features.empty:
        return pd.DataFrame()

    # Origens validas: precisam ter os h = 1..horizonte dias-alvo dentro do
    # proprio treino (por isso o recuo de `horizonte` dias no limite).
    # Series.unique() devolve Timestamps (nao numpy.datetime64), que e o que
    # montar_tabela_horizontes espera para somar timedelta.
    limite_origem = pd.Timestamp(treino_fim) - pd.Timedelta(days=horizonte)
    datas = sorted(d for d in df_features[COLS.data].unique() if d <= limite_origem)
    datas_origem = datas[::stride_origens]
    if not datas_origem:
        return pd.DataFrame()

    enc_sku = LabelEncoderJanela().fit(df_features[COLS.sku])
    enc_cat = LabelEncoderJanela().fit(df_features[COLS.categoria])

    tabela = montar_tabela_horizontes(df_features, datas_origem, horizonte=horizonte)
    if tabela.empty:
        return tabela

    tabela["sku_enc"] = enc_sku.transform(tabela[COLS.sku])
    tabela["categoria_enc"] = enc_cat.transform(tabela[COLS.categoria])
    return tabela


def _amostrar(df: pd.DataFrame, max_linhas: int | None) -> pd.DataFrame:
    """Amostragem aleatoria reprodutivel quando a tabela excede o teto de
    linhas configurado (protege a memoria em bases grandes)."""
    if max_linhas is None or len(df) <= max_linhas:
        return df
    return df.sample(n=max_linhas, random_state=config.RANDOM_STATE)


def gerar_heatmap_correlacao(
    matriz: pd.DataFrame,
    titulo: str,
    caminho_fig,
    anotar: bool = True,
) -> None:
    """Heatmap da matriz de correlacao em matplotlib puro (mesmo padrao dos
    demais graficos do pipeline, que usam o backend 'Agg' e nao dependem de
    seaborn). Valores anotados nas celulas para leitura direta."""
    rotulos = list(matriz.columns)
    n = len(rotulos)
    valores = matriz.to_numpy(dtype=float)

    lado = max(6.0, 0.62 * n + 2.0)
    fig, ax = plt.subplots(figsize=(lado, lado * 0.88))
    imagem = ax.imshow(valores, cmap="RdBu_r", vmin=-1.0, vmax=1.0)

    ax.set_xticks(range(n), labels=rotulos, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n), labels=rotulos, fontsize=8)
    ax.set_title(titulo, fontsize=11)

    if anotar:
        for i in range(n):
            for j in range(n):
                valor = valores[i, j]
                if np.isnan(valor):
                    texto, cor = "-", "gray"
                else:
                    texto = f"{valor:.2f}"
                    cor = "white" if abs(valor) > 0.6 else "black"
                ax.text(j, i, texto, ha="center", va="center", fontsize=6.5, color=cor)

    barra = fig.colorbar(imagem, ax=ax, shrink=0.8)
    barra.set_label("Correlação de Pearson", fontsize=9)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)


def top_correlacoes_com_target(
    matriz: pd.DataFrame, target: str = TARGET_CORRELACAO, n: int = 3
) -> pd.DataFrame:
    """As `n` features mais correlacionadas com o target (em modulo)."""
    if target not in matriz.columns:
        return pd.DataFrame(columns=["feature", "correlacao"])
    serie = matriz[target].drop(labels=[target], errors="ignore").dropna()
    if serie.empty:
        return pd.DataFrame(columns=["feature", "correlacao"])
    ordenada = serie.reindex(serie.abs().sort_values(ascending=False).index).head(n)
    return pd.DataFrame({"feature": ordenada.index, "correlacao": ordenada.to_numpy()})


def pares_multicolineares(
    matriz: pd.DataFrame,
    limiar: float | None = None,
    target: str = TARGET_CORRELACAO,
) -> pd.DataFrame:
    """Pares de features (alvo excluido) com |r| acima do limiar, ordenados do
    maior para o menor.

    Sao os candidatos a redundancia. |r| proximo de 1,000 costuma indicar
    identidade algebrica, nao apenas associacao forte -- e o criterio
    estrutural que o pipeline usa para excluir features redundantes."""
    limiar = config.LIMIAR_MULTICOLINEARIDADE if limiar is None else limiar
    features = [c for c in matriz.columns if c != target]
    registros = []
    for i, a in enumerate(features):
        for b in features[i + 1:]:
            r = matriz.loc[a, b]
            if pd.notna(r) and abs(r) >= limiar:
                registros.append({"feature_a": a, "feature_b": b, "correlacao": float(r)})
    df = pd.DataFrame(registros)
    if df.empty:
        return df
    return df.reindex(df["correlacao"].abs().sort_values(ascending=False).index).reset_index(drop=True)


def _texto_relatorio(
    matrizes: dict,
    tops: dict,
    redundancias: dict,
    n_linhas: dict,
    limiar: float,
    stride: int,
) -> str:
    partes = [
        "# Correlacao entre features e target (apoio a Secao 4.2.1)\n",
        "Correlacao de Pearson calculada sobre a tabela longa de treinamento "
        "(uma linha por SKU x data de origem x horizonte h), **restrita ao periodo "
        f"de treino** (ate {config.TREINO_FIM}), com passo de {stride} dias entre as "
        "datas de origem. O alvo `y` e a quantidade vendida na data (origem + h).\n",
        # Aviso condicional: um passo multiplo de 7 poe TODAS as origens no
        # mesmo dia da semana, o que torna o dia da semana do alvo uma funcao
        # exata do horizonte e faz o relatorio anunciar "r = 1,000" para um par
        # que nao e redundante na tabela de treino real (origens diarias).
        ("> **Atencao a amostragem:** o passo entre origens e multiplo de 7, "
         "entao todas as datas de origem caem no mesmo dia da semana e o dia da "
         "semana do alvo fica determinado pelo horizonte. Qualquer correlacao "
         "envolvendo `dia_semana_alvo` nesta tabela e artefato da amostragem, "
         "nao propriedade dos dados. Use um passo primo com 7 (ex.: 13).\n"
         if stride % 7 == 0 else ""),
        "As matrizes completas estao nos arquivos `correlacao_<quadrante>.csv` e nas "
        "figuras `fig_correlacao_<quadrante>.png`.\n",
    ]

    for quadrante in matrizes:
        partes.append(f"## Quadrante {quadrante}\n")
        partes.append(f"- Linhas usadas no calculo: **{n_linhas.get(quadrante, 0):,}**")

        top = tops.get(quadrante)
        if top is None or top.empty:
            partes.append("- Nao foi possivel calcular correlacoes com `y` (variancia nula).\n")
        else:
            partes.append("- Features mais correlacionadas com `y`:")
            for _, linha in top.iterrows():
                partes.append(f"  - `{linha['feature']}`: r = {linha['correlacao']:.3f}")
            partes.append("")

        red = redundancias.get(quadrante)
        if red is None or red.empty:
            partes.append(
                f"- Nenhum par de features com |r| >= {limiar:.2f} — "
                "sem indicio de multicolinearidade relevante neste quadrante.\n"
            )
        else:
            partes.append(
                f"- **Alerta de multicolinearidade** (|r| >= {limiar:.2f}) — pares candidatos "
                "a redundancia, avaliar mencao na Secao 4.2.1 (ou teste de remocao):"
            )
            for _, linha in red.iterrows():
                partes.append(
                    f"  - `{linha['feature_a']}` x `{linha['feature_b']}`: r = {linha['correlacao']:.3f}"
                )
            partes.append("")

    partes.append(
        "> Leitura recomendada para o TCC: correlacao alta com `y` justifica a "
        "presenca da feature na Tabela 4.3; correlacao alta ENTRE features nao "
        "invalida os modelos usados (arvores lidam com preditores correlacionados), "
        "mas explica por que a importancia de uma feature pode se diluir entre "
        "variaveis equivalentes (ver `fig_importancia_features_rf_xgb.png`).\n"
    )
    return "\n".join(partes)


def gerar_analise_correlacao(
    df_modelagem: pd.DataFrame,
    df_classes: pd.DataFrame,
    outputs_dir,
    colunas: list | None = None,
    quadrantes: tuple = config.QUADRANTES_FOCO,
) -> dict:
    """Gera a analise completa de correlacao.

    Produz, por quadrante: a matriz em CSV, o heatmap anotado em PNG e as
    secoes de correlacao_features.md com as features mais associadas ao alvo e
    os alertas de multicolinearidade.

    Precisa rodar DEPOIS da classificacao ADI/CV2 (que fornece `df_classes` com
    a coluna `quadrante`) e ANTES dos experimentos -- nao depende de resultado
    de modelo nenhum.

    Devolve {quadrante: matriz}, ou dicionario vazio se faltarem dados."""
    colunas = list(colunas or features_experimento1())

    tabela = montar_tabela_correlacao(df_modelagem)
    if tabela.empty:
        print("[correlacao] Tabela de treino vazia — analise de correlacao ignorada.")
        return {}

    tabela = tabela.merge(
        df_classes[[COLS.sku, "quadrante"]].drop_duplicates(), on=COLS.sku, how="left"
    )

    matrizes, tops, redundancias, n_linhas = {}, {}, {}, {}
    for quadrante in quadrantes:
        sub = tabela[tabela["quadrante"] == quadrante]
        if sub.empty:
            print(f"[correlacao] Nenhuma linha no quadrante {quadrante} — pulando.")
            continue
        sub = _amostrar(sub, config.CORRELACAO_MAX_LINHAS)

        matriz = calcular_matriz_correlacao(sub, colunas, target=TARGET_CORRELACAO)
        matrizes[quadrante] = matriz
        tops[quadrante] = top_correlacoes_com_target(matriz)
        redundancias[quadrante] = pares_multicolineares(matriz)
        n_linhas[quadrante] = len(sub)

        sufixo = quadrante.lower()
        matriz.to_csv(outputs_dir / f"correlacao_{sufixo}.csv")
        gerar_heatmap_correlacao(
            matriz,
            f"Correlação features × y — quadrante {quadrante}\n"
            f"(treino até {config.TREINO_FIM}, n = {len(sub):,} linhas)",
            outputs_dir / f"fig_correlacao_{sufixo}.png",
        )

    if not matrizes:
        return {}

    texto = _texto_relatorio(
        matrizes, tops, redundancias, n_linhas,
        config.LIMIAR_MULTICOLINEARIDADE, config.CORRELACAO_STRIDE_ORIGENS_DIAS,
    )
    (outputs_dir / "correlacao_features.md").write_text(texto, encoding="utf-8")
    print(f"[correlacao] Analise gerada para: {', '.join(matrizes)}")
    return matrizes

In [ ]:
%%writefile /content/src/score_relativo.py
"""
Fase 9b — Score relativo por SKU (complemento ao teste de Wilcoxon).

Motivacao: o teste de Wilcoxon responde apenas se a diferenca entre dois
modelos e estatisticamente significativa, nao QUANTO um e melhor que o outro.
O score relativo mede essa magnitude de forma comparavel entre SKUs de escalas
muito diferentes:

    score_relativo(modelo, sku) = metrica(modelo, sku) / min(metrica(*, sku))

Leitura: score = 1,00 significa que o modelo foi o MELHOR naquele SKU;
score = 2,00 significa erro duas vezes maior que o do melhor modelo naquele SKU.

POR QUE MEDIA GEOMETRICA E PISO NO DENOMINADOR

Razoes de erro sao multiplicativas, e agrega-las pela media aritmetica e
instavel: um unico SKU em que o melhor modelo erra quase nada domina o
resultado inteiro. O caso concreto nesta base e o TSB, que preve praticamente
zero em serie intermitente -- o MASE dele chega a ordem de 1e-8, vira
denominador, e a razao dos demais explode. Uma execucao com 412 SKUs chegou a
produzir media aritmetica na ordem de 1e+65.

Duas protecoes, ambas com respaldo de literatura:
  (a) PISO relativo no denominador (config.PISO_RELATIVO_SCORE): so serve de
      referencia o minimo maior que uma fracao da mediana daquele SKU;
  (b) MEDIA GEOMETRICA na agregacao. Hyndman e Koehler (2006) e Fildes (1992)
      alertam contra a media aritmetica de razoes de erro; a geometrica e o
      padrao (ver tambem Davydenko e Fildes, 2013, para o RelMAE).

A media aritmetica continua na tabela ao lado da geometrica, para que a
diferenca entre as duas fique visivel. Use a geometrica.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config
from .config import COLS
from .evaluate import METRICAS_WILCOXON


def montar_metricas_longas(
    dict_metricas_por_modelo: dict,
    df_classes: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Converte {'SBA': df_metricas, 'RF': df_metricas, ...} em um unico
    DataFrame no formato longo (uma linha por SKU x modelo), opcionalmente
    anotado com o quadrante ADI/CV2 de cada SKU."""
    partes = []
    for nome_modelo, df in dict_metricas_por_modelo.items():
        if df is None or df.empty:
            continue
        parte = df.copy()
        parte["modelo"] = nome_modelo
        partes.append(parte)

    if not partes:
        return pd.DataFrame(columns=[COLS.sku, "modelo"])

    longo = pd.concat(partes, ignore_index=True)
    if df_classes is not None and not df_classes.empty:
        longo = longo.merge(
            df_classes[[COLS.sku, "quadrante"]].drop_duplicates(), on=COLS.sku, how="left"
        )
    return longo


def calcular_score_relativo(df_resumo: pd.DataFrame, metrica: str = "WMAPE") -> pd.DataFrame:
    """Divide a metrica de cada modelo pela do melhor modelo daquele SKU.

    Acrescenta duas colunas: `score_relativo_<metrica>` e
    `melhor_<metrica>` (1 quando o modelo foi o melhor no SKU).

    O denominador so e aceito se ficar acima de PISO_RELATIVO_SCORE x mediana
    da metrica naquele SKU. Sem esse piso, um modelo que preve zero (metrica na
    ordem de 1e-8) vira referencia e faz as razoes dos demais explodirem."""
    df = df_resumo.copy()
    if metrica not in df.columns:
        return df

    minimo_por_sku = df.groupby(COLS.sku)[metrica].transform("min")

    if config.SCORE_RELATIVO_GEOMETRICO:
        mediana_por_sku = df.groupby(COLS.sku)[metrica].transform("median")
        piso = mediana_por_sku * float(config.PISO_RELATIVO_SCORE)
        denominador = minimo_por_sku.where(minimo_por_sku > piso.fillna(0.0))
    else:
        denominador = minimo_por_sku.where(minimo_por_sku > 0)  # ramo sem o piso

    coluna_score = f"score_relativo_{metrica}"
    df[coluna_score] = df[metrica] / denominador
    df[f"melhor_{metrica}"] = (
        df[metrica].notna() & np.isclose(df[metrica], minimo_por_sku)
    ).astype(int)
    return df


def _media_geometrica(valores: pd.Series) -> float:
    """Media geometrica de razoes estritamente positivas, ignorando NaN."""
    v = pd.to_numeric(valores, errors="coerce").dropna()
    v = v[v > 0]
    if v.empty:
        return float("nan")
    return float(np.exp(np.log(v).mean()))


def agregar_score_relativo(
    df_long: pd.DataFrame,
    metrica: str = "WMAPE",
    por_quadrante: bool = False,
) -> pd.DataFrame:
    """Agrega o score relativo por modelo, e por quadrante se pedido.

    Devolve as tres estatisticas lado a lado -- geometrica, mediana e
    aritmetica -- mais a proporcao de SKUs em que o modelo foi o melhor e a
    proporcao de scores indefinidos. Use a geometrica."""
    coluna_score = f"score_relativo_{metrica}"
    if coluna_score not in df_long.columns:
        return pd.DataFrame()

    chaves = ["modelo"] + (["quadrante"] if por_quadrante and "quadrante" in df_long.columns else [])

    linhas = []
    for chave, grupo in df_long.groupby(chaves, dropna=False):
        valores = tuple(chave) if isinstance(chave, tuple) else (chave,)
        registro = dict(zip(chaves, valores))
        registro.update({
            "metrica": metrica,
            "score_relativo_geometrico": _media_geometrica(grupo[coluna_score]),
            "score_relativo_mediano": float(grupo[coluna_score].median(skipna=True)),
            "score_relativo_medio": float(grupo[coluna_score].mean(skipna=True)),
            "pct_skus_melhor_modelo": float(grupo[f"melhor_{metrica}"].mean() * 100),
            "pct_score_indefinido": float(grupo[coluna_score].isna().mean() * 100),
            "n_skus": int(len(grupo)),
        })
        linhas.append(registro)

    ordenar_por = "score_relativo_geometrico" if config.SCORE_RELATIVO_GEOMETRICO else "score_relativo_medio"
    return pd.DataFrame(linhas).sort_values(ordenar_por).reset_index(drop=True)


def adicionar_score_relativo_ao_resumo(
    resumo: pd.DataFrame,
    agregados_por_metrica: dict,
) -> pd.DataFrame:
    """Acrescenta ao resumo agregado (resumo_experimento1.csv) as colunas de
    score relativo por metrica, ao lado das medianas de sMAPE/WMAPE/MASE."""
    if resumo is None or resumo.empty:
        return resumo

    saida = resumo.copy()
    for metrica, agregado in agregados_por_metrica.items():
        if agregado is None or agregado.empty or "quadrante" in agregado.columns:
            continue
        mapa_geo = agregado.set_index("modelo")["score_relativo_geometrico"]
        saida[f"score_rel_{metrica}_geometrico"] = saida["modelo"].map(mapa_geo)
        mapa_med = agregado.set_index("modelo")["score_relativo_medio"]
        saida[f"score_relativo_{metrica}_medio"] = saida["modelo"].map(mapa_med)
    return saida


def _texto_relatorio(agregados: dict, agregados_quadrante: dict) -> str:
    partes = [
        "# Score relativo por SKU (complemento ao Wilcoxon — Secao 4.2.7)\n",
        "Para cada SKU, a metrica de cada modelo e dividida pela metrica do melhor "
        "modelo naquele SKU. **1,00 = foi o melhor**; 1,50 = erro 50% maior que o do "
        "melhor modelo.\n",
        "**Use a coluna GEOMETRICA** (Item 13). A media aritmetica de razoes e "
        "instavel quando o melhor modelo tem erro proximo de zero — e o caso do TSB "
        "em serie intermitente — e produziu valores da ordem de 1e+65 na versao "
        "anterior. Hyndman e Koehler (2006) e Fildes (1992) recomendam a media "
        "geometrica para razoes de erro. A aritmetica fica na tabela apenas para "
        "tornar a diferenca visivel.\n",
        "SKUs em que o melhor modelo atinge metrica igual a 0 (ou abaixo do piso "
        "relativo) entram como score indefinido — ver `pct_score_indefinido`.\n",
    ]

    for metrica, agregado in agregados.items():
        if agregado is None or agregado.empty:
            continue
        partes.append(f"## {metrica} — geral\n")
        partes.append("| modelo | score GEOMETRICO | score mediano | score aritmetico (instavel) | "
                      "% SKUs em que foi o melhor | % score indefinido | n SKUs |")
        partes.append("| --- | --- | --- | --- | --- | --- | --- |")
        for _, r in agregado.iterrows():
            partes.append(
                f"| {r['modelo']} | {r['score_relativo_geometrico']:.3f} | "
                f"{r['score_relativo_mediano']:.3f} | {r['score_relativo_medio']:.3g} | "
                f"{r['pct_skus_melhor_modelo']:.1f}% | "
                f"{r['pct_score_indefinido']:.1f}% | {int(r['n_skus'])} |"
            )
        partes.append("")

        por_quad = agregados_quadrante.get(metrica)
        if por_quad is not None and not por_quad.empty and "quadrante" in por_quad.columns:
            partes.append(f"### {metrica} — por quadrante ADI/CV2\n")
            partes.append("| quadrante | modelo | score GEOMETRICO | score mediano | "
                          "% SKUs em que foi o melhor | n SKUs |")
            partes.append("| --- | --- | --- | --- | --- | --- |")
            for _, r in por_quad.sort_values(["quadrante", "score_relativo_geometrico"]).iterrows():
                partes.append(
                    f"| {r['quadrante']} | {r['modelo']} | {r['score_relativo_geometrico']:.3f} | "
                    f"{r['score_relativo_mediano']:.3f} | {r['pct_skus_melhor_modelo']:.1f}% | "
                    f"{int(r['n_skus'])} |"
                )
            partes.append("")

    return "\n".join(partes)


def gerar_analise_score_relativo(
    dict_metricas_por_modelo: dict,
    df_classes: pd.DataFrame | None,
    outputs_dir,
    resumo: pd.DataFrame | None = None,
    metricas: tuple = METRICAS_WILCOXON,
):
    """Calcula o score relativo, salva os artefatos na pasta da execucao e
    devolve `(resumo_atualizado, df_long, agregados)`."""
    df_long = montar_metricas_longas(dict_metricas_por_modelo, df_classes)
    if df_long.empty:
        return resumo, df_long, {}

    agregados, agregados_quadrante = {}, {}
    for metrica in metricas:
        if metrica not in df_long.columns:
            continue
        df_long = calcular_score_relativo(df_long, metrica)
        agregados[metrica] = agregar_score_relativo(df_long, metrica)
        agregados_quadrante[metrica] = agregar_score_relativo(df_long, metrica, por_quadrante=True)

    df_long.to_csv(outputs_dir / "score_relativo_por_sku.csv", index=False)

    partes_agregadas = [df for df in list(agregados.values()) + list(agregados_quadrante.values())
                        if df is not None and not df.empty]
    if partes_agregadas:
        pd.concat(partes_agregadas, ignore_index=True).to_csv(
            outputs_dir / "score_relativo_agregado.csv", index=False
        )

    (outputs_dir / "score_relativo.md").write_text(
        _texto_relatorio(agregados, agregados_quadrante), encoding="utf-8"
    )

    resumo_atualizado = adicionar_score_relativo_ao_resumo(resumo, agregados)
    return resumo_atualizado, df_long, agregados


In [ ]:
%%writefile /content/src/graficos_resultados.py
"""
Fase 10 — Graficos de resultados.

Cada figura responde a UMA pergunta de pesquisa — nao ha grafico decorativo:

  fig_mediana_wmape_modelos.png / fig_mediana_mase_modelos.png
      Qual modelo vence de forma geral?

  fig_boxplot_wmape_quadrante.png / fig_boxplot_mase_quadrante.png
      Os modelos diferem em mediana E em dispersao do erro, e isso muda entre
      Intermitente e Lumpy? Complementa o Wilcoxon, que so informa
      significancia, nao a forma da distribuicao.

  fig_tempo_execucao_modelos.png
      Qual o custo computacional pratico de cada modelo? Sustenta a discussao
      RF vs. XGBoost.

  fig_importancia_features_rf_xgb.png
      RF e XGBoost usam as mesmas informacoes para prever? E aqui que a
      diluicao de importancia entre features equivalentes fica visivel — o
      argumento para a remocao de features redundantes.

  fig_realvsprevisto_<sku>.png
      Como cada modelo se comporta na pratica, em um SKU tipico de cada
      quadrante (melhor e pior caso)?

Tudo em matplotlib puro com backend 'Agg' (mesmo padrao de eda.py,
sazonalidade.py e adi_cv2.py) — sem dependencia de seaborn.
"""
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config
from .config import COLS

config.configurar_matplotlib_pt_br()

# Ordem e cores fixas por modelo, para que todas as figuras sejam lidas com o
# mesmo codigo visual. Baselines em tons frios, modelos de ML em tons quentes.
ORDEM_MODELOS = ["SBA", "TSB", "RF", "XGBoost", "Zero"]
CORES_MODELOS = {
    "SBA": "#4C72B0",
    "TSB": "#64B5CD",
    "RF": "#55A868",
    "XGBoost": "#C44E52",
    "Zero": "#8C8C8C",   # baseline de controle
}
CORES_QUADRANTES = {"Intermitente": "#55A868", "Lumpy": "#C44E52"}
COR_PADRAO = "#8172B2"


def _ordenar_modelos(nomes) -> list:
    """Modelos conhecidos na ordem canonica; desconhecidos ao final (ordem
    alfabetica), para nunca perder um modelo novo silenciosamente."""
    presentes = [m for m in ORDEM_MODELOS if m in set(nomes)]
    extras = sorted(set(nomes) - set(ORDEM_MODELOS))
    return presentes + extras


def _cor(modelo: str) -> str:
    """Cor fixa de cada modelo, para que ele tenha a mesma cor em toda figura."""
    return CORES_MODELOS.get(modelo, COR_PADRAO)


# ---------------------------------------------------------------------------
# 3.4 — Barras de metrica agregada por modelo (mediana global + IQR)
# ---------------------------------------------------------------------------
def gerar_barras_mediana_por_modelo(
    df_long: pd.DataFrame,
    metrica: str,
    caminho_fig,
) -> pd.DataFrame:
    """Barras com a MEDIANA da metrica por modelo e barra de erro no
    intervalo interquartil (Q1-Q3, dispersao entre SKUs). Responde "qual
    modelo vence de forma geral" e resume os boxplots da Secao 3.1.

    Usa a mediana (nao a media): em demanda intermitente a distribuicao do
    erro por SKU e fortemente assimetrica."""
    modelos = _ordenar_modelos(df_long["modelo"].unique())
    linhas = []
    for modelo in modelos:
        valores = df_long.loc[df_long["modelo"] == modelo, metrica].dropna()
        if valores.empty:
            continue
        q1, mediana, q3 = np.percentile(valores, [25, 50, 75])
        linhas.append({"modelo": modelo, "mediana": mediana, "q1": q1, "q3": q3,
                       "n_skus": len(valores)})
    if not linhas:
        return pd.DataFrame()

    resumo = pd.DataFrame(linhas)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(resumo))
    erro = np.vstack([
        (resumo["mediana"] - resumo["q1"]).clip(lower=0).to_numpy(),
        (resumo["q3"] - resumo["mediana"]).clip(lower=0).to_numpy(),
    ])
    ax.bar(x, resumo["mediana"], color=[_cor(m) for m in resumo["modelo"]],
           yerr=erro, capsize=5, ecolor="#444444", edgecolor="white")

    for xi, (mediana, n) in enumerate(zip(resumo["mediana"], resumo["n_skus"])):
        ax.text(xi, mediana, f"{mediana:.2f}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x, labels=resumo["modelo"])
    ax.set_ylabel(f"{metrica} mediano entre SKUs")
    ax.set_title(f"{metrica} mediano por modelo (barras de erro = IQR entre SKUs)")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)
    return resumo


# ---------------------------------------------------------------------------
# 3.1 — Boxplot de metricas por modelo e por quadrante
# ---------------------------------------------------------------------------
def gerar_boxplot_por_quadrante(
    df_long: pd.DataFrame,
    metrica: str,
    caminho_fig,
    quadrantes: tuple = config.QUADRANTES_FOCO,
) -> None:
    """Boxplots agrupados: eixo X = modelo, uma caixa por quadrante dentro de
    cada modelo (equivalente ao `hue` do seaborn, feito em matplotlib puro
    via deslocamento das posicoes).

    Mostra mediana E dispersao do erro entre SKUs — a informacao que o teste
    de Wilcoxon nao da (ele responde apenas "a diferenca e significativa?")."""
    if "quadrante" not in df_long.columns:
        print("[graficos] Coluna 'quadrante' ausente — boxplot por quadrante ignorado.")
        return

    modelos = _ordenar_modelos(df_long["modelo"].unique())
    quadrantes_presentes = [q for q in quadrantes if (df_long["quadrante"] == q).any()]
    if not modelos or not quadrantes_presentes:
        return

    largura = 0.8 / len(quadrantes_presentes)
    fig, ax = plt.subplots(figsize=(1.9 * len(modelos) + 3.5, 5))

    for j, quadrante in enumerate(quadrantes_presentes):
        dados, posicoes = [], []
        for i, modelo in enumerate(modelos):
            valores = df_long.loc[
                (df_long["modelo"] == modelo) & (df_long["quadrante"] == quadrante), metrica
            ].dropna()
            dados.append(valores.to_numpy())
            posicoes.append(i + (j - (len(quadrantes_presentes) - 1) / 2) * largura)

        cor = CORES_QUADRANTES.get(quadrante, COR_PADRAO)
        caixas = ax.boxplot(
            dados, positions=posicoes, widths=largura * 0.85, patch_artist=True,
            showfliers=False, medianprops=dict(color="black", linewidth=1.4),
            whiskerprops=dict(color="#555555"), capprops=dict(color="#555555"),
        )
        for caixa in caixas["boxes"]:
            caixa.set_facecolor(cor)
            caixa.set_alpha(0.75)
            caixa.set_edgecolor("#333333")
        caixas["boxes"][0].set_label(quadrante)

    ax.set_xticks(range(len(modelos)), labels=modelos)
    ax.set_ylabel(f"{metrica} por SKU")
    ax.set_title(
        f"Distribuição de {metrica} por modelo e quadrante ADI/CV2\n"
        "(caixa = IQR, linha = mediana; outliers omitidos para leitura)"
    )
    ax.legend(title="Quadrante")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 3.3 — Barras de tempo de execucao por modelo
# ---------------------------------------------------------------------------
ETAPAS_CUSTO_MODELO = ("tuning", "fit_final")


def gerar_barras_tempo_execucao(
    df_tempos: pd.DataFrame,
    caminho_fig,
    modelos_baseline: tuple = ("SBA", "TSB"),
) -> pd.DataFrame:
    """Barras com o custo computacional por modelo.

    Destaca a parcela de tuning sobre o total. Para os baselines, que nao tem
    tuning nem fit no mesmo sentido, usa o tempo total do walk-forward.

    Passa a escala logaritmica no eixo Y quando a razao entre o maior e o menor
    tempo supera 10x, senao a barra dominante achata visualmente as demais.

    Devolve a tabela que vira tempos_por_modelo.csv."""
    if df_tempos is None or df_tempos.empty:
        print("[graficos] Registro de tempos vazio — figura de tempo ignorada.")
        return pd.DataFrame()

    linhas = []
    # "CACHE" nao e um modelo: e o registro do tempo de construcao das features
    # compartilhadas. Fica fora da tabela de custo por modelo.
    for modelo, grupo in df_tempos.groupby("modelo"):
        if modelo == "CACHE":
            continue
        if modelo in modelos_baseline:
            total = grupo.loc[grupo["etapa"] == "walk_forward", "segundos"].sum()
            linhas.append({"modelo": modelo, "tuning": 0.0, "fit_final": 0.0,
                           "walk_forward": float(total), "total": float(total)})
            continue
        tuning = float(grupo.loc[grupo["etapa"] == "tuning", "segundos"].sum())
        fit = float(grupo.loc[grupo["etapa"] == "fit_final", "segundos"].sum())
        total_wf = float(grupo.loc[grupo["etapa"] == "walk_forward", "segundos"].sum())
        linhas.append({"modelo": modelo, "tuning": tuning, "fit_final": fit,
                       "walk_forward": total_wf, "total": tuning + fit})

    resumo = pd.DataFrame(linhas)
    # `custo_total` = wall-clock do walk-forward, para TODOS os modelos. E a
    # unica coluna comparavel entre ML e baselines. As colunas tuning e
    # fit_final sao decomposicao auxiliar do custo de ML (os baselines nao tem
    # grade a buscar) e nao devem ser somadas entre modelos de tipos diferentes.
    resumo["custo_total"] = np.where(
        resumo["walk_forward"] > 0, resumo["walk_forward"], resumo["total"])
    resumo["modelo"] = pd.Categorical(
        resumo["modelo"], categories=_ordenar_modelos(resumo["modelo"]), ordered=True
    )
    resumo = resumo.sort_values("modelo").reset_index(drop=True)

    positivos = resumo.loc[resumo["custo_total"] > 0, "custo_total"]
    if positivos.empty:
        print("[graficos] Todos os tempos sao zero — figura de tempo ignorada.")
        return resumo
    # Escala log quando a diferenca entre o mais caro e o mais barato passa de
    # 10x: em escala linear a barra dominante achataria as demais. Uma barra por
    # modelo (nao empilhada) — a parcela de tuning entra como sobreposicao
    # hachurada, que funciona nas duas escalas.
    usar_log = bool(len(positivos) > 1 and positivos.max() / positivos.min() > 10)

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    x = np.arange(len(resumo))
    ax.bar(x, resumo["custo_total"], width=0.6,
           color=[_cor(str(m)) for m in resumo["modelo"]], edgecolor="white")
    ax.bar(x, resumo["tuning"], width=0.6, color="none", edgecolor="#333333",
           hatch="///", linewidth=0.8)

    if usar_log:
        ax.set_yscale("log")
        ax.set_ylim(positivos.min() * 0.5, positivos.max() * 3.0)
        ax.set_ylabel("Tempo acumulado (s, escala log)")
    else:
        ax.set_ylim(0, positivos.max() * 1.18)
        ax.set_ylabel("Tempo acumulado (s)")

    for xi, linha in resumo.iterrows():
        if linha["custo_total"] > 0:
            ax.text(xi, linha["custo_total"], f"{linha['custo_total']:.1f}s",
                    ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x, labels=[str(m) for m in resumo["modelo"]])
    ax.set_title(
        "Custo computacional por modelo no walk-forward\n"
        "ML: tuning + fit final | Baselines: tempo total (sem grade)",
        fontsize=10,
    )
    ax.legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, facecolor="#BBBBBB", edgecolor="white",
                          label="Custo total do modelo"),
            plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor="#333333",
                          hatch="///", label="Parcela gasta em tuning"),
        ],
        fontsize=8, loc="upper left",
    )
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)
    return resumo


# ---------------------------------------------------------------------------
# 3.5 — Importancia de features (RF vs. XGBoost)
# ---------------------------------------------------------------------------
def gerar_importancia_features(
    df_importancias: pd.DataFrame,
    caminho_fig,
    modelos: tuple = ("RF", "XGBoost"),
) -> None:
    """Barras horizontais lado a lado (1x2) com a importancia media de
    features de RF e XGBoost, na MESMA ordem de features nos dois paineis —
    a comparacao direta so faz sentido com os eixos alinhados.

    A ordem e definida pela importancia media do primeiro modelo presente;
    as barras trazem o desvio-padrao entre refits do walk-forward."""
    if df_importancias is None or df_importancias.empty:
        print("[graficos] Importancias vazias — figura de importancia ignorada.")
        return

    presentes = [m for m in modelos if (df_importancias["modelo"] == m).any()]
    if not presentes:
        return

    referencia = df_importancias[df_importancias["modelo"] == presentes[0]]
    ordem = (referencia.sort_values("importancia_media", ascending=True)["feature"].tolist())

    fig, eixos = plt.subplots(1, len(presentes), figsize=(6.2 * len(presentes), 0.42 * len(ordem) + 2.6),
                              sharey=True)
    eixos = np.atleast_1d(eixos)

    for ax, modelo in zip(eixos, presentes):
        sub = (df_importancias[df_importancias["modelo"] == modelo]
               .set_index("feature").reindex(ordem))
        valores = sub["importancia_media"].fillna(0).to_numpy()
        erros = sub.get("importancia_std")
        erros = None if erros is None else erros.fillna(0).to_numpy()
        ax.barh(np.arange(len(ordem)), valores, xerr=erros, color=_cor(modelo),
                edgecolor="white", ecolor="#444444", capsize=3)
        ax.set_yticks(np.arange(len(ordem)), labels=ordem, fontsize=8)
        ax.set_xlabel("Importância média (fits finais do walk-forward)")
        ax.set_title(modelo)
        ax.grid(axis="x", linestyle=":", alpha=0.5)
        ax.set_axisbelow(True)

    fig.suptitle("Importância de features: RF vs. XGBoost (mesma ordem de features nos dois painéis)")
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 3.2 — Real vs. previsto para SKUs representativos
# ---------------------------------------------------------------------------
def selecionar_skus_representativos(
    df_long: pd.DataFrame,
    metrica: str = "WMAPE",
    quadrantes: tuple = config.QUADRANTES_FOCO,
) -> pd.DataFrame:
    """Para cada quadrante, escolhe o SKU de MENOR e o de MAIOR metrica media
    entre os modelos (melhor e pior caso). So considera SKUs avaliados por
    TODOS os modelos, para que a media seja comparavel."""
    if df_long.empty or "quadrante" not in df_long.columns:
        return pd.DataFrame(columns=[COLS.sku, "quadrante", "caso", f"{metrica}_medio"])

    n_modelos = df_long["modelo"].nunique()
    contagem = df_long.groupby(COLS.sku)["modelo"].nunique()
    completos = contagem[contagem == n_modelos].index

    base = df_long[df_long[COLS.sku].isin(completos)]
    media = (base.groupby([COLS.sku, "quadrante"], observed=True)[metrica]
             .mean().reset_index(name=f"{metrica}_medio").dropna())

    escolhidos = []
    for quadrante in quadrantes:
        sub = media[media["quadrante"] == quadrante]
        if sub.empty:
            continue
        melhor = sub.loc[sub[f"{metrica}_medio"].idxmin()].to_dict()
        pior = sub.loc[sub[f"{metrica}_medio"].idxmax()].to_dict()
        melhor["caso"], pior["caso"] = "melhor_caso", "pior_caso"
        escolhidos.append(melhor)
        if pior[COLS.sku] != melhor[COLS.sku]:
            escolhidos.append(pior)

    return pd.DataFrame(escolhidos)


def gerar_real_vs_previsto(
    previsoes_por_modelo: dict,
    sku,
    caminho_fig,
    titulo_extra: str = "",
    agregacao: str = "auto",
) -> bool:
    """Serie real (linha solida) x previsoes dos modelos (linhas tracejadas,
    uma cor por modelo) no periodo de teste, para um SKU.

    Substitui deliberadamente as bandas de incerteza por simulacao de Monte
    Carlo do artigo de referencia: o desenho deste pipeline e um walk-forward
    deterministico com previsao pontual, e a comparacao direta entre as
    previsoes dos quatro modelos e mais informativa (e honesta) do que
    intervalos que o pipeline nao estima.

    `agregacao='auto'` (default) agrega em soma semanal quando o periodo de
    teste passa de 180 dias — com 1 ano de teste diario as linhas ficariam
    ilegiveis. O titulo sempre declara a agregacao usada."""
    series = {}
    real = None
    for modelo, prev in previsoes_por_modelo.items():
        if prev is None or prev.empty:
            continue
        sub = prev[prev[COLS.sku] == sku]
        if sub.empty:
            continue
        # Media quando ha mais de uma previsao para a mesma data (origens
        # diferentes do walk-forward podem cobrir a mesma data-alvo).
        agrupado = sub.groupby(COLS.data).agg(y_real=("y_real", "mean"),
                                              y_pred=("y_pred", "mean")).sort_index()
        series[modelo] = agrupado["y_pred"]
        real = agrupado["y_real"] if real is None else real.combine_first(agrupado["y_real"])

    if real is None or real.empty or not series:
        return False

    dias = (real.index.max() - real.index.min()).days
    semanal = agregacao == "semanal" or (agregacao == "auto" and dias > 180)
    if semanal:
        real = real.resample("W").sum()
        series = {m: s.resample("W").sum() for m, s in series.items()}
        nota = "agregado por soma semanal"
    else:
        nota = "granularidade diaria"

    fig, ax = plt.subplots(figsize=(11, 4.6))
    ax.plot(real.index, real.to_numpy(), color="black", linewidth=2.0, label="Real")
    for modelo in _ordenar_modelos(series):
        serie = series[modelo]
        ax.plot(serie.index, serie.to_numpy(), linestyle="--", linewidth=1.5,
                color=_cor(modelo), label=modelo)

    ax.set_xlabel(f"Período de teste ({nota})")
    ax.set_ylabel("Quantidade vendida")
    ax.set_title(f"Real vs. previsto — SKU {sku}{titulo_extra}")
    ax.legend(ncol=5, fontsize=9)
    ax.grid(linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)
    return True


def _sufixo_arquivo(sku) -> str:
    """SKU sanitizado para uso em nome de arquivo."""
    return "".join(c if (c.isalnum() or c in "-_") else "_" for c in str(sku))


# ---------------------------------------------------------------------------
# Orquestracao
# ---------------------------------------------------------------------------
def gerar_graficos_resultados(
    df_long: pd.DataFrame,
    previsoes_por_modelo: dict,
    df_tempos: pd.DataFrame | None,
    df_importancias: pd.DataFrame | None,
    outputs_dir,
    metricas: tuple = ("WMAPE", "MASE"),
) -> list:
    """Gera todas as figuras de resultado e devolve a lista de arquivos criados.

    Ordem: mediana por modelo, boxplots por quadrante, custo computacional,
    importancia de features e, por fim, real x previsto por SKU.

    `df_long`: metricas por SKU x modelo, com quadrante, vindo de
    score_relativo.montar_metricas_longas.
    `previsoes_por_modelo`: {modelo: DataFrame [sku, Data, y_real, y_pred]} do
    Experimento 1."""
    figuras = []

    # 3.4 — mediana por modelo
    for metrica in metricas:
        if metrica not in df_long.columns:
            continue
        caminho = outputs_dir / f"fig_mediana_{metrica.lower()}_modelos.png"
        if not gerar_barras_mediana_por_modelo(df_long, metrica, caminho).empty:
            figuras.append(caminho.name)

    # 3.1 — boxplots por quadrante
    for metrica in metricas:
        if metrica not in df_long.columns:
            continue
        caminho = outputs_dir / f"fig_boxplot_{metrica.lower()}_quadrante.png"
        gerar_boxplot_por_quadrante(df_long, metrica, caminho)
        if caminho.exists():
            figuras.append(caminho.name)

    # 3.3 — tempo de execucao
    caminho_tempo = outputs_dir / "fig_tempo_execucao_modelos.png"
    resumo_tempo = gerar_barras_tempo_execucao(df_tempos, caminho_tempo)
    if resumo_tempo is not None and not resumo_tempo.empty:
        resumo_tempo.to_csv(outputs_dir / "tempos_por_modelo.csv", index=False)
        figuras.append(caminho_tempo.name)

    # 3.5 — importancia de features
    caminho_imp = outputs_dir / "fig_importancia_features_rf_xgb.png"
    gerar_importancia_features(df_importancias, caminho_imp)
    if caminho_imp.exists():
        figuras.append(caminho_imp.name)

    # 3.2 — real vs. previsto (melhor e pior caso de cada quadrante)
    selecionados = selecionar_skus_representativos(df_long)
    if not selecionados.empty:
        selecionados.to_csv(outputs_dir / "skus_representativos.csv", index=False)
    for _, linha in selecionados.iterrows():
        sku = linha[COLS.sku]
        extra = (f" — {linha['quadrante']}, {linha['caso'].replace('_', ' ')} "
                 f"(WMAPE medio = {linha['WMAPE_medio']:.1f}%)")
        caminho = outputs_dir / f"fig_realvsprevisto_{_sufixo_arquivo(sku)}.png"
        if gerar_real_vs_previsto(previsoes_por_modelo, sku, caminho, titulo_extra=extra):
            figuras.append(caminho.name)

    print(f"[graficos] {len(figuras)} figura(s) de resultados gerada(s).")
    return figuras

In [ ]:
%%writefile /content/src/graficos_estratos.py
"""
FIGURAS DOS RECORTES POR ESTRATO.

Complementa `graficos_resultados.py` com as figuras dos dois recortes novos:

  Ativos x cessados (o produto continua vendendo? o modelo percebeu?)
  Faixas de volume de pedidos (onde a previsao vale mais?)

Figuras geradas:
  fig_pareto_pedidos.png              curva de concentracao (Pareto) dos pedidos
  fig_<metrica>_por_estrato_teste.png barras modelo x (Ativo/Cessado)
  fig_<metrica>_por_estrato_volume.png barras modelo x faixa ABC
  fig_boxplot_<metrica>_volume.png    distribuicao por faixa de volume
  fig_realvsprevisto_TOP<i>_<sku>.png real x previsto dos SKUs de maior volume
"""
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config
from .config import COLS

config.configurar_matplotlib_pt_br()
from .graficos_resultados import (
    _cor, _ordenar_modelos, _sufixo_arquivo, gerar_real_vs_previsto,
)


# ---------------------------------------------------------------------------
# Curva de Pareto dos pedidos
# ---------------------------------------------------------------------------
def gerar_pareto_pedidos(estratos_volume: pd.DataFrame, caminho_fig) -> bool:
    """Curva de concentracao: % acumulado dos pedidos contra % acumulado dos
    SKUs, ordenados do maior para o menor volume.

    Leitura: quanto mais a curva "abre" no inicio, mais concentrado
    esta o valor da previsao — e mais defensavel e priorizar os SKUs do topo."""
    if estratos_volume is None or estratos_volume.empty:
        return False
    if "pct_volume_acumulado" not in estratos_volume.columns:
        return False

    base = estratos_volume.sort_values("volume_pedidos", ascending=False).reset_index(drop=True)
    n = len(base)
    x = (np.arange(1, n + 1) / n) * 100
    y = base["pct_volume_acumulado"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x, y, color="#C44E52", linewidth=2.0, label="Pedidos acumulados")
    ax.plot([0, 100], [0, 100], color="gray", linestyle=":", linewidth=1.2,
            label="Distribuição uniforme (sem concentração)")

    for corte in (20, 50):
        if n >= 2:
            idx = max(int(np.ceil(corte / 100 * n)) - 1, 0)
            valor = float(y[idx])
            ax.axvline(corte, color="black", linestyle="--", linewidth=0.9, alpha=0.6)
            ax.plot([corte], [valor], marker="o", color="black", markersize=5)
            ax.annotate(f"{corte}% dos SKUs\n= {valor:.1f}% dos pedidos",
                        xy=(corte, valor), xytext=(corte + 4, max(valor - 18, 5)),
                        fontsize=9,
                        arrowprops=dict(arrowstyle="->", color="black", lw=0.8))

    ax.set_xlabel("% de SKUs (ordenados por nº de pedidos, do maior para o menor)")
    ax.set_ylabel("% acumulado dos pedidos")
    ax.set_title("Concentração dos pedidos entre os SKUs (curva de Pareto, período de treino)")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 101)
    ax.grid(linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)
    ax.legend(fontsize=9, loc="lower right")
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)
    return True


# ---------------------------------------------------------------------------
# Barras agrupadas: metrica por modelo dentro de cada estrato
# ---------------------------------------------------------------------------
def gerar_barras_por_estrato(
    resumo_estratos: pd.DataFrame,
    coluna_metrica: str,
    caminho_fig,
    titulo: str,
    rotulo_y: str | None = None,
) -> bool:
    """Barras agrupadas: um grupo por estrato, uma barra por modelo."""
    if resumo_estratos is None or resumo_estratos.empty:
        return False
    if coluna_metrica not in resumo_estratos.columns:
        return False

    dados = resumo_estratos[["modelo", "estrato", coluna_metrica]].dropna(subset=[coluna_metrica])
    if dados.empty:
        return False

    estratos = list(dict.fromkeys(dados["estrato"]))
    modelos = _ordenar_modelos(dados["modelo"])
    n_mod = len(modelos)
    if n_mod == 0:
        return False

    largura = 0.8 / n_mod
    x = np.arange(len(estratos))

    fig, ax = plt.subplots(figsize=(max(7.5, 2.6 * len(estratos)), 5))
    for i, modelo in enumerate(modelos):
        alturas = []
        for estrato in estratos:
            sel = dados[(dados["modelo"] == modelo) & (dados["estrato"] == estrato)]
            alturas.append(float(sel[coluna_metrica].iloc[0]) if not sel.empty else np.nan)
        posicoes = x - 0.4 + largura * (i + 0.5)
        barras = ax.bar(posicoes, alturas, width=largura * 0.92,
                        color=_cor(modelo), label=modelo)
        for barra, altura in zip(barras, alturas):
            if altura is None or (isinstance(altura, float) and np.isnan(altura)):
                continue
            ax.annotate(f"{altura:.3g}", xy=(barra.get_x() + barra.get_width() / 2, altura),
                        xytext=(0, 2), textcoords="offset points",
                        ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(x, labels=[str(e) for e in estratos], fontsize=9)
    ax.set_ylabel(rotulo_y or coluna_metrica)
    ax.set_title(titulo)
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.set_axisbelow(True)
    ax.legend(ncol=min(n_mod, 5), fontsize=9)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)
    return True


# ---------------------------------------------------------------------------
# Boxplot da distribuicao por estrato
# ---------------------------------------------------------------------------
def gerar_boxplot_por_estrato(
    df_long: pd.DataFrame,
    estratos: pd.DataFrame,
    metrica: str,
    coluna_estrato: str,
    caminho_fig,
    titulo: str,
) -> bool:
    """Distribuição da métrica por SKU, um painel por estrato."""
    if df_long is None or df_long.empty or metrica not in df_long.columns:
        return False
    if estratos is None or estratos.empty or coluna_estrato not in estratos.columns:
        return False

    base = df_long.merge(estratos[[COLS.sku, coluna_estrato]], on=COLS.sku, how="left")
    base = base.dropna(subset=[metrica, coluna_estrato])
    if base.empty:
        return False

    niveis = list(dict.fromkeys(base[coluna_estrato]))
    modelos = _ordenar_modelos(base["modelo"])
    fig, eixos = plt.subplots(1, len(niveis), figsize=(max(6.0, 4.4 * len(niveis)), 5),
                              sharey=True, squeeze=False)

    for ax, nivel in zip(eixos[0], niveis):
        sub = base[base[coluna_estrato] == nivel]
        dados = [sub.loc[sub["modelo"] == m, metrica].dropna().to_numpy() for m in modelos]
        presentes = [(m, d) for m, d in zip(modelos, dados) if len(d) > 0]
        if not presentes:
            ax.set_visible(False)
            continue
        nomes = [m for m, _ in presentes]
        valores = [d for _, d in presentes]
        bp = ax.boxplot(valores, patch_artist=True, showfliers=False,
                        tick_labels=nomes, medianprops=dict(color="black", linewidth=1.4))
        for caixa, nome in zip(bp["boxes"], nomes):
            caixa.set_facecolor(_cor(nome))
            caixa.set_alpha(0.75)
        ax.set_title(f"{nivel}\n(n = {sub[COLS.sku].nunique()} SKUs)", fontsize=9)
        ax.tick_params(axis="x", labelrotation=30, labelsize=8)
        ax.grid(axis="y", linestyle=":", alpha=0.5)
        ax.set_axisbelow(True)

    eixos[0][0].set_ylabel(metrica)
    fig.suptitle(titulo, fontsize=11)
    fig.tight_layout()
    fig.savefig(caminho_fig, dpi=150)
    plt.close(fig)
    return True


# ---------------------------------------------------------------------------
# Real x previsto dos SKUs de maior volume
# ---------------------------------------------------------------------------
def gerar_figuras_top_volume(
    previsoes_por_modelo: dict,
    estratos_volume: pd.DataFrame,
    outputs_dir,
    n_skus: int = 6,
) -> list:
    """Serie real x previsoes para os `n_skus` com MAIOR numero de pedidos
    distintos no treino.

    Motivacao: as figuras de `graficos_resultados` escolhem o melhor e o pior
    caso de cada quadrante ADI/CV2 — util para a discussao metodologica, mas
    esses SKUs costumam ser de baixissimo giro. Do ponto de vista gerencial,
    interessa ver o comportamento dos modelos justamente onde ha volume: sao
    os SKUs cujo erro afeta o maior numero de clientes finais."""
    if estratos_volume is None or estratos_volume.empty:
        return []

    topo = estratos_volume.nlargest(n_skus, "volume_pedidos")
    figuras = []
    for posicao, (_, linha) in enumerate(topo.iterrows(), start=1):
        sku = linha[COLS.sku]
        volume = float(linha["volume_pedidos"])
        extra = (f" — TOP {posicao} em pedidos "
                 f"({volume:.0f} pedidos no treino, {linha.get('estrato_volume', '')})")
        caminho = outputs_dir / f"fig_realvsprevisto_TOP{posicao:02d}_{_sufixo_arquivo(sku)}.png"
        if gerar_real_vs_previsto(previsoes_por_modelo, sku, caminho, titulo_extra=extra):
            figuras.append(caminho.name)
    return figuras


# ---------------------------------------------------------------------------
# Orquestracao
# ---------------------------------------------------------------------------
def gerar_graficos_estratos(
    df_long: pd.DataFrame,
    previsoes_por_modelo: dict,
    estratos_teste: pd.DataFrame | None,
    estratos_volume: pd.DataFrame | None,
    resumo_estratos_teste: pd.DataFrame | None,
    resumo_estratos_volume: pd.DataFrame | None,
    outputs_dir,
    n_skus_top: int = 6,
) -> list:
    """Gera todas as figuras dos recortes por estrato e devolve os nomes."""
    figuras = []

    # --- Pareto dos pedidos ---
    if estratos_volume is not None and not estratos_volume.empty:
        caminho = outputs_dir / "fig_pareto_pedidos.png"
        if gerar_pareto_pedidos(estratos_volume, caminho):
            figuras.append(caminho.name)

    # --- Barras por estrato ativo/cessado ---
    if resumo_estratos_teste is not None and not resumo_estratos_teste.empty:
        for coluna, rotulo in [("WMAPE_mediana", "WMAPE mediano por SKU (%)"),
                               ("MAE_mediana", "MAE mediano por SKU"),
                               ("WMAPE_pooled", "WMAPE agregado do estrato (%)")]:
            caminho = outputs_dir / f"fig_{coluna.lower()}_por_estrato_teste.png"
            ok = gerar_barras_por_estrato(
                resumo_estratos_teste, coluna, caminho,
                titulo=f"{rotulo} — SKUs ativos x cessados no teste",
                rotulo_y=rotulo)
            if ok:
                figuras.append(caminho.name)

    # --- Barras por faixa de volume ---
    if resumo_estratos_volume is not None and not resumo_estratos_volume.empty:
        for coluna, rotulo in [("WMAPE_mediana", "WMAPE mediano por SKU (%)"),
                               ("WMAPE_pooled", "WMAPE agregado do estrato (%)"),
                               ("MAE_mediana", "MAE mediano por SKU")]:
            caminho = outputs_dir / f"fig_{coluna.lower()}_por_estrato_volume.png"
            ok = gerar_barras_por_estrato(
                resumo_estratos_volume, coluna, caminho,
                titulo=f"{rotulo} — por faixa de volume de pedidos (ABC)",
                rotulo_y=rotulo)
            if ok:
                figuras.append(caminho.name)

    # --- Boxplots ---
    if estratos_volume is not None and not estratos_volume.empty:
        for metrica in ("WMAPE", "MASE"):
            caminho = outputs_dir / f"fig_boxplot_{metrica.lower()}_volume.png"
            ok = gerar_boxplot_por_estrato(
                df_long, estratos_volume, metrica, "estrato_volume", caminho,
                titulo=f"Distribuição de {metrica} por faixa de volume de pedidos")
            if ok:
                figuras.append(caminho.name)

    if estratos_teste is not None and not estratos_teste.empty:
        caminho = outputs_dir / "fig_boxplot_wmape_ativo_cessado.png"
        ok = gerar_boxplot_por_estrato(
            df_long, estratos_teste, "WMAPE", "estrato_teste", caminho,
            titulo="Distribuição de WMAPE — SKUs ativos x cessados")
        if ok:
            figuras.append(caminho.name)

    # --- Real x previsto dos SKUs de maior volume ---
    figuras += gerar_figuras_top_volume(previsoes_por_modelo, estratos_volume,
                                        outputs_dir, n_skus=n_skus_top)

    print(f"[graficos-estratos] {len(figuras)} figura(s) gerada(s).")
    return figuras


In [ ]:
%%writefile /content/src/consolidado.py
"""
CONSOLIDACAO DAS SAIDAS EM MARKDOWN.

Sem este modulo, boa parte do que o pipeline produz existiria apenas no console
do notebook -- resumos, diagnosticos, contagens -- e se perderia ao fechar a
sessao do Colab, restando os CSVs soltos, sem o contexto de leitura.

Ele garante DUAS coisas:

  1. `LOG_EXECUCAO.md` — captura literal de tudo o que foi impresso durante a
     execucao (via um "tee" no stdout). E o registro bruto e completo.

  2. `RESULTADOS_CONSOLIDADO.md` — todas as tabelas de resultado formatadas em
     markdown, na ordem em que foram produzidas, com as notas de leitura de
     cada uma. E o documento para consultar.

O mecanismo e por construcao: o pipeline usa `Consolidado.mostrar(...)`, que
IMPRIME e REGISTRA ao mesmo tempo. Nao ha como uma tabela aparecer no console
sem aparecer no markdown.
"""
from __future__ import annotations

import sys

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# 1) Captura do stdout (LOG_EXECUCAO.md)
# ---------------------------------------------------------------------------
class _Tee:
    """Duplica a escrita em stdout: uma copia vai para o console (para que o
    notebook continue mostrando tudo em tempo real) e outra para um buffer em
    memoria, gravado ao final como LOG_EXECUCAO.md."""

    def __init__(self, original):
        self._original = original
        self.buffer: list[str] = []
        self.run_dir = None
        self._salvo = False

    def write(self, texto):
        self._original.write(texto)
        self.buffer.append(texto)
        return len(texto)

    def flush(self):
        self._original.flush()

    def __getattr__(self, nome):
        # Delega isatty, encoding, reconfigure, etc. para o stream original.
        return getattr(self._original, nome)

    def texto(self) -> str:
        return "".join(self.buffer)


def iniciar_captura() -> _Tee:
    tee = _Tee(sys.stdout)
    sys.stdout = tee
    return tee


def salvar_log(tee, run_dir) -> None:
    if tee is None or run_dir is None or getattr(tee, "_salvo", False):
        return
    conteudo = (
        "# Log completo da execucao\n\n"
        "Captura literal de tudo o que o pipeline imprimiu durante esta execucao "
        "(Item 26). Para as tabelas formatadas, ver `RESULTADOS_CONSOLIDADO.md`; "
        "para a leitura interpretada, ver `RELATORIO_DECISAO.md`.\n\n"
        "```\n" + tee.texto() + "\n```\n"
    )
    (run_dir / "LOG_EXECUCAO.md").write_text(conteudo, encoding="utf-8")
    tee._salvo = True


def encerrar_captura(tee) -> None:
    """Restaura o stdout original. Grava o log se ainda nao tiver sido gravado
    (caso de excecao no meio da execucao — o log parcial e justamente o que
    interessa para diagnosticar)."""
    if tee is None:
        return
    try:
        salvar_log(tee, getattr(tee, "run_dir", None))
    finally:
        if sys.stdout is tee:
            sys.stdout = tee._original


# ---------------------------------------------------------------------------
# 2) Tabelas em markdown (RESULTADOS_CONSOLIDADO.md)
# ---------------------------------------------------------------------------
def df_para_markdown(df: pd.DataFrame, max_linhas: int = 200, casas: int = 4) -> str:
    """Tabela markdown sem depender do pacote opcional `tabulate`."""
    if df is None or len(df) == 0:
        return "_(vazio)_\n"
    d = df.head(max_linhas).copy()

    def _fmt(v):
        if v is None:
            return ""
        if isinstance(v, float) or isinstance(v, np.floating):
            if pd.isna(v):
                return ""
            if abs(v) >= 1e6 or (v != 0 and abs(v) < 1e-4):
                return f"{v:.3e}"
            return f"{v:.{casas}g}"
        if isinstance(v, (int, np.integer)):
            return str(int(v))
        return "" if (isinstance(v, float) and pd.isna(v)) else str(v)

    cabecalho = "| " + " | ".join(str(c) for c in d.columns) + " |"
    separador = "| " + " | ".join(["---"] * len(d.columns)) + " |"
    linhas = ["| " + " | ".join(_fmt(v) for v in row) + " |"
              for row in d.itertuples(index=False)]
    extra = ""
    if len(df) > max_linhas:
        extra = f"\n_({len(df) - max_linhas} linha(s) omitida(s) — ver o CSV correspondente)_\n"
    return "\n".join([cabecalho, separador] + linhas) + "\n" + extra


class Consolidado:
    """Acumula os blocos de resultado e escreve RESULTADOS_CONSOLIDADO.md.

    Uso no pipeline:
        cons = Consolidado()
        cons.secao("Experimento 1")
        cons.mostrar("Resumo Exp1", resumo1, nota="Medianas por SKU.")
    `mostrar` imprime no console E registra no markdown, garantindo que os dois
    nunca fiquem fora de sincronia."""

    def __init__(self):
        self.blocos: list[str] = []

    def secao(self, titulo: str, nivel: int = 2) -> None:
        self.blocos.append(f"\n{'#' * nivel} {titulo}\n")

    def texto(self, txt: str, imprimir: bool = False) -> None:
        self.blocos.append(txt.rstrip() + "\n")
        if imprimir:
            print(txt)

    def tabela(self, titulo: str, df: pd.DataFrame, nota: str | None = None,
               arquivo: str | None = None) -> None:
        self.blocos.append(f"\n### {titulo}\n")
        if nota:
            self.blocos.append(f"_{nota}_\n")
        self.blocos.append(df_para_markdown(df))
        if arquivo:
            self.blocos.append(f"\n_Arquivo: `{arquivo}`_\n")

    def mostrar(self, titulo: str, df: pd.DataFrame, nota: str | None = None,
                arquivo: str | None = None) -> None:
        """Imprime no console e registra no markdown — a mesma tabela."""
        print(f"\n=== {titulo} ===")
        if nota:
            print(f"  ({nota})")
        if df is None or len(df) == 0:
            print("  (vazio)")
        else:
            print(df.to_string(index=False))
        self.tabela(titulo, df, nota=nota, arquivo=arquivo)

    def figuras(self, titulo: str, nomes: list) -> None:
        if not nomes:
            return
        self.blocos.append(f"\n### {titulo}\n")
        for nome in nomes:
            self.blocos.append(f"![{nome}]({nome})\n")

    def escrever(self, run_dir, metadata: dict | None = None) -> str:
        cabecalho = [
            "# Resultados consolidados da execucao\n",
            "_Gerado automaticamente (Item 26). Reune, em um unico documento, todas as "
            "tabelas que o pipeline produziu — as mesmas que aparecem no console e nos "
            "CSVs da pasta. Para a leitura interpretada (o que usar no TCC), ver "
            "`RELATORIO_DECISAO.md`; para o log bruto, `LOG_EXECUCAO.md`._\n",
        ]
        if metadata:
            cabecalho.append(f"- Execucao: `{metadata.get('timestamp', '')}`")
            cabecalho.append(f"- Treino: {metadata.get('treino', '')} | "
                             f"Teste: {metadata.get('teste', '')}")
            cabecalho.append(f"- SKUs modelados: {metadata.get('amostragem_skus', '')} | "
                             f"Duracao: {metadata.get('duracao_s', '')} s\n")
        texto = "\n".join(cabecalho + self.blocos) + "\n"
        (run_dir / "RESULTADOS_CONSOLIDADO.md").write_text(texto, encoding="utf-8")
        return texto


In [ ]:
%%writefile /content/src/decisao.py
"""
RELATORIO DE DECISAO.

Consolida, em um unico arquivo (RELATORIO_DECISAO.md na pasta da execucao),
tudo o que e preciso ler para interpretar os resultados desta execucao:

  1. Quais opcoes estavam ligadas nesta execucao (e o que cada uma implica em
     termos de mudanca na documentacao);
  2. Ranking dos modelos por metrica, com destaque para o vencedor;
  3. O resultado contra o baseline Zero — o teste que diz se a metrica esta
     medindo capacidade preditiva ou apenas proximidade de zero;
  4. A estratificacao ativos x cessados;
  5. Um bloco de "leitura sugerida" com as conclusoes que os numeros
     sustentam e as que NAO sustentam.

Este modulo nao calcula nada de novo — apenas organiza o que os demais
modulos ja produziram.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config
from .config import COLS
from .evaluate import ESTRATO_ATIVO, ESTRATO_CESSADO


# Mapa: flag -> (rotulo, impacto na documentacao)
_IMPACTO_FLAGS = {
    # Bloco A — correcoes
    "ORDENAR_TABELA_CRONOLOGICA": ("Item 11 — CV cronologica na busca", "Nenhum (corrige o que o TCC ja afirma)"),
    "TRUNCAR_PRE_LANCAMENTO": ("Item 12 — truncagem pre-lancamento", "Nenhum (1 linha no funil da Secao 4.1.2)"),
    "SCORE_RELATIVO_GEOMETRICO": ("Item 13 — score relativo geometrico", "Nenhum (nota de rodape)"),
    "CORTE_ATRIBUTOS_TREINO": ("Item 14 — corte de treino nos atributos", "Nenhum"),
    "SALVAR_PREVISOES_BRUTAS": ("Item 15 — salvar previsoes brutas", "Nenhum"),
    # Bloco B — analises novas
    "ESTRATIFICAR_ATIVOS_CESSADOS": ("Item 16 — estratos ativo/cessado", "Opcional: 1 subsecao nos resultados"),
    "BASELINE_ZERO": ("Item 17 — baseline Zero (controle)", "Opcional: 1 linha nas tabelas + 1 paragrafo"),
    "METRICAS_POOLED": ("Item 18 — metricas agregadas (pooled)", "Opcional: colunas na tabela de resultados"),
    "CLASSIFICACAO_ROBUSTEZ_DIARIA": ("Item 19 — ADI/CV2 diario (robustez)", "Opcional: 1 paragrafo na Secao 4.1.4"),
    # Bloco B2 — recorte gerencial e consolidacao
    "ESTRATIFICAR_POR_VOLUME": ("Item 25 — faixas de volume de pedidos (ABC)", "Opcional: 1 subsecao gerencial"),
    "CONSOLIDAR_MARKDOWN": ("Item 26 — consolidacao em markdown", "Nenhum"),
    "GRAFICOS_POR_ESTRATO": ("Item 27 — figuras por estrato e top volume", "Opcional: figuras adicionais"),
    # Bloco C — modelagem
    "FEATURES_INTERMITENCIA": ("Item 21 — features de intermitencia", "EXIGE atualizar a Tabela 4.3"),
    "TUNAR_UMA_VEZ_GLOBAL": ("Item 24 — tuning unico entre experimentos", "EXIGE 1 frase na Secao 4.2.3"),
    # Bloco D — parcimonia das variaveis
    "REMOVER_FEATURES_REDUNDANTES": ("Item 28 — remocao de features redundantes",
                                     "EXIGE atualizar a Tabela 4.3 (ver selecao_variaveis.md)"),
    "METRICA_SELECAO_HIPER": ("Item E4 — metrica de selecao dos hiperparametros",
                              "EXIGE 1 paragrafo na Secao 4.3.3"),
}


def _bloco_configuracao(metadata: dict | None = None) -> list:
    """Secao 1: quais opcoes estavam ligadas e o que cada uma custa em texto.

    Le o estado direto do modulo `config`, ja reatribuido pelo pipeline, entao
    reflete o que a execucao de fato usou -- nao os defaults. A cadencia do
    walk-forward vem do `metadata`, por ser local a execucao."""
    linhas = ["## 1. Configuracao desta execucao\n",
              "| Opcao | Estado | Impacto no texto do TCC |",
              "| --- | --- | --- |"]
    for flag, (rotulo, impacto) in _IMPACTO_FLAGS.items():
        estado = "LIGADO" if getattr(config, flag, False) else "desligado"
        linhas.append(f"| {rotulo} | {estado} | {impacto} |")

    linhas.append(f"| Item 22 — encoding do SKU | `{config.ENCODING_SKU}` | "
                  f"{'EXIGE 1 linha na Tabela 4.3' if config.ENCODING_SKU != 'ordinal' else 'Nenhum'} |")
    linhas.append(f"| Item 23 — grade do XGBoost | `{config.GRADE_XGB_MODO}` | "
                  f"{'Opcional: reportar cobertura' if config.GRADE_XGB_MODO != 'completa' else 'Nenhum'} |")
    linhas.append(f"| Item 10 — objetivo do XGBoost | `{config.XGB_OBJETIVO}` | "
                  f"{'EXIGE 1 paragrafo (ou apresentar como analise de sensibilidade)' if config.XGB_OBJETIVO != 'reg:squarederror' else 'Nenhum'} |")
    _n_exo = len(config.colunas_calendario_rico())
    _n_exo_total = len(config.COLUNAS_CALENDARIO_RICO)
    linhas.append(f"| Item 29 — exogenas do Cenario 2 | {_n_exo} de {_n_exo_total} | "
                  f"{'EXIGE 1 frase justificando o corte (ver selecao_variaveis.md)' if _n_exo != _n_exo_total else 'Nenhum'} |")
    linhas.append("")
    if metadata and metadata.get("item31_stride_origem_treino_dias") is not None:
        _st = metadata["item31_stride_origem_treino_dias"]
        _rf = metadata.get("item31_janelas_por_refit")
        _n = metadata.get("item31_refits_no_teste")
        _cadencia = {1: "semanal", 2: "quinzenal", 4: "mensal"}.get(_rf, f"a cada {_rf} semanas")
        linhas.append(
            f"- **Cadencia do walk-forward (Item 31):** reajuste **{_cadencia}** "
            f"({_n} retreinos no ano de teste); passo entre origens de treino = {_st} dia(s). "
            f"Declare a cadencia na Secao 4.2.2 — o texto precisa dizer a mesma coisa."
        )
        if _st % 7 == 0:
            linhas.append(
                "  - **ATENCAO:** passo multiplo de 7 — todas as origens de treino caem no "
                "mesmo dia da semana e `dia_semana_alvo` fica determinado pelo `horizonte`. "
                "O Item 9 nao tem efeito nesta execucao."
            )
        elif _st > 1:
            linhas.append(
                f"  - O passo de {_st} dias e primo com 7, entao as origens circulam pelos dias "
                "da semana e todas as combinacoes (dia da semana, horizonte) aparecem no treino. "
                "A cobertura do periodo e integral: nenhuma venda e descartada, apenas se "
                "amostram as datas de origem."
            )
    linhas.append(f"- Busca de hiperparametros: {config.descrever_busca_rf()}")
    _teto = (metadata or {}).get("max_linhas_tuning", "(nao registrado)")
    linhas.append(
        f"- Teto de linhas na busca (max_linhas_tuning): **{_teto}** — o ajuste "
        f"final usa sempre a tabela de treino inteira. Duas execucoes so sao "
        f"comparaveis entre si com o mesmo teto; declare o valor na Secao 4.3.3.")
    _reap = (metadata or {}).get("reaproveitar_tabela_treino")
    if _reap is not None:
        linhas.append(
            f"- Tabela de treino compartilhada entre modelos: **{_reap}**"
            + ("" if not _reap else
               " — as previsoes sao as mesmas, mas o `custo_total` da Secao 4.3.1 "
               "NAO inclui a montagem da tabela (ver a linha `CACHE` em "
               "tempos_execucao.csv) e nao e comparavel ao de uma execucao com "
               "este parametro desligado."))
    linhas.append(f"- Metrica de selecao dos hiperparametros (Item E4): "
                  f"{config.METRICA_SELECAO_HIPER}")
    linhas.append(f"- Granularidade da classificacao ADI/CV2: `{config.GRANULARIDADE_CLASSIFICACAO}`")
    if config.EXOGENAS_EXCLUIDAS:
        linhas.append("- Exogenas excluidas do Cenario 2 (criterio medido no treino): "
                      + ", ".join(f"`{c}`" for c in config.EXOGENAS_EXCLUIDAS))
    linhas.append("")
    return linhas


def _ranking(resumo: pd.DataFrame, coluna: str, rotulo: str) -> list:
    """Ordena os modelos por uma coluna de erro (menor = melhor) e marca o
    primeiro. Devolve lista vazia quando a coluna nao existe na tabela."""
    if resumo is None or resumo.empty or coluna not in resumo.columns:
        return []
    ordenado = resumo[["modelo", coluna]].dropna().sort_values(coluna)
    if ordenado.empty:
        return []
    linhas = [f"**{rotulo}** (menor = melhor):", ""]
    for pos, (_, r) in enumerate(ordenado.iterrows(), start=1):
        marca = " ← melhor" if pos == 1 else ""
        linhas.append(f"{pos}. {r['modelo']}: {r[coluna]:.4g}{marca}")
    linhas.append("")
    return linhas


def _bloco_ranking(resumo1: pd.DataFrame, resumo_pooled: pd.DataFrame | None,
                   resumo_semanal: pd.DataFrame | None) -> list:
    """Secao 2: rankings por metrica, em tres niveis -- mediana por SKU,
    agregado global e semanal.

    Divergencia entre os tres niveis nao e erro: cada um responde a uma
    pergunta diferente. A secao 3 (baseline Zero) e que diz quais deles
    realmente medem capacidade preditiva."""
    linhas = ["## 2. Ranking dos modelos\n"]
    for coluna, rotulo in [("WMAPE_mediana", "WMAPE mediano por SKU (diario)"),
                           ("MASE_mediana", "MASE mediano por SKU (diario)"),
                           ("RMSSE_mediana", "RMSSE mediano por SKU (diario)"),
                           ("MAE_mediana", "MAE mediano por SKU (diario)")]:
        linhas += _ranking(resumo1, coluna, rotulo)

    if resumo_pooled is not None and not resumo_pooled.empty:
        linhas.append("### Metricas agregadas globais — Item 18\n")
        for coluna, rotulo in [("WMAPE_pooled", "WMAPE agregado (soma|e| / soma y)"),
                               ("MAE_pooled", "MAE agregado")]:
            linhas += _ranking(resumo_pooled, coluna, rotulo)

    if resumo_semanal is not None and not resumo_semanal.empty:
        linhas.append("### Nivel semanal — Item 7 (comparacao mais justa; ver evaluate.avaliar_nivel_semanal)\n")
        for coluna, rotulo in [("WMAPE_sem_mediana", "WMAPE semanal mediano"),
                               ("MASE_sem_mediana", "MASE semanal mediano")]:
            linhas += _ranking(resumo_semanal, coluna, rotulo)
    return linhas


def _bloco_zero(resumo1: pd.DataFrame, resumo_pooled: pd.DataFrame | None) -> list:
    """Compara cada modelo com o baseline Zero e diz, em texto, o que isso
    significa para a leitura do resultado."""
    if resumo1 is None or resumo1.empty or "Zero" not in set(resumo1["modelo"]):
        return ["## 3. Baseline Zero\n",
                "_Nao executado nesta rodada (Item 17 desligado)._\n"]

    linhas = ["## 3. Baseline Zero — o controle (Item 17)\n",
              "O baseline Zero preve 0 sempre. Nao e um modelo proposto: e o analogo do "
              "classificador de classe majoritaria. A pergunta que ele responde e "
              "**\"a metrica esta medindo capacidade preditiva, ou apenas proximidade de zero?\"**\n"]

    zero = resumo1[resumo1["modelo"] == "Zero"].iloc[0]
    linhas.append("| Metrica | Zero | Melhor modelo (exceto Zero) | Zero vence? |")
    linhas.append("| --- | --- | --- | --- |")

    veredictos = {}
    for coluna, rotulo in [("WMAPE_mediana", "WMAPE mediano"),
                           ("MASE_mediana", "MASE mediano"),
                           ("RMSSE_mediana", "RMSSE mediano"),
                           ("MAE_mediana", "MAE mediano")]:
        if coluna not in resumo1.columns:
            continue
        outros = resumo1[resumo1["modelo"] != "Zero"][["modelo", coluna]].dropna()
        if outros.empty or pd.isna(zero.get(coluna)):
            continue
        melhor = outros.sort_values(coluna).iloc[0]
        zero_vence = float(zero[coluna]) <= float(melhor[coluna])
        veredictos[rotulo] = zero_vence
        linhas.append(f"| {rotulo} | {zero[coluna]:.4g} | {melhor['modelo']} "
                      f"({melhor[coluna]:.4g}) | {'SIM' if zero_vence else 'nao'} |")

    if resumo_pooled is not None and not resumo_pooled.empty and "Zero" in set(resumo_pooled["modelo"]):
        z = resumo_pooled[resumo_pooled["modelo"] == "Zero"].iloc[0]
        outros = resumo_pooled[resumo_pooled["modelo"] != "Zero"][["modelo", "WMAPE_pooled"]].dropna()
        if not outros.empty:
            melhor = outros.sort_values("WMAPE_pooled").iloc[0]
            zero_vence = float(z["WMAPE_pooled"]) <= float(melhor["WMAPE_pooled"])
            veredictos["WMAPE agregado"] = zero_vence
            linhas.append(f"| WMAPE agregado | {z['WMAPE_pooled']:.4g} | {melhor['modelo']} "
                          f"({melhor['WMAPE_pooled']:.4g}) | {'SIM' if zero_vence else 'nao'} |")

    linhas.append("")
    n_vence = sum(1 for v in veredictos.values() if v)
    if n_vence == len(veredictos) and veredictos:
        linhas.append("> **Leitura:** o baseline Zero vence em TODAS as metricas pontuais. "
                      "Afirmar que um modelo \"venceu\" nessas metricas equivale a afirmar que "
                      "prever zero venceu. A conclusao defensavel do trabalho passa a ser sobre a "
                      "INADEQUACAO das metricas pontuais em cauda longa (Kolassa, 2016), com "
                      "evidencia gerada nestes dados — o que e um resultado mais forte, nao mais "
                      "fraco. Verifique tambem a estratificacao da secao 4 e o nivel semanal.\n")
    elif n_vence == 0 and veredictos:
        linhas.append("> **Leitura:** nenhum modelo perde para o Zero. As metricas estao medindo "
                      "capacidade preditiva de fato, e o ranking da secao 2 pode ser usado "
                      "diretamente no TCC.\n")
    elif veredictos:
        vencidas = [k for k, v in veredictos.items() if v]
        preservadas = [k for k, v in veredictos.items() if not v]
        linhas.append(f"> **Leitura:** o Zero vence em {', '.join(vencidas)} mas NAO em "
                      f"{', '.join(preservadas)}. Use {', '.join(preservadas)} como metrica "
                      "principal do TCC e reporte as demais com a ressalva de que sao "
                      "degeneradas nesta base.\n")
    return linhas


def _bloco_estratos(resumo_estratos: pd.DataFrame | None) -> list:
    """Secao 4: resultados separados entre SKUs ativos e cessados no teste.

    Cada estrato e lido por uma metrica diferente -- WMAPE no Ativo, MAE no
    Cessado, onde o WMAPE e indefinido."""
    if resumo_estratos is None or resumo_estratos.empty:
        return ["## 4. Estratificacao ativos x cessados\n",
                "_Nao executada nesta rodada (Item 16 desligado)._\n"]

    linhas = ["## 4. Estratificacao ativos x cessados (Item 16)\n",
              "**Ativo** = vendeu pelo menos 1 unidade no teste. "
              "**Cessado** = zero vendas em todo o teste.\n",
              "Estratificacao de RELATORIO — todos os SKUs foram modelados; nenhuma decisao de "
              "modelagem usou o periodo de teste, portanto nao ha vazamento.\n"]

    for estrato, pergunta, coluna in [
        (ESTRATO_ATIVO, "qual modelo preve melhor a demanda que de fato ocorreu?", "WMAPE_mediana"),
        (ESTRATO_CESSADO, "qual modelo reconheceu a obsolescencia mais rapido?", "MAE_mediana"),
    ]:
        sub = resumo_estratos[resumo_estratos["estrato"] == estrato]
        if sub.empty:
            continue
        n = int(sub["n_skus"].iloc[0])
        linhas.append(f"### {estrato} (n = {n} SKUs) — {pergunta}\n")
        if coluna in sub.columns:
            linhas += _ranking(sub, coluna, f"{coluna.replace('_mediana', '')} mediano")
        else:
            linhas.append("_(metrica indisponivel — ligue METRICAS_EXTRAS)_\n")
    return linhas


def _bloco_volume(resumo_volume: pd.DataFrame | None,
                  concentracao: pd.DataFrame | None,
                  cruzamento: pd.DataFrame | None,
                  fonte_volume: str | None) -> list:
    """O recorte gerencial: onde a previsao vale mais."""
    if resumo_volume is None or resumo_volume.empty:
        return ["## 5. Faixas de volume de pedidos (visao gerencial)\n",
                "_Nao executada nesta rodada (Item 25 desligado)._\n"]

    linhas = ["## 5. Faixas de volume de pedidos — visao gerencial (Item 25)\n",
              "A mediana entre SKUs trata igualmente um produto que concentra milhares de "
              "pedidos e um que vendeu 3 vezes no ano. Isso responde a pergunta cientifica "
              "(QP1), mas nao a gerencial: **prever melhor onde ha volume significa atender "
              "melhor um numero muito maior de clientes finais**.\n",
              "Faixas ABC definidas pelo numero de pedidos no periodo de TREINO "
              "(sem vazamento — e a informacao que o gestor teria em maos).\n"]
    if fonte_volume:
        linhas.append(f"- Fonte do volume: _{fonte_volume}_\n")

    if concentracao is not None and not concentracao.empty:
        linhas.append("### Concentracao (Pareto)\n")
        linhas.append("| Faixa | n SKUs | % SKUs | % dos pedidos | % da demanda |")
        linhas.append("| --- | --- | --- | --- | --- |")
        for _, r in concentracao.iterrows():
            linhas.append(f"| {r['estrato']} | {int(r['n_skus'])} | {r['pct_skus']:.1f}% | "
                          f"{r['pct_volume_pedidos']:.1f}% | {r['pct_demanda_treino']:.1f}% |")
        topo = concentracao.iloc[0]
        linhas.append("")
        linhas.append(f"> **Leitura:** os {topo['pct_skus']:.0f}% de SKUs do topo concentram "
                      f"**{topo['pct_volume_pedidos']:.1f}% dos pedidos** e "
                      f"{topo['pct_demanda_treino']:.1f}% da demanda. Quanto maior essa "
                      "concentracao, mais defensavel e priorizar a acuracia nessa faixa — e "
                      "mais relevante e o ranking dela do que o ranking geral.\n")

    for _, faixa in (resumo_volume[["estrato"]].drop_duplicates().iterrows()
                     if "estrato" in resumo_volume.columns else []):
        nome = faixa["estrato"]
        sub = resumo_volume[resumo_volume["estrato"] == nome]
        if sub.empty:
            continue
        n = int(sub["n_skus"].iloc[0])
        linhas.append(f"### {nome} (n = {n} SKUs)\n")
        for coluna, rotulo in [("WMAPE_pooled", "WMAPE agregado da faixa"),
                               ("WMAPE_mediana", "WMAPE mediano por SKU"),
                               ("MAE_mediana", "MAE mediano por SKU")]:
            linhas += _ranking(sub, coluna, rotulo)

    if cruzamento is not None and not cruzamento.empty:
        linhas.append("### Obsolescencia por faixa de volume\n")
        linhas.append("_A obsolescencia se concentra na cauda (esperado) ou tambem atinge "
                      "SKUs de alto giro (achado relevante)?_\n")
        linhas.append("| Faixa de volume | Estrato no teste | n SKUs | % da faixa |")
        linhas.append("| --- | --- | --- | --- |")
        for _, r in cruzamento.iterrows():
            linhas.append(f"| {r['estrato_volume']} | {r['estrato_teste']} | "
                          f"{int(r['n_skus'])} | {r['pct_no_estrato_volume']:.1f}% |")
        linhas.append("")
    return linhas


def _bloco_leitura(resumo1: pd.DataFrame) -> list:
    """Secao 6: o que se pode e o que nao se pode afirmar com estes numeros.

    E a secao que existe para evitar a conclusao errada -- os limites sao
    propriedades conhecidas das metricas e do desenho, nao dependem do
    resultado de uma rodada especifica."""
    linhas = ["## 6. O que os numeros sustentam (e o que nao sustentam)\n"]
    linhas.append("Sustentam (se o Wilcoxon confirmar e o Zero nao vencer a metrica em questao):")
    linhas.append("- a ordenacao PAREADA entre modelos (Wilcoxon + r bisserial), nao a ordenacao "
                  "por mediana marginal — as duas podem divergir e respondem perguntas diferentes;")
    linhas.append("- a comparacao de custo computacional (Secao 4.3.1), que agora usa wall-clock "
                  "comparavel para todos os modelos;")
    linhas.append("- o efeito das variaveis exogenas (QP2), desde que o Item 24 esteja ligado — "
                  "sem ele, a diferenca entre Cenario 1 e Cenario 2 se mistura com o sorteio da "
                  "busca de hiperparametros.\n")
    linhas.append("NAO sustentam:")
    linhas.append("- conclusoes a partir do sMAPE: com ~200 para todos os modelos, ele esta "
                  "saturado (200% e o valor limite quando um dos lados e zero) e nao discrimina;")
    linhas.append("- conclusoes a partir da mediana entre SKUs sem olhar a estratificacao, quando "
                  "a fracao de SKUs cessados esta proxima de 50% — a mediana cai exatamente na "
                  "fronteira entre os dois grupos;")
    if resumo1 is not None and not resumo1.empty and "pct_WMAPE_indefinido" in resumo1.columns:
        pct = float(resumo1["pct_WMAPE_indefinido"].max())
        linhas.append(f"- comparar o n do Wilcoxon de WMAPE com o de MASE: sao populacoes "
                      f"diferentes ({pct:.1f}% dos SKUs tem WMAPE indefinido). Declare isso.\n")
    return linhas


def gerar_relatorio_decisao(
    outputs_dir,
    resumo1: pd.DataFrame | None = None,
    resumo_pooled: pd.DataFrame | None = None,
    resumo_semanal: pd.DataFrame | None = None,
    resumo_estratos: pd.DataFrame | None = None,
    metadata: dict | None = None,
    resumo_volume: pd.DataFrame | None = None,
    concentracao_volume: pd.DataFrame | None = None,
    cruzamento_estratos: pd.DataFrame | None = None,
    fonte_volume: str | None = None,
) -> str:
    """Escreve RELATORIO_DECISAO.md na pasta da execucao e devolve o texto."""
    partes = ["# Relatorio de decisao — o que usar no TCC\n",
              "_Gerado automaticamente pelo pipeline (Item 20). Este arquivo nao calcula nada "
              "de novo: organiza o que os demais modulos produziram, para que a escolha do que "
              "entra no TCC seja feita a partir de um unico documento._\n",
              "_Companheiros: `RESULTADOS_CONSOLIDADO.md` (todas as tabelas) e "
              "`LOG_EXECUCAO.md` (log bruto)._\n"]
    if metadata:
        partes.append(f"- Execucao: `{metadata.get('timestamp', '')}`")
        partes.append(f"- Periodo de teste: {metadata.get('teste', '')}")
        partes.append(f"- SKUs modelados: {metadata.get('amostragem_skus', '')}")
        partes.append(f"- Duracao: {metadata.get('duracao_s', '')} s\n")

    partes += _bloco_configuracao(metadata)
    partes += _bloco_ranking(resumo1, resumo_pooled, resumo_semanal)
    partes += _bloco_zero(resumo1, resumo_pooled)
    partes += _bloco_estratos(resumo_estratos)
    partes += _bloco_volume(resumo_volume, concentracao_volume, cruzamento_estratos, fonte_volume)
    partes += _bloco_leitura(resumo1)

    texto = "\n".join(partes) + "\n"
    (outputs_dir / "RELATORIO_DECISAO.md").write_text(texto, encoding="utf-8")
    return texto

## 5. Orquestração da pipeline

Nesta etapa é definido o orquestrador (`rodar(...)`), responsável por executar o fluxo completo e salvar os artefatos da execução.

In [ ]:
%%writefile /content/pipeline_colab.py
"""Orquestrador do pipeline para Colab: uma funcao `rodar()`, sem argparse.

ORDEM DE EXECUCAO
  1. aplica os parametros recebidos sobre o modulo `config`
  2. carrega, agrega, expande e trunca as series          (preprocess)
  3. aplica elegibilidade e classifica ADI/CV2            (adi_cv2)
  4. estratifica por volume de pedidos, no treino
  5. constroi as features de origem, uma unica vez        (cache)
  6. Experimento 1: SBA, TSB, XGBoost, RF e o baseline Zero
  7. Experimento 2: cenarios de variaveis exogenas
  8. grava metricas, testes, figuras e relatorios

Cada opcao da celula de parametros esta documentada em src/config.py.

O QUE LER DEPOIS DE CADA EXECUCAO, nesta ordem:
  RELATORIO_DECISAO.md       o que os numeros sustentam e o que nao sustentam
  selecao_variaveis.md       quais variaveis entraram, quais sairam e por que
  RESULTADOS_CONSOLIDADO.md  todas as tabelas formatadas
  METADATA.md                a configuracao exata e as versoes de biblioteca
  LOG_EXECUCAO.md            o log bruto
"""
from __future__ import annotations
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src import (config, preprocess, eda, sazonalidade, adi_cv2, evaluate, historico,
                 ml_models, correlacao, score_relativo, graficos_resultados,
                 graficos_estratos, decisao, consolidado)
from src.config import COLS
from src.features import (features_experimento1, features_cenario_1, features_cenario_2,
                          diagnosticar_origem_idade)
from src.ml_models import (ConfigWalkForward, EspecificacaoModelo, executar_walk_forward,
                          executar_walk_forward_multi, construir_cache_features)
from src.baselines import executar_walk_forward_baseline


def _amostra_estratificada(df_modelagem, df_classes, n):
    """Sorteia ~n SKUs preservando a proporcao de cada quadrante ADI/CV2.

    Usada so em rodadas de teste (parametro `amostra`), para que a amostra
    reduzida ainda represente os dois quadrantes de interesse."""
    skus = set(df_modelagem[COLS.sku].unique())
    cm = df_classes[df_classes["sku"].isin(skus)].copy()
    frac = n / len(cm)
    am = (cm.groupby("quadrante", group_keys=False)
            .apply(lambda g: g.sample(frac=frac, random_state=config.RANDOM_STATE), include_groups=False))
    am = am.merge(cm[["sku", "quadrante"]], on="sku", how="left")
    df = df_modelagem[df_modelagem[COLS.sku].isin(am["sku"])].reset_index(drop=True)
    return df, am


def _versoes_bibliotecas() -> str:
    """Versoes das bibliotecas que produzem numeros, para gravar no METADATA.

    Por que importa: duas execucoes sobre a MESMA base, com a mesma semente,
    ja produziram previsoes diferentes de SBA e TSB — modelos deterministicos,
    sem GPU e sem componente aleatorio — em 18% e 43% das linhas, porque o
    ambiente do Colab foi reinstalado entre as sessoes. Sem esta informacao no
    METADATA nao ha como saber, meses depois, se uma diferenca veio de uma
    opcao do pipeline ou de uma atualizacao de biblioteca."""
    import platform
    partes = [f"python {platform.python_version()}"]
    for nome, modulo in (("numpy", "numpy"), ("pandas", "pandas"),
                         ("sklearn", "sklearn"), ("xgboost", "xgboost"),
                         ("statsforecast", "statsforecast"), ("scipy", "scipy")):
        try:
            partes.append(f"{nome} {__import__(modulo).__version__}")
        except Exception:
            partes.append(f"{nome} ausente")
    return " | ".join(partes)


def _relatorio_selecao_variaveis(features_ativas, removidas, cal_exogenas) -> str:
    """Monta o selecao_variaveis.md: o que entrou, o que saiu e por que.

    Existe para responder a pergunta que se faz sobre qualquer selecao de
    variaveis: "com base em que voce cortou?". A resposta tem de ser
    verificavel e tem de vir do treino -- por isso o suporte amostral das
    exogenas e recalculado aqui, sobre a janela de treino desta execucao, em
    vez de ficar congelado num comentario do codigo."""
    linhas = [
        "# Selecao de variaveis (Items 28 e 29) — apoio a Secao 4.2.1\n",
        "Toda exclusao listada abaixo usa um criterio **estrutural** (identidade "
        "algebrica ou janela equivalente) ou **medido no periodo de treino** "
        f"(ate {config.TREINO_FIM}). Nenhuma usa metrica, teste de significancia "
        "ou importancia calculada sobre o periodo de teste — se usasse, a selecao "
        "de variaveis passaria a olhar o teste.\n",
    ]

    linhas.append("## Item 28 — features redundantes\n")
    if not removidas:
        linhas.append("_Nenhuma feature removida nesta execucao "
                      "(REMOVER_FEATURES_REDUNDANTES=False ou substituta ausente)._\n")
    else:
        linhas.append("| Feature removida | Justificativa |")
        linhas.append("| --- | --- |")
        for feature, motivo in removidas.items():
            linhas.append(f"| `{feature}` | {motivo} |")
        linhas.append("")
        linhas.append("> A remocao age sobre a lista de ENTRADA do modelo. As colunas "
                      "continuam sendo calculadas e aparecendo em `correlacao_features.md`, "
                      "que e o artefato que justifica a exclusao.\n")
    linhas.append(f"**Features do Experimento 1 nesta execucao ({len(features_ativas)}):** "
                  + ", ".join(f"`{c}`" for c in features_ativas) + "\n")

    linhas.append("## Item 29 — variaveis exogenas do Cenario 2\n")
    usadas = config.colunas_calendario_rico()
    excluidas = [c for c in config.COLUNAS_CALENDARIO_RICO if c not in usadas]
    linhas.append(
        "Escolher quais variaveis de calendario fazem sentido para o contexto e "
        "parte do desenho do Cenario 2, e nao um cenario novo: o Cenario 2 "
        "continua sendo 'o cenario das exogenas'. A escolha e explicita e "
        "verificavel.\n")

    if cal_exogenas is not None:
        try:
            cal = cal_exogenas.reset_index()
            col_data = cal.columns[0]
            cal[col_data] = pd.to_datetime(cal[col_data])
            trn = cal[(cal[col_data] >= pd.Timestamp(config.TREINO_INICIO))
                      & (cal[col_data] <= pd.Timestamp(config.TREINO_FIM))]
            tst = cal[(cal[col_data] >= pd.Timestamp(config.TESTE_INICIO))
                      & (cal[col_data] <= pd.Timestamp(config.TESTE_FIM))]
            linhas.append("| Variavel | Situacao | Dias ativos no TREINO | Dias ativos no teste |")
            linhas.append("| --- | --- | ---: | ---: |")
            for col in config.COLUNAS_CALENDARIO_RICO:
                if col not in cal.columns:
                    continue
                n_tr = int((pd.to_numeric(trn[col], errors="coerce").fillna(0) != 0).sum())
                n_te = int((pd.to_numeric(tst[col], errors="coerce").fillna(0) != 0).sum())
                situacao = "usada" if col in usadas else "**excluida**"
                linhas.append(f"| `{col}` | {situacao} | {n_tr:,} | {n_te:,} |")
            linhas.append("")
            linhas.append("> A coluna 'dias ativos no teste' e informativa (o calendario e "
                          "conhecido com antecedencia, entao nao ha vazamento em olha-la), "
                          "mas o CRITERIO de exclusao usa apenas a coluna do treino.\n")
        except Exception as exc:      # pragma: no cover — diagnostico opcional
            linhas.append(f"_Nao foi possivel calcular o suporte amostral: {exc}_\n")

    linhas.append(f"- **Usadas ({len(usadas)}):** " + ", ".join(f"`{c}`" for c in usadas))
    linhas.append(f"- **Excluidas ({len(excluidas)}):** "
                  + (", ".join(f"`{c}`" for c in excluidas) or "nenhuma") + "\n")
    return "\n".join(linhas)


def _resolver_cadencia_walk_forward(modo_fiel, stride_param, refit_param,
                                    permitir_alias) -> tuple:
    """Decide o passo entre origens de treino e a frequencia de refit.

    As duas grandezas sao independentes e respondem a perguntas diferentes:

      `janelas_por_refit`  de quantas em quantas semanas do teste o modelo e
                           retreinado. E DESENHO EXPERIMENTAL: 1 = reajuste
                           semanal (53 treinos no ano de teste), 4 = mensal
                           (14 treinos).

      `stride_origem_treino`  de quantos em quantos dias se toma uma data de
                           origem para montar a tabela de treino. E DENSIDADE
                           DA AMOSTRA: 1 = toda data (~16,9 M linhas nesta
                           base), 3 = a cada 3 dias (~5,6 M). Nenhuma venda e
                           descartada em qualquer passo -- toda venda continua
                           entrando nos lags e medias moveis de todas as
                           origens posteriores; o que muda e quantas fotos
                           quase identicas do historico entram como linha.

    `modo_fiel` continua funcionando como atalho: True equivale a (1, 1) e
    False a (7, 4). Qualquer um dos dois parametros explicitos tem precedencia
    sobre ele.

    POR QUE UM PASSO MULTIPLO DE 7 E REJEITADO
    O passo e aplicado sobre a lista ordenada de datas, entao um multiplo de 7
    poe TODAS as origens no mesmo dia da semana. Como o alvo e origem + h com
    h = 1..7, o dia da semana do alvo vira funcao deterministica do horizonte:
    `dia_semana_alvo` e `horizonte` passam a carregar a mesma informacao e o
    modelo nao consegue separar os dois efeitos. Isso anula o efeito da feature
    `dia_semana_alvo`, que existe justamente para dar ao ML o padrao semanal
    que SBA e TSB ja recebem via `proporcao_dia_semana` -- ou seja, reintroduz
    a assimetria informacional que ela pretendia eliminar.

    Um passo primo com 7 (3, 5, 13...) faz as origens circularem pelos dias da
    semana e cobre todas as combinacoes (dia da semana, horizonte).

    `permitir_alias_dia_semana=True` desativa a rejeicao. Existe para reproduzir
    execucoes anteriores de proposito, nao para uso corrente."""
    stride = int(stride_param) if stride_param is not None else (1 if modo_fiel else 7)
    refit = int(refit_param) if refit_param is not None else (1 if modo_fiel else 4)

    if stride < 1:
        raise ValueError(f"stride_origem_treino deve ser >= 1 (recebido: {stride}).")
    if refit < 1:
        raise ValueError(f"janelas_por_refit deve ser >= 1 (recebido: {refit}).")

    if stride % 7 == 0 and not permitir_alias:
        raise ValueError(
            f"[Item 31] stride_origem_treino={stride} e multiplo de 7: todas as datas de "
            f"origem cairiam no mesmo dia da semana, e o dia da semana do alvo viraria "
            f"funcao exata do horizonte (alvo = origem + h, h = 1..7). Com isso "
            f"`dia_semana_alvo` deixa de acrescentar informacao ao `horizonte` e o Item 9 "
            f"perde o efeito.\n"
            f"  Use um passo primo com 7 — stride_origem_treino=3 e o mais proximo em "
            f"custo (~2,3x o de 7).\n"
            f"  Para reproduzir uma execucao anterior mesmo assim, passe "
            f"permitir_alias_dia_semana=True."
        )

    origem = "modo_fiel" if (stride_param is None and refit_param is None) else "explicito"
    alias = " | ATENCAO: passo multiplo de 7, dia_semana_alvo aliased com horizonte" \
        if stride % 7 == 0 else ""
    print(f"[Item 31] Cadencia do walk-forward ({origem}): passo entre origens de treino "
          f"= {stride} dia(s) | refit a cada {refit} janela(s) de 7 dias{alias}")
    return stride, refit


def _salvar_resultados_modelo(run_dir, prefixo, nome, previsoes, metricas):
    """Grava metricas e previsoes em arquivos distintos.

      metricas_<prefixo>_<modelo>.csv       uma linha por SKU
      previsoes_<prefixo>_<modelo>.parquet  uma linha por SKU x data

    Sao as previsoes brutas que permitem refazer qualquer analise -- agregacao
    semanal, metricas novas, graficos -- sem re-executar o pipeline. Cai para
    CSV se o parquet nao estiver disponivel no ambiente."""
    metricas.to_csv(run_dir / f"metricas_{prefixo}_{nome.lower()}.csv", index=False)
    if config.SALVAR_PREVISOES_BRUTAS and previsoes is not None and not previsoes.empty:
        try:
            previsoes.to_parquet(run_dir / f"previsoes_{prefixo}_{nome.lower()}.parquet", index=False)
        except Exception:
            previsoes.to_csv(run_dir / f"previsoes_{prefixo}_{nome.lower()}.csv", index=False)


def _rodar_grupo_modelos(df_modelagem, especificacoes, cfg, reaproveitar,
                         rotulos_log=None, cols_exogenas_alvo=None,
                         df_features_pronto=None) -> dict:
    """Executa os modelos do grupo e devolve {rotulo_tempos: previsoes}.

    `reaproveitar=False` (default do pipeline) mantem o caminho de sempre: uma
    chamada de executar_walk_forward por modelo, cada uma montando a propria
    tabela de treino, com o mesmo log e os mesmos registros de tempo das versoes
    anteriores.

    `reaproveitar=True` monta a tabela UMA vez por janela para o grupo inteiro.
    So agrupa modelos que compartilham df_features, features e cols_exogenas —
    e o que garante que a tabela e a mesma para todos eles."""
    rotulos_log = rotulos_log or {}
    if reaproveitar and len(especificacoes) > 1:
        t = time.time()
        saidas = executar_walk_forward_multi(
            df_modelagem, especificacoes, cfg,
            teste_inicio=config.TESTE_INICIO, teste_fim=config.TESTE_FIM,
            cols_exogenas_alvo=cols_exogenas_alvo, df_features_pronto=df_features_pronto)
        for e in especificacoes:
            nome = rotulos_log.get(e.rotulo_tempos, e.rotulo_tempos)
            print(f"{nome}: {len(saidas[e.rotulo_tempos])} previsoes")
        print(f"  [tabela de treino compartilhada] {len(especificacoes)} modelos "
              f"em {time.time() - t:.1f}s")
        return saidas

    saidas = {}
    for e in especificacoes:
        t = time.time()
        saidas[e.rotulo_tempos] = executar_walk_forward(
            df_modelagem, e.colunas_features, e.nome_modelo, cfg,
            teste_inicio=config.TESTE_INICIO, teste_fim=config.TESTE_FIM,
            rotulo_tempos=e.rotulo_tempos, max_estimators_final=e.max_estimators_final,
            cols_exogenas_alvo=cols_exogenas_alvo, df_features_pronto=df_features_pronto)
        nome = rotulos_log.get(e.rotulo_tempos, e.rotulo_tempos)
        print(f"{nome}: {len(saidas[e.rotulo_tempos])} previsoes ({time.time() - t:.1f}s)")
    return saidas


def rodar(*args, **kwargs):
    """Ponto de entrada do pipeline. Devolve o caminho da pasta da execucao.

    Instala a captura de stdout e delega para `_rodar` dentro de um
    try/finally, para que o LOG_EXECUCAO.md seja gravado mesmo quando uma
    excecao interrompe a execucao no meio -- o log parcial e justamente o que
    interessa para diagnosticar.

    Todos os parametros aceitos estao documentados em `_rodar`."""
    tee = consolidado.iniciar_captura()
    try:
        return _rodar(*args, _tee=tee, **kwargs)
    finally:
        consolidado.encerrar_captura(tee)


def _rodar(dados, drive_out, rotulo="colab", amostra=None, experimentos=("1",),
           modo_fiel=False, n_jobs=-1, n_iter=15, cv=3,
           gpu_xgb=False, treino_inicio=None, treino_fim=None,
           teste_inicio=None, teste_fim=None, n_iter_rf=None, grade_rf=None,
           gerar_correlacao=True, gerar_graficos=True,
           calendario_exogenas=None, cenarios_exp2=("1",),
           # --- Metricas e diagnosticos ---
           diagnostico_wmape_zero=True,
           metricas_extras=False,
           avaliar_semanal=False,
           tamanho_efeito=False,
           adicionar_dia_semana=False,
           xgb_objetivo="reg:squarederror",
           # --- Bloco A: correcoes ---
           ordenar_tabela_cronologica=True,
           truncar_pre_lancamento=True,
           score_relativo_geometrico=True,
           corte_atributos_treino=True,
           salvar_previsoes_brutas=True,
           # --- Bloco B: analises novas ---
           estratificar_ativos_cessados=True,
           baseline_zero=True,
           metricas_pooled=True,
           classificacao_robustez_diaria=True,
           relatorio_decisao=True,
           # --- Bloco B2: recorte gerencial e consolidacao ---
           estratificar_por_volume=True,
           consolidar_markdown=True,
           graficos_por_estrato=True,
           n_skus_top_figuras=6,
           # --- Bloco C: modelagem ---
           features_intermitencia=False,
           encoding_sku="ordinal",
           grade_xgb="completa",
           tunar_uma_vez_global=False,
           # --- Bloco D: parcimonia das variaveis ---
           remover_features_redundantes=True,
           features_excluidas_extra=(),
           exogenas_excluidas=None,
           # --- Cadencia do walk-forward ---
           stride_origem_treino=None,
           janelas_por_refit=None,
           permitir_alias_dia_semana=False,
           # --- Metrica de selecao dos hiperparametros ---
           metrica_selecao_hiper="WMAPE",
           diagnostico_selecao_hiper=True,
           # --- Teto de linhas da busca de hiperparametros ---
           max_linhas_tuning=40_000,
           # --- Tabela de treino compartilhada entre modelos ---
           reaproveitar_tabela_treino=False,
           _tee=None):
    """Executa o pipeline completo e devolve o caminho da pasta da execucao.

    ESCOPO E DESEMPENHO
      dados, drive_out    caminho do CSV e pasta raiz das saidas
      rotulo              sufixo do nome da pasta desta execucao
      amostra             None = todos os SKUs elegiveis; inteiro = amostra
                          estratificada por quadrante, para teste rapido
      experimentos        ("1",) so o Experimento 1; ("1","2") inclui as exogenas
      modo_fiel           atalho: True = (passo 1, refit 1); False = (7, 4)
      stride_origem_treino  passo entre origens de treino, em dias. Tem
                          precedencia sobre modo_fiel. NAO pode ser multiplo
                          de 7 — ver _resolver_cadencia_walk_forward
      janelas_por_refit   de quantas em quantas janelas de 7 dias do teste o
                          modelo e retreinado. Tem precedencia sobre modo_fiel
      permitir_alias_dia_semana  libera passo multiplo de 7, so para reproduzir
                          execucoes anteriores
      n_jobs, n_iter, cv, gpu_xgb, n_iter_rf, grade_rf   custo da busca
      reaproveitar_tabela_treino  False (default) = cada modelo monta a
                          propria tabela de treino, como sempre. True = os
                          modelos do mesmo experimento/cenario dividem UMA
                          montagem por janela (mesma tabela por construcao;
                          cada um recebe uma copia). Mais rapido, mas muda o
                          significado do custo_total registrado
      max_linhas_tuning   teto de linhas da tabela de treino usadas na BUSCA
                          de hiperparametros; None = sem teto. O ajuste final
                          usa sempre a tabela inteira. Fica no METADATA.md,
                          sem o que duas execucoes que buscaram sobre amostras
                          de tamanhos diferentes ficam incomparaveis
      treino_*/teste_*    janelas; None usa o valor de config.py

    ANALISES DE APOIO
      gerar_correlacao    matriz features x alvo (evidencia para a remocao de
                          features redundantes)
      gerar_graficos      figuras de resultado
      calendario_exogenas CSV do calendario rico, para o Cenario 2
      cenarios_exp2       quais cenarios do Experimento 2 rodar

    OPCOES POR BLOCO — a descricao completa de cada opcao esta em src/config.py.
      Metricas e diagnosticos
      Bloco A  correcoes             default True   ajustes de comportamento
      Bloco B  recortes de relatorio default True   opcionais
      Bloco B2 recorte gerencial     default True   opcionais
      Bloco C  modelagem             default False  mudam o que entra no modelo
      Bloco D  parcimonia            default True   remocao de features redundantes

    Todos os parametros sao gravados em METADATA.md. O criterio de exclusao do
    Bloco D e estrutural ou medido no TREINO; nenhum corte usa desempenho no
    periodo de teste. Ver selecao_variaveis.md.
    """
    t0 = time.time()
    config.configurar_saida_utf8()
    ml_models.resetar_registros()
    cons = consolidado.Consolidado()

    # ---- Metricas e diagnosticos ----
    config.DIAGNOSTICO_WMAPE_ZERO = diagnostico_wmape_zero
    config.METRICAS_EXTRAS = metricas_extras
    config.AVALIAR_SEMANAL = avaliar_semanal
    config.TAMANHO_EFEITO = tamanho_efeito
    config.ADICIONAR_DIA_SEMANA = adicionar_dia_semana
    config.XGB_OBJETIVO = xgb_objetivo

    # ---- Bloco A: correcoes ----
    config.ORDENAR_TABELA_CRONOLOGICA = ordenar_tabela_cronologica
    config.TRUNCAR_PRE_LANCAMENTO = truncar_pre_lancamento
    config.SCORE_RELATIVO_GEOMETRICO = score_relativo_geometrico
    config.CORTE_ATRIBUTOS_TREINO = corte_atributos_treino
    config.SALVAR_PREVISOES_BRUTAS = salvar_previsoes_brutas

    # ---- Bloco B: analises novas ----
    config.ESTRATIFICAR_ATIVOS_CESSADOS = estratificar_ativos_cessados
    config.BASELINE_ZERO = baseline_zero
    config.METRICAS_POOLED = metricas_pooled
    config.CLASSIFICACAO_ROBUSTEZ_DIARIA = classificacao_robustez_diaria
    config.RELATORIO_DECISAO = relatorio_decisao

    # ---- Bloco B2: recorte gerencial ----
    config.ESTRATIFICAR_POR_VOLUME = estratificar_por_volume
    config.CONSOLIDAR_MARKDOWN = consolidar_markdown
    config.GRAFICOS_POR_ESTRATO = graficos_por_estrato
    config.N_SKUS_TOP_FIGURAS = n_skus_top_figuras

    # ---- Bloco C: modelagem ----
    config.FEATURES_INTERMITENCIA = features_intermitencia
    config.ENCODING_SKU = encoding_sku
    config.definir_grade_xgb(grade_xgb)
    config.TUNAR_UMA_VEZ_GLOBAL = tunar_uma_vez_global

    # ---- Bloco D: parcimonia das variaveis ----
    config.REMOVER_FEATURES_REDUNDANTES = remover_features_redundantes
    config.FEATURES_EXCLUIDAS_EXTRA = tuple(features_excluidas_extra or ())
    # Reatribuicao INCONDICIONAL. `config` e estado global do processo, entao
    # uma chamada anterior de `rodar()` na mesma sessao pode ter deixado outro
    # valor aqui. Sem esta linha, `exogenas_excluidas=None` herdaria o valor da
    # execucao anterior em vez do default, e o A/B entre duas configuracoes
    # passaria a depender da ordem em que os cenarios fossem rodados.
    config.EXOGENAS_EXCLUIDAS = (
        tuple(exogenas_excluidas) if exogenas_excluidas is not None
        else config.EXOGENAS_EXCLUIDAS_PADRAO)
    _exo_desconhecidas = [c for c in config.EXOGENAS_EXCLUIDAS
                          if c not in config.COLUNAS_CALENDARIO_RICO]
    if _exo_desconhecidas:
        raise ValueError(
            f"[Item 29] EXOGENAS_EXCLUIDAS cita colunas que nao existem em "
            f"COLUNAS_CALENDARIO_RICO: {_exo_desconhecidas}. Corrija o nome — "
            f"uma exclusao silenciosamente ignorada faria o METADATA declarar "
            f"um conjunto de variaveis diferente do que foi de fato usado."
        )

    # Redireciona todas as saidas para o Drive
    drive_out = Path(drive_out)
    config.OUTPUTS_DIR = drive_out
    config.EXECUCOES_DIR = drive_out / "execucoes"
    config.HISTORICO_CSV = drive_out / "historico_execucoes.csv"
    config.EXECUCOES_DIR.mkdir(parents=True, exist_ok=True)

    # Parametros de desempenho/escopo
    config.N_JOBS = n_jobs
    config.N_ITER_RANDOM_SEARCH = n_iter
    config.N_ITER_RANDOM_SEARCH_RF = n_iter_rf
    # Reatribuicao INCONDICIONAL, pelo mesmo motivo das demais: `config` e
    # estado global do processo, e uma chamada anterior de rodar() na mesma
    # sessao pode ter deixado outro teto aqui.
    if max_linhas_tuning is not None and int(max_linhas_tuning) < 1:
        raise ValueError(
            f"max_linhas_tuning={max_linhas_tuning!r} invalido; use None "
            f"(sem teto) ou um inteiro >= 1.")
    config.MAX_LINHAS_TUNING = (None if max_linhas_tuning is None
                                else int(max_linhas_tuning))
    if grade_rf is not None:
        config.definir_grade_rf(grade_rf)
    print(f"Busca de hiperparametros — {config.descrever_busca_rf()}")
    _teto_busca = ("sem teto" if config.MAX_LINHAS_TUNING is None
                   else f"{config.MAX_LINHAS_TUNING:,} linhas")
    print(f"  Teto de linhas na busca (max_linhas_tuning): {_teto_busca} "
          f"— o ajuste final usa sempre a tabela inteira")
    config.METRICA_SELECAO_HIPER = str(metrica_selecao_hiper).upper()
    config.DIAGNOSTICO_SELECAO_HIPER = diagnostico_selecao_hiper
    if config.METRICA_SELECAO_HIPER not in config.METRICAS_SELECAO_VALIDAS:
        raise ValueError(
            f"metrica_selecao_hiper='{metrica_selecao_hiper}' invalida; "
            f"use uma de {config.METRICAS_SELECAO_VALIDAS}")
    print(f"[Item E4] Metrica de selecao dos hiperparametros: "
          f"{config.METRICA_SELECAO_HIPER}")
    config.N_SPLITS_TIME_SERIES_CV = cv
    config.USE_GPU_XGB = gpu_xgb
    if gpu_xgb:
        print("[GPU] USE_GPU_XGB=True — XGBoost tentara device='cuda'.")
    if xgb_objetivo != "reg:squarederror":
        print(f"[Item 10] XGBoost objetivo: {xgb_objetivo}")

    # ---- Painel das opcoes ativas, no topo do log ----
    # Serve para conferir, ao abrir o LOG_EXECUCAO.md, o que esta rodada de
    # fato usou — sem depender de lembrar o que estava na celula de parametros.
    print("\n--- Opcoes v10 ativas ---")
    print(f"  [A] Item 11 CV cronologica={ordenar_tabela_cronologica} | "
          f"Item 12 truncagem={truncar_pre_lancamento} | "
          f"Item 13 score geometrico={score_relativo_geometrico}")
    print(f"      Item 14 corte atributos={corte_atributos_treino} | "
          f"Item 15 previsoes brutas={salvar_previsoes_brutas}")
    print(f"  [B] Item 16 estratos ativo/cessado={estratificar_ativos_cessados} | "
          f"Item 17 baseline Zero={baseline_zero} | "
          f"Item 18 pooled={metricas_pooled}")
    print(f"      Item 19 ADI/CV2 diario={classificacao_robustez_diaria} | "
          f"Item 20 relatorio decisao={relatorio_decisao}")
    print(f" [B2] Item 25 faixas de volume={estratificar_por_volume} | "
          f"Item 26 consolidado md={consolidar_markdown} | "
          f"Item 27 graficos por estrato={graficos_por_estrato}")
    print(f"  [C] Item 21 features intermitencia={features_intermitencia} | "
          f"Item 22 encoding SKU='{encoding_sku}' | "
          f"Item 23 grade XGB='{grade_xgb}' | "
          f"Item 24 tuning global={tunar_uma_vez_global}")
    print(f"  [D] Item 28 remover features redundantes={remover_features_redundantes} | "
          f"Item 29 exogenas: {len(config.colunas_calendario_rico())} de "
          f"{len(config.COLUNAS_CALENDARIO_RICO)} mantidas")
    print(f"  [E] Teto da busca (max_linhas_tuning)={_teto_busca} | "
          f"tabela de treino compartilhada={reaproveitar_tabela_treino}")
    if reaproveitar_tabela_treino:
        print("  [tabela compartilhada] os modelos de cada experimento dividem uma "
              "unica montagem da tabela de treino por janela.\n"
              "      As previsoes sao as mesmas; o custo_total por modelo passa a "
              "excluir a montagem (linha CACHE\n"
              "      em tempos_execucao.csv) e NAO e comparavel ao de uma execucao "
              "sem compartilhamento.")
    print("-------------------------\n")

    # Cadencia do walk-forward — resolvida antes de qualquer processamento: um
    # passo invalido deve interromper a execucao agora, e nao depois de horas
    # de walk-forward.
    stride, refit = _resolver_cadencia_walk_forward(
        modo_fiel, stride_origem_treino, janelas_por_refit, permitir_alias_dia_semana)

    if treino_inicio is not None:
        config.TREINO_INICIO = treino_inicio
    if treino_fim is not None:
        config.TREINO_FIM = treino_fim
    if teste_inicio is not None:
        config.TESTE_INICIO = teste_inicio
    if teste_fim is not None:
        config.TESTE_FIM = teste_fim
    print(f"Periodo: treino {config.TREINO_INICIO} a {config.TREINO_FIM} | "
          f"teste {config.TESTE_INICIO} a {config.TESTE_FIM}")

    # Bateria de metricas dos testes de Wilcoxon. Definida uma unica vez e
    # usada nos QUATRO relatorios (geral, por estrato, por faixa de volume e
    # Exp2), para que nenhum deles decida por um subconjunto diferente.
    colunas_wilcoxon = ("WMAPE", "MASE") + (("RMSSE", "MAE") if metricas_extras else ())
    colunas_wilcoxon_exp2 = colunas_wilcoxon

    # ---- Features redundantes: o que sai do conjunto, e por que ----
    # Calculado ANTES de qualquer experimento: e uma funcao apenas da
    # configuracao, e precisa aparecer no METADATA mesmo quando so o Exp2 roda.
    _features_removidas_exp1 = config.features_a_excluir(
        features_experimento1(aplicar_exclusoes=False))
    if _features_removidas_exp1:
        print(f"[Item 28] {len(_features_removidas_exp1)} features redundantes removidas: "
              + ", ".join(_features_removidas_exp1))
        for _f, _motivo in _features_removidas_exp1.items():
            print(f"          - {_f}: {_motivo}")
    if config.EXOGENAS_EXCLUIDAS:
        print(f"[Item 29] {len(config.EXOGENAS_EXCLUIDAS)} exogenas excluidas do Cenario 2: "
              + ", ".join(config.EXOGENAS_EXCLUIDAS))
    experimentos = set(experimentos)
    cenarios_exp2 = tuple(cenarios_exp2)

    # --- Calendario externo (Cenario 2 do Experimento 2) ---
    cal_exogenas = None
    if "2" in experimentos and (calendario_exogenas is not None):
        _cal_path = Path(calendario_exogenas)
        if _cal_path.exists():
            from src.exogenas import carregar_calendario
            cal_exogenas = carregar_calendario(calendario_exogenas)
            print(f"Calendario externo carregado: {len(cal_exogenas)} datas, "
                  f"{len(config.COLUNAS_CALENDARIO_RICO)} colunas exogenas "
                  f"({len(config.colunas_calendario_rico())} usadas no Cenario 2 — Item 29)")
        else:
            print(f"[AVISO] Arquivo de calendario nao encontrado: {_cal_path}")
    if "2" in experimentos and cal_exogenas is None and "2" in cenarios_exp2:
        print("[AVISO] Cenario 2 solicitado mas calendario_exogenas nao encontrado. "
              "Cenario 2 sera ignorado nesta execucao.")
        cenarios_exp2 = tuple(c for c in cenarios_exp2 if c != "2")

    run_dir = config.criar_pasta_execucao(rotulo=rotulo)
    if _tee is not None:
        _tee.run_dir = run_dir          # permite salvar o log mesmo com excecao
    print(f"Pasta desta execucao: {run_dir}")

    # ---- Relatorio de selecao de variaveis ----
    (run_dir / "selecao_variaveis.md").write_text(
        _relatorio_selecao_variaveis(features_experimento1(),
                                     _features_removidas_exp1, cal_exogenas),
        encoding="utf-8")

    # ---- Dados brutos + EDA sazonal ----
    df_raw = preprocess.carregar_extracao(dados)
    print(f"raw: {len(df_raw):,} linhas, {df_raw[COLS.sku].nunique():,} SKUs ({time.time()-t0:.1f}s)")
    sazonalidade.gerar_relatorio_sazonalidade(df_raw, outputs_dir=run_dir)

    # ---- Pre-processamento + elegibilidade ----
    cons.secao("Caracterizacao da base")
    corte_attr = config.TREINO_FIM if corte_atributos_treino else None
    df_agg = preprocess.agregar_por_sku(df_raw, corte_atributos=corte_attr)
    df_full = preprocess.expandir_series_diarias(df_agg)

    # ---- Truncagem pre-lancamento ----
    resumo_trunc = None
    if truncar_pre_lancamento:
        df_full, resumo_trunc = preprocess.truncar_antes_do_lancamento(df_full)
        print(f"[Item 12] Truncagem pre-lancamento: {resumo_trunc['linhas_removidas']:,} linhas "
              f"removidas ({resumo_trunc['pct_removido']:.1f}%) — dias anteriores ao cadastro/1a venda. "
              f"SKUs {resumo_trunc['skus_antes']} -> {resumo_trunc['skus_depois']}; "
              f"vendas removidas: {resumo_trunc['linhas_com_venda_removidas']}; "
              f"SKUs com cadastro posterior a 1a venda: {resumo_trunc['skus_cadastro_apos_1a_venda']}.")
        cons.texto(f"**Item 12 — truncagem pre-lancamento:** "
                   f"{resumo_trunc['linhas_removidas']:,} de {resumo_trunc['linhas_antes']:,} linhas "
                   f"removidas ({resumo_trunc['pct_removido']:.1f}%).")

    df_full = preprocess.detectar_outliers(df_full)
    df_filtrado, resumo_elig = preprocess.aplicar_criterios_elegibilidade(df_full)
    funil = preprocess.funil_filtragem_texto(df_full, resumo_elig, resumo_truncagem=resumo_trunc)
    print(funil)
    cons.texto(funil)

    if corte_atributos_treino:
        msg = diagnosticar_origem_idade(df_filtrado, corte=config.TREINO_FIM)
        print(msg)
        cons.texto(msg)

    # ---- EDA + classificacao ADI/CV2 ----
    eda.gerar_relatorio_eda(df_full, funil, outputs_dir=run_dir)
    df_modelagem, df_classes, texto_classes = adi_cv2.executar_classificacao(df_filtrado, outputs_dir=run_dir)
    print(texto_classes)
    cons.texto(texto_classes)

    # ---- Classificacao de robustez em granularidade diaria ----
    if classificacao_robustez_diaria:
        _, texto_rob = adi_cv2.executar_classificacao_robustez(df_filtrado, run_dir, freq="D")
        if texto_rob:
            print("[Item 19] Robustez da classificacao (granularidade diaria):")
            print(texto_rob)
            cons.texto(texto_rob)

    if df_modelagem[COLS.sku].nunique() == 0:
        print("Nenhum SKU Intermitente/Lumpy — encerrando.")
        return run_dir

    # ---- Amostra (opcional) ----
    if amostra:
        df_modelagem, am = _amostra_estratificada(df_modelagem, df_classes, amostra)
        am[["sku", "quadrante", "ADI", "CV2"]].to_csv(run_dir / "amostra_sku.csv", index=False)
        print(f"Amostra estratificada: {df_modelagem[COLS.sku].nunique()} SKUs")

    df_treino_hist = df_modelagem[df_modelagem[COLS.data] <= config.TREINO_FIM]
    resumo1 = resumo2 = None
    resumo_pooled = resumo_sem = resumo_estratos = None
    estratos_teste = estratos_volume = None
    resumo_volume = concentracao = cruzamento = None
    fonte_volume = None

    # =======================================================================
    # Faixas de volume de pedidos (ABC / Pareto)
    # Calculado sobre o TREINO, antes de qualquer modelagem: e a informacao
    # que um gestor teria em maos ao decidir onde investir em previsao.
    # =======================================================================
    if estratificar_por_volume:
        cons.secao("Recorte gerencial — faixas de volume de pedidos (Item 25)")
        estratos_volume, fonte_volume = evaluate.classificar_estratos_volume(df_treino_hist)
        if estratos_volume is not None and not estratos_volume.empty:
            estratos_volume.to_csv(run_dir / "estratos_volume_pedidos.csv", index=False)
            concentracao = evaluate.resumir_concentracao_volume(estratos_volume)
            concentracao.to_csv(run_dir / "concentracao_volume_pedidos.csv", index=False)
            print(f"[Item 25] Fonte do volume: {fonte_volume}")
            cons.texto(f"Fonte do volume de pedidos: _{fonte_volume}_ "
                       "(medido no periodo de treino — sem vazamento).")
            cons.mostrar("Concentracao dos pedidos por faixa (Pareto)", concentracao,
                         nota="Quanto mais concentrado, mais defensavel priorizar a acuracia "
                              "nos SKUs do topo.",
                         arquivo="concentracao_volume_pedidos.csv")
            if not concentracao.empty:
                topo = concentracao.iloc[0]
                print(f"[Item 25] Os {topo['pct_skus']:.0f}% de SKUs do topo concentram "
                      f"{topo['pct_volume_pedidos']:.1f}% dos pedidos e "
                      f"{topo['pct_demanda_treino']:.1f}% da demanda.")

    # ---- Correlacao features x target ----
    if gerar_correlacao:
        t = time.time()
        correlacao.gerar_analise_correlacao(df_modelagem, df_classes, run_dir)
        print(f"Correlacao features x target: {time.time()-t:.1f}s")

    # ---- Cache de features (construidas uma unica vez, compartilhadas) ----
    print("Construindo features de origem (cache compartilhado)...")
    df_features_cache = construir_cache_features(df_modelagem)
    print(f"  {len(df_features_cache):,} linhas, "
          f"{df_features_cache[COLS.sku].nunique():,} SKUs")

    # =======================================================================
    # EXPERIMENTO 1 — ML vs. baselines
    # =======================================================================
    if "1" in experimentos:
        print("\n=== EXPERIMENTO 1 — ML vs. baselines (QP1) ===")
        cons.secao("Experimento 1 — ML vs. baselines (QP1)")

        features_exp1 = features_experimento1()
        print(f"Features do Exp1 ({len(features_exp1)}): {features_exp1}")
        cons.texto(f"**Features do Experimento 1 ({len(features_exp1)}):** "
                   + ", ".join(f"`{f}`" for f in features_exp1))

        cfg = ConfigWalkForward(features_exp1, stride, refit)
        prevs = {}
        n_skus_modelagem = int(df_modelagem[COLS.sku].nunique())
        for nome, metodo in [("SBA", "sba"), ("TSB", "tsb")]:
            t = time.time()
            prevs[nome] = executar_walk_forward_baseline(
                df_modelagem, metodo, janelas_por_refit=refit,
                teste_inicio=config.TESTE_INICIO, teste_fim=config.TESTE_FIM)
            duracao_baseline = time.time() - t
            ml_models.registrar_tempo(nome, "walk_forward", duracao_baseline,
                                      n_skus=n_skus_modelagem)
            print(f"{nome}: {len(prevs[nome])} previsoes ({duracao_baseline:.1f}s)")
        # Os dois modelos usam o MESMO df_features, as MESMAS features e as
        # mesmas origens: a tabela de treino de cada janela e identica para
        # ambos, e por isso podem dividi-la quando o parametro estiver ligado.
        especificacoes_exp1 = [
            EspecificacaoModelo(nome_modelo="xgb", colunas_features=features_exp1,
                                rotulo_tempos="XGBoost"),
            EspecificacaoModelo(nome_modelo="rf", colunas_features=features_exp1,
                                rotulo_tempos="RF"),
        ]
        prevs.update(_rodar_grupo_modelos(
            df_modelagem, especificacoes_exp1, cfg,
            reaproveitar=reaproveitar_tabela_treino,
            df_features_pronto=df_features_cache))

        # ---- Verificacao de alinhamento dos conjuntos de avaliacao ----
        n_por_modelo = {nome: len(prev) for nome, prev in prevs.items() if not prev.empty}
        if len(n_por_modelo) > 1:
            n_max = max(n_por_modelo.values())
            n_min = min(n_por_modelo.values())
            divergencia = (n_max - n_min) / n_max if n_max > 0 else 0
            if divergencia > 0.05:
                print(f"\n[AVISO CRITICO] Conjuntos de avaliacao divergentes (>{divergencia:.1%}): {n_por_modelo}")
                print("  Possivel desalinhamento de granularidade. Verifique freq= em baselines.py.")
            else:
                print(f"[OK] Alinhamento de conjuntos de avaliacao verificado: {n_por_modelo}")

        # ---- Baseline Zero ----
        if baseline_zero:
            referencia = next((p for p in prevs.values() if p is not None and not p.empty), None)
            if referencia is not None:
                prevs["Zero"] = evaluate.construir_baseline_zero(referencia)
                ml_models.registrar_tempo("Zero", "walk_forward", 0.0, n_skus=n_skus_modelagem,
                                          detalhe="baseline de controle (previsao identicamente nula)")
                print(f"[Item 17] Baseline Zero: {len(prevs['Zero'])} previsoes (controle, custo ~0s)")

        metricas1 = {}
        for nome, prev in prevs.items():
            if prev.empty:
                continue
            m = evaluate.calcular_metricas_por_sku(prev, df_treino_hist,
                                                    metricas_extras=metricas_extras)
            _salvar_resultados_modelo(run_dir, "exp1", nome, prev, m)
            metricas1[nome] = m

        # ---- Diagnostico WMAPE indefinido ----
        if diagnostico_wmape_zero and metricas1:
            diag = pd.DataFrame([{
                "modelo": nome,
                "n_skus": len(df),
                "pct_WMAPE_indefinido": df["WMAPE"].isna().mean() * 100,
                "n_obs_mediano": int(df["n_observacoes"].median()),
            } for nome, df in metricas1.items()])
            cons.mostrar("Diagnostico Item 1 — WMAPE indefinido por modelo", diag,
                         nota="WMAPE = NaN significa SKU sem demanda no conjunto de avaliacao.")

        resumo1 = evaluate.resumir_medianas(metricas1)

        # ---- Metricas agregadas globais ----
        if metricas_pooled:
            resumo_pooled = evaluate.resumir_pooled(prevs)
            resumo_pooled.to_csv(run_dir / "resumo_pooled_experimento1.csv", index=False)
            cons.mostrar("Item 18 — Metricas agregadas globais (pooled)", resumo_pooled,
                         nota="WMAPE_pooled = soma|y-yhat| / soma(y) sobre TODOS os SKUs e dias. "
                              "Definida enquanto algum SKU vender.",
                         arquivo="resumo_pooled_experimento1.csv")
            resumo1 = resumo1.merge(
                resumo_pooled[["modelo", "WMAPE_pooled", "MAE_pooled", "RMSE_pooled"]],
                on="modelo", how="left")

        # ---- Wilcoxon geral ----
        # O RMSSE entra na bateria de testes, e nao so nas tabelas de mediana.
        # WMAPE, MASE e MAE sao todos o erro ABSOLUTO reescalado por uma
        # constante do SKU: tem otimo na mediana condicional, que e zero numa
        # serie ~95% esparsa, e por isso o baseline Zero vence os tres. O RMSSE
        # e quadratico (otimo na media condicional) e e a unica metrica desta
        # bateria em que os modelos chegam a vencer o Zero — testar so as
        # outras deixaria a conclusao defensavel do trabalho sem teste de
        # significancia.
        (run_dir / "testes_wilcoxon_experimento1.md").write_text(
            evaluate.gerar_relatorio_wilcoxon_metricas(
                metricas1,
                colunas=colunas_wilcoxon,
                aplicar_correcao_multipla=tamanho_efeito,
                calcular_efeito=tamanho_efeito,
            ), encoding="utf-8")

        esp = evaluate.calcular_esparsidade_treino(df_treino_hist)
        evaluate.estratificar_por_esparsidade(metricas1, esp).to_csv(
            run_dir / "analise_estratificada.csv", index=False)

        # ---- Estratificacao ativos x cessados ----
        if estratificar_ativos_cessados and metricas1:
            cons.secao("Item 16 — SKUs ativos x cessados (obsolescencia)")
            referencia = next((p for p in prevs.values() if p is not None and not p.empty), None)
            estratos_teste = evaluate.classificar_estratos_teste(referencia)
            estratos_teste.to_csv(run_dir / "estratos_ativo_cessado.csv", index=False)

            n_ativo = int((estratos_teste["estrato_teste"] == evaluate.ESTRATO_ATIVO).sum())
            n_cess = int((estratos_teste["estrato_teste"] == evaluate.ESTRATO_CESSADO).sum())
            print(f"[Item 16] Estratos no teste: Ativo={n_ativo} SKUs | Cessado={n_cess} SKUs "
                  f"({n_cess / max(n_ativo + n_cess, 1) * 100:.1f}% sem nenhuma venda)")
            cons.texto(f"**Ativo** = {n_ativo} SKUs (venderam ao menos 1 vez no teste); "
                       f"**Cessado** = {n_cess} SKUs "
                       f"({n_cess / max(n_ativo + n_cess, 1) * 100:.1f}% do total).\n\n"
                       "Estratificacao de RELATORIO: todos os SKUs foram modelados; o recorte e "
                       "feito apenas na apresentacao. Nenhuma decisao de modelagem usou o "
                       "periodo de teste, portanto nao ha vazamento.")

            colunas_wilcoxon_estrato = colunas_wilcoxon
            resumo_estratos = evaluate.resumir_por_estrato(
                metricas1, estratos_teste, prevs,
                coluna_estrato="estrato_teste",
                ordem=(evaluate.ESTRATO_ATIVO, evaluate.ESTRATO_CESSADO))
            resumo_estratos.to_csv(run_dir / "resumo_experimento1_por_estrato.csv", index=False)
            (run_dir / "testes_wilcoxon_por_estrato.md").write_text(
                evaluate.gerar_relatorio_wilcoxon_por_estrato(
                    metricas1, estratos_teste, colunas=colunas_wilcoxon_estrato,
                    aplicar_correcao_multipla=tamanho_efeito,
                    calcular_efeito=tamanho_efeito,
                ), encoding="utf-8")
            cons.mostrar("Resumo por estrato (Ativo x Cessado)", resumo_estratos,
                         nota="No estrato Cessado o WMAPE e indefinido (demanda zero) — "
                              "a comparacao relevante ali e por MAE: menor = reconheceu a "
                              "obsolescencia mais rapido.",
                         arquivo="resumo_experimento1_por_estrato.csv")

        # ---- Resultados por faixa de volume de pedidos ----
        if estratificar_por_volume and metricas1 and estratos_volume is not None \
                and not estratos_volume.empty:
            cons.secao("Item 25 — Resultados por faixa de volume de pedidos")
            ordem_vol = tuple(f[2] for f in config.FAIXAS_VOLUME_PEDIDOS)
            resumo_volume = evaluate.resumir_por_estrato(
                metricas1, estratos_volume, prevs,
                coluna_estrato="estrato_volume", ordem=ordem_vol)
            resumo_volume.to_csv(run_dir / "resumo_experimento1_por_volume.csv", index=False)
            (run_dir / "testes_wilcoxon_por_volume.md").write_text(
                evaluate.gerar_relatorio_wilcoxon_por_estrato(
                    metricas1, estratos_volume,
                    colunas=colunas_wilcoxon,
                    aplicar_correcao_multipla=tamanho_efeito,
                    calcular_efeito=tamanho_efeito,
                    coluna_estrato="estrato_volume", ordem=ordem_vol,
                    titulo="# Testes de Wilcoxon por faixa de volume de pedidos (Item 25)\n",
                    nota="Faixas ABC pelo numero de pedidos distintos no periodo de TREINO. "
                         "A faixa A concentra os SKUs cujo erro de previsao afeta o maior "
                         "numero de clientes finais.",
                ), encoding="utf-8")
            cons.mostrar("Resumo por faixa de volume de pedidos (ABC)", resumo_volume,
                         nota="A pergunta gerencial: nos SKUs que mais vendem, qual modelo "
                              "preve melhor? Ver WMAPE_pooled da faixa A.",
                         arquivo="resumo_experimento1_por_volume.csv")

            if estratos_teste is not None and not estratos_teste.empty:
                cruzamento = evaluate.cruzar_estratos(estratos_teste, estratos_volume)
                if not cruzamento.empty:
                    cruzamento.to_csv(run_dir / "cruzamento_volume_x_obsolescencia.csv", index=False)
                    cons.mostrar("Obsolescencia por faixa de volume", cruzamento,
                                 nota="A obsolescencia se concentra na cauda (esperado) ou "
                                      "tambem atinge SKUs de alto giro (achado relevante)?",
                                 arquivo="cruzamento_volume_x_obsolescencia.csv")

        # ---- Avaliacao no nivel semanal ----
        if avaliar_semanal:
            resumo_sem = evaluate.avaliar_nivel_semanal(prevs, df_treino_hist, run_dir)
            cons.mostrar("Item 7 — Avaliacao no nivel semanal", resumo_sem,
                         nota="Comparacao mais justa: e o unico nivel em que os quatro modelos "
                              "operam nativamente (SBA/TSB preveem semanal e sao desagregados "
                              "para o dia). Equivale ao agregado do lead time de 7 dias.",
                         arquivo="resumo_semanal_agregado.csv")

        # ---- Score relativo por SKU ----
        resumo1, metricas1_longo, _ = score_relativo.gerar_analise_score_relativo(
            metricas1, df_classes, run_dir, resumo=resumo1)
        resumo1.to_csv(run_dir / "resumo_experimento1.csv", index=False)
        cons.mostrar("Resumo do Experimento 1", resumo1,
                     nota="Medianas por SKU + metricas agregadas + score relativo. "
                          "Use a coluna score_rel_*_geometrico (Item 13).",
                     arquivo="resumo_experimento1.csv")

        # ---- Tempos, importancia de features e figuras ----
        ml_models.salvar_tempos_csv(run_dir / "tempos_execucao.csv")
        ml_models.salvar_importancias_csv(run_dir / "importancia_features.csv")
        figuras_geradas = []
        if gerar_graficos:
            figuras_geradas += graficos_resultados.gerar_graficos_resultados(
                metricas1_longo, prevs,
                ml_models.obter_tempos_df(), ml_models.obter_importancias_df(),
                run_dir)
            # ---- Figuras dos recortes por estrato ----
            if graficos_por_estrato:
                figuras_geradas += graficos_estratos.gerar_graficos_estratos(
                    metricas1_longo, prevs, estratos_teste, estratos_volume,
                    resumo_estratos, resumo_volume, run_dir,
                    n_skus_top=n_skus_top_figuras)
            cons.figuras("Figuras geradas", figuras_geradas)

        tempos_modelo = run_dir / "tempos_por_modelo.csv"
        if tempos_modelo.exists():
            cons.mostrar("Custo computacional por modelo (Secao 4.3.1)",
                         pd.read_csv(tempos_modelo),
                         nota="custo_total = wall-clock do walk-forward, comparavel entre TODOS "
                              "os modelos; tuning/fit_final sao a decomposicao do custo de ML.",
                         arquivo="tempos_por_modelo.csv")

    # ---- Diagnostico da metrica de selecao ----
    if diagnostico_selecao_hiper:
        (run_dir / "item_e4_metrica_selecao.md").write_text(
            ml_models.relatorio_diagnostico_selecao(), encoding="utf-8")
        _diag_e4 = ml_models.diagnostico_selecao_df()
        if not _diag_e4.empty:
            _diag_e4.to_csv(run_dir / "item_e4_metrica_selecao.csv", index=False)

    # =======================================================================
    # EXPERIMENTO 2 — variaveis exogenas
    # =======================================================================
    metricas2 = {}
    if "2" in experimentos:
        print("\n=== EXPERIMENTO 2 — variaveis exogenas (QP2) ===")
        cons.secao("Experimento 2 — variaveis exogenas (QP2)")
        if tunar_uma_vez_global:
            print("[Item 24] Hiperparametros do Exp1 reaproveitados — a comparacao "
                  "Cenario 1 vs 2 isola o efeito das exogenas.")

        df_modelagem_rico = None
        if cal_exogenas is not None and "2" in cenarios_exp2:
            from src.features import adicionar_calendario_exogeno
            df_modelagem_rico = adicionar_calendario_exogeno(df_modelagem.copy(), cal_exogenas)
            cal_exogenas.reset_index().to_csv(run_dir / "calendario_exogenas_usado.csv", index=False)

        if "1" in cenarios_exp2:
            features_1 = features_cenario_1()
            cfg_1 = ConfigWalkForward(features_1, stride, refit)
            saidas_1 = _rodar_grupo_modelos(
                df_modelagem,
                [EspecificacaoModelo(nome_modelo=m, colunas_features=features_1,
                                     rotulo_tempos=f"{r}_Cenario1")
                 for m, r in [("xgb", "XGBoost"), ("rf", "RF")]],
                cfg_1, reaproveitar=reaproveitar_tabela_treino,
                rotulos_log={f"{r}_Cenario1": f"{r} Cenario 1"
                             for r in ("XGBoost", "RF")},
                df_features_pronto=df_features_cache)
            for mod, rot in [("xgb", "XGBoost"), ("rf", "RF")]:
                prev = saidas_1[f"{rot}_Cenario1"]
                if not prev.empty:
                    m = evaluate.calcular_metricas_por_sku(prev, df_treino_hist,
                                                            metricas_extras=metricas_extras)
                    _salvar_resultados_modelo(run_dir, "exp2", f"{mod}_cenario1", prev, m)
                    metricas2[f"{rot}_Cenario1"] = m

        if df_modelagem_rico is not None and "2" in cenarios_exp2:
            features_2 = features_cenario_2()
            cfg_2 = ConfigWalkForward(features_2, stride, refit)
            # O Cenario 2 tem colunas exogenas extras, entao precisa do seu
            # proprio cache de features (nao pode reusar o do Exp1).
            df_features_cache_2 = construir_cache_features(df_modelagem_rico)
            saidas_2 = _rodar_grupo_modelos(
                df_modelagem_rico,
                [EspecificacaoModelo(nome_modelo=m, colunas_features=features_2,
                                     rotulo_tempos=f"{r}_Cenario2")
                 for m, r in [("xgb", "XGBoost"), ("rf", "RF")]],
                cfg_2, reaproveitar=reaproveitar_tabela_treino,
                rotulos_log={f"{r}_Cenario2": f"{r} Cenario 2"
                             for r in ("XGBoost", "RF")},
                cols_exogenas_alvo=config.colunas_calendario_rico(),
                df_features_pronto=df_features_cache_2)
            for mod, rot in [("xgb", "XGBoost"), ("rf", "RF")]:
                prev = saidas_2[f"{rot}_Cenario2"]
                if not prev.empty:
                    m = evaluate.calcular_metricas_por_sku(prev, df_treino_hist,
                                                            metricas_extras=metricas_extras)
                    _salvar_resultados_modelo(run_dir, "exp2", f"{mod}_cenario2", prev, m)
                    metricas2[f"{rot}_Cenario2"] = m

        if metricas2:
            resumo2 = evaluate.resumir_medianas(metricas2)
            resumo2.to_csv(run_dir / "resumo_experimento2.csv", index=False)

            comparacoes = []
            for rot in ["XGBoost", "RF"]:
                if f"{rot}_Cenario1" in metricas2 and f"{rot}_Cenario2" in metricas2:
                    comparacoes.append((
                        f"{rot}_Cenario1", f"{rot}_Cenario2",
                        f"Q2.3 {rot}: calendario simples->rico"
                    ))

            linhas = ["# Testes de Wilcoxon — Experimento 2 (Secao 4.2.7)\n",
                      "Cenario 1 (calendario simples: feriados BR + Black Friday) vs. "
                      "Cenario 2 (calendario rico via CSV externo).\n",
                      "_Leitura do r bisserial: r > 0 = o SEGUNDO cenario do par e o melhor "
                      "(a metrica e de erro), ou seja, o calendario rico ajudou._\n"]
            # Mesmas metricas e mesma correcao de Holm do Exp1, incluindo o
            # RMSSE. Testar o efeito das exogenas so com metricas de erro
            # absoluto decidiria justamente pelas metricas em que o baseline
            # Zero vence todos os modelos — ver a nota do Wilcoxon do Exp1.
            for coluna in colunas_wilcoxon_exp2:
                linhas.append(f"\n## Metrica: {coluna}\n")
                resultados = [evaluate.teste_wilcoxon_pareado(metricas2[a], metricas2[b], coluna)
                              for a, b, _ in comparacoes]
                if tamanho_efeito and resultados:
                    p_adj = evaluate.aplicar_holm_bonferroni([r["p_valor"] for r in resultados])
                    for r, p in zip(resultados, p_adj):
                        r["p_valor_adj"] = p
                        if p is not None:
                            r["significativo_adj"] = p < 0.05
                    linhas.append(f"_(correcao de Holm-Bonferroni aplicada — "
                                  f"{len(comparacoes)} comparacoes)_\n")
                for (a_key, b_key, descricao), r in zip(comparacoes, resultados):
                    linhas.append(f"### {descricao}")
                    linhas.append(evaluate._formatar_linha_wilcoxon(
                        a_key, b_key, r, coluna,
                        calcular_efeito=tamanho_efeito,
                        com_correcao=tamanho_efeito))
            (run_dir / "testes_wilcoxon_experimento2.md").write_text("\n".join(linhas), encoding="utf-8")
            cons.mostrar("Resumo do Experimento 2", resumo2,
                         nota="Cenario 1 = calendario simples; Cenario 2 = calendario rico.",
                         arquivo="resumo_experimento2.csv")

    # ---- Tempos consolidados ----
    ml_models.salvar_tempos_csv(run_dir / "tempos_execucao.csv")
    ml_models.salvar_importancias_csv(run_dir / "importancia_features.csv")

    # ---- Historico + metadata ----
    duracao = round(time.time() - t0, 1)
    meta = {"timestamp": run_dir.name, "fonte_dados": str(dados),
            "modo": "fiel" if modo_fiel else "rapido",
            # Cadencia efetiva do walk-forward, que pode divergir do atalho
            # `modo`. O numero de retreinos e derivado do periodo de teste
            # desta execucao, e nao de um ano fixo.
            "item31_stride_origem_treino_dias": stride,
            "item31_janelas_por_refit": refit,
            "item31_refits_no_teste": len(range(0, len(pd.date_range(
                config.TESTE_INICIO, config.TESTE_FIM, freq="7D")), refit)),
            "n_iter_busca": n_iter,
            "n_iter_busca_rf": config.n_iter_rf(),
            # Sem este registro nao da para comparar duas execucoes que
            # buscaram hiperparametros sobre amostras de tamanhos diferentes.
            # Lido de `config` para refletir o teto que de fato valeu aqui.
            "max_linhas_tuning": ("sem teto" if config.MAX_LINHAS_TUNING is None
                                  else config.MAX_LINHAS_TUNING),
            # Com True, custo_total por modelo exclui a montagem da tabela.
            "reaproveitar_tabela_treino": reaproveitar_tabela_treino,
            "grade_rf": config.GRADE_RF_MODO,
            "busca_hiperparametros": config.descrever_busca_rf(),
            "itemE4_metrica_selecao_hiper": config.METRICA_SELECAO_HIPER,
            "cv_splits": cv, "amostragem_skus": df_modelagem[COLS.sku].nunique(),
            "gpu_xgb": gpu_xgb,
            "analise_correlacao": gerar_correlacao,
            "graficos_resultados": gerar_graficos,
            "treino": f"{config.TREINO_INICIO} a {config.TREINO_FIM}",
            "teste": f"{config.TESTE_INICIO} a {config.TESTE_FIM}",
            # Metricas e diagnosticos
            "diagnostico_wmape_zero": diagnostico_wmape_zero,
            "metricas_extras": metricas_extras,
            "avaliar_semanal": avaliar_semanal,
            "tamanho_efeito": tamanho_efeito,
            "adicionar_dia_semana": adicionar_dia_semana,
            "xgb_objetivo": xgb_objetivo,
            # Bloco A: correcoes
            "item11_ordenar_tabela_cronologica": ordenar_tabela_cronologica,
            "item12_truncar_pre_lancamento": truncar_pre_lancamento,
            "item13_score_relativo_geometrico": score_relativo_geometrico,
            "item14_corte_atributos_treino": corte_atributos_treino,
            "item15_salvar_previsoes_brutas": salvar_previsoes_brutas,
            # Bloco B: analises novas
            "item16_estratificar_ativos_cessados": estratificar_ativos_cessados,
            "item17_baseline_zero": baseline_zero,
            "item18_metricas_pooled": metricas_pooled,
            "item19_classificacao_robustez_diaria": classificacao_robustez_diaria,
            "item20_relatorio_decisao": relatorio_decisao,
            # Bloco B2: recorte gerencial
            "item25_estratificar_por_volume": estratificar_por_volume,
            "item25_fonte_volume": fonte_volume,
            "item26_consolidar_markdown": consolidar_markdown,
            "item27_graficos_por_estrato": graficos_por_estrato,
            # Bloco C: modelagem
            "item21_features_intermitencia": features_intermitencia,
            "item22_encoding_sku": encoding_sku,
            "item23_grade_xgb": grade_xgb,
            "item24_tunar_uma_vez_global": tunar_uma_vez_global,
            # Bloco D: parcimonia das variaveis
            "item28_remover_features_redundantes": remover_features_redundantes,
            "item28_features_removidas": ", ".join(_features_removidas_exp1) or "(nenhuma)",
            "item29_exogenas_excluidas": ", ".join(config.EXOGENAS_EXCLUIDAS) or "(nenhuma)",
            "item29_exogenas_usadas": ", ".join(config.colunas_calendario_rico()),
            # Sem as versoes a execucao nao e reproduzivel: ja houve duas
            # rodadas com a MESMA base e a MESMA semente em que as previsoes de
            # SBA e TSB (deterministicas, sem GPU) divergiram em 18% e 43% das
            # linhas, por reinstalacao do ambiente do Colab entre elas.
            "versoes_bibliotecas": _versoes_bibliotecas(),
            "duracao_s": duracao}
    historico.registrar_execucao(run_dir.name, meta, resumo1, resumo2)
    (run_dir / "METADATA.md").write_text(
        "\n".join(f"- **{k}**: {v}" for k, v in meta.items()) + "\n", encoding="utf-8")

    # ---- Consolidacao em markdown ----
    if consolidar_markdown:
        cons.escrever(run_dir, metadata=meta)
        print("[Item 26] RESULTADOS_CONSOLIDADO.md gerado (todas as tabelas em um so arquivo).")

    # ---- Relatorio de decisao ----
    if relatorio_decisao:
        decisao.gerar_relatorio_decisao(
            run_dir, resumo1=resumo1, resumo_pooled=resumo_pooled,
            resumo_semanal=resumo_sem, resumo_estratos=resumo_estratos, metadata=meta,
            resumo_volume=resumo_volume, concentracao_volume=concentracao,
            cruzamento_estratos=cruzamento, fonte_volume=fonte_volume)
        print("[Item 20] RELATORIO_DECISAO.md gerado — leia este arquivo primeiro.")

    print(f"\nConcluido em {duracao:.1f}s. Saidas em: {run_dir}")
    # Grava o log com tudo o que foi impresso ate aqui.
    consolidado.salvar_log(_tee, run_dir)
    return run_dir

## 6. Executar pipeline

In [ ]:
# Executa a pipeline com os parâmetros definidos na seção 2.
import sys

for module in list(sys.modules.keys()):
    if module.startswith('src') or module == 'pipeline_colab':
        del sys.modules[module]
sys.path.insert(0, '/content')
from pipeline_colab import rodar

run_dir = rodar(
    dados=DADOS,
    drive_out=DRIVE_OUT,
    rotulo=ROTULO,
    amostra=AMOSTRA,
    experimentos=EXPERIMENTOS,
    modo_fiel=MODO_FIEL,
    n_jobs=N_JOBS,
    n_iter=N_ITER_BUSCA,
    n_iter_rf=N_ITER_BUSCA_RF,
    grade_rf=GRADE_RF,
    cv=CV_SPLITS,
    max_linhas_tuning=MAX_LINHAS_TUNING,
    reaproveitar_tabela_treino=REAPROVEITAR_TABELA_TREINO,
    gpu_xgb=GPU_XGB,
    treino_inicio=TREINO_INICIO,
    treino_fim=TREINO_FIM,
    teste_inicio=TESTE_INICIO,
    teste_fim=TESTE_FIM,
    gerar_correlacao=GERAR_CORRELACAO,
    gerar_graficos=GERAR_GRAFICOS,
    calendario_exogenas=CALENDARIO_EXOGENAS,
    cenarios_exp2=CENARIOS_EXP2,
    # --- Metricas e diagnosticos ---
    diagnostico_wmape_zero=DIAGNOSTICO_WMAPE_ZERO,
    metricas_extras=METRICAS_EXTRAS,
    avaliar_semanal=AVALIAR_SEMANAL,
    tamanho_efeito=TAMANHO_EFEITO,
    adicionar_dia_semana=ADICIONAR_DIA_SEMANA,
    xgb_objetivo=XGB_OBJETIVO,
    # --- BLOCO A: correcoes ---
    ordenar_tabela_cronologica=ORDENAR_TABELA_CRONOLOGICA,
    truncar_pre_lancamento=TRUNCAR_PRE_LANCAMENTO,
    score_relativo_geometrico=SCORE_RELATIVO_GEOMETRICO,
    corte_atributos_treino=CORTE_ATRIBUTOS_TREINO,
    salvar_previsoes_brutas=SALVAR_PREVISOES_BRUTAS,
    # --- BLOCO B: analises novas ---
    estratificar_ativos_cessados=ESTRATIFICAR_ATIVOS_CESSADOS,
    baseline_zero=BASELINE_ZERO,
    metricas_pooled=METRICAS_POOLED,
    classificacao_robustez_diaria=CLASSIFICACAO_ROBUSTEZ_DIARIA,
    relatorio_decisao=RELATORIO_DECISAO,
    # --- BLOCO B2: recorte gerencial e consolidacao ---
    estratificar_por_volume=ESTRATIFICAR_POR_VOLUME,
    consolidar_markdown=CONSOLIDAR_MARKDOWN,
    graficos_por_estrato=GRAFICOS_POR_ESTRATO,
    n_skus_top_figuras=N_SKUS_TOP_FIGURAS,
    # --- BLOCO C: modelagem ---
    features_intermitencia=FEATURES_INTERMITENCIA,
    encoding_sku=ENCODING_SKU,
    grade_xgb=GRADE_XGB,
    tunar_uma_vez_global=TUNAR_UMA_VEZ_GLOBAL,
    # --- BLOCO D: parcimonia das variaveis ---
    remover_features_redundantes=REMOVER_FEATURES_REDUNDANTES,
    features_excluidas_extra=FEATURES_EXCLUIDAS_EXTRA,
    exogenas_excluidas=EXOGENAS_EXCLUIDAS,
    # --- Cadencia do walk-forward ---
    stride_origem_treino=STRIDE_ORIGEM_TREINO,
    janelas_por_refit=JANELAS_POR_REFIT,
    permitir_alias_dia_semana=PERMITIR_ALIAS_DIA_SEMANA,
    # --- Metrica de selecao dos hiperparametros ---
    metrica_selecao_hiper=METRICA_SELECAO_HIPER,
    diagnostico_selecao_hiper=DIAGNOSTICO_SELECAO_HIPER,
)

## 7. Conferir resultados

Exibe os principais artefatos da execução (relatórios, tabelas de métricas e testes estatísticos).

In [ ]:
# Exibe os principais artefatos da execução.
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

RUN = Path(run_dir)

rd = RUN / "RELATORIO_DECISAO.md"
if rd.exists():
    display(Markdown(rd.read_text(encoding="utf-8")))
else:
    print("(RELATORIO_DECISAO.md nao gerado — Item 20 desligado)")

print("\nArquivos gerados nesta execucao:")
for f in sorted(RUN.iterdir()):
    print("  ", f.name)

e4 = RUN / "item_e4_metrica_selecao.md"
if e4.exists():
    display(Markdown(e4.read_text(encoding="utf-8")))
else:
    print("(item_e4_metrica_selecao.md nao gerado — diagnostico desligado)")

print("\n=== Resumo Experimento 1 (medianas por SKU + pooled + score relativo) ===")
r1 = RUN / "resumo_experimento1.csv"
if r1.exists():
    display(pd.read_csv(r1))

print("\n=== Item 18 — Metricas agregadas globais (pooled) ===")
rp = RUN / "resumo_pooled_experimento1.csv"
if rp.exists():
    display(pd.read_csv(rp))

print("\n=== Item 7 — Avaliacao no nivel semanal (comparacao mais justa) ===")
rs = RUN / "resumo_semanal_agregado.csv"
if rs.exists():
    display(pd.read_csv(rs))

print("\n=== Item 16 — Resumo por estrato (Ativo x Cessado) ===")
re_ = RUN / "resumo_experimento1_por_estrato.csv"
if re_.exists():
    display(pd.read_csv(re_))

we = RUN / "testes_wilcoxon_por_estrato.md"
if we.exists():
    display(Markdown(we.read_text(encoding="utf-8")))

print("\n=== Item 25 — Concentracao dos pedidos (Pareto) ===")
cv_ = RUN / "concentracao_volume_pedidos.csv"
if cv_.exists():
    display(pd.read_csv(cv_))

print("\n=== Item 25 — Resultados por faixa de volume (ABC) ===")
rv = RUN / "resumo_experimento1_por_volume.csv"
if rv.exists():
    display(pd.read_csv(rv))

print("\n=== Item 25 — Obsolescencia por faixa de volume ===")
cx = RUN / "cruzamento_volume_x_obsolescencia.csv"
if cx.exists():
    display(pd.read_csv(cx))

wv = RUN / "testes_wilcoxon_por_volume.md"
if wv.exists():
    display(Markdown(wv.read_text(encoding="utf-8")))

print("\n=== Testes de Wilcoxon — Experimento 1 (WMAPE e MASE) ===")
w1 = RUN / "testes_wilcoxon_experimento1.md"
if w1.exists():
    display(Markdown(w1.read_text(encoding="utf-8")))

print("\n=== Testes de Wilcoxon — Experimento 2 (QP2: calendario simples -> rico) ===")
w2 = RUN / "testes_wilcoxon_experimento2.md"
if w2.exists():
    display(Markdown(w2.read_text(encoding="utf-8")))

print("\n=== Item 13 — Score relativo (USE A COLUNA GEOMETRICA) ===")
srm = RUN / "score_relativo.md"
if srm.exists():
    display(Markdown(srm.read_text(encoding="utf-8")))

print("\n=== Custo computacional por modelo (Secao 4.3.1) ===")
print("Nota: `custo_total` = wall-clock do walk-forward, comparavel entre TODOS")
print("os modelos. As colunas tuning/fit_final sao a decomposicao do custo de ML.")
tm = RUN / "tempos_por_modelo.csv"
if tm.exists():
    display(pd.read_csv(tm))

print("\n=== Item 19 — Robustez da classificacao ADI/CV2 (granularidade diaria) ===")
cd = RUN / "classificacao_adi_cv2_diaria.md"
if cd.exists():
    display(Markdown(cd.read_text(encoding="utf-8")))

print("\n=== Correlacao features x target (apoio a Secao 4.2.1) ===")
cm = RUN / "correlacao_features.md"
if cm.exists():
    display(Markdown(cm.read_text(encoding="utf-8")))

print("\n=== Items 28/29 — Selecao de variaveis (o que entrou, o que saiu e por que) ===")
sv = RUN / "selecao_variaveis.md"
if sv.exists():
    display(Markdown(sv.read_text(encoding="utf-8")))

## 8. Visualizar figuras

Mostra as figuras geradas para:

- caracterização da base;
- comparação de desempenho dos modelos;
- recortes por estrato (ativo/cessado e volume);
- exemplos de séries real vs. previsto.

In [ ]:
# Exibe as figuras geradas na execução atual.
from IPython.display import Image, display, Markdown
from pathlib import Path

RUN = Path(run_dir)


def mostrar(titulo, arquivos):
    presentes = [RUN / f for f in arquivos if (RUN / f).exists()]
    if not presentes:
        return
    display(Markdown(f"## {titulo}"))
    for p in presentes:
        print(p.name)
        display(Image(filename=str(p)))


mostrar("Caracterização da base", [
    "fig_cauda_longa.png", "fig_volume_diario.png", "fig_sazonalidade_dia_ano.png",
    "fig_volume_por_campanha.png", "fig_dispersao_adi_cv2.png", "fig_esparsidade.png",
    "fig_dispersao_adi_cv2_diaria.png",
])

mostrar("Recorte gerencial — volume de pedidos (Item 25)", [
    "fig_pareto_pedidos.png",
    "fig_wmape_pooled_por_estrato_volume.png",
    "fig_wmape_mediana_por_estrato_volume.png",
    "fig_mae_mediana_por_estrato_volume.png",
    "fig_boxplot_wmape_volume.png",
    "fig_boxplot_mase_volume.png",
])

mostrar("SKUs ativos x cessados (Item 16)", [
    "fig_wmape_mediana_por_estrato_teste.png",
    "fig_mae_mediana_por_estrato_teste.png",
    "fig_wmape_pooled_por_estrato_teste.png",
    "fig_boxplot_wmape_ativo_cessado.png",
])

mostrar("Resultados gerais", [
    "fig_correlacao_intermitente.png",
    "fig_correlacao_lumpy.png",
    "fig_mediana_wmape_modelos.png",
    "fig_mediana_mase_modelos.png",
    "fig_boxplot_wmape_quadrante.png",
    "fig_boxplot_mase_quadrante.png",
    "fig_tempo_execucao_modelos.png",
    "fig_importancia_features_rf_xgb.png",
])

topo = sorted(RUN.glob("fig_realvsprevisto_TOP*.png"))
if topo:
    display(Markdown("## Real x previsto — SKUs com maior número de pedidos (Item 27)"))
    for p in topo:
        print(p.name)
        display(Image(filename=str(p)))

outros = [p for p in sorted(RUN.glob("fig_realvsprevisto_*.png")) if "_TOP" not in p.name]
if outros:
    display(Markdown("## Real x previsto — melhor e pior caso por quadrante ADI/CV2"))
    for p in outros:
        print(p.name)
        display(Image(filename=str(p)))